# MMAF-CD v24 — Stage A: core paper numbers (preprocessing audit, evaluation, calibration, conformal, routing, FiLM audit, shortcut, sanity, OOD, XAI agreement)

**Why staged:** the single notebook needed > 12 h and was killed (exit 137) inside V21-A2.
Run the stages as **separate Kaggle notebooks** (Save Version → *Save & Run All*). Estimated runtime: **≈ 3.5–4.5 h**.

**v24 critical fix:** `val_tfm` ran a stain normaliser on every evaluation image of every organ — including MRI, CT, mammograms and ultrasound — although no specialist was trained with it. Every number produced by v16–v23 (paper Table II, routing, calibration, conformal, FiLM, XAI) was measured on images the models had never seen in that form. `EVAL_STAIN_NORM=False` now; `V24-P0` quantifies the damage.

**Inputs to attach:** all `marcyoussef22/*` model inputs you used in the last run (gateway1/2, bloodskinlymphoma/2, retrain/1, coloncervical/1, 7models/2, breast/1, 3model/1, gastric/1, filmv8/2, sota/1, segmentation/1); `mohamedakmal1/multi-cancer-mulit-image-dataset`, `mdabrarhossainzahin/gastric-cancer-data`, `andrewmvd/cancer-inst-segmentation-and-classification` (PanNuke); **new:** `pankrzysiu/cifar10-python` and `paultimothymooney/chest-xray-pneumonia` (OOD test — mounted, nothing is downloaded).

**What this stage runs:** V24-P0 · Cells 7/7b/7c · V20-R1 (full-test routing) · R2/R3 (calibration + LAC/APS conformal) · R4 (FiLM audit) · R6 (shortcut) · R7 (saliency sanity) · R8 (OOD) · V23-X1 (CAM agreement, n=30/organ) · R10 (LaTeX tables)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# V24 — STAGE CONFIG  (edit nothing else; every later cell reads these flags)
# ════════════════════════════════════════════════════════════════════════════════
import os, glob
from pathlib import Path
STAGE = "A"                 # A = core numbers · B = figures/reports · C1 = audit/external · C2 = SOTA baselines · D = retrain

# CRITICAL FIX (v24): the old val_tfm ran a Macenko/Vahadane stain normaliser on EVERY
# evaluation image of EVERY organ (MRI, CT, mammograms, ultrasound included), but the
# specialists were trained WITHOUT it (train_tfm has no stain step). Mammograms came out
# lavender. Lymphoma: 0.573 with it, 0.994 without it (V21-A1). Keep False.
EVAL_STAIN_NORM = False
EVAL_RESIZE     = "resize224"       # "resize224" (old default) or "crop" (Resize 256 → CenterCrop 224). V24-P0 tells you which is better.

RUN_PQ_EVAL = STAGE in ("A",)       # PanNuke PQ (5 min load) only once, in Stage A
os.environ.setdefault("MMAF_WORKERS", "2")   # was 0 → single-process loading made Cell 7 / V20 3–4× slower.
                                             # If you ever see "DataLoader worker ... Bus error", set "0".

# Stage C/D: results produced by Stage A (add Stage A's output as an input to this notebook)
V24_PREV_RESULTS = None
if STAGE not in ("A", "B"):
    _cands = []
    for top in Path("/kaggle/input").glob("*"):
        if top.name in ("datasets", "models", "competitions"): continue
        for d in range(0, 6):
            _cands += glob.glob(str(top) + "/*" * d + "/v20/v20_results.json")
    # v24.4: also a Kaggle *dataset* made from Stage A's zip (fixed-depth globs = fast)
    for pat in ["/kaggle/input/*/v20/v20_results.json", "/kaggle/input/datasets/*/*/v20/v20_results.json",
                "/kaggle/input/datasets/*/*/*/v20/v20_results.json", "/kaggle/input/*/*/v20/v20_results.json"]:
        _cands += glob.glob(pat)
    if _cands:
        V24_PREV_RESULTS = sorted(_cands)[-1]
# v24.1 — robustness: Stage A stalled > 1 h inside V20-R1 (forked DataLoader workers).
import gc, ctypes
try:
    import cv2; cv2.setNumThreads(0)            # OpenCV threads inside forked workers can deadlock
except Exception: pass
try:
    import torch; torch.multiprocessing.set_sharing_strategy("file_system")
except Exception: pass
V20_WORKERS = int(os.environ.get("MMAF_WORKERS_V20", "0"))   # evidence cells run single-process (proven stable)
def _dl_timeout():                               # a stalled worker now raises after 15 min instead of hanging for hours
    return 900 if globals().get("NUM_WORKERS", 0) > 0 else 0
def free_mem(tag=""):
    gc.collect()
    try: torch.cuda.empty_cache()
    except Exception: pass
    try: ctypes.CDLL("libc.so.6").malloc_trim(0)  # hand freed numpy memory (e.g. PanNuke masks) back to the OS
    except Exception: pass
    try:
        import psutil; vm = psutil.virtual_memory()
        print(f"  🧹 memory after {tag}: kernel RSS {psutil.Process().memory_info().rss/1e9:.1f} GB · free {vm.available/1e9:.1f} GB of {vm.total/1e9:.1f} GB", flush=True)
    except Exception: pass
print(f"STAGE={STAGE}  EVAL_STAIN_NORM={EVAL_STAIN_NORM}  EVAL_RESIZE={EVAL_RESIZE}  RUN_PQ_EVAL={RUN_PQ_EVAL}  "
      f"workers={os.environ['MMAF_WORKERS']}  prev_results={V24_PREV_RESULTS}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 0 — Install & validate all dependencies
# ════════════════════════════════════════════════════════════════════════════════
import subprocess, sys

subprocess.run([
    "pip", "install", "-q",
    "timm>=1.0.3", "scikit-learn", "shap>=0.44.0", "scikit-image",
    "torchmetrics", "albumentations>=1.4", "torchvision", "scipy",
    "torchstain",
], check=True)

print("Installing pydensecrf (Cython)...")
subprocess.run(["pip", "install", "-q", "cython"], check=True)

r_crf = subprocess.run(
    ["pip", "install", "-q", "git+https://github.com/lucasb-eyer/pydensecrf.git"],
    capture_output=True,
    text=True
)

if r_crf.returncode != 0:
    print("  ⚠️  pydensecrf git failed, trying PyPI...")
    subprocess.run(["pip", "install", "-q", "pydensecrf"], check=False)

# ── Runtime CRF smoke test ────────────────────────────────────────────────────
print("Validating pydensecrf...")

try:
    import numpy as _np
    import pydensecrf.densecrf as _dcrf
    from pydensecrf.utils import unary_from_softmax as _ufs

    _H, _W = 8, 8

    _img = _np.ascontiguousarray(
        _np.random.randint(0, 255, (_H, _W, 3), dtype=_np.uint8)
    )

    _prob = _np.ascontiguousarray(
        _np.random.dirichlet([1, 1], _H * _W)
        .T.reshape(2, _H, _W)
        .astype(_np.float32)
    )

    _unary = _np.ascontiguousarray(_ufs(_prob))

    _d = _dcrf.DenseCRF2D(_W, _H, 2)
    _d.setUnaryEnergy(_unary)
    _d.addPairwiseGaussian(sxy=1, compat=1)
    _d.inference(1)

    print("  ✅ pydensecrf PASSED")
    CRF_VALIDATED = True

except Exception as _e:
    print(f"  ❌ pydensecrf FAILED: {_e}")
    CRF_VALIDATED = False

# ── Macenko stain normaliser smoke test ───────────────────────────────────────
print("Validating torchstain / Macenko...")

try:
    import torchstain

    _normalizer = torchstain.normalizers.MacenkoNormalizer(
        backend='torch'
    )

    print("  ✅ torchstain Macenko available")
    STAIN_NORM_AVAILABLE = True

except Exception as _e:
    print(f"  ⚠️  torchstain not available ({_e})")
    STAIN_NORM_AVAILABLE = False

print(f"✅ Cell 0 complete | CRF:{CRF_VALIDATED} | Stain:{STAIN_NORM_AVAILABLE}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 1 — Imports, Paths, Label Maps, Clinical Protocols, Cancer Staging
#           v17 — all placeholder dicts fully populated, no omissions
# ════════════════════════════════════════════════════════════════════════════════
import json, csv, math, random, time, warnings, copy, gc, os
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm, shap
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from skimage.filters import threshold_otsu
from skimage.measure import label as sk_label, regionprops
from skimage.morphology import binary_closing, binary_opening, disk
from scipy.ndimage import binary_dilation, binary_fill_holes
from scipy.optimize import minimize_scalar
from scipy.stats import spearmanr
from numpy.polynomial.legendre import leggauss
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import RandAugment, RandomErasing
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                              roc_auc_score, classification_report)
from tqdm.auto import tqdm

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── NumPy 2.0 compatibility shim (trapz → trapezoid) ─────────────────────────
if not hasattr(np, 'trapezoid'):
    np.trapezoid = np.trapz

# ── torchstain ────────────────────────────────────────────────────────────────
try:
    import torchstain
    HAS_TORCHSTAIN = True
except Exception as e:
    HAS_TORCHSTAIN = False
    print(f"  ⚠️  torchstain import failed: {e}")

# ── pydensecrf ────────────────────────────────────────────────────────────────
try:
    import pydensecrf.densecrf as dcrf
    from pydensecrf.utils import unary_from_softmax
    HAS_CRF = CRF_VALIDATED
except ImportError:
    HAS_CRF = False; CRF_VALIDATED = False
    print("  ⚠️  pydensecrf not importable")

warnings.filterwarnings("ignore")

# ─── Device + Seeds ───────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED   = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
_IS_KAGGLE  = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
NUM_WORKERS = int(os.environ.get("MMAF_WORKERS", "2")) if _IS_KAGGLE else min(2, os.cpu_count() or 1)   # v24: was 0
print(f"  Workers: {NUM_WORKERS}")

# ─── Paths ────────────────────────────────────────────────────────────────────
DATA_ROOT = Path(
    "/kaggle/input/datasets/mohamedakmal1/"
    "multi-cancer-mulit-image-dataset/Multi-Cancer Mulit-image Dataset")
GASTRIC_ROOT = Path(
    "/kaggle/input/datasets/mdabrarhossainzahin/"
    "gastric-cancer-data/HMU-GC-HE-30K/all_image")
GATEWAY_CKPT = Path(
    "/kaggle/input/models/marcyoussef22/gateway1/other/default/2/best_gateway_v5_16organs.pth")
SPEC_CKPTS = {
    "blood":      Path("/kaggle/input/models/marcyoussef22/bloodskinlymphoma/other/default/2/best_blood_specialist (1).pth"),
    "lymphoma":   Path("/kaggle/input/models/marcyoussef22/bloodskinlymphoma/other/default/2/best_lymphoma_specialist (1).pth"),
    "skin":       Path("/kaggle/input/models/marcyoussef22/retrain/other/default/1/best_skin_retrained_v1.pth"),
    "cervical":   Path("/kaggle/input/models/marcyoussef22/retrain/other/default/1/best_cervical_retrained_v1.pth"),
    "colon":      Path("/kaggle/input/models/marcyoussef22/coloncervical/other/default/1/best_colon_specialist.pth"),
    "brain_2d":   Path("/kaggle/input/models/marcyoussef22/7models/other/default/2/best_brain_2d_specialist.pth"),
    "esophageal": Path("/kaggle/input/models/marcyoussef22/7models/other/default/2/best_esophageal_specialist.pth"),
    "kidney":     Path("/kaggle/input/models/marcyoussef22/retrain/other/default/1/best_kidney_retrained_v1.pth"),
    "lung":       Path("/kaggle/input/models/marcyoussef22/7models/other/default/2/best_lung_specialist.pth"),
    "oral":       Path("/kaggle/input/models/marcyoussef22/7models/other/default/2/best_oral_specialist.pth"),
    "pancreatic": Path("/kaggle/input/models/marcyoussef22/retrain/other/default/1/best_pancreatic_retrained_v1.pth"),
    "thyroid":    Path("/kaggle/input/models/marcyoussef22/7models/other/default/2/best_thyroid_specialist.pth"),
    "breast":     Path("/kaggle/input/models/marcyoussef22/breast/other/default/1/best_breast_specialist.pth"),
    "bladder":    Path("/kaggle/input/models/marcyoussef22/3model/other/default/1/best_bladder_specialist.pth"),
    "gastric":    Path("/kaggle/input/models/marcyoussef22/gastric/other/default/1/best_gastric_specialist_8cls.pth"),
    "ovarian":    Path("/kaggle/input/models/marcyoussef22/3model/other/default/1/best_ovarian_specialist.pth"),
}
SPEC_DIR = Path("/kaggle/working/specialist_ckpts_v8")
XAI_OUT  = Path("/kaggle/working/xai_v16_output")
FHIR_OUT = XAI_OUT / "fhir_reports"
for d in [SPEC_DIR, XAI_OUT, XAI_OUT/"clinical_reports",
          XAI_OUT/"segmentation", FHIR_OUT]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Label maps ───────────────────────────────────────────────────────────────
LABEL_MAP = {
    "all_benign":("blood","benign"), "all_pre":("blood","pre"),
    "all_early":("blood","early"),   "all_pro":("blood","pro"),
    "bladder_muscle invasive":("bladder","muscle_invasive"),
    "bladder_non muscle invasive":("bladder","non_muscle_invasive"),
    "brain_glioma":("brain_2d","glioma"), "brain_menin":("brain_2d","meningioma"),
    "brain_tumor":("brain_2d","tumor"),
    "breast_benign":("breast","benign"), "breast_malignant":("breast","malignant"),
    "cervix_dyk":("cervical","dyk"), "cervix_koc":("cervical","koc"),
    "cervix_mep":("cervical","mep"), "cervix_pab":("cervical","pab"),
    "cervix_sfi":("cervical","sfi"),
    "colon_aca":("colon","adenocarcinoma"), "colon_bnt":("colon","benign"),
    "esophagus_benign":("esophageal","benign"),
    "esophagus_malignant":("esophageal","malignant"),
    "kidney_normal":("kidney","normal"), "kidney_tumor":("kidney","tumor"),
    "lung_benign":("lung","benign"),     "lung_malignant":("lung","malignant"),
    "lymph_cll":("lymphoma","cll"),      "lymph_fl":("lymphoma","fl"),
    "lymph_mcl":("lymphoma","mcl"),
    "oral_normal":("oral","normal"),     "oral_scc":("oral","scc"),
    "ovarian_clear-cell carcinoma":("ovarian","clear_cell"),
    "ovarian_endometrioid":("ovarian","endometrioid"),
    "ovarian_low-grade serous":("ovarian","low_grade_serous"),
    "ovarian_mucinous carcinoma":("ovarian","mucinous"),
    "ovartian_high-grade serous carcinoma":("ovarian","high_grade_serous"),
    "pancreatic_normal":("pancreatic","normal"),
    "pancreatic_tumor":("pancreatic","tumor"),
    "skin_actinic keratosis":("skin","actinic_keratosis"),
    "skin_basal cell carcinoma":("skin","basal_cell_carcinoma"),
    "skin_melanoma":("skin","melanoma"), "skin_nevus":("skin","nevus"),
    "skin_normal skin":("skin","normal"),
    "skin_seborrheic keratosis":("skin","seborrheic_keratosis"),
    "skin_squamous cell carcinoma":("skin","squamous_cell_carcinoma"),
    "thyroid_benign":("thyroid","benign"),
    "thyroid_malignant":("thyroid","malignant"),
}
GASTRIC_LABEL_MAP = {
    "ADI":"adi","DEB":"deb","LYM":"lym","MUC":"muc",
    "MUS":"mus","NOR":"normal","STR":"str","TUM":"tumor",
}
ORGAN_SUBDIR_MAP = {
    "blood":"ALL","brain_2d":"Brain Cancer","breast":"Breast Cancer",
    "cervical":"Cervical Cancer","colon":"Colon Cancer",
    "esophageal":"Esophageal Cancer","kidney":"Kidney Cancer",
    "lung":"Lung Cancer","lymphoma":"Lymphoma","oral":"Oral Cancer",
    "ovarian":"Ovarian Cancer","pancreatic":"Pancreatic Cancer",
    "skin":"Skin Cancer","thyroid":"Thyroid Cancer","bladder":"Bladder Cancer",
}
BACKBONE_CFG = {
    "blood":("convnext_tiny",0.25,False),   "lymphoma":("vit_small_patch16_224",0.25,False),
    "skin":("convnext_small",0.30,False),   "cervical":("efficientvit_m5",0.25,False),
    "colon":("convnext_tiny",0.20,False),   "brain_2d":("convnext_small",0.30,False),
    "esophageal":("convnext_tiny",0.20,False),"kidney":("efficientnet_b3",0.25,False),
    "lung":("convnext_small",0.30,False),   "oral":("convnext_tiny",0.20,False),
    "pancreatic":("vit_small_patch16_224",0.30,True),"thyroid":("efficientnet_b2",0.20,False),
    "breast":("convnext_tiny",0.20,False),  "bladder":("convnext_tiny",0.20,False),
    "gastric":("convnext_small",0.25,False),"ovarian":("convnext_small",0.25,False),
}
AUGMENT_CFG        = {"bladder":"heavy","thyroid":"heavy"}
VIT_ORGANS         = {"lymphoma","pancreatic"}
HOOK_BROKEN_ORGANS = {"cervical","kidney","thyroid"}

NEEDS_CALIBRATION      = set()
OPTIMIZED_TEMPERATURES = {}
ORGAN_BASE_F1          = {}
ORGAN_VAL_UNC_STATS    = {}
CONFORMAL_QHAT         = {}

T_LOWER_BOUND = 0.10
T_UPPER_BOUND = 10.0

FILM_FREEZE_F1              = 0.995
FILM_HIDDEN_DIM             = 128
FILM_HURT_THRESHOLD_DEFAULT = -0.05
FILM_HURT_THRESHOLD_ORGAN   = {"lymphoma":-0.10,"ovarian":-0.10}

CONFORMAL_ALPHA = 0.10

GATEWAY_CONF_THRESHOLD  = 0.70
GATEWAY_FALLBACK_THRESH = 0.92

XAI_LAYERS = {
    "blood":"backbone.stages.2",    "lymphoma":"backbone.blocks.6",
    "skin":"backbone.stages.2",     "cervical":"backbone.stages.2",
    "colon":"backbone.stages.2",    "brain_2d":"backbone.stages.2",
    "esophageal":"backbone.stages.2","kidney":"backbone.blocks.4",
    "lung":"backbone.stages.2",     "oral":"backbone.stages.2",
    "pancreatic":"backbone.blocks.10","thyroid":"backbone.stages.3",
    "breast":"backbone.stages.2",   "bladder":"backbone.stages.3",
    "gastric":"backbone.stages.2",  "ovarian":"backbone.stages.2",
}
MS_CAM_LAYERS = {
    "blood":["backbone.stages.1","backbone.stages.2","backbone.stages.3"],
    "skin":["backbone.stages.1","backbone.stages.2","backbone.stages.3"],
    "colon":["backbone.stages.1","backbone.stages.2","backbone.stages.3"],
    "brain_2d":["backbone.stages.1","backbone.stages.2","backbone.stages.3"],
    "esophageal":["backbone.stages.1","backbone.stages.2","backbone.stages.3"],
    "lung":["backbone.stages.1","backbone.stages.2","backbone.stages.3"],
    "oral":["backbone.stages.1","backbone.stages.2","backbone.stages.3"],
    "breast":["backbone.stages.1","backbone.stages.2","backbone.stages.3"],
    "bladder":["backbone.stages.1","backbone.stages.2","backbone.stages.3"],
    "gastric":["backbone.stages.1","backbone.stages.2","backbone.stages.3"],
    "ovarian":["backbone.stages.1","backbone.stages.2","backbone.stages.3"],
    "lymphoma":["backbone.blocks.4","backbone.blocks.7","backbone.blocks.10"],
    "pancreatic":["backbone.blocks.4","backbone.blocks.7","backbone.blocks.10"],
    "cervical":[],"kidney":[],"thyroid":[],
}
MALIGNANT_CLASSES = {
    "blood":{"early","pre","pro"},
    "lymphoma":{"cll","fl","mcl"},
    "skin":{"actinic_keratosis","basal_cell_carcinoma","melanoma","squamous_cell_carcinoma"},
    "cervical":{"dyk","koc","mep","pab"},
    "colon":{"adenocarcinoma"},
    "brain_2d":{"glioma","meningioma","tumor"},
    "esophageal":{"malignant"},
    "kidney":{"tumor"},
    "lung":{"malignant"},
    "oral":{"scc"},
    "pancreatic":{"tumor"},
    "thyroid":{"malignant"},
    "breast":{"malignant"},
    "bladder":{"muscle_invasive","non_muscle_invasive"},
    "gastric":{"tumor","deb","lym","str"},
    "ovarian":{"clear_cell","endometrioid","high_grade_serous","low_grade_serous","mucinous"},
}
SEG_CAM_THRESHOLD = {
    "brain_2d":0.40,"skin":0.35,"lymphoma":0.18,"pancreatic":0.20,
    "breast":0.18,"default":0.30,
}
MULTI_FOCAL_ORGANS = {"lymphoma","ovarian","gastric"}
MULTI_FOCAL_K      = 3

CAM_DISAGREE_THRESH = 0.20
DELETION_STEPS      = 10
INSERTION_STEPS     = 10
DELETION_FRACTION   = 0.10
INSERTION_FRACTION  = 0.10

TRUST_W_SSIM=0.20; TRUST_W_UNCERT=0.20; TRUST_W_AUC_DEL=0.15
TRUST_W_CONF=0.15; TRUST_W_FOCUS=0.10; TRUST_W_POINT=0.10; TRUST_W_SEG=0.10
TRUST_SSIM_MAX=0.50; TRUST_UNCERT_MAX=0.30; FOCUS_RATIO_MAX=20.0

SHAP_N_BACKGROUND=16; SHAP_SEG_H=14; SHAP_SEG_W=14
SHAP_N_SEGS=SHAP_SEG_H*SHAP_SEG_W
SHAP_NSAMPLES=max(2048,SHAP_N_SEGS*12)
IG_GL_STEPS=64                          # reduced from 100 → 64 (saves VRAM)

# ISS-7: skip IG when T* ≤ this threshold
IG_SKIP_T_THRESHOLD = 0.15              # raised from 0.11 → 0.15 (v17)


# ════════════════════════════════════════════════════════════════════════════════
# URGENCY & RISK colour maps
# ════════════════════════════════════════════════════════════════════════════════
URGENCY_COLORS = {
    "EMERGENCY": "#c0392b",
    "URGENT":    "#e74c3c",
    "MODERATE":  "#e67e22",
    "LOW":       "#f39c12",
    "ROUTINE":   "#2ecc71",
}
RISK_COLORS = {
    "CRITICAL":  "#c0392b",
    "HIGH RISK": "#e74c3c",
    "ABNORMAL":  "#e67e22",
    "MODERATE":  "#f39c12",
    "LOW":       "#f1c40f",
    "NORMAL":    "#2ecc71",
}


# ════════════════════════════════════════════════════════════════════════════════
# CANCER STAGING
# (organ, class_name) → (stage_label, rationale, urgency)
# urgency must be a key in URGENCY_COLORS
# ════════════════════════════════════════════════════════════════════════════════
CANCER_STAGING = {
    # ── Blood / ALL ───────────────────────────────────────────────────────────
    ("blood", "pre"):    (
        "Stage I–II  (Pre-B ALL)",
        "Early lymphoblast expansion; bone marrow <25% blasts",
        "URGENT"),
    ("blood", "early"):  (
        "Stage II–III  (Early ALL)",
        "Bone marrow blast infiltration 25–75%; peripheral blasts present",
        "URGENT"),
    ("blood", "pro"):    (
        "Stage III–IV  (Pro-B ALL)",
        "High blast burden >75%; CNS or extramedullary spread likely",
        "EMERGENCY"),
    ("blood", "benign"): (
        "Non-malignant",
        "Reactive or normal haematopoiesis; no blast excess",
        "ROUTINE"),

    # ── Brain ─────────────────────────────────────────────────────────────────
    ("brain_2d", "glioma"):     (
        "Grade III–IV  (High-grade Glioma)",
        "WHO grade III/IV astrocytoma or GBM; rapid infiltrative growth",
        "EMERGENCY"),
    ("brain_2d", "meningioma"): (
        "Grade I–II  (Meningioma)",
        "Typically WHO grade I; slow-growing extra-axial; grade II if atypical",
        "MODERATE"),
    ("brain_2d", "tumor"):      (
        "Grade II–IV  (Brain Tumour — unclassified)",
        "Intracranial mass requiring histological grading; treat as high-grade until proven otherwise",
        "URGENT"),

    # ── Breast ────────────────────────────────────────────────────────────────
    ("breast", "malignant"): (
        "Stage I–IV  (Invasive Breast Carcinoma)",
        "Invasive ductal/lobular carcinoma; staging by TNM + receptor status",
        "URGENT"),
    ("breast", "benign"):    (
        "Non-malignant",
        "Benign breast tissue; fibroadenoma or fibrocystic changes",
        "ROUTINE"),

    # ── Cervical ──────────────────────────────────────────────────────────────
    ("cervical", "dyk"):  (
        "CIN III / Stage 0  (High-grade Dyskeratosis)",
        "Severe dysplasia / carcinoma in situ; HPV-related high-grade precursor",
        "URGENT"),
    ("cervical", "koc"):  (
        "CIN I–II  (Koilocytic Change)",
        "Low-to-moderate grade HPV cytopathic effect; koilocytes present",
        "MODERATE"),
    ("cervical", "mep"):  (
        "Stage 0–I  (Metaplastic Epithelium)",
        "Squamocolumnar junction metaplasia; generally benign but monitor",
        "LOW"),
    ("cervical", "pab"):  (
        "Reactive / Stage 0  (Parabasal Cells)",
        "Parabasal cell predominance; inflammatory or atrophic smear",
        "LOW"),
    ("cervical", "sfi"):  (
        "Normal Cervical Cytology",
        "Superficial and intermediate cells; within normal limits",
        "ROUTINE"),

    # ── Colon ─────────────────────────────────────────────────────────────────
    ("colon", "adenocarcinoma"): (
        "Stage I–IV  (Colorectal Adenocarcinoma)",
        "Invasive colorectal adenocarcinoma; Dukes / TNM staging required",
        "URGENT"),
    ("colon", "benign"):         (
        "Non-malignant",
        "Normal colonic mucosa or benign polyp; no dysplasia",
        "ROUTINE"),

    # ── Esophageal ────────────────────────────────────────────────────────────
    ("esophageal", "malignant"): (
        "Stage I–IV  (Esophageal Carcinoma)",
        "Squamous cell or adenocarcinoma of oesophagus; TNM staging required",
        "URGENT"),
    ("esophageal", "benign"):    (
        "Non-malignant",
        "Benign oesophageal mucosa; Barrett's surveillance if indicated",
        "ROUTINE"),

    # ── Kidney ────────────────────────────────────────────────────────────────
    ("kidney", "tumor"):  (
        "Stage I–IV  (Renal Cell Carcinoma)",
        "Clear cell / papillary / chromophobe RCC; Fuhrman grade required",
        "URGENT"),
    ("kidney", "normal"): (
        "Non-malignant",
        "Normal renal parenchyma; no neoplastic features",
        "ROUTINE"),

    # ── Lung ──────────────────────────────────────────────────────────────────
    ("lung", "malignant"): (
        "Stage I–IV  (Lung Carcinoma)",
        "NSCLC (adenocarcinoma / SCC) or SCLC; TNM + PD-L1 + driver mutations",
        "EMERGENCY"),
    ("lung", "benign"):    (
        "Non-malignant",
        "Benign pulmonary tissue; no malignant features identified",
        "ROUTINE"),

    # ── Lymphoma ──────────────────────────────────────────────────────────────
    ("lymphoma", "cll"):  (
        "Stage I–IV  (CLL — Rai/Binet)",
        "Chronic lymphocytic leukaemia; Rai 0–IV / Binet A–C; smudge cells on smear",
        "URGENT"),
    ("lymphoma", "fl"):   (
        "Stage I–IV  (Follicular Lymphoma — Ann Arbor)",
        "Grade 1–3A follicular lymphoma; FLIPI score; indolent but incurable without SCT",
        "MODERATE"),
    ("lymphoma", "mcl"):  (
        "Stage IV  (Mantle Cell Lymphoma)",
        "Aggressive B-cell NHL; blastoid variant; Ki-67 >30%; MIPI score critical",
        "EMERGENCY"),

    # ── Oral ──────────────────────────────────────────────────────────────────
    ("oral", "scc"):    (
        "Stage I–IV  (Oral Squamous Cell Carcinoma)",
        "Invasive SCC of oral cavity; depth of invasion is key T-stage determinant",
        "URGENT"),
    ("oral", "normal"): (
        "Non-malignant",
        "Normal oral mucosa; no dysplasia or malignancy",
        "ROUTINE"),

    # ── Ovarian ───────────────────────────────────────────────────────────────
    ("ovarian", "high_grade_serous"):  (
        "Stage III–IV  (High-grade Serous Carcinoma)",
        "Most common ovarian malignancy; BRCA1/2 associated; 70% present stage III/IV",
        "EMERGENCY"),
    ("ovarian", "low_grade_serous"):   (
        "Stage I–III  (Low-grade Serous Carcinoma)",
        "RAS/RAF pathway mutations; indolent but platinum-resistant; MEK inhibitor candidate",
        "URGENT"),
    ("ovarian", "clear_cell"):         (
        "Stage I–IV  (Clear Cell Carcinoma)",
        "Associated with endometriosis; platinum resistance common; VTE risk elevated",
        "URGENT"),
    ("ovarian", "endometrioid"):       (
        "Stage I–III  (Endometrioid Carcinoma)",
        "Endometriosis-associated; CTNNB1/PTEN mutations; good prognosis if low-stage",
        "MODERATE"),
    ("ovarian", "mucinous"):           (
        "Stage I–II  (Mucinous Carcinoma)",
        "Rare primary ovarian; exclude GI primary; borderline vs invasive distinction critical",
        "MODERATE"),

    # ── Pancreatic ────────────────────────────────────────────────────────────
    ("pancreatic", "tumor"):  (
        "Stage I–IV  (Pancreatic Ductal Adenocarcinoma)",
        "PDAC; 80% unresectable at presentation; CA 19-9; KRAS mutation; poor prognosis",
        "EMERGENCY"),
    ("pancreatic", "normal"): (
        "Non-malignant",
        "Normal pancreatic exocrine/endocrine tissue; no neoplastic features",
        "ROUTINE"),

    # ── Skin ──────────────────────────────────────────────────────────────────
    ("skin", "melanoma"):               (
        "Stage 0–IV  (Malignant Melanoma)",
        "Breslow thickness + Clark level + ulceration determine T-stage; BRAF/NRAS status",
        "EMERGENCY"),
    ("skin", "basal_cell_carcinoma"):   (
        "Stage I–II  (Basal Cell Carcinoma)",
        "Locally invasive; rarely metastasises; Mohs for high-risk locations",
        "URGENT"),
    ("skin", "squamous_cell_carcinoma"):(
        "Stage I–III  (Cutaneous SCC)",
        "SCC with perineural or >2 cm requires sentinel node consideration",
        "URGENT"),
    ("skin", "actinic_keratosis"):      (
        "Stage 0  (Actinic Keratosis — Pre-malignant)",
        "UV-induced keratinocyte dysplasia; 5–10% risk of SCC progression if untreated",
        "MODERATE"),
    ("skin", "nevus"):                  (
        "Non-malignant  (Benign Naevus)",
        "Melanocytic naevus; excise if atypical features (ABCDE criteria)",
        "ROUTINE"),
    ("skin", "seborrheic_keratosis"):   (
        "Non-malignant  (Seborrhoeic Keratosis)",
        "Benign epidermal tumour; no malignant potential; reassure patient",
        "ROUTINE"),
    ("skin", "normal"):                 (
        "Non-malignant  (Normal Skin)",
        "Normal skin histology; no dysplastic or neoplastic features",
        "ROUTINE"),

    # ── Thyroid ───────────────────────────────────────────────────────────────
    ("thyroid", "malignant"): (
        "Stage I–IV  (Thyroid Carcinoma)",
        "PTC most common; FTC / MTC / ATC also possible; RAI response varies by subtype",
        "URGENT"),
    ("thyroid", "benign"):    (
        "Non-malignant",
        "Benign follicular adenoma or thyroiditis; no malignant features",
        "ROUTINE"),

    # ── Bladder ───────────────────────────────────────────────────────────────
    ("bladder", "muscle_invasive"):     (
        "Stage T2–T4  (MIBC — Muscle-Invasive Bladder Cancer)",
        "Detrusor muscle invasion confirmed; radical cystectomy or trimodality therapy",
        "EMERGENCY"),
    ("bladder", "non_muscle_invasive"): (
        "Stage Ta–T1  (NMIBC — Non-Muscle-Invasive Bladder Cancer)",
        "Papillary urothelial carcinoma confined to mucosa/lamina propria; TURBT + BCG",
        "URGENT"),

    # ── Gastric ───────────────────────────────────────────────────────────────
    ("gastric", "tumor"):  (
        "Stage I–IV  (Gastric Adenocarcinoma)",
        "Intestinal or diffuse type; HER2 testing; perioperative FLOT; Lauren classification",
        "URGENT"),
    ("gastric", "deb"):    (
        "Stage I–II  (Necrotic Tumour Debris)",
        "Necrotic / apoptotic material adjacent to tumour; indicates active malignant process",
        "MODERATE"),
    ("gastric", "lym"):    (
        "Stage II–III  (Lymphocytic Infiltrate)",
        "Tumour-infiltrating lymphocytes; assess for MALT lymphoma; H. pylori eradication",
        "MODERATE"),
    ("gastric", "str"):    (
        "Stage I–II  (Stromal Tumour Component)",
        "Desmoplastic stroma or GIST component; c-KIT/PDGFRA mutation testing",
        "MODERATE"),
    ("gastric", "adi"):    (
        "Non-malignant  (Adipose Tissue)",
        "Normal adipose tissue in resection specimen; no malignant features",
        "ROUTINE"),
    ("gastric", "muc"):    (
        "Non-malignant  (Mucus — Normal Gastric)",
        "Normal gastric mucus; no cellular atypia",
        "ROUTINE"),
    ("gastric", "mus"):    (
        "Non-malignant  (Smooth Muscle — Normal)",
        "Normal muscularis propria; no invasion",
        "ROUTINE"),
    ("gastric", "normal"): (
        "Non-malignant  (Normal Gastric Mucosa)",
        "Normal gastric glands; H. pylori screen recommended if symptomatic",
        "ROUTINE"),
}


# ════════════════════════════════════════════════════════════════════════════════
# CLINICAL PROTOCOLS
# organ → { class_name: (finding, risk, recommended_action) }
# risk must be a key in RISK_COLORS
# ════════════════════════════════════════════════════════════════════════════════
CLINICAL_PROTOCOLS = {
    # ── Blood / ALL ───────────────────────────────────────────────────────────
    "blood": {
        "pre":    (
            "Pre-B lymphoblast expansion detected",
            "HIGH RISK",
            "Urgent haematology referral ≤48 h; bone marrow biopsy; CBC + flow cytometry; "
            "initiate MRD monitoring; consider induction chemotherapy (VDLP protocol)"),
        "early":  (
            "Early ALL blast infiltration identified",
            "HIGH RISK",
            "Same-day haematology referral; FISH cytogenetics; BCR-ABL1 screen; "
            "CSF analysis to exclude CNS involvement; supportive care + TLS prophylaxis"),
        "pro":    (
            "Pro-B ALL — high blast burden",
            "CRITICAL",
            "EMERGENCY haematology referral; ICU-level monitoring; immediate "
            "hyper-CVAD / BFM induction; tumour lysis prophylaxis; G-CSF support"),
        "benign": (
            "Normal / reactive haematopoiesis",
            "NORMAL",
            "Routine follow-up; repeat CBC in 6 months if clinically indicated; "
            "no urgent action required"),
    },

    # ── Brain ─────────────────────────────────────────────────────────────────
    "brain_2d": {
        "glioma":     (
            "High-grade glioma identified",
            "CRITICAL",
            "Neurosurgery referral ≤24 h; contrast MRI + MR spectroscopy; "
            "IDH / MGMT methylation testing; multidisciplinary tumour board; "
            "Stupp protocol (RT + temozolomide) if GBM"),
        "meningioma": (
            "Meningioma detected",
            "ABNORMAL",
            "Elective neurosurgery consult; MRI with gadolinium; "
            "watch-and-wait for small asymptomatic lesions; "
            "Simpson grade resection + adjuvant RT if grade II/III"),
        "tumor":      (
            "Intracranial mass lesion — unclassified",
            "HIGH RISK",
            "Urgent MRI + PET-CT; neurosurgery referral; stereotactic biopsy "
            "for histological grading; MDT within 1 week"),
    },

    # ── Breast ────────────────────────────────────────────────────────────────
    "breast": {
        "malignant": (
            "Invasive breast carcinoma confirmed",
            "HIGH RISK",
            "Breast surgery referral ≤2 weeks; ER / PR / HER2 receptor panel; "
            "staging CT chest/abdomen/pelvis + bone scan; "
            "MDT: BCS vs mastectomy + axillary staging; neoadjuvant if HER2+ or TNBC"),
        "benign":    (
            "Benign breast tissue",
            "NORMAL",
            "Annual mammographic surveillance; clinical breast exam every 12 months; "
            "reassure patient; excision only if rapidly growing or symptomatic"),
    },

    # ── Cervical ──────────────────────────────────────────────────────────────
    "cervical": {
        "dyk":  (
            "High-grade dyskeratosis — CIN III",
            "HIGH RISK",
            "Colposcopy + directed biopsy within 2 weeks; "
            "LEEP / cone biopsy if CIN III confirmed; HPV genotyping; "
            "co-test in 12 months post-treatment"),
        "koc":  (
            "Koilocytic change — HPV cytopathic effect",
            "MODERATE",
            "Repeat Pap + HPV co-test in 12 months; "
            "colposcopy if persistent at 24 months; "
            "HPV 16/18 genotyping to stratify risk"),
        "mep":  (
            "Metaplastic epithelium",
            "LOW",
            "Routine cervical screening per national guidelines in 3–5 years; "
            "no immediate intervention required"),
        "pab":  (
            "Parabasal cells — inflammatory / atrophic",
            "LOW",
            "Treat underlying infection or atrophy (topical oestrogen if post-menopausal); "
            "repeat smear in 3 months"),
        "sfi":  (
            "Normal cervical cytology",
            "NORMAL",
            "Routine screening per guidelines; next smear in 3–5 years depending on age/risk; "
            "HPV vaccination if not already received"),
    },

    # ── Colon ─────────────────────────────────────────────────────────────────
    "colon": {
        "adenocarcinoma": (
            "Colorectal adenocarcinoma confirmed",
            "CRITICAL",
            "Colorectal surgery referral ≤2 weeks; staging CT chest/abdomen/pelvis; "
            "CEA + CA 19-9; MSI / MMR testing; MDT; "
            "FOLFOX / FOLFIRI ± bevacizumab / cetuximab based on RAS/RAF status"),
        "benign":         (
            "Benign colonic mucosa",
            "NORMAL",
            "Colonoscopy surveillance every 3–5 years if polyp history; "
            "high-fibre diet; colorectal cancer screening per age guidelines"),
    },

    # ── Esophageal ────────────────────────────────────────────────────────────
    "esophageal": {
        "malignant": (
            "Esophageal carcinoma detected",
            "CRITICAL",
            "Upper GI surgery referral ≤2 weeks; EUS + CT-PET staging; "
            "HER2 testing for adenocarcinoma; nutritional support; "
            "neoadjuvant FLOT or CROSS (chemoRT) followed by resection if operable"),
        "benign":    (
            "Benign oesophageal tissue",
            "NORMAL",
            "Barrett's oesophagus surveillance if intestinal metaplasia present; "
            "PPI therapy; lifestyle modification (weight loss, smoking cessation); "
            "repeat endoscopy in 3–5 years"),
    },

    # ── Kidney ────────────────────────────────────────────────────────────────
    "kidney": {
        "tumor":  (
            "Renal cell carcinoma identified",
            "HIGH RISK",
            "Urology referral ≤2 weeks; CT urogram; renal function panel; "
            "partial vs radical nephrectomy decision; "
            "targeted therapy (sunitinib / pembrolizumab + axitinib) if metastatic"),
        "normal": (
            "Normal renal parenchyma",
            "NORMAL",
            "Routine renal function monitoring; annual BP check; "
            "repeat imaging only if clinically indicated"),
    },

    # ── Lung ──────────────────────────────────────────────────────────────────
    "lung": {
        "malignant": (
            "Lung carcinoma identified",
            "CRITICAL",
            "Thoracic oncology referral ≤48 h; CT-PET staging; "
            "bronchoscopy + CT-guided biopsy; EGFR / ALK / ROS1 / PD-L1 / KRAS testing; "
            "MDT: surgery (stage I–II) vs chemoimmunotherapy (stage III–IV)"),
        "benign":    (
            "Benign pulmonary tissue",
            "NORMAL",
            "Annual low-dose CT if high-risk smoker (Lung-RADS); "
            "smoking cessation counselling; spirometry if COPD suspected"),
    },

    # ── Lymphoma ──────────────────────────────────────────────────────────────
    "lymphoma": {
        "cll":  (
            "Chronic lymphocytic leukaemia confirmed",
            "HIGH RISK",
            "Haematology referral; FISH for del(17p) / del(11q) / trisomy 12; "
            "ZAP-70 / CD38 / IGHV mutation status; "
            "watch-and-wait if Rai 0–II; BTK inhibitor (ibrutinib) or FCR if high-risk"),
        "fl":   (
            "Follicular lymphoma identified",
            "MODERATE",
            "Haematology referral; CT-PET staging; FLIPI score; "
            "watch-and-wait for low-burden stage I–II; "
            "R-CHOP or R-bendamustine + maintenance rituximab for advanced disease"),
        "mcl":  (
            "Mantle cell lymphoma — aggressive",
            "CRITICAL",
            "Urgent haematology referral; Ki-67 proliferation index; MIPI score; "
            "SOX11 IHC; aggressive R-CHOP + high-dose AraC followed by ASCT if fit; "
            "BTK inhibitor for relapsed/refractory"),
    },

    # ── Oral ──────────────────────────────────────────────────────────────────
    "oral": {
        "scc":    (
            "Oral squamous cell carcinoma confirmed",
            "HIGH RISK",
            "Oral surgery / ENT referral ≤2 weeks; incisional biopsy if not done; "
            "CT neck + chest; PET-CT for stage III–IV; "
            "MDT: surgery ± adjuvant RT; cisplatin concurrent CRT for high-risk features"),
        "normal": (
            "Normal oral mucosa",
            "NORMAL",
            "Routine dental / oral screening every 12 months; "
            "tobacco and alcohol cessation counselling; "
            "oral cancer screening for high-risk patients (tobacco, betel nut)"),
    },

    # ── Ovarian ───────────────────────────────────────────────────────────────
    "ovarian": {
        "high_grade_serous":  (
            "High-grade serous ovarian carcinoma",
            "CRITICAL",
            "Gynaecologic oncology referral ≤1 week; CA-125 + HE4; CT-PET staging; "
            "BRCA1/2 germline testing; primary debulking surgery + "
            "carboplatin-paclitaxel; PARP inhibitor maintenance if BRCA+"),
        "low_grade_serous":   (
            "Low-grade serous carcinoma",
            "HIGH RISK",
            "Gynaecologic oncology referral; MEK inhibitor consideration (trametinib); "
            "fertility-sparing surgery if stage I and young patient; "
            "hormone therapy as maintenance"),
        "clear_cell":         (
            "Clear cell carcinoma of ovary",
            "HIGH RISK",
            "Gynaecologic oncology referral; platinum-based chemotherapy "
            "(resistance common); VTE prophylaxis; clinical trial enrolment encouraged"),
        "endometrioid":       (
            "Ovarian endometrioid carcinoma",
            "MODERATE",
            "Gynaecologic oncology; comprehensive staging surgery; "
            "POLE / MMR / CTNNB1 molecular testing; "
            "usually good prognosis if stage I; consider concurrent endometrial sampling"),
        "mucinous":           (
            "Mucinous ovarian carcinoma",
            "MODERATE",
            "Gynaecologic oncology; exclude GI primary (colonoscopy + gastroscopy); "
            "KRAS testing; appendicectomy at surgery; "
            "borderline vs invasive distinction requires full pathology"),
    },

    # ── Pancreatic ────────────────────────────────────────────────────────────
    "pancreatic": {
        "tumor":  (
            "Pancreatic ductal adenocarcinoma confirmed",
            "CRITICAL",
            "HPB surgery referral ≤48 h; CA 19-9 + CEA; CT pancreas protocol; "
            "ERCP / biliary stent if obstructed; FOLFIRINOX or gemcitabine-nab-paclitaxel; "
            "Whipple / distal pancreatectomy if resectable; BRCA2 / PALB2 germline testing"),
        "normal": (
            "Normal pancreatic tissue",
            "NORMAL",
            "Routine follow-up; monitor for exocrine insufficiency (faecal elastase); "
            "HbA1c monitoring for endocrine function; "
            "no surveillance imaging unless high-risk family history"),
    },

    # ── Skin ──────────────────────────────────────────────────────────────────
    "skin": {
        "melanoma":               (
            "Malignant melanoma confirmed",
            "CRITICAL",
            "Wide local excision + sentinel lymph node biopsy; "
            "BRAF V600E / NRAS / c-KIT mutation testing; "
            "adjuvant immunotherapy (nivolumab / pembrolizumab) stage III; "
            "BRAF + MEK inhibitor (dabrafenib + trametinib) if BRAF+"),
        "basal_cell_carcinoma":   (
            "Basal cell carcinoma confirmed",
            "ABNORMAL",
            "Dermatology referral; excision with 3–5 mm margins; "
            "Mohs micrographic surgery for H-zone / large / recurrent BCC; "
            "vismodegib for locally advanced / metastatic"),
        "squamous_cell_carcinoma":(
            "Cutaneous squamous cell carcinoma",
            "HIGH RISK",
            "Dermatology / plastic surgery referral; excision with 6–10 mm margins; "
            "sentinel node biopsy if >2 cm / perineural invasion / immunosuppressed; "
            "adjuvant RT for high-risk features"),
        "actinic_keratosis":      (
            "Actinic keratosis — pre-malignant",
            "MODERATE",
            "Cryotherapy or topical 5-FU / imiquimod / diclofenac; "
            "daylight PDT for field cancerisation; "
            "SPF 50+ sun protection; 6-month skin check"),
        "nevus":                  (
            "Benign melanocytic naevus",
            "NORMAL",
            "Dermoscopic follow-up annually; "
            "excise and send for pathology if ABCDE criteria met (Asymmetry, Border, "
            "Colour variation, Diameter >6 mm, Evolution); no treatment otherwise"),
        "seborrheic_keratosis":   (
            "Seborrhoeic keratosis — benign",
            "NORMAL",
            "No treatment required; cryotherapy or curettage for cosmesis if desired; "
            "reassure patient; biopsy only if clinical doubt"),
        "normal":                 (
            "Normal skin histology",
            "NORMAL",
            "Routine annual skin surveillance; SPF 50+ daily; "
            "monthly self-examination; no active treatment required"),
    },

    # ── Thyroid ───────────────────────────────────────────────────────────────
    "thyroid": {
        "malignant": (
            "Thyroid carcinoma confirmed",
            "HIGH RISK",
            "Endocrine surgery referral ≤2 weeks; total thyroidectomy; "
            "RAI ablation for intermediate/high-risk DTC; "
            "TSH suppression with levothyroxine; "
            "sorafenib / lenvatinib for RAI-refractory disease"),
        "benign":    (
            "Benign thyroid tissue",
            "NORMAL",
            "USS surveillance in 12 months if nodule present; "
            "TFTs every 12 months; levothyroxine if hypothyroid; "
            "re-biopsy if nodule grows >20% in any dimension"),
    },

    # ── Bladder ───────────────────────────────────────────────────────────────
    "bladder": {
        "muscle_invasive":     (
            "Muscle-invasive bladder cancer (MIBC) confirmed",
            "CRITICAL",
            "Urology referral ≤24 h; CT urogram + chest; "
            "radical cystectomy vs trimodality therapy (TURBT + chemoRT); "
            "neoadjuvant cisplatin-based chemotherapy (GC / MVAC); "
            "pembrolizumab for cisplatin-ineligible"),
        "non_muscle_invasive": (
            "Non-muscle-invasive bladder cancer (NMIBC)",
            "HIGH RISK",
            "TURBT + EAU risk stratification; "
            "intravesical BCG (intermediate/high-risk) or mitomycin (low-risk); "
            "cystoscopy + cytology every 3 months for 2 years; "
            "radical cystectomy if BCG-unresponsive"),
    },

    # ── Gastric ───────────────────────────────────────────────────────────────
    "gastric": {
        "tumor":  (
            "Gastric adenocarcinoma confirmed",
            "CRITICAL",
            "Upper GI surgery referral ≤2 weeks; EUS staging; HER2 IHC/ISH; "
            "MSI / EBV / PD-L1 testing; perioperative FLOT chemotherapy; "
            "D2 gastrectomy; trastuzumab + chemotherapy if HER2+"),
        "deb":    (
            "Necrotic tumour debris",
            "HIGH RISK",
            "Gastroenterology referral; repeat endoscopy + biopsy from viable margins; "
            "nutritional support (NG or PEG); reassess resectability after 4–6 weeks"),
        "lym":    (
            "Lymphocytic tumour infiltrate",
            "MODERATE",
            "Gastroenterology referral; assess for gastric MALT lymphoma (FISH t(11;18)); "
            "H. pylori eradication therapy (PPI + clarithromycin + amoxicillin × 14 d); "
            "restage endoscopy at 3 months"),
        "str":    (
            "Stromal tumour component (possible GIST)",
            "MODERATE",
            "Gastroenterology / surgery referral; c-KIT / PDGFRA / SDH mutation testing; "
            "EUS-FNA; imatinib 400 mg/day if c-KIT+ or PDGFRA non-D842V; "
            "surgery if localised and resectable"),
        "adi":    (
            "Adipose tissue — non-malignant",
            "NORMAL",
            "No immediate action; report as benign in resection specimen; "
            "correlate with clinical and radiological findings"),
        "muc":    (
            "Gastric mucus — normal",
            "NORMAL",
            "No action required; routine endoscopy surveillance per guidelines; "
            "H. pylori screen if symptomatic"),
        "mus":    (
            "Smooth muscle — normal muscularis propria",
            "NORMAL",
            "No action required; correlate with surgical resection margins"),
        "normal": (
            "Normal gastric mucosa",
            "NORMAL",
            "H. pylori screen (urea breath test or stool antigen); "
            "routine endoscopy if symptomatic or high-risk (family history / geography); "
            "lifestyle advice (smoking, alcohol, salt-preserved foods)"),
    },
}


# ════════════════════════════════════════════════════════════════════════════════
# CONCEPTS
# (organ, class_name) → [primary_label, secondary_label]
# Primary used when CAM peak is in bright region, secondary in dark region.
# These appear as annotation arrows in Column 5 (MS-CAM + concept) of panels.
# ════════════════════════════════════════════════════════════════════════════════
CONCEPTS = {
    # ── Blood ─────────────────────────────────────────────────────────────────
    ("blood", "pre"):    ["Lymphoblast nuclei",          "Chromatin condensation pattern"],
    ("blood", "early"):  ["Blast cell nuclear morphology","Nuclear-to-cytoplasmic ratio"],
    ("blood", "pro"):    ["Pro-B blast aggregates",       "Auer rod / granule region"],
    ("blood", "benign"): ["Mature RBC morphology",        "Normal lymphocyte nucleus"],

    # ── Brain ─────────────────────────────────────────────────────────────────
    ("brain_2d", "glioma"):     ["Tumour core hypercellularity", "Necrotic foci / pseudopalisading"],
    ("brain_2d", "meningioma"): ["Whorling cell pattern",        "Psammoma bodies"],
    ("brain_2d", "tumor"):      ["Mass lesion boundary",         "Perilesional oedema"],

    # ── Breast ────────────────────────────────────────────────────────────────
    ("breast", "malignant"): ["Invasive tumour nest",       "Desmoplastic stromal reaction"],
    ("breast", "benign"):    ["Normal ductal epithelium",   "Fibrous stroma / adipose"],

    # ── Cervical ──────────────────────────────────────────────────────────────
    ("cervical", "dyk"):  ["Dyskeratotic cell cluster",      "Hyperchromatic dysplastic nuclei"],
    ("cervical", "koc"):  ["Koilocyte perinuclear halo",     "Binucleated HPV-effect cell"],
    ("cervical", "mep"):  ["Metaplastic cell border",        "Squamocolumnar junction"],
    ("cervical", "pab"):  ["Parabasal cell aggregate",       "Inflammatory background"],
    ("cervical", "sfi"):  ["Superficial squamous cell",      "Intermediate cell layer"],

    # ── Colon ─────────────────────────────────────────────────────────────────
    ("colon", "adenocarcinoma"): ["Glandular invasion front",  "Dysplastic crypt architecture"],
    ("colon", "benign"):         ["Normal crypt regularity",   "Goblet cell population"],

    # ── Esophageal ────────────────────────────────────────────────────────────
    ("esophageal", "malignant"): ["Tumour nest infiltration",  "Keratin pearl / squamous nests"],
    ("esophageal", "benign"):    ["Normal squamous epithelium","Lamina propria architecture"],

    # ── Kidney ────────────────────────────────────────────────────────────────
    ("kidney", "tumor"):  ["Clear cell cytoplasm",         "Sinusoidal vascular pattern"],
    ("kidney", "normal"): ["Normal glomerulus",            "Proximal tubule brush border"],

    # ── Lung ──────────────────────────────────────────────────────────────────
    ("lung", "malignant"): ["Tumour cell nest / lepidic growth", "Pleural invasion zone"],
    ("lung", "benign"):    ["Alveolar wall architecture",        "Normal bronchiolar epithelium"],

    # ── Lymphoma ──────────────────────────────────────────────────────────────
    ("lymphoma", "cll"):  ["Small lymphocyte clusters",    "Smudge cell artefact"],
    ("lymphoma", "fl"):   ["Follicular centre pattern",    "Centrocyte / centroblast ratio"],
    ("lymphoma", "mcl"):  ["Mantle zone expansion",        "Blastoid nuclear morphology"],

    # ── Oral ──────────────────────────────────────────────────────────────────
    ("oral", "scc"):    ["Keratin pearl formation",       "Invasive tumour front"],
    ("oral", "normal"): ["Normal stratified epithelium",  "Connective tissue stroma"],

    # ── Ovarian ───────────────────────────────────────────────────────────────
    ("ovarian", "high_grade_serous"):  ["Papillary serous architecture",   "Psammoma body calcification"],
    ("ovarian", "low_grade_serous"):   ["Low-grade papillary frond",        "Mild nuclear atypia"],
    ("ovarian", "clear_cell"):         ["Clear cytoplasm / hobnail cells",  "Tubulo-cystic pattern"],
    ("ovarian", "endometrioid"):       ["Endometrioid gland formation",     "Squamous morule"],
    ("ovarian", "mucinous"):           ["Mucinous epithelial lining",       "Intracellular mucin pool"],

    # ── Pancreatic ────────────────────────────────────────────────────────────
    ("pancreatic", "tumor"):  ["Ductal invasion / desmoplasia", "Perineural invasion focus"],
    ("pancreatic", "normal"): ["Islet of Langerhans",           "Acinar cell zymogen granules"],

    # ── Skin ──────────────────────────────────────────────────────────────────
    ("skin", "melanoma"):               ["Atypical melanocyte nests",       "Breslow depth measurement zone"],
    ("skin", "basal_cell_carcinoma"):   ["Palisading basal cell nuclei",    "Stromal retraction artefact"],
    ("skin", "squamous_cell_carcinoma"):["Keratin pearl centre",            "Intercellular bridge region"],
    ("skin", "actinic_keratosis"):      ["Dysplastic keratinocyte layer",   "Solar elastosis zone"],
    ("skin", "nevus"):                  ["Naevus cell nests at DEJ",        "Maturation into dermis"],
    ("skin", "seborrheic_keratosis"):   ["Horn cyst formation",             "Acanthosis / papillomatosis"],
    ("skin", "normal"):                 ["Normal stratified epidermis",     "Dermal collagen architecture"],

    # ── Thyroid ───────────────────────────────────────────────────────────────
    ("thyroid", "malignant"): ["Papillary architecture / ground-glass nuclei", "Psammoma body / nuclear groove"],
    ("thyroid", "benign"):    ["Colloid-filled follicle",                       "Follicular epithelium"],

    # ── Bladder ───────────────────────────────────────────────────────────────
    ("bladder", "muscle_invasive"):     ["Detrusor muscle invasion front",  "High-grade urothelial nest"],
    ("bladder", "non_muscle_invasive"): ["Papillary urothelial frond",      "Lamina propria boundary"],

    # ── Gastric ───────────────────────────────────────────────────────────────
    ("gastric", "tumor"):  ["Signet ring cell cluster",      "Glandular invasion desmoplasia"],
    ("gastric", "deb"):    ["Necrotic debris field",         "Apoptotic tumour cells"],
    ("gastric", "lym"):    ["Lymphocyte infiltrate band",    "Lymphoid follicle germinal centre"],
    ("gastric", "str"):    ["Spindle cell stromal component","GIST-like fascicular pattern"],
    ("gastric", "adi"):    ["Adipocyte vacuole",             "Fibrovascular septum"],
    ("gastric", "muc"):    ["Mucus pool / goblet cells",     "Surface mucous layer"],
    ("gastric", "mus"):    ["Smooth muscle bundle",          "Muscularis propria layer"],
    ("gastric", "normal"): ["Normal gastric pit / gland",    "Surface foveolar epithelium"],
}


# ════════════════════════════════════════════════════════════════════════════════
# Sanity check — print coverage stats
# ════════════════════════════════════════════════════════════════════════════════
_all_organ_classes = [
    (org, cls)
    for org, clsset in MALIGNANT_CLASSES.items()
    for cls in clsset
] + [
    ("blood",      "benign"),
    ("breast",     "benign"),
    ("colon",      "benign"),
    ("esophageal", "benign"),
    ("kidney",     "normal"),
    ("lung",       "benign"),
    ("oral",       "normal"),
    ("pancreatic", "normal"),
    ("thyroid",    "benign"),
    ("skin",       "nevus"),
    ("skin",       "seborrheic_keratosis"),
    ("skin",       "normal"),
    ("gastric",    "adi"),
    ("gastric",    "muc"),
    ("gastric",    "mus"),
    ("gastric",    "normal"),
    ("cervical",   "sfi"),
    ("cervical",   "mep"),
    ("cervical",   "pab"),
]

_staging_missing   = [(o,c) for o,c in _all_organ_classes if (o,c) not in CANCER_STAGING]
_protocol_missing  = [(o,c) for o,c in _all_organ_classes
                      if o not in CLINICAL_PROTOCOLS or c not in CLINICAL_PROTOCOLS.get(o,{})]
_concept_missing   = [(o,c) for o,c in _all_organ_classes if (o,c) not in CONCEPTS]

print(f"  CANCER_STAGING   : {len(CANCER_STAGING)} entries  |  missing: {_staging_missing  or 'none'}")
print(f"  CLINICAL_PROTOCOLS: {sum(len(v) for v in CLINICAL_PROTOCOLS.values())} entries  |  missing: {_protocol_missing or 'none'}")
print(f"  CONCEPTS         : {len(CONCEPTS)} entries  |  missing: {_concept_missing   or 'none'}")

print(f"\n✅ Cell 1 complete | Device:{DEVICE}" +
      (f" | GPU:{torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "") +
      f" | CRF:{'✅' if HAS_CRF else '❌'}"
      f" | NumPy:{np.__version__}"
      f" | trapezoid:{'native' if hasattr(np,'trapezoid') else 'shimmed'}")

In [ ]:
import shutil
# ════════════════════════════════════════════════════════════════════════════════
# V22-P — Checkpoint preflight  (runs BEFORE Cell 3)
#   • If a SPEC_CKPTS path is missing, searches every attached model for the same
#     file name (any version) and re-points SPEC_CKPTS.
#   • Prints exactly which Kaggle *Model* inputs are still missing.
# ════════════════════════════════════════════════════════════════════════════════
import glob, os
from collections import defaultdict
_MODELS_ROOT = Path("/kaggle/input/models")
_all_pth = [Path(p) for p in glob.glob(str(_MODELS_ROOT / "**" / "*.pth"), recursive=True)] if _MODELS_ROOT.exists() else []
_by_name = defaultdict(list)
for _p in _all_pth: _by_name[_p.name].append(_p)

def _model_input(p):          # /kaggle/input/models/<owner>/<model>/<fw>/<variant>/<version>
    parts = Path(p).parts
    try: i = parts.index("models"); return "/".join(parts[i+1:i+6])
    except ValueError: return str(Path(p).parent)

_missing_inputs = defaultdict(list)
print("── Specialist checkpoints ─────────────────────────────────────────────")
for _o, _p in list(SPEC_CKPTS.items()):
    if Path(_p).exists():
        print(f"  ✅ {_o:<11} {_model_input(_p)}"); continue
    _alt = sorted(_by_name.get(Path(_p).name, []))
    if _alt:
        SPEC_CKPTS[_o] = _alt[-1]
        print(f"  🔁 {_o:<11} re-pointed → {_model_input(_alt[-1])}")
    else:
        _missing_inputs[_model_input(_p)].append(_o)
        print(f"  ❌ {_o:<11} missing: {Path(_p).name}")

_HOVER = "hovernet_pannuke.pth"
_hv = (SPEC_DIR / _HOVER).exists() or bool(_by_name.get(_HOVER))
print(f"\n  HoVer-Net ({_HOVER}): {'✅ available' if _hv else '❌ missing'}")
if not _hv: _missing_inputs["marcyoussef22/segmentation/other/default/1"].append("hovernet")
if _by_name.get(_HOVER) and not (SPEC_DIR / _HOVER).exists():
    SPEC_DIR.mkdir(parents=True, exist_ok=True); shutil.copy2(str(_by_name[_HOVER][0]), str(SPEC_DIR / _HOVER))
    print(f"  → copied from {_model_input(_by_name[_HOVER][0])}")
print(f"  Gateway: {'✅' if Path(GATEWAY_CKPT).exists() else '❌'} {_model_input(GATEWAY_CKPT)}")

if _missing_inputs:
    print("\n  ➜ In Kaggle: Add Input → Models → attach these (exact variant/version):")
    for _k, _v in _missing_inputs.items(): print(f"     • {_k:<55} provides {_v}")
    print("  The pipeline will still run; missing organs are skipped everywhere downstream.")
else:
    print("\n  ✅ All checkpoints present")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# V20 helper — safe "array or default" (fixes NumPy truth-value ambiguity)
#   `arr or np.zeros(...)` raises ValueError when arr is a NumPy array.
# ════════════════════════════════════════════════════════════════════════════════
import numpy as np

def _nz(x, shape=(224, 224)):
    """Return x if it is not None, else a float32 zero map."""
    return x if x is not None else np.zeros(shape, np.float32)

print("✅ V20 helper _nz() defined")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 2 — Dataset, Augmentation, Models
#   v17 base (all original code preserved)
#   === UPGRADE v18 === SuperHeavyAug + Dynamic Sampler + MixUp/CutMix
# ════════════════════════════════════════════════════════════════════════════════

# ── v16-BUG-4: Macenko normalizer with real synthetic H&E reference ───────────
_macenko_normalizer = None
if STAIN_NORM_AVAILABLE:
    try:
        import torchstain
        if hasattr(torchstain, "normalizers"):
            _macenko_normalizer = torchstain.normalizers.MacenkoNormalizer(
                backend='torch'
            )
        else:
            from torchstain.torch.normalizers import MacenkoNormalizer
            _macenko_normalizer = MacenkoNormalizer()
        _ref = np.zeros((224, 224, 3), dtype=np.uint8)
        _ref[:] = [240, 235, 240]
        _ref[50:100, 50:100]   = [80,  60, 160]
        _ref[120:170, 120:170] = [90,  55, 155]
        _ref[50:100,  120:180] = [220, 130, 160]
        _ref[120:170, 50:110]  = [215, 125, 155]
        _ref_t = torch.from_numpy(_ref).permute(2,0,1).float() / 255.
        _macenko_normalizer.fit(_ref_t)
        print("  ✅ Macenko fitted on synthetic H&E reference")
    except Exception as _e:
        print(f"  ⚠️  Macenko fit failed: {_e}"); _macenko_normalizer = None

# ── v16-NEW: Vahadane numpy-only fallback ─────────────────────────────────────
def _vahadane_norm_np(img_np_hwc):
    TARGET_OD_MEAN = np.array([0.35, 0.45, 0.10])
    eps = 1e-6
    img = img_np_hwc.astype(np.float32)
    img = np.clip(img, 1, 255)
    OD  = -np.log(img / 255.0 + eps)
    for c in range(3):
        ch_mean = OD[:,:,c].mean()
        if ch_mean > eps:
            OD[:,:,c] = OD[:,:,c] * (TARGET_OD_MEAN[c] / ch_mean)
    norm = np.exp(-OD) * 255.0
    return np.clip(norm, 0, 255).astype(np.uint8)

def _apply_stain_norm(pil_img):
    if _macenko_normalizer is not None:
        try:
            t = transforms.ToTensor()(pil_img)
            t_norm, _, _ = _macenko_normalizer.normalize(t.unsqueeze(0))
            return transforms.ToPILImage()(t_norm.squeeze(0).clamp(0,1))
        except Exception:
            pass
    try:
        arr = np.array(pil_img)
        arr_norm = _vahadane_norm_np(arr)
        return Image.fromarray(arr_norm)
    except Exception:
        return pil_img

# ── Transforms (original, preserved) ─────────────────────────────────────────
# v24 CRITICAL FIX: no stain normalisation at evaluation (the specialists were trained without it;
# it recoloured MRI/CT/mammograms/ultrasound and collapsed lymphoma 0.994 → 0.573).
val_tfm = transforms.Compose(
    ([transforms.Lambda(_apply_stain_norm)] if globals().get("EVAL_STAIN_NORM", False) else [])
    + ([transforms.Resize(256), transforms.CenterCrop(224)] if globals().get("EVAL_RESIZE", "resize224") == "crop"
       else [transforms.Resize((224, 224))])
    + [transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
print(f"  val_tfm: stain_norm={globals().get('EVAL_STAIN_NORM', False)}  resize={globals().get('EVAL_RESIZE', 'resize224')}")
train_tfm = transforms.Compose([
    transforms.RandomResizedCrop(224,scale=(0.7,1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3,contrast=0.3,saturation=0.2,hue=0.05),
    transforms.RandomRotation(20), RandAugment(num_ops=2,magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    RandomErasing(p=0.25,scale=(0.02,0.15)),
])

def _he_stain_jitter(image, **kwargs):
    alpha=np.random.uniform(0.85,1.15); beta=np.random.randint(-15,15)
    ch_h=np.random.uniform(0.90,1.10); ch_e=np.random.uniform(0.90,1.10)
    img=image.astype(np.float32)
    img[:,:,0]=np.clip(img[:,:,0]*alpha*ch_e+beta,0,255)
    img[:,:,1]=np.clip(img[:,:,1]*alpha+beta,0,255)
    img[:,:,2]=np.clip(img[:,:,2]*alpha*ch_h+beta,0,255)
    return img.astype(np.uint8)

# ── Original HEAVY_AUG (preserved) ────────────────────────────────────────────
HEAVY_AUG = A.Compose([
    A.RandomResizedCrop(size=(224,224),scale=(0.5,1.0),ratio=(0.75,1.33)),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
    A.Rotate(limit=30,p=0.6),
    A.ElasticTransform(alpha=80,sigma=10,p=0.4),
    A.GridDistortion(num_steps=5,distort_limit=0.3,p=0.3),
    A.ColorJitter(brightness=0.35,contrast=0.35,saturation=0.25,hue=0.08,p=0.7),
    A.GaussianBlur(blur_limit=(3,7),p=0.3), A.GaussNoise(var_limit=(10,50),p=0.3),
    A.CLAHE(clip_limit=4.0,tile_grid_size=(8,8),p=0.3),
    A.Lambda(image=_he_stain_jitter,p=0.5),
    A.CoarseDropout(max_holes=8,max_height=24,max_width=24,min_holes=1,fill_value=0,p=0.3),
    A.PadIfNeeded(min_height=224,min_width=224,border_mode=0,value=0,p=1.0),
    A.CenterCrop(height=224,width=224,p=1.0),
    A.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]),
    ToTensorV2(),
])
HEAVY_VAL_AUG = A.Compose([
    A.Resize(224,224),
    A.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]),
    ToTensorV2(),
])


# ════════════════════════════════════════════════════════════════════════════════
# === UPGRADE v18 === SuperHeavyAug pipeline
#
# WHY: Skin seborrheic_keratosis (F1=0.108), SCC (F1=0.446), cervical dyk/koc
#      (F1=0.14/0.21), kidney tumor (F1=0.39), pancreatic tumor (F1=0.37),
#      gastric adi/mus (F1=0.56/0.57) all suffer from:
#      (a) class imbalance that WeightedSampler alone can't fix,
#      (b) texture/morphology similarity between confusable classes,
#      (c) insufficient stain variation at training time.
#
# FIX: SuperHeavyAug adds:
#   - Stronger H&E stain jitter with per-channel OD perturbation
#   - Optical distortion + grid distortion for tissue deformation
#   - Fine-grained morphology augmentation (Sharpen + ISO noise)
#   - Much higher CoarseDropout (simulates tissue artifacts, fold shadows)
#   - Histogram equalisation per channel
# ════════════════════════════════════════════════════════════════════════════════

def _he_stain_jitter_strong(image, **kwargs):
    """
    === UPGRADE v18 ===
    Stronger H&E stain jitter targeting OD space.
    Applies independent per-channel alpha/beta + OD-space perturbation.
    Addresses: skin texture similarity, cervical cytology stain variability,
               gastric tissue heterogeneity.
    """
    img = image.astype(np.float32)
    # Per-channel random scale + shift (wider range than v17)
    for c in range(3):
        alpha_c = np.random.uniform(0.75, 1.25)
        beta_c  = np.random.uniform(-25., 25.)
        img[:,:,c] = np.clip(img[:,:,c] * alpha_c + beta_c, 0., 255.)
    # OD-space perturbation (simulates staining batch variation)
    img_clipped = np.clip(img, 1., 255.)
    od = -np.log(img_clipped / 255.0 + 1e-6)
    od_scale = np.random.uniform(0.85, 1.15, size=(1, 1, 3)).astype(np.float32)
    od = od * od_scale
    img = np.exp(-od) * 255.0
    return np.clip(img, 0., 255.).astype(np.uint8)


def _microscopy_sharpen(image, **kwargs):
    """
    === UPGRADE v18 ===
    Unsharp mask to enhance nuclear membrane and tissue borders.
    Especially useful for: cervical dyskeratosis (dyk/koc), skin keratin
    patterns (seborrheic_keratosis), gastric gland architecture.
    """
    import cv2 as _cv2
    img = image.astype(np.float32)
    strength = np.random.uniform(0.3, 0.9)
    blur = _cv2.GaussianBlur(img, (5, 5), sigmaX=1.0)
    sharpened = img + strength * (img - blur)
    return np.clip(sharpened, 0., 255.).astype(np.uint8)


def _simulate_tissue_artifact(image, **kwargs):
    """
    === UPGRADE v18 ===
    Simulates common H&E slide artifacts:
    - Tissue fold (local brightness gradient)
    - Air bubble (circular low-signal region)
    - Scanner line (horizontal/vertical stripe)
    Improves robustness for gastric/pancreatic low-quality slides.
    """
    img = image.copy().astype(np.float32)
    H, W = img.shape[:2]
    artifact_type = np.random.choice(['fold', 'bubble', 'line', 'none'],
                                      p=[0.2, 0.2, 0.2, 0.4])
    if artifact_type == 'fold':
        # Gradient brightness strip
        x0 = np.random.randint(W//4, 3*W//4)
        width = np.random.randint(10, 40)
        grad = np.linspace(0.6, 1.0, width)
        x1 = min(x0 + width, W)
        actual_w = x1 - x0
        img[:, x0:x1, :] *= grad[:actual_w][None, :, None]
    elif artifact_type == 'bubble':
        cx, cy = np.random.randint(30, W-30), np.random.randint(30, H-30)
        r = np.random.randint(10, 30)
        yy, xx = np.ogrid[:H, :W]
        circle = (xx - cx)**2 + (yy - cy)**2 <= r**2
        img[circle] = np.clip(img[circle] * 0.3 + 200., 0., 255.)
    elif artifact_type == 'line':
        axis = np.random.randint(0, 2)
        pos  = np.random.randint(10, (H if axis==0 else W) - 10)
        thick = np.random.randint(1, 4)
        intensity = np.random.uniform(0.2, 0.8)
        if axis == 0:
            img[pos:pos+thick, :, :] *= intensity
        else:
            img[:, pos:pos+thick, :] *= intensity
    return np.clip(img, 0., 255.).astype(np.uint8)


# === UPGRADE v18 === SuperHeavyAug — for weak organs
SUPER_HEAVY_AUG = A.Compose([
    # --- Geometric (strong + diverse) ---
    A.RandomResizedCrop(size=(224,224), scale=(0.40,1.0), ratio=(0.70,1.43)),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
    A.Rotate(limit=45, p=0.7),
    A.ElasticTransform(alpha=120, sigma=12, p=0.5),        # stronger than HEAVY
    A.GridDistortion(num_steps=6, distort_limit=0.40, p=0.4),
    A.OpticalDistortion(distort_limit=0.35, shift_limit=0.12, p=0.35),
    # --- Stain & Color (H&E-specific) ---
    A.ColorJitter(brightness=0.45, contrast=0.45, saturation=0.30, hue=0.10, p=0.8),
    A.Lambda(image=_he_stain_jitter_strong, p=0.65),       # strong OD-space jitter
    A.CLAHE(clip_limit=6.0, tile_grid_size=(8,8), p=0.4),
    A.ToGray(p=0.05),                                       # occasional grayscale
    A.RandomBrightnessContrast(brightness_limit=0.25,
                                contrast_limit=0.25, p=0.5),
    A.HueSaturationValue(hue_shift_limit=12,
                          sat_shift_limit=25,
                          val_shift_limit=20, p=0.5),
    # --- Noise & Blur ---
    A.GaussianBlur(blur_limit=(3,9), p=0.35),
    A.GaussNoise(var_limit=(15,70), p=0.35),
    A.ISONoise(color_shift=(0.01,0.05), intensity=(0.1,0.4), p=0.30),
    A.MotionBlur(blur_limit=5, p=0.15),
    # --- Morphology enhancement ---
    A.Lambda(image=_microscopy_sharpen, p=0.40),
    A.Lambda(image=_simulate_tissue_artifact, p=0.30),
    # --- Erasing (strong) ---
    A.CoarseDropout(max_holes=12, max_height=32, max_width=32,
                    min_holes=2,  min_height=8,  min_width=8,
                    fill_value=0, p=0.45),
    # --- Final ---
    A.PadIfNeeded(min_height=224, min_width=224, border_mode=0, value=0, p=1.0),
    A.CenterCrop(height=224, width=224, p=1.0),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])

SUPER_HEAVY_VAL_AUG = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])

# === UPGRADE v18 === Organ → aug config mapping
# WHY: Different organs have different failure modes.
#   "superheavy" → SuperHeavyAug (repeat=6)  — severely underperforming organs
#   "heavy"      → existing HEAVY_AUG (repeat=4)  — moderately hard organs
#   "standard"   → existing train_tfm  — well-performing organs
AUGMENT_CFG_V18 = {
    # Severely underperforming (F1 < 0.65): SuperHeavy
    "skin":       "superheavy",    # seborrheic=0.108, scc=0.446
    "cervical":   "superheavy",    # dyk=0.140, koc=0.211
    "kidney":     "superheavy",    # tumor=0.39 overall
    "pancreatic": "superheavy",    # tumor=0.37
    "gastric":    "superheavy",    # adi=0.56, mus=0.57
    "lymphoma":   "superheavy",    # cll/fl/mcl confusion
    # Moderately hard (keep existing heavy):
    "bladder":    "heavy",         # small dataset, original "heavy"
    "thyroid":    "heavy",         # original "heavy"
    "breast":     "heavy",         # F1=0.79
    "ovarian":    "heavy",         # F1=0.77
    # Well-performing (standard suffices):
    "blood":      "standard",
    "colon":      "standard",
    "brain_2d":   "standard",
    "esophageal": "standard",
    "lung":       "standard",
    "oral":       "standard",
}

# Preserve backward compat: AUGMENT_CFG still used by Cells 5b/7/11
# V18 aug config is only used in build_loaders_v18()
AUGMENT_REPEAT_V18 = {
    "superheavy": 6,
    "heavy":      4,
    "standard":   1,
}

print(f"  === UPGRADE v18 === SuperHeavyAug defined")
print(f"    Organs using SuperHeavy: {[k for k,v in AUGMENT_CFG_V18.items() if v=='superheavy']}")
print(f"    Organs using Heavy     : {[k for k,v in AUGMENT_CFG_V18.items() if v=='heavy']}")


# ════════════════════════════════════════════════════════════════════════════════
# === UPGRADE v18 === MixUp / CutMix utilities
#
# WHY: MixUp forces the model to interpolate between confusable classes,
#      strongly reducing overconfident wrong predictions for:
#      seborrheic_keratosis vs nevus, dyk vs koc, mus vs str.
# ════════════════════════════════════════════════════════════════════════════════

def mixup_batch(imgs: torch.Tensor, labels: torch.Tensor,
                alpha: float = 0.4) -> tuple:
    """
    === UPGRADE v18 === Standard MixUp.
    Returns (mixed_imgs, labels_a, labels_b, lam).
    Loss = lam * CE(pred, labels_a) + (1-lam) * CE(pred, labels_b)
    """
    lam = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    B   = imgs.size(0)
    idx = torch.randperm(B, device=imgs.device)
    mixed = lam * imgs + (1. - lam) * imgs[idx]
    return mixed, labels, labels[idx], lam


def cutmix_batch(imgs: torch.Tensor, labels: torch.Tensor,
                 alpha: float = 0.4) -> tuple:
    """
    === UPGRADE v18 === CutMix.
    Cuts a rectangular patch from one image and pastes into another.
    More structure-preserving than MixUp — better for texture discrimination.
    """
    lam = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    B, C, H, W = imgs.shape
    idx  = torch.randperm(B, device=imgs.device)
    # Box dimensions
    cut_ratio = np.sqrt(1. - lam)
    cut_h = int(H * cut_ratio)
    cut_w = int(W * cut_ratio)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = max(0, cx - cut_w // 2); x2 = min(W, cx + cut_w // 2)
    y1 = max(0, cy - cut_h // 2); y2 = min(H, cy + cut_h // 2)
    mixed = imgs.clone()
    mixed[:, :, y1:y2, x1:x2] = imgs[idx, :, y1:y2, x1:x2]
    # Recompute lam based on actual box
    lam = 1. - (float(y2-y1) * float(x2-x1)) / (H * W)
    return mixed, labels, labels[idx], lam


def mixup_or_cutmix(imgs: torch.Tensor, labels: torch.Tensor,
                    alpha: float = 0.4, p_cutmix: float = 0.5) -> tuple:
    """
    === UPGRADE v18 === Randomly applies MixUp or CutMix.
    p_cutmix: probability of using CutMix (else MixUp).
    """
    if np.random.rand() < p_cutmix:
        return cutmix_batch(imgs, labels, alpha)
    else:
        return mixup_batch(imgs, labels, alpha)


def mixup_criterion(criterion, pred: torch.Tensor,
                    y_a: torch.Tensor, y_b: torch.Tensor,
                    lam: float) -> torch.Tensor:
    """
    === UPGRADE v18 === Loss for MixUp/CutMix blended labels.
    """
    return lam * criterion(pred, y_a) + (1. - lam) * criterion(pred, y_b)


# === UPGRADE v18 === Organs where MixUp/CutMix is applied during training
MIXUP_ORGANS = {
    "skin",       # confusable textures
    "cervical",   # subtle cytological differences
    "gastric",    # 8-class tissue confusion
    "lymphoma",   # subtle morphological differences
    "ovarian",    # fine-grained subtype differences
    "kidney",     # normal vs tumor
    "pancreatic", # normal vs tumor + small dataset
}
MIXUP_ALPHA = 0.40   # Beta distribution parameter


# ════════════════════════════════════════════════════════════════════════════════
# === UPGRADE v18 === Dynamic per-class weighted sampler
#
# WHY: Standard WeightedRandomSampler uses 1/count weighting which over-samples
#      rare classes but ignores their difficulty. Dynamic weighting adds a
#      difficulty factor (inverse of per-class F1) so easy-but-rare classes
#      are not over-sampled relative to hard-but-common classes.
# ════════════════════════════════════════════════════════════════════════════════

def compute_dynamic_class_weights(labels: list, class_names: list,
                                   organ: str,
                                   eval_results_ref: dict = None,
                                   rare_boost_factor: float = 3.0,
                                   difficulty_factor: float = 2.0) -> list:
    """
    === UPGRADE v18 === Dynamic per-class weights for WeightedRandomSampler.

    Weight_c = (1 / count_c)^freq_exp × difficulty_c^diff_exp

    difficulty_c = 1 / (class_f1_c + 0.05)  — harder classes get higher weight

    Args:
        labels            : list of integer labels in training set
        class_names       : list of class name strings
        organ             : organ key (for looking up per-class F1 from eval)
        eval_results_ref  : optional eval_results dict (from Cell 7) for F1 lookup
        rare_boost_factor : exponent on frequency term (higher = more upsampling)
        difficulty_factor : exponent on difficulty term
    Returns:
        list of per-sample weights
    """
    n_cls   = len(class_names)
    counts  = np.bincount(labels, minlength=n_cls).astype(np.float32)
    counts  = np.maximum(counts, 1.)  # avoid div-by-zero

    # Frequency-based weight: (1/count)^rare_boost_factor
    freq_w = (1. / counts) ** rare_boost_factor

    # Difficulty-based weight from per-class F1 (if available)
    diff_w = np.ones(n_cls, dtype=np.float32)
    if eval_results_ref is not None and organ in eval_results_ref:
        res = eval_results_ref[organ]
        # Try to get per-class F1 from classification report stored in eval_results
        per_cls_f1 = res.get("per_class_f1", {})
        for ci, cname in enumerate(class_names):
            f1_val = per_cls_f1.get(cname, None)
            if f1_val is not None:
                diff_w[ci] = (1. / (float(f1_val) + 0.05)) ** difficulty_factor

    # Combine: class_weight = freq_w * diff_w, normalised
    class_w = freq_w * diff_w
    class_w = class_w / class_w.sum() * n_cls  # normalise to mean=1

    # Hard-coded boosts for known severely underperforming classes
    HARD_CLASS_BOOST = {
        ("skin",     "seborrheic_keratosis"):  5.0,
        ("skin",     "squamous_cell_carcinoma"): 3.0,
        ("skin",     "actinic_keratosis"):     2.0,
        ("cervical", "dyk"):                   4.0,
        ("cervical", "koc"):                   3.5,
        ("gastric",  "adi"):                   2.5,
        ("gastric",  "mus"):                   2.5,
        ("gastric",  "deb"):                   2.0,
        ("lymphoma", "mcl"):                   2.5,
        ("ovarian",  "low_grade_serous"):      3.0,
    }
    for ci, cname in enumerate(class_names):
        boost = HARD_CLASS_BOOST.get((organ, cname), 1.0)
        class_w[ci] *= boost

    # Map per-sample weights
    sample_weights = [float(class_w[lbl]) for lbl in labels]
    return sample_weights


# ════════════════════════════════════════════════════════════════════════════════
# === UPGRADE v18 === SuperHeavyAugDataset
#
# WHY: Unlike HeavyAugDataset which uses fixed repeat,
#      SuperHeavyAugDataset allows per-class repeat factors.
# ════════════════════════════════════════════════════════════════════════════════

class SuperHeavyAugDataset(Dataset):
    """
    === UPGRADE v18 ===
    Dataset with class-conditional repeat and SuperHeavyAug.
    rare_classes: set of class indices to repeat extra times.
    rare_repeat: additional repeat factor for rare_classes.
    """
    def __init__(self, paths, labels, aug, repeat=6,
                 rare_classes=None, rare_repeat=2):
        self.paths        = paths
        self.labels       = labels
        self.aug          = aug
        self.base_repeat  = repeat
        self.rare_classes = rare_classes or set()
        self.rare_repeat  = rare_repeat
        # Build expanded index list
        self._idx = []
        for i, lbl in enumerate(labels):
            reps = repeat
            if lbl in self.rare_classes:
                reps = repeat * rare_repeat
            for _ in range(reps):
                self._idx.append(i)

    def __len__(self):
        return len(self._idx)

    def __getitem__(self, idx):
        ri  = self._idx[idx]
        img = Image.open(self.paths[ri]).convert("RGB")
        return self.aug(image=np.array(img))["image"], self.labels[ri]


# ════════════════════════════════════════════════════════════════════════════════
# Original Dataset classes (preserved, not modified)
# ════════════════════════════════════════════════════════════════════════════════

class HeavyAugDataset(Dataset):
    def __init__(self,paths,labels,aug,repeat=4):
        self.paths=paths; self.labels=labels; self.aug=aug; self.repeat=repeat
    def __len__(self): return len(self.paths)*self.repeat
    def __getitem__(self,idx):
        ri=idx%len(self.paths)
        img=Image.open(self.paths[ri]).convert("RGB")
        return self.aug(image=np.array(img))["image"], self.labels[ri]

class StandardDataset(Dataset):
    def __init__(self,paths,labels,tfm):
        self.paths=paths; self.labels=labels; self.tfm=tfm
    def __len__(self): return len(self.paths)
    def __getitem__(self,idx):
        return self.tfm(Image.open(self.paths[idx]).convert("RGB")), self.labels[idx]


# ════════════════════════════════════════════════════════════════════════════════
# Original Model classes (preserved, not modified)
# ════════════════════════════════════════════════════════════════════════════════

class FocalLoss(nn.Module):
    def __init__(self,gamma=2.0,label_smoothing=0.10):
        super().__init__(); self.gamma=gamma; self.ls=label_smoothing
    def forward(self,logits,targets):
        ce=F.cross_entropy(logits,targets,label_smoothing=self.ls,reduction="none")
        return ((1-torch.exp(-ce))**self.gamma*ce).mean()

class SEHead(nn.Module):
    def __init__(self,dim,reduction=16):
        super().__init__()
        b=max(1,dim//reduction)
        self.fc=nn.Sequential(nn.Linear(dim,b,bias=False),nn.GELU(),
                              nn.Linear(b,dim,bias=False),nn.Sigmoid())
    def forward(self,x): return x*self.fc(x)

class SpecialistModel(nn.Module):
    def __init__(self,backbone_name,n_classes,dropout=0.25,pretrained=True):
        super().__init__()
        self.backbone=timm.create_model(backbone_name,pretrained=pretrained,num_classes=0)
        D=self.backbone.num_features
        self.head=nn.Sequential(nn.Dropout(dropout),SEHead(D),
                                nn.LayerNorm(D),nn.Dropout(dropout/2),nn.Linear(D,n_classes))
    def forward(self,x): return self.head(self.backbone(x))

class FiLMLayer(nn.Module):
    def __init__(self,feat_dim,n_hint_dims,hidden=FILM_HIDDEN_DIM):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(n_hint_dims,hidden),nn.LayerNorm(hidden),
                               nn.GELU(),nn.Linear(hidden,feat_dim*2))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)
    def forward(self,feat,hint):
        gb=self.net(hint); g,b=gb.chunk(2,dim=-1)
        return (g+1.0)*feat+b

class FiLMConditionedSpecialist(nn.Module):
    def __init__(self,backbone_name,n_classes,n_hint_dims,dropout=0.25,pretrained=True):
        super().__init__()
        try:
            img_size=336 if "336" in backbone_name else (256 if "swinv2" in backbone_name else 224)
            self.needs_resize=(img_size!=224); self.img_size=img_size
            extra={}
            if any(x in backbone_name for x in ["eva02","swinv2"]): extra["img_size"]=img_size
            self.backbone=timm.create_model(backbone_name,pretrained=pretrained,num_classes=0,**extra)
        except Exception as e:
            print(f"  ⚠️  FiLM backbone {backbone_name} failed ({e}), fallback convnext_small")
            self.backbone=timm.create_model("convnext_small",pretrained=pretrained,num_classes=0)
            self.needs_resize=False; self.img_size=224
        D=self.backbone.num_features
        self.film=FiLMLayer(D,n_hint_dims)
        self.head=nn.Sequential(nn.Dropout(dropout),SEHead(D),
                                nn.LayerNorm(D),nn.Dropout(dropout/2),nn.Linear(D,n_classes))
    def forward(self,x,hint_logits=None):
        if self.needs_resize:
            x=F.interpolate(x,(self.img_size,self.img_size),mode="bilinear",align_corners=False)
        feat=self.backbone(x)
        if hint_logits is not None: feat=self.film(feat,hint_logits.float())
        return self.head(feat)

def _raw_fwd(model, inp):
    with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
        logits = model(inp)
    logits = torch.nan_to_num(logits.float(), nan=0., posinf=30., neginf=-30.)
    return torch.clamp(logits, -30., 30.)

def _safe_fwd(model, inp, hint=None, organ=None):
    with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
        if hint is not None and hasattr(model,"film"):
            logits=model(inp,hint_logits=hint)
        else:
            logits=model(inp)
    logits=torch.nan_to_num(logits.float(),nan=0.,posinf=30.,neginf=-30.)
    logits=torch.clamp(logits,-30.,30.)
    T=OPTIMIZED_TEMPERATURES.get(organ,1.0)
    if abs(T-1.0)>0.01:
        logits=logits/T
    return logits

_safe_forward = _safe_fwd

def smart_fwd(base_model, film_model, inp, hint, organ, class_names):
    threshold=FILM_HURT_THRESHOLD_ORGAN.get(organ,FILM_HURT_THRESHOLD_DEFAULT)
    base_f1=ORGAN_BASE_F1.get(organ,0.0)
    film_frozen=base_f1>=FILM_FREEZE_F1
    with torch.no_grad():
        base_logits=_safe_fwd(base_model,inp,None,organ)
        base_prbs=torch.softmax(base_logits,1)[0].cpu().numpy()
        base_pred=int(base_prbs.argmax()); base_conf=float(base_prbs[base_pred])
        film_disabled = (film_model is None)
        if film_disabled or film_frozen:
            return base_logits,base_prbs,base_pred,base_conf,False
        film_logits=_safe_fwd(film_model,inp,hint,organ)
        film_prbs=torch.softmax(film_logits,1)[0].cpu().numpy()
        film_pred=int(film_prbs.argmax()); film_conf=float(film_prbs[film_pred])
        if film_pred!=base_pred:
            base_ent=float(-np.sum(base_prbs*np.log(base_prbs+1e-8)))
            film_ent=float(-np.sum(film_prbs*np.log(film_prbs+1e-8)))
            if film_ent<base_ent:
                return film_logits,film_prbs,film_pred,film_conf,True
            return base_logits,base_prbs,base_pred,base_conf,False
        if film_conf-base_conf<threshold:
            return base_logits,base_prbs,base_pred,base_conf,False
        return film_logits,film_prbs,film_pred,film_conf,True


def _collect_paths(organ):
    paths,labels,class_names_set=[],[],set()
    if organ=="gastric":
        for folder in GASTRIC_ROOT.iterdir():
            if not folder.is_dir(): continue
            key=folder.name.upper()
            if key not in GASTRIC_LABEL_MAP: continue
            lbl=GASTRIC_LABEL_MAP[key]; class_names_set.add(lbl)
            for ext in ("*.jpg","*.jpeg","*.png","*.bmp"):
                for p in folder.glob(ext): paths.append(str(p)); labels.append(lbl)
        return paths,labels,sorted(class_names_set)
    if organ=="bladder":
        bladder_root=DATA_ROOT/"Bladder Cancer"
        if not bladder_root.exists():
            for alt in ["bladder","Bladder","bladder cancer"]:
                p=DATA_ROOT/alt
                if p.exists(): bladder_root=p; break
        if bladder_root.exists():
            for sf in bladder_root.iterdir():
                if not sf.is_dir(): continue
                fn=sf.name.lower()
                if "muscle" in fn and "invasive" in fn:
                    lbl="non_muscle_invasive" if "non" in fn else "muscle_invasive"
                    class_names_set.add(lbl)
                    for ext in ("*.jpg","*.jpeg","*.png","*.bmp"):
                        for p in sf.glob(ext): paths.append(str(p)); labels.append(lbl)
        return paths,labels,sorted(class_names_set)
    subdir_name=ORGAN_SUBDIR_MAP.get(organ)
    subdir_root=(DATA_ROOT/subdir_name) if subdir_name else None
    subdir_children={}
    if subdir_root and subdir_root.exists():
        for child in subdir_root.iterdir():
            if child.is_dir(): subdir_children[child.name.lower()]=child
    for folder_name,(org,lbl) in LABEL_MAP.items():
        if org!=organ: continue
        class_names_set.add(lbl)
        candidates=[DATA_ROOT/folder_name]
        if subdir_root:
            candidates.append(subdir_root/folder_name)
            if folder_name.lower() in subdir_children:
                candidates.append(subdir_children[folder_name.lower()])
            bare=folder_name.replace(f"{organ}_","").replace(f"{organ.split('_')[0]}_","")
            if bare.lower() in subdir_children: candidates.append(subdir_children[bare.lower()])
        found=False
        for candidate in candidates:
            if not candidate.exists(): continue
            cb=len(paths)
            for ext in ("*.jpg","*.jpeg","*.png","*.bmp"):
                for p in candidate.glob(ext): paths.append(str(p)); labels.append(lbl)
            if len(paths)>cb: found=True; break
        if not found and subdir_root and subdir_root.exists():
            for child in subdir_root.iterdir():
                if not child.is_dir(): continue
                if (lbl.replace("_"," ").lower() in child.name.lower() or
                        folder_name.lower() in child.name.lower()):
                    cb=len(paths)
                    for ext in ("*.jpg","*.jpeg","*.png","*.bmp"):
                        for p in child.glob(ext): paths.append(str(p)); labels.append(lbl)
                    if len(paths)>cb: break
    return paths,labels,sorted(class_names_set)


# ════════════════════════════════════════════════════════════════════════════════
# === UPGRADE v18 === build_loaders_v18
#
# WHY: New build_loaders_v18 uses:
#   1. AUGMENT_CFG_V18 to select aug strength per organ
#   2. compute_dynamic_class_weights() for smarter sampling
#   3. SuperHeavyAugDataset with per-class rare repeat
#   The original build_loaders() is fully preserved for backward compat.
# ════════════════════════════════════════════════════════════════════════════════

def build_loaders_v18(organ, batch_size=64, eval_results_ref=None):
    """
    === UPGRADE v18 ===
    Upgraded loader with dynamic augmentation selection and smart sampling.

    Backward compatible: if organ not in AUGMENT_CFG_V18, falls back to
    original build_loaders() behavior.

    eval_results_ref: optional dict from Cell 7 for difficulty-aware sampling.
    """
    paths, str_labels, class_names = _collect_paths(organ)
    if not paths:
        raise ValueError(f"No images found for organ: {organ}")

    l2i = {c:i for i,c in enumerate(class_names)}
    int_labels = [l2i[l] for l in str_labels]
    tr_p, te_p, tr_l, te_l = train_test_split(
        paths, int_labels, test_size=0.15, stratify=int_labels, random_state=SEED)
    tr_p, vl_p, tr_l, vl_l = train_test_split(
        tr_p, tr_l, test_size=0.15, stratify=tr_l, random_state=SEED)

    # === Determine aug mode ===
    aug_mode = AUGMENT_CFG_V18.get(organ, "standard")
    repeat   = AUGMENT_REPEAT_V18.get(aug_mode, 1)

    # === Identify rare class indices for extra repeat ===
    counts = np.bincount(tr_l, minlength=len(class_names))
    median_count = float(np.median(counts[counts > 0]))
    # Classes with count < 40% of median are "rare"
    rare_class_indices = set(
        ci for ci, cnt in enumerate(counts)
        if cnt > 0 and cnt < 0.40 * median_count
    )
    if rare_class_indices:
        rare_names = [class_names[ci] for ci in rare_class_indices]
        print(f"  [{organ}] v18 rare classes: {rare_names}")

    # === Build datasets ===
    if aug_mode == "superheavy":
        tr_ds = SuperHeavyAugDataset(
            tr_p, tr_l, SUPER_HEAVY_AUG,
            repeat=repeat,
            rare_classes=rare_class_indices,
            rare_repeat=2)
        vl_ds = HeavyAugDataset(vl_p, vl_l, SUPER_HEAVY_VAL_AUG, repeat=1)
        te_ds = HeavyAugDataset(te_p, te_l, SUPER_HEAVY_VAL_AUG, repeat=1)
    elif aug_mode == "heavy":
        tr_ds = HeavyAugDataset(tr_p, tr_l, HEAVY_AUG, repeat=repeat)
        vl_ds = HeavyAugDataset(vl_p, vl_l, HEAVY_VAL_AUG, repeat=1)
        te_ds = HeavyAugDataset(te_p, te_l, HEAVY_VAL_AUG, repeat=1)
    else:
        tr_ds = StandardDataset(tr_p, tr_l, train_tfm)
        vl_ds = StandardDataset(vl_p, vl_l, val_tfm)
        te_ds = StandardDataset(te_p, te_l, val_tfm)

    # === Dynamic class weights ===
    # For superheavy/heavy: use all expanded labels from dataset
    if aug_mode == "superheavy" and isinstance(tr_ds, SuperHeavyAugDataset):
        expanded_labels = [tr_ds.labels[i] for i in tr_ds._idx]
    elif aug_mode == "heavy" and isinstance(tr_ds, HeavyAugDataset):
        expanded_labels = list(tr_l) * repeat
    else:
        expanded_labels = list(tr_l)

    sample_weights = compute_dynamic_class_weights(
        expanded_labels, class_names, organ,
        eval_results_ref=eval_results_ref,
        rare_boost_factor=2.5,
        difficulty_factor=1.5,
    )

    sampler = WeightedRandomSampler(
        sample_weights,
        num_samples=len(sample_weights),
        replacement=True)

    tr_loader = DataLoader(tr_ds, batch_size=batch_size, sampler=sampler,
                           num_workers=NUM_WORKERS, timeout=_dl_timeout(), pin_memory=True,
                           drop_last=True)
    vl_loader = DataLoader(vl_ds, batch_size=batch_size, shuffle=False,
                           num_workers=NUM_WORKERS, timeout=_dl_timeout(), pin_memory=True)
    te_loader = DataLoader(te_ds, batch_size=batch_size, shuffle=False,
                           num_workers=NUM_WORKERS, timeout=_dl_timeout(), pin_memory=True)

    print(f"  [{organ}] v18 aug={aug_mode} repeat={repeat} "
          f"train={len(tr_ds)} val={len(vl_ds)} test={len(te_ds)} "
          f"classes={class_names}")
    return tr_loader, vl_loader, te_loader, class_names


# ── Original build_loaders (preserved, not modified) ─────────────────────────
def build_loaders(organ, batch_size=64, heavy=False, repeat=4):
    paths,str_labels,class_names=_collect_paths(organ)
    if not paths: raise ValueError(f"No images found for organ: {organ}")
    l2i={c:i for i,c in enumerate(class_names)}
    int_labels=[l2i[l] for l in str_labels]
    tr_p,te_p,tr_l,te_l=train_test_split(paths,int_labels,test_size=0.15,stratify=int_labels,random_state=SEED)
    tr_p,vl_p,tr_l,vl_l=train_test_split(tr_p,tr_l,test_size=0.15,stratify=tr_l,random_state=SEED)
    if heavy:
        tr_ds=HeavyAugDataset(tr_p,tr_l,HEAVY_AUG,repeat=repeat)
        vl_ds=HeavyAugDataset(vl_p,vl_l,HEAVY_VAL_AUG,repeat=1)
        te_ds=HeavyAugDataset(te_p,te_l,HEAVY_VAL_AUG,repeat=1)
    else:
        tr_ds=StandardDataset(tr_p,tr_l,train_tfm)
        vl_ds=StandardDataset(vl_p,vl_l,val_tfm)
        te_ds=StandardDataset(te_p,te_l,val_tfm)
    cc=np.bincount(tr_l,minlength=len(class_names))
    weights=[1./(cc[l]+1) for l in tr_l]
    if heavy: weights=list(np.array(weights).tolist())*repeat
    sampler=WeightedRandomSampler(weights,num_samples=len(weights),replacement=True)
    tr_loader=DataLoader(tr_ds,batch_size=batch_size,sampler=sampler,
                         num_workers=NUM_WORKERS, timeout=_dl_timeout(),pin_memory=True,drop_last=True)
    vl_loader=DataLoader(vl_ds,batch_size=batch_size,shuffle=False,
                         num_workers=NUM_WORKERS, timeout=_dl_timeout(),pin_memory=True)
    te_loader=DataLoader(te_ds,batch_size=batch_size,shuffle=False,
                         num_workers=NUM_WORKERS, timeout=_dl_timeout(),pin_memory=True)
    print(f"  [{organ}] train={len(tr_ds)} val={len(vl_ds)} test={len(te_ds)} classes={class_names}")
    return tr_loader,vl_loader,te_loader,class_names


# === UPGRADE v18 === Smoke tests
print("\n  === UPGRADE v18 Cell 2 === Smoke tests...")
_test_img = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
try:
    _aug_res = SUPER_HEAVY_AUG(image=_test_img)
    assert _aug_res["image"].shape == (3, 224, 224), "Shape mismatch"
    print("  ✅ SuperHeavyAug smoke test passed")
except Exception as _e:
    print(f"  ❌ SuperHeavyAug smoke test FAILED: {_e}")

try:
    _imgs_t  = torch.randn(4, 3, 224, 224)
    _lbls_t  = torch.randint(0, 3, (4,))
    _mixed, _ya, _yb, _lam = mixup_or_cutmix(_imgs_t, _lbls_t)
    assert _mixed.shape == (4, 3, 224, 224)
    assert 0. <= _lam <= 1.
    print(f"  ✅ MixUp/CutMix smoke test passed  lam={_lam:.3f}")
except Exception as _e:
    print(f"  ❌ MixUp/CutMix smoke test FAILED: {_e}")

try:
    _w = compute_dynamic_class_weights(
        [0,0,1,1,2,2,2,2,2], ["a","b","c"], "skin")
    assert len(_w) == 9
    print(f"  ✅ Dynamic weights smoke test passed  sample_w={[round(x,2) for x in _w[:5]]}")
except Exception as _e:
    print(f"  ❌ Dynamic weights smoke test FAILED: {_e}")

print("\n✅ Cell 2 v18 complete")
print("   New: SuperHeavyAug | MixUp/CutMix | DynamicSampler | build_loaders_v18")
print("   Preserved: val_tfm | train_tfm | HEAVY_AUG | build_loaders | all models")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 2b — Adaptive Per-Epoch Weighted Sampler
#
# PROBLEM: WeightedRandomSampler with replacement=True + fixed weights means
#   the effective class distribution drifts and never adapts to what the model
#   is currently struggling with. SuperHeavyAug repeat=6 means some images
#   are seen 12× per epoch vs once — compounding bias without any feedback.
#
# FIX:
#   AdaptiveSampler   — subclass of WeightedRandomSampler that accepts
#                       per-class accuracy at set_epoch() and re-weights
#                       dynamically: w_c = (1/count)^freq × (1/acc+0.05)^diff
#   build_loaders_adaptive() — drop-in replacement for build_loaders_v18()
#                              used by Cell 5c retrain and Cell 7 eval
#   Context manager   _shap_speed_cap() — safe SHAP cap that always restores
#                     even on exception (fixes Cell 11 state-corruption risk)
# ════════════════════════════════════════════════════════════════════════════════

import numpy as np
import torch
from torch.utils.data import WeightedRandomSampler, DataLoader
from contextlib import contextmanager

print("═" * 68)
print("  CELL 2b — Adaptive Sampler + Safe SHAP Cap")
print("═" * 68)


# ════════════════════════════════════════════════════════════════════════════════
# PART 1 — AdaptiveSampler
# ════════════════════════════════════════════════════════════════════════════════

class AdaptiveSampler(WeightedRandomSampler):
    """
    WeightedRandomSampler that re-weights per epoch based on running
    per-class accuracy from the previous epoch.

    Usage:
        sampler = AdaptiveSampler(
            initial_weights, num_samples=len(weights),
            labels=tr_labels, class_names=class_names, organ=organ)
        # At end of each epoch, after computing val metrics:
        sampler.set_epoch(epoch, per_class_acc={'benign': 0.98, 'malignant': 0.61})
        # DataLoader uses new weights next epoch automatically
    """
    def __init__(self, weights, num_samples, labels,
                 class_names, organ,
                 freq_exp=2.5, diff_exp=1.5):
        super().__init__(weights, num_samples=num_samples,
                         replacement=True)
        self._labels       = np.array(labels)
        self._class_names  = class_names
        self._organ        = organ
        self._freq_exp     = freq_exp
        self._diff_exp     = diff_exp
        self._epoch        = 0
        self._counts       = np.bincount(self._labels,
                                         minlength=len(class_names)
                                         ).astype(np.float32)
        self._counts       = np.maximum(self._counts, 1.)
        # Hard-coded boosts for persistently hard classes (from audit)
        self._HARD_BOOST = {
            ("skin",     "seborrheic_keratosis"):    5.0,
            ("skin",     "squamous_cell_carcinoma"): 3.0,
            ("skin",     "actinic_keratosis"):        2.0,
            ("cervical", "dyk"):                      4.0,
            ("cervical", "koc"):                      3.5,
            ("gastric",  "adi"):                      2.5,
            ("gastric",  "mus"):                      2.5,
            ("gastric",  "deb"):                      2.0,
            ("lymphoma", "mcl"):                      2.5,
            ("ovarian",  "low_grade_serous"):         3.0,
        }

    def set_epoch(self, epoch: int, per_class_acc: dict = None):
        """
        Call at the end of each training epoch.
        per_class_acc: dict {class_name: float accuracy 0-1}
        If None, uses frequency-only weighting.
        """
        self._epoch = epoch
        n_cls = len(self._class_names)

        # Frequency weight
        freq_w = (1. / self._counts) ** self._freq_exp

        # Difficulty weight from last epoch's per-class accuracy
        diff_w = np.ones(n_cls, dtype=np.float32)
        if per_class_acc is not None:
            for ci, cname in enumerate(self._class_names):
                acc = per_class_acc.get(cname, None)
                if acc is not None:
                    diff_w[ci] = (1. / (float(acc) + 0.05)) ** self._diff_exp

        # Hard boost
        for ci, cname in enumerate(self._class_names):
            boost = self._HARD_BOOST.get((self._organ, cname), 1.0)
            diff_w[ci] *= boost

        # Combined, normalised
        class_w = freq_w * diff_w
        class_w = class_w / class_w.sum() * n_cls

        # Map to per-sample weights and update parent
        sample_w = np.array([float(class_w[lbl]) for lbl in self._labels],
                             dtype=np.float64)
        # WeightedRandomSampler stores weights as tensor
        self.weights = torch.as_tensor(sample_w, dtype=torch.double)
        self.num_samples = len(sample_w)

        if epoch % 5 == 0:
            print(f"    [{self._organ}] epoch={epoch} "
                  f"class_weights={[round(float(x),2) for x in class_w]}")


def build_loaders_adaptive(organ, batch_size=64, eval_results_ref=None):
    """
    Drop-in replacement for build_loaders_v18() that uses AdaptiveSampler.
    Returns (tr_loader, vl_loader, te_loader, class_names, sampler)
    — sampler is exposed so trainers can call sampler.set_epoch() each epoch.
    """
    paths, str_labels, class_names = _collect_paths(organ)
    if not paths:
        raise ValueError(f"No images found for organ: {organ}")

    l2i = {c: i for i, c in enumerate(class_names)}
    int_labels = [l2i[l] for l in str_labels]

    tr_p, te_p, tr_l, te_l = train_test_split(
        paths, int_labels, test_size=0.15,
        stratify=int_labels, random_state=SEED)
    tr_p, vl_p, tr_l, vl_l = train_test_split(
        tr_p, tr_l, test_size=0.15,
        stratify=tr_l, random_state=SEED)

    aug_mode = AUGMENT_CFG_V18.get(organ, "standard")
    repeat   = AUGMENT_REPEAT_V18.get(aug_mode, 1)

    counts      = np.bincount(tr_l, minlength=len(class_names))
    median_cnt  = float(np.median(counts[counts > 0]))
    rare_cls    = set(ci for ci, cnt in enumerate(counts)
                      if cnt > 0 and cnt < 0.40 * median_cnt)

    if aug_mode == "superheavy":
        tr_ds = SuperHeavyAugDataset(tr_p, tr_l, SUPER_HEAVY_AUG,
                                     repeat=repeat,
                                     rare_classes=rare_cls,
                                     rare_repeat=2)
        vl_ds = HeavyAugDataset(vl_p, vl_l, SUPER_HEAVY_VAL_AUG, repeat=1)
        te_ds = HeavyAugDataset(te_p, te_l, SUPER_HEAVY_VAL_AUG, repeat=1)
    elif aug_mode == "heavy":
        tr_ds = HeavyAugDataset(tr_p, tr_l, HEAVY_AUG, repeat=repeat)
        vl_ds = HeavyAugDataset(vl_p, vl_l, HEAVY_VAL_AUG, repeat=1)
        te_ds = HeavyAugDataset(te_p, te_l, HEAVY_VAL_AUG, repeat=1)
    else:
        tr_ds = StandardDataset(tr_p, tr_l, train_tfm)
        vl_ds = StandardDataset(vl_p, vl_l, val_tfm)
        te_ds = StandardDataset(te_p, te_l, val_tfm)

    if aug_mode == "superheavy" and hasattr(tr_ds, '_idx'):
        expanded = [tr_ds.labels[i] for i in tr_ds._idx]
    elif aug_mode == "heavy":
        expanded = list(tr_l) * repeat
    else:
        expanded = list(tr_l)

    # Initial frequency-only weights
    init_w = compute_dynamic_class_weights(
        expanded, class_names, organ,
        eval_results_ref=eval_results_ref,
        rare_boost_factor=2.5, difficulty_factor=1.5)

    sampler = AdaptiveSampler(
        init_w, num_samples=len(init_w),
        labels=expanded,
        class_names=class_names, organ=organ)

    tr_loader = DataLoader(tr_ds, batch_size=batch_size, sampler=sampler,
                           num_workers=NUM_WORKERS, timeout=_dl_timeout(), pin_memory=True,
                           drop_last=True)
    vl_loader = DataLoader(vl_ds, batch_size=batch_size, shuffle=False,
                           num_workers=NUM_WORKERS, timeout=_dl_timeout(), pin_memory=True)
    te_loader = DataLoader(te_ds, batch_size=batch_size, shuffle=False,
                           num_workers=NUM_WORKERS, timeout=_dl_timeout(), pin_memory=True)

    print(f"  [{organ}] adaptive aug={aug_mode} repeat={repeat} "
          f"train={len(tr_ds)} val={len(vl_ds)} test={len(te_ds)} "
          f"classes={class_names}")
    return tr_loader, vl_loader, te_loader, class_names, sampler


# ════════════════════════════════════════════════════════════════════════════════
# PART 2 — Context manager for safe SHAP cap (fixes Cell 11 state corruption)
# ════════════════════════════════════════════════════════════════════════════════

@contextmanager
def _shap_speed_cap(cnn_cap=1024, vit_cap=400):
    """
    Temporarily caps SHAP sample counts and ALWAYS restores on exit,
    even if an exception is raised mid-panel.

    Usage:
        with _shap_speed_cap(1024, 400):
            make_xai_panel_v19(...)
        # SHAP_NSAMPLES is guaranteed restored here
    """
    global SHAP_NSAMPLES, SHAP_NSAMPLES_VIT
    orig_cnn = SHAP_NSAMPLES     if 'SHAP_NSAMPLES'     in dir() else 2352
    orig_vit = SHAP_NSAMPLES_VIT if 'SHAP_NSAMPLES_VIT' in dir() else 600
    try:
        SHAP_NSAMPLES     = min(orig_cnn, cnn_cap)
        SHAP_NSAMPLES_VIT = min(orig_vit, vit_cap)
        yield
    finally:
        SHAP_NSAMPLES     = orig_cnn
        SHAP_NSAMPLES_VIT = orig_vit
        print(f"  _shap_speed_cap: restored SHAP_NSAMPLES={orig_cnn} "
              f"VIT={orig_vit}")


# ════════════════════════════════════════════════════════════════════════════════
# PART 3 — Smoke tests
# ════════════════════════════════════════════════════════════════════════════════

print("\n  Smoke tests...")
try:
    _w_test = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
    _samp   = AdaptiveSampler(
        _w_test, num_samples=6,
        labels=[0,0,1,1,2,2],
        class_names=['a','b','c'],
        organ='skin')
    _samp.set_epoch(1, per_class_acc={'a': 0.95, 'b': 0.50, 'c': 0.30})
    assert len(_samp.weights) == 6
    print("  ✅ AdaptiveSampler smoke test passed")
except Exception as _e:
    print(f"  ❌ AdaptiveSampler FAILED: {_e}")

try:
    _SHAP_NSAMPLES_ORIG = SHAP_NSAMPLES if 'SHAP_NSAMPLES' in dir() else 2352
    with _shap_speed_cap(512):
        assert SHAP_NSAMPLES == min(_SHAP_NSAMPLES_ORIG, 512)
    assert SHAP_NSAMPLES == _SHAP_NSAMPLES_ORIG
    print("  ✅ _shap_speed_cap smoke test passed (restore verified)")
except Exception as _e:
    print(f"  ❌ _shap_speed_cap FAILED: {_e}")

print("\n✅ Cell 2b complete")
print("   AdaptiveSampler     — re-weights each epoch from per-class accuracy")
print("   build_loaders_adaptive() — exposes sampler for trainer feedback")
print("   _shap_speed_cap()   — context manager, always restores SHAP state")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 3 — Load all 16 specialist checkpoints
# v18 — self-describing checkpoint loading:
#   FIX-1: reads backbone_name + dropout FROM checkpoint (written by Cell 5c)
#           instead of always using BACKBONE_CFG → fixes size-mismatch errors
#           for pancreatic (vit→convnext_tiny), cervical (efficientvit→convnext_small),
#           kidney (efficientnet_b3→convnext_small) after Cell 5c retraining
#   FIX-2: head-rebuild guard now checks actual feature dim D from the loaded
#           backbone, not from a BACKBONE_CFG that may be stale
#   FIX-3: BACKBONE_CFG fallback preserved → organs NOT retrained by Cell 5c
#           (blood, colon, brain_2d, etc.) load exactly as before — zero regression
#   FIX-4: per-organ load summary now shows which backbone was actually used
#   FIX-5: RETRAIN_RESULTS hot-swap — if Cell 5c ran in this session,
#           SPEC_CKPTS is updated to point at retrained checkpoints automatically
# ════════════════════════════════════════════════════════════════════════════════

SPECIALISTS  = {}
LOAD_REPORT  = {}

# ── FIX-5: hot-swap checkpoint paths if Cell 5c already ran ──────────────────
# RETRAIN_RESULTS is populated by Cell 5c; if it exists, update SPEC_CKPTS so
# retrained organs use the new weights without manually editing path dicts.
if "RETRAIN_RESULTS" in dir():
    for _organ, _r in RETRAIN_RESULTS.items():
        _new_path = _r.get("ckpt_path")
        if _new_path is not None and Path(_new_path).exists():
            SPEC_CKPTS[_organ] = Path(_new_path)
            print(f"  [hot-swap] {_organ} → {Path(_new_path).name}")


def _warmup_model(model):
    """Forward pass on a zero tensor to catch NaN/Inf before deployment."""
    model.eval()
    dummy = torch.zeros(1, 3, 224, 224).to(DEVICE)
    with torch.no_grad():
        out = model(dummy)
    out = torch.nan_to_num(out.float(), nan=0., posinf=30., neginf=-30.)
    return bool(torch.isfinite(out).all().item())


def _resolve_class_names(organ, ckpt):
    """
    Return the canonical class-name list for an organ.
    Priority: (1) ckpt["class_names"], (2) organ-specific override,
              (3) empty list (caller must handle).
    """
    class_names = ckpt.get("class_names", [])

    if organ == "cervical":
        gt = sorted(["dyk", "koc", "mep", "pab", "sfi"])
        if "class_to_idx" in ckpt:
            itoc = {v: k for k, v in ckpt["class_to_idx"].items()}
            class_names = [itoc[i] for i in range(len(itoc))]
        elif sorted(class_names) != gt:
            class_names = gt

    if organ == "gastric" and len(class_names) != 8:
        class_names = sorted(GASTRIC_LABEL_MAP.values())

    return class_names


def _resolve_backbone(organ, ckpt):
    """
    FIX-1 + FIX-3: return (backbone_name, dropout) by reading from the
    checkpoint first, then falling back to BACKBONE_CFG.
    Checkpoints saved by Cell 5c store "backbone_name" and "dropout".
    Checkpoints from earlier cells do not — BACKBONE_CFG is the fallback.
    """
    cfg_bb, cfg_do, _ = BACKBONE_CFG.get(organ, ("convnext_tiny", 0.25, False))
    bb_name = ckpt.get("backbone_name", cfg_bb)   # Cell 5c key
    dropout  = ckpt.get("dropout",       cfg_do)   # Cell 5c key
    source   = "ckpt" if "backbone_name" in ckpt else "cfg"
    return bb_name, dropout, source


def _load_specialist(organ, ckpt_path):
    if not ckpt_path.exists():
        print(f"  ⚠️  [{organ:<12}] NOT FOUND: {ckpt_path}")
        LOAD_REPORT[organ] = {"status": "NOT_FOUND"}
        return

    ckpt = torch.load(str(ckpt_path), map_location=DEVICE, weights_only=False)

    # ── Resolve class names ───────────────────────────────────────────────────
    class_names = _resolve_class_names(organ, ckpt)
    val_f1      = ckpt.get("val_f1", 0.0)
    ORGAN_BASE_F1[organ] = val_f1

    # ── FIX-1: resolve backbone from checkpoint, not from BACKBONE_CFG alone ─
    bb_name, dropout, bb_source = _resolve_backbone(organ, ckpt)

    # ── Build model with correct backbone ────────────────────────────────────
    model = SpecialistModel(
        bb_name, len(class_names), dropout, pretrained=False
    ).to(DEVICE)

    # ── Load state dict ───────────────────────────────────────────────────────
    state = ckpt.get("model_state_dict", ckpt)
    # Strip DDP / DataParallel prefix if present
    state = {k.replace("module.", ""): v for k, v in state.items()}

    miss, unex = model.load_state_dict(state, strict=False)

    # ── FIX-2: head rebuild uses actual D from loaded backbone ────────────────
    # Only rebuild if there are genuinely unexpected keys (>8 is a heuristic
    # that filters normal minor mismatches from a full architecture mismatch).
    if len(unex) > 8:
        print(f"  ⚠️  [{organ}] {len(unex)} unexpected keys — rebuilding head "
              f"(backbone keys loaded separately)")
        backbone_state = {
            k: v for k, v in state.items() if k.startswith("backbone.")
        }
        model.backbone.load_state_dict(backbone_state, strict=False)
        D = model.backbone.num_features          # FIX-2: read from actual model
        model.head = nn.Sequential(
            nn.Dropout(dropout),
            SEHead(D),
            nn.LayerNorm(D),
            nn.Dropout(dropout / 2),
            nn.Linear(D, len(class_names)),
        ).to(DEVICE)
        # Warn if head was rebuilt: weights are random — inference will be poor
        print(f"  ⚠️  [{organ}] head rebuilt with random weights "
              f"(D={D}, n_cls={len(class_names)}) — retraining recommended")

    # ── Warmup ────────────────────────────────────────────────────────────────
    valid = _warmup_model(model)
    model.eval()

    # ── FiLM eligibility flag ─────────────────────────────────────────────────
    if val_f1 >= FILM_FREEZE_F1:
        freeze_flag = "🔒 FROZEN"
    else:
        freeze_flag = "🎬 FiLM"

    # ── Register ──────────────────────────────────────────────────────────────
    SPECIALISTS[organ] = {
        "model":       model,
        "class_names": class_names,
        "film_model":  None,
        "film_status": "PENDING",
    }
    LOAD_REPORT[organ] = {
        "status":       "OK",
        "val_f1":       val_f1,
        "class_names":  class_names,
        "warmup_valid": valid,
        "backbone":     bb_name,       # FIX-4: log actual backbone used
        "bb_source":    bb_source,     # "ckpt" or "cfg"
        "unexpected_keys": len(unex),
        "missing_keys":    len(miss),
    }

    warmup_tag = "OK" if valid else "⚠️ NaN!"
    print(
        f"  ✅ [{organ:<12}] "
        f"val_f1={val_f1:.4f} | "
        f"bb={bb_name}({bb_source}) | "
        f"cls={class_names} | "
        f"warmup={warmup_tag} | "
        f"{freeze_flag}"
    )


# ── Load all 16 ──────────────────────────────────────────────────────────────
print("═" * 72)
print("  CELL 3 — Loading all 16 specialists")
print("═" * 72)

for organ, ckpt_path in SPEC_CKPTS.items():
    _load_specialist(organ, ckpt_path)

# ── Summary ───────────────────────────────────────────────────────────────────
ok        = sum(1 for v in LOAD_REPORT.values() if v["status"] == "OK")
not_found = sum(1 for v in LOAD_REPORT.values() if v["status"] == "NOT_FOUND")
nan_warn  = [o for o, v in LOAD_REPORT.items()
             if v.get("warmup_valid") is False]
head_reb  = [o for o, v in LOAD_REPORT.items()
             if v.get("unexpected_keys", 0) > 8]

print(f"\n{'─'*72}")
print(f"  {'Organ':<14} {'Backbone':<32} {'Src':<5} {'F1':>6}  {'Status'}")
print(f"{'─'*72}")
for organ in SPEC_CKPTS:
    r = LOAD_REPORT.get(organ, {})
    if r.get("status") == "NOT_FOUND":
        print(f"  {organ:<14} {'—':<32} {'—':<5} {'—':>6}  NOT FOUND")
    else:
        bb  = r.get("backbone",  "?")
        src = r.get("bb_source", "?")
        f1  = r.get("val_f1",    0.)
        wup = "OK" if r.get("warmup_valid") else "⚠️ NaN"
        print(f"  {organ:<14} {bb:<32} {src:<5} {f1:>6.4f}  {wup}")

print(f"\n  Loaded:    {ok}/{len(SPEC_CKPTS)}")
if not_found:
    missing = [o for o, v in LOAD_REPORT.items() if v["status"] == "NOT_FOUND"]
    print(f"  Not found: {not_found}  → {missing}")
if nan_warn:
    print(f"  ⚠️  NaN warmup:      {nan_warn}")
if head_reb:
    print(f"  ⚠️  Head rebuilt:    {head_reb}  (random head weights — retrain!)")

print(f"\n✅ Cell 3 complete — {ok}/{len(SPEC_CKPTS)} specialists loaded")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 4-NEW — HoVer-Net Inference Engine  (v18-FIX)
#
# FIXES vs Cell 4-NEW original:
#   FIX-NEO-1  Confidence gate threshold raised 0.55 → 0.72 (random images
#              had all 45 nuclei labelled neoplastic due to softmax mode collapse)
#   FIX-NEO-2  Confidence computation changed from pixel-mean to max-pooled
#              spatial max within each nucleus mask — more robust to small
#              high-confidence regions within noisy nuclei
#   FIX-NEO-3  seg_net_used now set based on gated n_neoplastic > 0 with
#              organ-aware min threshold; fixes 0% seg_net_used bug
#   FIX-WS-1   Watershed gradient threshold lowered 0.40 → 0.30, reduces
#              over-segmentation; PQ DQ improvement expected ~0.39→0.48
#   FIX-WS-2   Min nucleus area increased 20 → 40px to filter watershed noise
#   FIX-PQ-1   evaluate_hovernet_pq now uses n_samples=150 for better estimate
#   All original FIX-1/2/3/4/5 from Cell 4-NEW preserved
# ════════════════════════════════════════════════════════════════════════════════

import math, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage as ndi
from torchvision import transforms
import cv2
from pathlib import Path

print("\n" + "═" * 68)
print("  CELL 4-NEW — HoVer-Net Inference Engine  (v18-FIX)")
print("═" * 68)

# ── FIX-1: correct paths ──────────────────────────────────────────────────────
DATASET_BASE = Path(
    "/kaggle/input/datasets/andrewmvd/"
    "cancer-inst-segmentation-and-classification")
PANNUKE_ROOT = DATASET_BASE
HOVERNET_CKPT = SPEC_DIR / "hovernet_pannuke.pth"

# ── FIX-2: re-declare constants ───────────────────────────────────────────────
N_NUCLEAR_CLASSES = 5
NUCLEAR_CLASSES   = {
    0: "neoplastic",
    1: "inflammatory",
    2: "connective",
    3: "dead",
    4: "epithelial",
}

# FIX-NEO-1: raised from 0.55 → 0.72
# WHY: With an undertrained nuclear classifier, softmax for neoplastic
# saturates near 0.6-0.65 even for non-neoplastic cells on random images.
# Raising to 0.72 requires a more decisive prediction.
HOVERNET_NEO_CONF_THRESHOLD_V2 = 0.72

# FIX-WS-1: tighter gradient threshold
# WHY: DQ=0.397 means too many nuclei are split or missed.
# Lowering from 0.40 → 0.30 reduces over-segmentation of single nuclei.
_WS_GRADIENT_THRESHOLD = 0.30

# FIX-WS-2: larger min area
# WHY: Watershed fragments produce many tiny (<20px) false positive instances
# that tank DQ. Raising to 40px removes most noise fragments.
_MIN_NUCLEUS_AREA = 40

# ── FIX-2: re-declare HoVerNet architecture ────────────────────────────────────
if "HoVerNet" not in dir():
    class _PreActBottleneck(nn.Module):
        def __init__(self, c_in, c_mid, stride=1):
            super().__init__()
            c_out = c_mid * 4
            self.bn1   = nn.BatchNorm2d(c_in)
            self.conv1 = nn.Conv2d(c_in,  c_mid, 1, bias=False)
            self.bn2   = nn.BatchNorm2d(c_mid)
            self.conv2 = nn.Conv2d(c_mid, c_mid, 3, stride=stride,
                                   padding=1, bias=False)
            self.bn3   = nn.BatchNorm2d(c_mid)
            self.conv3 = nn.Conv2d(c_mid, c_out, 1, bias=False)
            self.skip  = (nn.Conv2d(c_in, c_out, 1, stride=stride, bias=False)
                          if stride != 1 or c_in != c_out else nn.Identity())
        def forward(self, x):
            h = F.relu(self.bn1(x))
            return (self.conv3(F.relu(self.bn3(
                    self.conv2(F.relu(self.bn2(self.conv1(h)))))))
                    + self.skip(h))

    def _make_layer_4(c_in, c_mid, n, stride=1):
        layers = [_PreActBottleneck(c_in, c_mid, stride)]
        for _ in range(1, n):
            layers.append(_PreActBottleneck(c_mid * 4, c_mid))
        return nn.Sequential(*layers)

    class _DecoderBranch4(nn.Module):
        def __init__(self, enc_ch, out_ch, hidden=256):
            super().__init__()
            self.lat = nn.ModuleList([
                nn.Sequential(nn.Conv2d(c, hidden, 1, bias=False),
                              nn.BatchNorm2d(hidden), nn.ReLU())
                for c in enc_ch])
            self.up = nn.Sequential(
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
                nn.Conv2d(hidden, hidden, 3, padding=1, bias=False),
                nn.BatchNorm2d(hidden), nn.ReLU(),
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
                nn.Conv2d(hidden, hidden, 3, padding=1, bias=False),
                nn.BatchNorm2d(hidden), nn.ReLU())
            self.head = nn.Conv2d(hidden, out_ch, 1)
        def forward(self, feats):
            x = self.lat[0](feats[0])
            for i in range(1, len(feats)):
                x = F.interpolate(x, feats[i].shape[2:],
                                  mode='bilinear', align_corners=False)
                x = x + self.lat[i](feats[i])
            return self.head(self.up(x))

    class HoVerNet(nn.Module):
        def __init__(self, n_classes=N_NUCLEAR_CLASSES, pretrained=False):
            super().__init__()
            self.stem = nn.Sequential(
                nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
                nn.BatchNorm2d(64), nn.ReLU(),
                nn.MaxPool2d(3, stride=2, padding=1))
            self.layer1 = _make_layer_4( 64,  64, 3, stride=1)
            self.layer2 = _make_layer_4(256, 128, 4, stride=2)
            self.layer3 = _make_layer_4(512, 256, 6, stride=2)
            self.layer4 = _make_layer_4(1024,512, 3, stride=2)
            enc_ch = [2048, 1024, 512, 256]
            self.np_branch = _DecoderBranch4(enc_ch, out_ch=1)
            self.hv_branch = _DecoderBranch4(enc_ch, out_ch=2)
            self.nc_branch = _DecoderBranch4(enc_ch, out_ch=n_classes + 1)

        def _encode(self, x):
            x = self.stem(x)
            e1 = self.layer1(x); e2 = self.layer2(e1)
            e3 = self.layer3(e2); e4 = self.layer4(e3)
            return [e4, e3, e2, e1]

        def forward(self, x):
            H, W  = x.shape[2], x.shape[3]
            feats = self._encode(x)
            np_logit = F.interpolate(self.np_branch(feats), (H,W),
                                     mode='bilinear', align_corners=False)
            hv_logit = F.interpolate(self.hv_branch(feats), (H,W),
                                     mode='bilinear', align_corners=False)
            nc_logit = F.interpolate(self.nc_branch(feats), (H,W),
                                     mode='bilinear', align_corners=False)
            return {"np_logit": np_logit,
                    "hv_logit": hv_logit,
                    "nc_logit": nc_logit}

        @torch.no_grad()
        def predict(self, x):
            out = self.forward(x)
            return {"np_map": torch.sigmoid(out["np_logit"]),
                    "hv_map": torch.tanh(  out["hv_logit"]),
                    "nc_map": out["nc_logit"]}

    print("  ✅ HoVerNet class defined")
else:
    print("  ✅ HoVerNet already in scope")

# ── Load checkpoint ────────────────────────────────────────────────────────────
if "HOVERNET_MODEL" not in dir() or HOVERNET_MODEL is None:
    if HOVERNET_CKPT.exists():
        print(f"  📦 Loading checkpoint: {HOVERNET_CKPT}")
        HOVERNET_MODEL = HoVerNet(n_classes=N_NUCLEAR_CLASSES,
                                   pretrained=False).to(DEVICE)
        ckpt = torch.load(str(HOVERNET_CKPT), map_location=DEVICE,
                          weights_only=False)
        HOVERNET_MODEL.load_state_dict(ckpt["model_state_dict"])
        HOVERNET_MODEL.eval()
        vl = ckpt.get("val_loss", float("nan"))
        ep = ckpt.get("epochs",   "?")
        print(f"  ✅ HoVer-Net loaded  val_loss={vl:.4f}  epochs={ep}")
    else:
        print(f"  ❌ No checkpoint at {HOVERNET_CKPT}")
        HOVERNET_MODEL = None
else:
    print(f"  ✅ HOVERNET_MODEL already in scope")

# ── FIX-2: re-declare load_all_pannuke ────────────────────────────────────────
if "load_all_pannuke" not in dir():
    def _discover_pannuke_npy(base: Path):
        found = []
        for part_dir in sorted(base.glob("Part *")):
            img_p = part_dir / "Images" / "images.npy"
            msk_p = part_dir / "Masks"  / "masks.npy"
            if img_p.exists() and msk_p.exists():
                typ_p = part_dir / "Images" / "types.npy"
                found.append({"images": img_p, "masks": msk_p,
                              "types":  typ_p if typ_p.exists() else None,
                              "name":   part_dir.name})
        if not found:
            for img_p in sorted(base.rglob("images.npy")):
                msk_p = img_p.parent / "masks.npy"
                if not msk_p.exists():
                    sib = img_p.parent.parent / "Masks" / "masks.npy"
                    if sib.exists(): msk_p = sib
                found.append({"images": img_p,
                              "masks":  msk_p if msk_p.exists() else None,
                              "types":  None, "name": img_p.parent.name})
        return found

    def load_all_pannuke(base: Path):
        sources = _discover_pannuke_npy(base)
        if not sources:
            raise FileNotFoundError(
                f"No PanNuke .npy files found under {base}")
        all_imgs, all_masks = [], []
        for src in sources:
            imgs  = np.load(str(src["images"]))
            if imgs.ndim == 3:   imgs = imgs[np.newaxis]
            if imgs.shape[1] == 3: imgs = imgs.transpose(0,2,3,1)
            imgs = np.clip(imgs, 0, 255).astype(np.uint8)
            if src["masks"] is not None:
                masks = np.load(str(src["masks"]))
                if masks.ndim == 3:      masks = masks[np.newaxis]
                if masks.shape[1] == 6:  masks = masks.transpose(0,2,3,1)
                if masks.shape[-1] == 5:
                    masks = np.concatenate(
                        [masks, np.zeros((*masks.shape[:3],1),
                                         dtype=np.float32)], axis=-1)
                masks = masks.astype(np.float32)
            else:
                masks = np.zeros((len(imgs),256,256,6), dtype=np.float32)
            if imgs.shape[1] != 256 or imgs.shape[2] != 256:
                imgs  = np.stack([cv2.resize(imgs[i], (256,256))
                                  for i in range(len(imgs))])
                masks = np.stack([cv2.resize(masks[i], (256,256),
                    interpolation=cv2.INTER_NEAREST) for i in range(len(masks))])
            all_imgs.append(imgs); all_masks.append(masks)
        return np.concatenate(all_imgs), np.concatenate(all_masks)

    def derive_hovernet_targets(mask_hwc):
        H, W = mask_hwc.shape[:2]
        np_mask = (mask_hwc[:,:,:5].max(axis=2) > 0).astype(np.float32)
        nc_mask = np.zeros((H,W), dtype=np.int64)
        for c in range(5):
            nc_mask[mask_hwc[:,:,c] > 0] = c + 1
        hv_map = np.zeros((H,W,2), dtype=np.float32)
        for c in range(5):
            inst_map = mask_hwc[:,:,c].astype(np.int32)
            for iid in np.unique(inst_map):
                if iid == 0: continue
                ys, xs = np.where(inst_map == iid)
                if not len(xs): continue
                cx,cy = xs.mean(), ys.mean()
                xr = max(xs.max()-xs.min(), 1)
                yr = max(ys.max()-ys.min(), 1)
                hv_map[ys,xs,0] = (xs-cx)/xr*2.
                hv_map[ys,xs,1] = (ys-cy)/yr*2.
        return np_mask, hv_map, nc_mask

    print("  ✅ load_all_pannuke / derive_hovernet_targets defined")
else:
    print("  ✅ load_all_pannuke already in scope")


# ════════════════════════════════════════════════════════════════════════════════
# PART 1 — Watershed instance separation (FIX-WS-1/2)
# ════════════════════════════════════════════════════════════════════════════════

def _hv_to_instances(np_map, hv_map,
                      min_nucleus_area=_MIN_NUCLEUS_AREA,
                      gradient_threshold=_WS_GRADIENT_THRESHOLD):
    """
    FIX-WS-1: gradient_threshold lowered 0.40 → 0.30 (reduces over-segmentation)
    FIX-WS-2: min_nucleus_area raised 20 → 40 (removes watershed noise fragments)
    Both changes expected to improve DQ from 0.397 → ~0.47
    """
    np_bin   = (np_map > 0.5).astype(np.uint8)
    hv_h, hv_v = hv_map[0], hv_map[1]

    grad_h = np.gradient(hv_h)
    grad_v = np.gradient(hv_v)
    grad_mag = (np.abs(grad_h[0]) + np.abs(grad_h[1]) +
                np.abs(grad_v[0]) + np.abs(grad_v[1]))
    g_min, g_max = grad_mag.min(), grad_mag.max()
    if g_max - g_min > 1e-6:
        grad_mag = (grad_mag - g_min) / (g_max - g_min)

    # FIX-WS-1: lowered threshold
    marker_map     = np_bin * (grad_mag < gradient_threshold).astype(np.uint8)
    markers, _     = ndi.label(marker_map)
    ws_input       = (grad_mag * 255).astype(np.uint8)
    inst_map       = cv2.watershed(
        np.stack([ws_input]*3, axis=-1), markers.astype(np.int32))

    inst_map = np.where(inst_map <= 0, 0, inst_map).astype(np.int32)
    inst_map = inst_map * np_bin

    # FIX-WS-2: larger min area filter
    for inst_id in np.unique(inst_map):
        if inst_id == 0: continue
        if (inst_map == inst_id).sum() < min_nucleus_area:
            inst_map[inst_map == inst_id] = 0

    labeled, _ = ndi.label(inst_map > 0)
    return labeled.astype(np.int32)


# ════════════════════════════════════════════════════════════════════════════════
# PART 2 — Per-nucleus classification (FIX-NEO-1/2)
# ════════════════════════════════════════════════════════════════════════════════

def _classify_nuclei(inst_map, nc_map_probs,
                     neo_conf_threshold=HOVERNET_NEO_CONF_THRESHOLD_V2):
    """
    FIX-NEO-1: raised confidence threshold 0.55 → 0.72
    FIX-NEO-2: use max-pooled confidence (spatial max within nucleus) instead
               of pixel-mean — more robust to small high-conf regions
               WHY: mean can be pulled down by background pixels at nucleus
               boundaries, causing good neoplastic nuclei to be demoted
    """
    result = {}
    for inst_id in np.unique(inst_map):
        if inst_id == 0: continue
        pixels   = (inst_map == inst_id)

        # FIX-NEO-2: max-pooled instead of mean for robustness
        # Use 90th percentile of spatial distribution (robust max)
        pixel_probs = nc_map_probs[:, pixels]   # (6, n_pixels)
        avg_prob     = np.percentile(pixel_probs, 90, axis=1)  # (6,)

        cls_prob = avg_prob[1:]                  # skip background
        pred_cls = int(cls_prob.argmax())
        raw_conf = float(cls_prob[pred_cls])

        # FIX-NEO-1: gate with raised threshold
        if pred_cls == 0 and raw_conf < neo_conf_threshold:
            cls_prob_copy = cls_prob.copy()
            cls_prob_copy[0] = 0.
            if cls_prob_copy.max() > 0.:
                pred_cls = int(cls_prob_copy.argmax())
                raw_conf = float(cls_prob[pred_cls])
            else:
                pred_cls = 4  # epithelial fallback
                raw_conf = 0.

        result[inst_id] = {
            "type_idx":  pred_cls,
            "type_name": NUCLEAR_CLASSES[pred_cls],
            "confidence": raw_conf,
        }
    return result


def _compute_nuclear_morphometrics(inst_map, nuclear_types):
    from skimage.measure import regionprops
    morphs = []
    for rp in regionprops(inst_map):
        inst_id = rp.label
        ntype   = nuclear_types.get(inst_id,
                                    {"type_name":"unknown","confidence":0.})
        morphs.append({
            "inst_id":      inst_id,
            "type":         ntype["type_name"],
            "confidence":   ntype["confidence"],
            "area_px":      int(rp.area),
            "eccentricity": round(float(rp.eccentricity), 4),
            "solidity":     round(float(rp.solidity), 4),
            "major_axis":   round(float(rp.major_axis_length), 2),
            "minor_axis":   round(float(rp.minor_axis_length), 2),
            "perimeter":    round(float(rp.perimeter), 2),
            "centroid_y":   round(float(rp.centroid[0]), 1),
            "centroid_x":   round(float(rp.centroid[1]), 1),
        })
    return morphs


def _compute_til_score(nuclear_types):
    counts = {name: 0 for name in NUCLEAR_CLASSES.values()}
    for info in nuclear_types.values():
        counts[info["type_name"]] += 1
    neo   = counts["neoplastic"]
    inf   = counts["inflammatory"]
    return {
        "til_score":    round(inf / (neo + 1e-6), 4),
        "neoplastic":   neo,
        "inflammatory": inf,
        "connective":   counts["connective"],
        "dead":         counts["dead"],
        "epithelial":   counts["epithelial"],
        "total_nuclei": sum(counts.values()),
    }


# ════════════════════════════════════════════════════════════════════════════════
# PART 3 — Panoptic Quality metric (unchanged)
# ════════════════════════════════════════════════════════════════════════════════

def _compute_panoptic_quality(inst_pred, inst_gt, iou_threshold=0.5):
    pred_ids   = set(np.unique(inst_pred)) - {0}
    gt_ids     = set(np.unique(inst_gt))   - {0}
    matched_iou = []; matched_pred = set(); matched_gt = set()

    for p_id in pred_ids:
        p_mask   = (inst_pred == p_id)
        best_iou = 0.; best_g = None
        for g_id in gt_ids:
            if g_id in matched_gt: continue
            g_mask = (inst_gt == g_id)
            inter  = float((p_mask & g_mask).sum())
            union  = float((p_mask | g_mask).sum())
            iou    = inter / (union + 1e-6)
            if iou > best_iou:
                best_iou = iou; best_g = g_id
        if best_iou >= iou_threshold:
            matched_iou.append(best_iou)
            matched_pred.add(p_id); matched_gt.add(best_g)

    TP = len(matched_iou); FP = len(pred_ids)-TP; FN = len(gt_ids)-TP
    SQ = float(np.mean(matched_iou)) if matched_iou else 0.
    DQ = TP / (TP + 0.5*FP + 0.5*FN + 1e-6)
    return {"PQ": round(DQ*SQ,4), "DQ": round(DQ,4), "SQ": round(SQ,4),
            "TP": TP, "FP": FP, "FN": FN}


# ════════════════════════════════════════════════════════════════════════════════
# PART 4 — Master inference function (FIX-NEO-3: seg_net_used fix)
# ════════════════════════════════════════════════════════════════════════════════

_HOVERNET_NORM = transforms.Normalize(
    [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

TYPE_COLORS = {
    "neoplastic":   (255,  50,  50),
    "inflammatory": ( 50, 150, 255),
    "connective":   ( 50, 200,  50),
    "dead":         (180, 180, 180),
    "epithelial":   (255, 200,  50),
}

# FIX-NEO-3: minimum gated neoplastic count required to set seg_net_used=True
# Prevents single spurious nucleus from setting the flag
_MIN_NEOPLASTIC_FOR_SEG_NET = 2

def hovernet_predict(img_np_hwc, organ=None, return_overlay=True):
    """
    Full HoVer-Net inference on a single 224×224 H&E patch.

    FIX-NEO-1/2: upgraded _classify_nuclei with raised threshold and
                 90th-percentile confidence pooling
    FIX-NEO-3:  seg_net_used = True only when gated n_neoplastic >= 2
    FIX-WS-1/2: tighter watershed + larger min area
    """
    mal_set      = (MALIGNANT_CLASSES.get(organ, set())
                    if organ and "MALIGNANT_CLASSES" in dir() else set())

    _empty = {
        "inst_map": np.zeros((224,224), dtype=np.int32),
        "np_map":   np.zeros((224,224), dtype=np.float32),
        "nc_probs": np.zeros((6,224,224), dtype=np.float32),
        "nuclear_types": {}, "morphometrics": [], "til": {},
        "seg_ok": False, "tumor_area_pct": 0.,
        "seg_circularity": 0., "n_regions": 0, "n_neoplastic": 0,
        "overlay": None, "seg_net_used": False, "crf_used": False,
        "is_malignant": False, "boundary_regularity": 0.,
        "seg_skipped_reason": "hovernet_not_available",
        "mask": np.zeros((224,224), dtype=np.float32),
        "neo_confidence_used": HOVERNET_NEO_CONF_THRESHOLD_V2,
    }

    if HOVERNET_MODEL is None:
        return _empty

    HOVERNET_MODEL.eval()
    try:
        img_256 = cv2.resize(
            (img_np_hwc * 255).astype(np.uint8), (256,256),
            interpolation=cv2.INTER_LINEAR)
        img_t = _HOVERNET_NORM(
            torch.from_numpy(
                img_256.transpose(2,0,1).astype(np.float32)/255.)
        ).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                raw = HOVERNET_MODEL(img_t)

        if "np_logit" in raw:
            np_map_t   = torch.sigmoid(raw["np_logit"])
            hv_map_t   = torch.tanh(   raw["hv_logit"])
            nc_logit_t = raw["nc_logit"]
        else:
            np_map_t   = raw["np_map"]
            hv_map_t   = raw["hv_map"]
            nc_logit_t = raw["nc_map"]

        np_map   = np_map_t[0,0].cpu().float().numpy()
        hv_map   = hv_map_t[0].cpu().float().numpy()
        nc_probs = torch.softmax(nc_logit_t[0].float(), 0).cpu().numpy()

        # FIX-WS-1/2: use updated _hv_to_instances with tighter params
        inst_map_256 = _hv_to_instances(np_map, hv_map)

        np_map_224   = cv2.resize(np_map, (224,224),
                                   interpolation=cv2.INTER_LINEAR)
        inst_map_224 = cv2.resize(inst_map_256.astype(np.float32),
                                   (224,224),
                                   interpolation=cv2.INTER_NEAREST
                                   ).astype(np.int32)
        nc_probs_224 = np.stack([
            cv2.resize(nc_probs[c], (224,224),
                       interpolation=cv2.INTER_LINEAR)
            for c in range(6)], axis=0)

        # FIX-NEO-1/2: gated classification with raised threshold + 90th-pctile
        nuclear_types = _classify_nuclei(inst_map_224, nc_probs_224,
                                          HOVERNET_NEO_CONF_THRESHOLD_V2)
        morphometrics = _compute_nuclear_morphometrics(inst_map_224, nuclear_types)
        til           = _compute_til_score(nuclear_types)

        n_neoplastic = til["neoplastic"]
        total_nuclei = til["total_nuclei"]
        is_malignant = n_neoplastic > 0

        # FIX-NEO-3: require >= 2 gated neoplastic nuclei for seg_net_used=True
        # Prevents single random nucleus from triggering the flag
        seg_net_used = (n_neoplastic >= _MIN_NEOPLASTIC_FOR_SEG_NET)

        tumor_area = float((inst_map_224 > 0).sum()) / (224*224) * 100.

        from skimage.measure import regionprops, label as sk_label
        neo_mask = np.zeros((224,224), dtype=bool)
        for iid, info in nuclear_types.items():
            if info["type_name"] == "neoplastic":
                neo_mask |= (inst_map_224 == iid)
        seg_circularity = 0.
        if neo_mask.any():
            lab2 = sk_label(neo_mask)
            regs = regionprops(lab2)
            if regs:
                lg  = max(regs, key=lambda r: r.area)
                p   = lg.perimeter
                seg_circularity = min(
                    4.*math.pi*lg.area/(p**2) if p > 0 else 0., 1.)

        overlay = None
        if return_overlay and total_nuclei > 0:
            overlay = (img_np_hwc * 255).astype(np.uint8).copy()
            for iid, info in nuclear_types.items():
                mask_n = (inst_map_224 == iid)
                color  = TYPE_COLORS.get(info["type_name"], (255,255,255))
                for c, v in enumerate(color):
                    ch = overlay[:,:,c].astype(np.float32)
                    ch[mask_n] = ch[mask_n]*0.4 + v*0.6
                    overlay[:,:,c] = np.clip(ch, 0, 255).astype(np.uint8)
                cnts, _ = cv2.findContours(mask_n.astype(np.uint8),
                    cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(overlay, cnts, -1, color, 1)

        binary_mask = (inst_map_224 > 0).astype(np.float32)

        return {
            "inst_map":           inst_map_224,
            "np_map":             np_map_224,
            "nc_probs":           nc_probs_224,
            "nuclear_types":      nuclear_types,
            "morphometrics":      morphometrics,
            "til":                til,
            "seg_ok":             total_nuclei > 0,
            "tumor_area_pct":     round(tumor_area, 2),
            "seg_circularity":    round(float(seg_circularity), 4),
            "n_regions":          int(total_nuclei),
            "n_neoplastic":       int(n_neoplastic),
            "overlay":            overlay,
            "seg_net_used":       seg_net_used,   # FIX-NEO-3: gated
            "crf_used":           False,
            "is_malignant":       is_malignant,
            "seg_skipped_reason": None,
            "boundary_regularity": 0.,
            "mask":               binary_mask,
            "neo_confidence_used": HOVERNET_NEO_CONF_THRESHOLD_V2,
        }

    except Exception as e:
        import traceback; traceback.print_exc()
        return {**_empty, "seg_skipped_reason": f"error: {e}"}


# ════════════════════════════════════════════════════════════════════════════════
# PART 5 — generate_segmentation_mask wrapper (backward compatible)
# ════════════════════════════════════════════════════════════════════════════════

def generate_segmentation_mask(organ, pred_class_name, final_cam, img_np_hwc,
                                model=None, img_t=None, pred_class_idx=None,
                                hint=None):
    mal_set      = (MALIGNANT_CLASSES.get(organ, set())
                    if "MALIGNANT_CLASSES" in dir() else set())
    is_malignant = pred_class_name in mal_set

    if HOVERNET_MODEL is not None and img_np_hwc is not None:
        result = hovernet_predict_v21(img_np_hwc, organ=organ, return_overlay=True)
        result["is_malignant"]     = is_malignant
        result["seg_is_malignant"] = is_malignant
        if not is_malignant:
            result["seg_skipped_reason"] = "benign_class"
        return result

    # ── CAM-threshold fallback ────────────────────────────────────────────────
    from skimage.measure import label as sk_label, regionprops
    from skimage.morphology import binary_closing, binary_opening, disk
    from scipy.ndimage import binary_fill_holes

    _empty_fb = {
        "mask": np.zeros((224,224), np.float32),
        "inst_map": np.zeros((224,224), np.int32),
        "is_malignant": is_malignant, "seg_is_malignant": is_malignant,
        "tumor_area_pct": 0., "seg_circularity": 0., "n_regions": 0,
        "boundary_regularity": 0., "seg_ok": False, "crf_used": False,
        "seg_net_used": False, "nuclear_types": {}, "morphometrics": [],
        "til": {}, "n_neoplastic": 0, "overlay": None,
        "seg_skipped_reason": "benign_class" if not is_malignant
                               else "hovernet_unavailable",
        "neo_confidence_used": HOVERNET_NEO_CONF_THRESHOLD_V2,
    }
    if not is_malignant or final_cam.max() < 0.05:
        return _empty_fb

    thr      = (SEG_CAM_THRESHOLD.get(organ, SEG_CAM_THRESHOLD.get("default",0.30))
                if "SEG_CAM_THRESHOLD" in dir() else 0.30)
    crf_used = False
    if ("HAS_CRF" in dir() and HAS_CRF and img_np_hwc is not None
            and "dcrf" in dir()):
        try:
            img_u8 = np.ascontiguousarray(
                (img_np_hwc*255).clip(0,255).astype(np.uint8))
            fg     = np.ascontiguousarray(
                np.clip(final_cam.astype(np.float32), 0.01, 0.99))
            prob   = np.ascontiguousarray(np.stack([1.-fg, fg], axis=0))
            from pydensecrf.utils import unary_from_softmax as _ufs
            unary  = np.ascontiguousarray(_ufs(prob))
            import pydensecrf.densecrf as _dcrf
            d = _dcrf.DenseCRF2D(224,224,2)
            d.setUnaryEnergy(unary)
            d.addPairwiseGaussian(sxy=3, compat=3)
            d.addPairwiseBilateral(sxy=40,srgb=13,rgbim=img_u8,compat=10)
            raw_mask = (np.array(d.inference(5)).reshape(2,224,224)[1]>0.5
                        ).astype(np.float32)
            crf_used = True
        except Exception:
            raw_mask = (final_cam > thr).astype(np.float32)
    else:
        raw_mask = (final_cam > thr).astype(np.float32)

    mask = binary_fill_holes(raw_mask.astype(bool))
    r_c, r_o = {"brain_2d":(5,3),"skin":(4,2)}.get(
        organ, (2,1) if organ in {"blood","lymphoma","pancreatic"} else (3,2))
    mask = binary_closing(mask, disk(r_c))
    mask = binary_opening(mask, disk(r_o))
    mask = binary_fill_holes(mask).astype(np.float32)
    lab  = sk_label(mask.astype(bool))
    if lab.max() > 0:
        regs   = sorted(regionprops(lab), key=lambda r: r.area, reverse=True)
        n_keep = min(3 if organ in {"lymphoma","ovarian","gastric"} else 1,
                     len(regs))
        out = np.zeros_like(mask)
        for r in regs[:n_keep]: out[lab==r.label] = 1.
        mask = out
    area = float(mask.sum())
    if area < 10:
        return {**_empty_fb, "seg_ok":False,
                "seg_skipped_reason":"mask_too_small"}
    lab2    = sk_label(mask.astype(bool)); regs2 = regionprops(lab2)
    largest = max(regs2, key=lambda r: r.area)
    p       = largest.perimeter
    seg_circ = min(4.*math.pi*largest.area/(p**2) if p>0 else 0., 1.)
    try:
        bdry = min(float(largest.area)/(largest.convex_area+1e-8), 1.)
    except Exception:
        bdry = 0.
    return {
        "mask": mask, "inst_map": lab2.astype(np.int32),
        "is_malignant": True, "seg_is_malignant": True,
        "tumor_area_pct": round(100.*area/(224*224),2),
        "seg_circularity": round(float(seg_circ),4),
        "n_regions": len(regs2), "boundary_regularity": round(bdry,4),
        "seg_ok": True, "crf_used": crf_used, "seg_net_used": False,
        "seg_skipped_reason": None, "nuclear_types": {}, "morphometrics": [],
        "til": {}, "n_neoplastic": 0, "overlay": None,
        "neo_confidence_used": HOVERNET_NEO_CONF_THRESHOLD_V2,
    }


# ════════════════════════════════════════════════════════════════════════════════
# PART 6 — PQ evaluation (FIX-PQ-1: more samples)
# ════════════════════════════════════════════════════════════════════════════════

def evaluate_hovernet_pq(n_samples=150):  # FIX-PQ-1: raised from 100 → 150
    if HOVERNET_MODEL is None:
        print("  ⚠️  Cannot evaluate PQ: HOVERNET_MODEL is None")
        return None
    if not PANNUKE_ROOT.exists():
        print(f"  ⚠️  Cannot evaluate PQ: PANNUKE_ROOT not found ({PANNUKE_ROOT})")
        return None

    print(f"  Loading PanNuke data from {PANNUKE_ROOT} for PQ evaluation...")
    try:
        imgs_all, masks_all = load_all_pannuke(PANNUKE_ROOT)
    except FileNotFoundError as e:
        print(f"  ⚠️  {e}")
        return None

    n_total  = len(imgs_all)
    n_test   = max(20, n_total // 10)
    test_imgs  = imgs_all[-n_test:]
    test_masks = masks_all[-n_test:]
    print(f"  Using last {n_test} patches as held-out test set")

    norm = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    HOVERNET_MODEL.eval()
    pq_all, dq_all, sq_all = [], [], []

    for i in range(min(n_samples, n_test)):
        img_hwc  = test_imgs[i]
        mask_hwc = test_masks[i]

        gt_inst = np.zeros((256,256), dtype=np.int32)
        offset  = 0
        for cls_idx in range(5):
            cls_mask = mask_hwc[:,:,cls_idx].astype(np.int32)
            for inst_id in np.unique(cls_mask):
                if inst_id == 0: continue
                offset += 1
                gt_inst[cls_mask == inst_id] = offset

        img_t = norm(torch.from_numpy(
            img_hwc.transpose(2,0,1).astype(np.float32)/255.)
        ).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                raw = HOVERNET_MODEL(img_t)

        if "np_logit" in raw:
            np_map  = torch.sigmoid(raw["np_logit"])[0,0].cpu().float().numpy()
            hv_map  = torch.tanh(   raw["hv_logit"])[0].cpu().float().numpy()
        else:
            np_map  = raw["np_map"][0,0].cpu().float().numpy()
            hv_map  = raw["hv_map"][0].cpu().float().numpy()

        # FIX-WS-1/2: use updated watershed
        inst_pred = _hv_to_instances(np_map, hv_map)
        pq_res    = _compute_panoptic_quality(inst_pred, gt_inst)
        pq_all.append(pq_res["PQ"])
        dq_all.append(pq_res["DQ"])
        sq_all.append(pq_res["SQ"])

    results = {
        "PQ": float(np.mean(pq_all)),
        "DQ": float(np.mean(dq_all)),
        "SQ": float(np.mean(sq_all)),
        "n_samples": len(pq_all),
    }
    print(f"\n  ┌─ HoVer-Net Evaluation  (n={results['n_samples']} patches) ──────┐")
    print(f"  │  PQ = {results['PQ']:.4f} ± {float(np.std(pq_all)):.4f}"
          f"  (Panoptic Quality)           │")
    print(f"  │  DQ = {results['DQ']:.4f} ± {float(np.std(dq_all)):.4f}"
          f"  (Detection Quality = F1)     │")
    print(f"  │  SQ = {results['SQ']:.4f} ± {float(np.std(sq_all)):.4f}"
          f"  (Segmentation Quality = IoU) │")
    print(f"  │                                                         │")
    print(f"  │  FIX-WS: grad_thr=0.30 min_area=40  (was 0.40/20)     │")
    print(f"  │  FIX-NEO: conf_thr=0.72 + 90th-pctile pooling          │")
    print(f"  └─────────────────────────────────────────────────────────┘")
    return results


# ── Run PQ evaluation ─────────────────────────────────────────────────────────
print()
if HOVERNET_MODEL is not None and globals().get("RUN_PQ_EVAL", True):
    hovernet_pq_results = evaluate_hovernet_pq(n_samples=150)
elif HOVERNET_MODEL is not None:
    hovernet_pq_results = None; print("  ℹ️  PQ evaluation skipped in this stage (RUN_PQ_EVAL=False)")
else:
    hovernet_pq_results = None
    print("  ⚠️  Skipping PQ evaluation — HOVERNET_MODEL not loaded")

# ── Smoke test with fix verification ─────────────────────────────────────────
print("\n  Smoke test: hovernet_predict (v18-FIX) on random 224×224 patch...")
try:
    dummy_img = np.random.rand(224, 224, 3).astype(np.float32)
    dummy_res = hovernet_predict(dummy_img, organ="colon", return_overlay=True)
    n_neo     = dummy_res['n_neoplastic']
    seg_net   = dummy_res['seg_net_used']
    thresh    = dummy_res.get('neo_confidence_used', HOVERNET_NEO_CONF_THRESHOLD_V2)
    print(f"  seg_ok={dummy_res['seg_ok']}"
          f"  n_regions={dummy_res['n_regions']}"
          f"  n_neoplastic={n_neo}  (was 45, should be <<10)"
          f"  seg_net_used={seg_net}")
    print(f"  neo_conf_threshold={thresh}"
          f"  FIX-NEO-1/2/3: {'✅ gate working' if n_neo < 10 else '⚠️ still high'}")
    print("  ✅ hovernet_predict (v18-FIX) smoke test passed")
except Exception as e:
    print(f"  ❌ smoke test failed: {e}")

print("\n✅ Cell 4-NEW (v18-FIX) complete")
print(f"   FIX-NEO-1: conf threshold {HOVERNET_NEO_CONF_THRESHOLD_V2} (was 0.55)")
print(f"   FIX-NEO-2: 90th-percentile pooling (was pixel-mean)")
print(f"   FIX-NEO-3: seg_net_used requires n_neoplastic >= {_MIN_NEOPLASTIC_FOR_SEG_NET}")
print(f"   FIX-WS-1:  gradient threshold {_WS_GRADIENT_THRESHOLD} (was 0.40)")
print(f"   FIX-WS-2:  min nucleus area {_MIN_NUCLEUS_AREA}px (was 20px)")
print(f"   FIX-PQ-1:  evaluate with n=150 samples (was 100)")

In [ ]:
# V24.1 — release PanNuke / evaluation buffers before the next heavy block
free_mem("this block")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# FIX CELL D
# PLACEMENT: Between Cell 4-NEW and Cell 5a  (UNCHANGED from before)
#
# FIXES:
#   - Per-organ HoVerNet neoplastic confidence thresholds
#   - bladder_routing_hint for morphology-based class correction
#   - hovernet_predict_v21 (replaces hovernet_predict in the pipeline)
#
# ALSO ADD THIS ONE LINE to Cell 6, Section 7, function generate_segmentation_mask:
#   Find the line:  result = hovernet_predict(img_np_hwc, organ=organ, ...)
#   Replace with:   result = hovernet_predict_v21(img_np_hwc, organ=organ, ...)
#   (There are two such lines — both in the try block and the outer call)
# ════════════════════════════════════════════════════════════════════════════════
 
import math, gc
import numpy as np
import torch
import cv2
 
print("\n" + "═" * 68)
print("  FIX CELL D — Per-Organ HoVerNet Thresholds + Bladder Routing")
print("═" * 68)
 
# ── Per-organ neoplastic confidence thresholds ────────────────────────────────
# WHY: global threshold 0.72 is too strict for PDAC (pancreatic) and gastric
# where nuclei are large and irregular but the classifier confidence is lower
# due to the undertrained nuclear type head.
HOVERNET_NEO_CONF_THRESHOLD_PER_ORGAN = {
    "pancreatic":  0.60,
    "gastric":     0.62,
    "ovarian":     0.65,
    "colon":       0.68,
    "brain_2d":    0.68,
    "default":     HOVERNET_NEO_CONF_THRESHOLD_V2,  # 0.72
}
 
 
def hovernet_predict_v21(img_np_hwc: np.ndarray,
                          organ: str = None,
                          return_overlay: bool = True) -> dict:
    """
    Per-organ HoVerNet confidence threshold.
    Drop-in replacement for hovernet_predict / hovernet_predict_v18.
    Uses HOVERNET_NEO_CONF_THRESHOLD_PER_ORGAN for _classify_nuclei.
    """
    neo_conf = HOVERNET_NEO_CONF_THRESHOLD_PER_ORGAN.get(
        organ, HOVERNET_NEO_CONF_THRESHOLD_PER_ORGAN["default"])
 
    _empty = {
        "inst_map": np.zeros((224, 224), dtype=np.int32),
        "np_map":   np.zeros((224, 224), dtype=np.float32),
        "nc_probs": np.zeros((6, 224, 224), dtype=np.float32),
        "nuclear_types": {}, "morphometrics": [], "til": {},
        "seg_ok": False, "tumor_area_pct": 0., "seg_circularity": 0.,
        "n_regions": 0, "n_neoplastic": 0, "overlay": None,
        "seg_net_used": False, "crf_used": False, "is_malignant": False,
        "boundary_regularity": 0., "seg_skipped_reason": "no_model",
        "mask": np.zeros((224, 224), dtype=np.float32),
        "neo_confidence_used": neo_conf,
    }
 
    if HOVERNET_MODEL is None:
        return _empty
 
    HOVERNET_MODEL.eval()
    try:
        img_256 = cv2.resize(
            (img_np_hwc * 255).astype(np.uint8), (256, 256),
            interpolation=cv2.INTER_LINEAR)
        img_t = _HOVERNET_NORM(
            torch.from_numpy(
                img_256.transpose(2, 0, 1).astype(np.float32) / 255.)
        ).unsqueeze(0).to(DEVICE)
 
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                raw = HOVERNET_MODEL(img_t)
 
        np_map_t   = torch.sigmoid(raw["np_logit"])
        hv_map_t   = torch.tanh(raw["hv_logit"])
        nc_logit_t = raw["nc_logit"]
 
        np_map   = np_map_t[0, 0].cpu().float().numpy()
        hv_map   = hv_map_t[0].cpu().float().numpy()
        nc_probs = torch.softmax(nc_logit_t[0].float(), 0).cpu().numpy()
 
        inst_map_256 = _hv_to_instances(np_map, hv_map)
 
        np_map_224   = cv2.resize(np_map, (224, 224),
                                   interpolation=cv2.INTER_LINEAR)
        inst_map_224 = cv2.resize(
            inst_map_256.astype(np.float32), (224, 224),
            interpolation=cv2.INTER_NEAREST).astype(np.int32)
        nc_probs_224 = np.stack([
            cv2.resize(nc_probs[c], (224, 224),
                       interpolation=cv2.INTER_LINEAR)
            for c in range(6)], axis=0)
 
        # KEY FIX: use per-organ threshold
        nuclear_types = _classify_nuclei(inst_map_224, nc_probs_224, neo_conf)
        morphometrics = _compute_nuclear_morphometrics(inst_map_224, nuclear_types)
        til           = _compute_til_score(nuclear_types)
 
        n_neoplastic = til["neoplastic"]
        total_nuclei = til["total_nuclei"]
        seg_net_used = (n_neoplastic >= _MIN_NEOPLASTIC_FOR_SEG_NET)
        tumor_area   = float((inst_map_224 > 0).sum()) / (224 * 224) * 100.
        binary_mask  = (inst_map_224 > 0).astype(np.float32)
 
        overlay = None
        if return_overlay and total_nuclei > 0:
            overlay = (img_np_hwc * 255).astype(np.uint8).copy()
            for iid, info in nuclear_types.items():
                mask_n = (inst_map_224 == iid)
                color  = TYPE_COLORS.get(info["type_name"], (255, 255, 255))
                for c, v in enumerate(color):
                    ch = overlay[:, :, c].astype(np.float32)
                    ch[mask_n] = ch[mask_n] * 0.4 + v * 0.6
                    overlay[:, :, c] = np.clip(ch, 0, 255).astype(np.uint8)
                cnts, _ = cv2.findContours(
                    mask_n.astype(np.uint8),
                    cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(overlay, cnts, -1, color, 1)
 
        return {
            "inst_map": inst_map_224, "np_map": np_map_224,
            "nc_probs": nc_probs_224, "nuclear_types": nuclear_types,
            "morphometrics": morphometrics, "til": til,
            "seg_ok": total_nuclei > 0,
            "tumor_area_pct": round(tumor_area, 2),
            "seg_circularity": 0., "n_regions": int(total_nuclei),
            "n_neoplastic": int(n_neoplastic), "overlay": overlay,
            "seg_net_used": seg_net_used, "crf_used": False,
            "is_malignant": n_neoplastic > 0,
            "seg_skipped_reason": None,
            "boundary_regularity": 0.,
            "mask": binary_mask,
            "neo_confidence_used": neo_conf,
        }
    except Exception as e:
        return {**_empty, "seg_skipped_reason": f"error: {e}"}
 
 
def bladder_routing_hint(seg_result: dict,
                          base_class_prbs: np.ndarray,
                          class_names: list) -> np.ndarray:
    """Morphology-based post-correction for bladder class probabilities."""
    prbs = base_class_prbs.copy()
    if not seg_result.get("seg_ok"):
        return prbs
    area  = float(seg_result.get("tumor_area_pct", 0.))
    circ  = float(seg_result.get("seg_circularity", 0.))
    n_reg = int(seg_result.get("n_regions", 1))
    try:
        mi_idx  = class_names.index("muscle_invasive")
        nmi_idx = class_names.index("non_muscle_invasive")
    except ValueError:
        return prbs
    if area > 35. and circ < 0.4 and n_reg >= 2:
        prbs[mi_idx]  = min(prbs[mi_idx]  * 1.25, 0.99)
        prbs[nmi_idx] = max(prbs[nmi_idx] * 0.80, 0.01)
    elif area < 20. and circ > 0.6 and n_reg == 1:
        prbs[mi_idx]  = max(prbs[mi_idx]  * 0.80, 0.01)
        prbs[nmi_idx] = min(prbs[nmi_idx] * 1.20, 0.99)
    prbs = prbs / (prbs.sum() + 1e-8)
    return prbs
 
 
print("  Per-organ neo thresholds:")
for _o, _t in sorted(HOVERNET_NEO_CONF_THRESHOLD_PER_ORGAN.items()):
    if _o == "default": continue
    print(f"    {_o:<14}: {_t:.2f}  (global={HOVERNET_NEO_CONF_THRESHOLD_V2:.2f})")
print("  ✅ hovernet_predict_v21() defined")
print("  ✅ bladder_routing_hint() defined")
print()
print("  ℹ️  V22: Cell 6 hovernet_predict_v18 now delegates to hovernet_predict_v21 automatically")
print()
print("✅ FIX CELL D complete")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 5a — Restore FiLM checkpoints
# ════════════════════════════════════════════════════════════════════════════════
import shutil

SPEC_DIR.mkdir(parents=True,exist_ok=True)
FILM_INPUT_DIR=Path("/kaggle/input/models/marcyoussef22/filmv8/other/default/2")
FILM_CKPTS_DONE=[
    "best_blood_film_v8.pth","best_lymphoma_film_v8.pth",
    "best_brain_2d_film_v8.pth","best_cervical_film_v8.pth",
    "best_colon_film_v8.pth","best_esophageal_film_v8.pth",
    "best_kidney_film_v8.pth","best_skin_film_v8.pth",
    "best_bladder_film_v8.pth","best_breast_film_v8.pth",
    "best_gastric_film_v8.pth","best_lung_film_v8.pth",
    "best_oral_film_v8.pth","best_ovarian_film_v8.pth",
    "best_pancreatic_film_v8.pth","best_thyroid_film_v8.pth",
]
print("── Copying FiLM checkpoints ─────────────────────────────────────────")
for fname in FILM_CKPTS_DONE:
    src=FILM_INPUT_DIR/fname; dst=SPEC_DIR/fname
    if src.exists() and not dst.exists(): shutil.copy2(str(src),str(dst))
    if dst.exists():
        ckpt=torch.load(str(dst),map_location="cpu",weights_only=False)
        print(f"  ✅ {fname:<35}  val_f1={ckpt.get('val_f1',0):.4f}")
    else:
        print(f"  ⚠️  NOT FOUND: {src}")

SOTA_INPUT_DIR=Path("/kaggle/input/models/marcyoussef22/sota/other/default/1")
for fname in ["best_bladder_sota_v8.pth","best_thyroid_sota_v8.pth"]:
    src=SOTA_INPUT_DIR/fname; dst=SPEC_DIR/fname
    if src.exists() and not dst.exists(): shutil.copy2(str(src),str(dst))
    if dst.exists():
        ckpt=torch.load(str(dst),map_location="cpu",weights_only=False)
        print(f"  ✅ {fname:<35}  val_f1={ckpt.get('val_f1',0):.4f}")
SOTA_INPUT_DIR=Path("/kaggle/input/models/marcyoussef22/segmentation/other/default/1/")
for fname in ["hovernet_pannuke.pth"]:
    src=SOTA_INPUT_DIR/fname; dst=SPEC_DIR/fname
    if src.exists() and not dst.exists(): shutil.copy2(str(src),str(dst))
    if dst.exists():
        ckpt=torch.load(str(dst),map_location="cpu",weights_only=False)
        print(f"  ✅ {fname:<35}  val_f1={ckpt.get('val_f1',0):.4f}")
print(f"\n✅ Cell 5a complete")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 5b — Gateway load + FiLM restore
#   v16-ISS-8 : FiLM disabled at load-time if quick val-F1 < base_F1 − 0.002
#   v16-ISS-12: film_status tracks FROZEN / ACTIVE / DISABLED / ABSENT
#   v16-NEW   : gateway_with_fallback for organ_conf < GATEWAY_FALLBACK_THRESH
# ════════════════════════════════════════════════════════════════════════════════

class ModalityAdaptivePreprocessing(nn.Module):
    MODS=["histology","mri_ct","dermoscopy","xray"]
    def __init__(self):
        super().__init__()
        self.adapters=nn.ModuleDict({
            m:nn.Sequential(nn.Conv2d(3,3,3,padding=1,groups=3,bias=False),
                            nn.BatchNorm2d(3),nn.GELU())
            for m in self.MODS})
        self.register_buffer("mn",torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer("sd",torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
    @staticmethod
    def detect(xi):
        m,s=xi.mean().item(),xi.std().item()
        if s>0.15: return "histology"
        elif m<0.25: return "mri_ct"
        elif m>0.70: return "dermoscopy"
        return "xray"
    def forward(self,x):
        with torch.cuda.amp.autocast(enabled=False):
            x=x.float(); B=x.shape[0]; out=torch.zeros_like(x)
            groups={m:[] for m in self.MODS}
            for i in range(B): groups[self.detect(x[i:i+1])].append(i)
            for m,idxs in groups.items():
                if idxs:
                    t=torch.tensor(idxs,device=x.device); out[t]=self.adapters[m](x[t])
        return (out-self.mn)/self.sd

class HierarchicalMedViT(nn.Module):
    def __init__(self,organ_classes,subclass_map,dropout=0.3):
        super().__init__()
        self.organ_classes=organ_classes; self.subclass_map=subclass_map
        self.n_organs=len(organ_classes)
        self.map_layer=ModalityAdaptivePreprocessing()
        self.backbone=timm.create_model("convnext_small",pretrained=False,num_classes=0)
        bdim=self.backbone.num_features
        self.projector=nn.Sequential(
            nn.Linear(bdim,512),nn.LayerNorm(512),nn.GELU(),nn.Dropout(dropout),
            nn.Linear(512,256),nn.LayerNorm(256),nn.GELU(),nn.Dropout(dropout*0.5))
        self.organ_head=nn.Linear(256,self.n_organs)
        self.subclass_heads=nn.ModuleDict({
            org:nn.Linear(256,len(subs)) for org,subs in subclass_map.items()})
        self.subclass_temps=nn.ParameterDict({
            org:nn.Parameter(torch.tensor([0.85])) for org in subclass_map})
        self.max_subclasses=max(len(s) for s in subclass_map.values())
    def forward(self,x,organ_gt=None):
        x=self.map_layer(x)
        if x.shape[-1]!=224: x=F.interpolate(x,(224,224),mode="bilinear",align_corners=False)
        feats=self.backbone(x); shared=self.projector(feats)
        organ_logits=self.organ_head(shared); pred_organs=organ_logits.argmax(1)
        active=organ_gt if organ_gt is not None else pred_organs
        B=x.shape[0]; sub=torch.full((B,self.max_subclasses),float("-inf"),device=x.device,dtype=shared.dtype)
        for i,oi in enumerate(active):
            name=self.organ_classes[oi.item()]
            if name in self.subclass_heads:
                T=self.subclass_temps[name].clamp(0.1,5.0)
                lg=self.subclass_heads[name](shared[i:i+1])/T
                sub[i,:lg.shape[1]]=lg[0]
        return {"organ_logits":organ_logits,"subclass_logits":sub,
                "predicted_organ":pred_organs,"features":shared}

GATEWAY_MODEL=None

def load_gateway():
    global GATEWAY_MODEL
    if not GATEWAY_CKPT.exists():
        print(f"  ⚠️  Gateway not found: {GATEWAY_CKPT}"); return None
    ckpt=torch.load(str(GATEWAY_CKPT),map_location=DEVICE,weights_only=False)
    organ_classes=ckpt.get("organ_classes",list(SPECIALISTS.keys()))
    subclass_map=ckpt.get("subclass_map",{
        org:info["class_names"] for org,info in SPECIALISTS.items() if info.get("class_names")})
    try:
        gw=HierarchicalMedViT(organ_classes,subclass_map).to(DEVICE)
        state=ckpt.get("model_state_dict",ckpt)
        state={k.replace("module.",""):v for k,v in state.items()}
        gw.load_state_dict(state,strict=False); gw.eval()
        for p in gw.parameters(): p.requires_grad_(False)
        print("  ✅ Gateway loaded + frozen")
        GATEWAY_MODEL=gw; return gw
    except Exception as e:
        print(f"  ⚠️  Gateway load error: {e}"); return None

def get_gateway_hint(gateway,imgs,organ):
    if gateway is None: return None
    try:
        with torch.no_grad():
            out=gateway(imgs); n_subs=len(gateway.subclass_map.get(organ,[]))
            if n_subs==0: return None
            raw=out["subclass_logits"][:,:n_subs]
            raw=raw.masked_fill(raw==float("-inf"),-1e4)
            hint=torch.softmax(raw.float(),dim=-1)
        return hint.detach()
    except Exception as e:
        print(f"  ⚠️  get_gateway_hint failed for {organ}: {e}"); return None

# ─── v16-NEW: gateway with top-2 fallback ────────────────────────────────────
def gateway_with_fallback(gateway, imgs, threshold=GATEWAY_CONF_THRESHOLD,
                          fallback_thresh=GATEWAY_FALLBACK_THRESH):
    """
    Returns (organ_list, routing_mode).
    routing_mode: 'CONFIDENT' | 'ENSEMBLE' | 'UNCERTAIN'
    For CONFIDENT: organ_list has 1 element.
    For ENSEMBLE : organ_list has 2 elements — run both, average softmax outputs.
    For UNCERTAIN: below threshold entirely — flag for pathologist.
    """
    if gateway is None:
        return ["unknown"], "NO_GATEWAY"
    try:
        with torch.no_grad():
            out = gateway(imgs)
            organ_probs = torch.softmax(out["organ_logits"], 1)[0]
            top2 = torch.topk(organ_probs, min(2, len(gateway.organ_classes)))
            conf1 = float(top2.values[0])
            organ1 = gateway.organ_classes[int(top2.indices[0])]
        if conf1 >= fallback_thresh:
            return [organ1], "CONFIDENT"
        elif conf1 >= threshold:
            organ2 = gateway.organ_classes[int(top2.indices[1])]
            return [organ1, organ2], "ENSEMBLE"
        else:
            return [organ1], "UNCERTAIN"
    except Exception as e:
        print(f"  ⚠️  gateway_with_fallback: {e}")
        return ["unknown"], "ERROR"

# ─── v16-ISS-8: quick val F1 check before activating FiLM ────────────────────
def _quick_eval_f1(model, val_loader, organ, hint_fn=None, max_batches=20):
    model.eval(); preds, labels_all = [], []
    with torch.no_grad():
        for i, (imgs, lbs) in enumerate(val_loader):
            if i >= max_batches: break
            imgs = imgs.to(DEVICE)
            hint = hint_fn(imgs) if hint_fn else None
            out  = _safe_fwd(model, imgs, hint, organ)
            preds.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(lbs.numpy())
    return f1_score(labels_all, preds, average="macro", zero_division=0)

print("\nLoading frozen gateway...")
gw = load_gateway()

print("\nRestoring FiLM models (with val-F1 gate)...")
for organ in list(SPECIALISTS.keys()):
    done_ckpt = SPEC_DIR / f"best_{organ}_film_v8.pth"
    if not done_ckpt.exists():
        SPECIALISTS[organ]["film_status"] = "ABSENT"
        continue
    try:
        saved     = torch.load(str(done_ckpt), map_location=DEVICE, weights_only=False)
        saved_f1  = saved.get("val_f1", 0.0)
        note      = saved.get("note","")
        if saved_f1 <= 0.0 or note == "baseline_copy":
            SPECIALISTS[organ]["film_status"] = "ABSENT"
            continue

        base_f1 = ORGAN_BASE_F1.get(organ, 0.0)

        # Frozen: base is near-perfect
        if base_f1 >= FILM_FREEZE_F1:
            SPECIALISTS[organ]["film_status"] = "FROZEN"
            print(f"  🔒 [{organ:<12}] FROZEN (base F1={base_f1:.4f})")
            continue

        bb_name  = saved.get("backbone_name", BACKBONE_CFG.get(organ,("convnext_small",0.25,False))[0])
        dropout  = BACKBONE_CFG.get(organ, ("convnext_small",0.25,False))[1]
        n_hint   = saved.get("n_hint_dims", len(gw.subclass_map.get(organ,[])) if gw else 0)
        n_cls    = len(SPECIALISTS[organ]["class_names"])
        if n_hint == 0:
            SPECIALISTS[organ]["film_status"] = "ABSENT"
            continue

        fm = FiLMConditionedSpecialist(bb_name, n_cls, n_hint_dims=n_hint,
                                        dropout=dropout, pretrained=False).to(DEVICE)
        fm.load_state_dict(saved["model_state_dict"]); fm.eval()

        # v16-ISS-8: quick val check — disable if FiLM hurts
        hint_fn = (lambda imgs, o=organ: get_gateway_hint(gw, imgs, o)) if gw else None
        try:
            heavy = AUGMENT_CFG.get(organ) == "heavy"
            _, vl_chk, _, _ = build_loaders(organ, batch_size=32, heavy=heavy)
            film_val_f1 = _quick_eval_f1(fm, vl_chk, organ, hint_fn)
            if film_val_f1 < base_f1 - 0.002:
                print(f"  ⚠️  [{organ:<12}] FiLM val_f1={film_val_f1:.4f} < base {base_f1:.4f} - 0.002 — DISABLED")
                SPECIALISTS[organ]["film_status"] = "DISABLED"
                continue
        except Exception as chk_e:
            print(f"  ⚠️  [{organ}] FiLM val check failed ({chk_e}) — loading anyway")

        SPECIALISTS[organ]["film_model"]  = fm
        SPECIALISTS[organ]["film_status"] = "ACTIVE"
        LOAD_REPORT[organ]["film_val_f1"] = saved_f1
        print(f"  ✅ [{organ:<12}] FiLM ACTIVE  film_val_f1={saved_f1:.4f}")
    except Exception as e:
        print(f"  ⚠️  [{organ}] FiLM load failed: {e}")
        SPECIALISTS[organ]["film_status"] = "ABSENT"

print("\n✅ Cell 5b complete")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 5c — FiLM Entropy-Gated Hint Injection
#
# PROBLEM: 4 organs DISABLED (skin/breast/gastric/ovarian) because
#   FiLM val_f1 dropped 15-20 points below base.
#   ROOT CAUSE: Gateway subclass hints for 7-8 class organs are nearly
#   uniform (high entropy). FiLM interprets flat hint → harmful feature shift.
#   skin (7 cls): gateway entropy ≈ log(7)=1.95 bits, max = 1.95 → always flat
#   gastric (8 cls): same problem
#
# FIX: Patch FiLMConditionedSpecialist.forward() to gate hint injection
#   by hint entropy. When entropy > 80% of theoretical maximum, skip FiLM
#   and return raw backbone features (identical to base model behaviour).
#   This means FiLM only activates when the gateway is actually informative.
#
# ADDITIONAL: Re-run Cell 5b's val-F1 check for the 4 disabled organs
#   using the gated forward — expected to re-enable skin/breast/gastric/ovarian
#
# EXPECTED:
#   skin:    DISABLED → ACTIVE  (FiLM only helps when gateway conf > 20%)
#   breast:  DISABLED → ACTIVE
#   gastric: DISABLED → ACTIVE
#   ovarian: DISABLED → ACTIVE
# ════════════════════════════════════════════════════════════════════════════════

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

print("═" * 68)
print("  CELL 5c — FiLM Entropy-Gated Hint Injection")
print("═" * 68)


# ════════════════════════════════════════════════════════════════════════════════
# PART 1 — Patch FiLMConditionedSpecialist.forward()
# ════════════════════════════════════════════════════════════════════════════════

# Configuration: per-organ gate threshold (fraction of max entropy)
# Higher = more permissive (FiLM activates even for uncertain hints)
# Lower  = stricter (FiLM only activates when gateway is very confident)
FILM_ENTROPY_GATE_THRESHOLD = {
    "skin":     0.70,   # 7 classes, max H=1.946 → gate at H < 1.362
    "gastric":  0.70,   # 8 classes, max H=2.079 → gate at H < 1.455
    "breast":   0.75,   # 2 classes, max H=0.693 → gate at H < 0.520
    "ovarian":  0.72,   # 5 classes, max H=1.609 → gate at H < 1.159
    "thyroid":  0.75,   # 2 classes
    "bladder":  0.75,   # 2 classes
    "oral":     0.75,   # 2 classes
    "default":  0.75,
}

# Hint temperature sharpening (applied before entropy check)
# WHY: Gateway logits tend to be overconfident-but-wrong for hard organs.
# Temperature > 1.0 softens hints slightly, making them more honest.
FILM_HINT_TEMPERATURE = {
    "skin":     1.2,
    "gastric":  1.3,
    "breast":   1.0,
    "ovarian":  1.1,
    "default":  1.0,
}


def _film_entropy_gate(hint_logits: torch.Tensor,
                        n_classes: int,
                        organ: str) -> torch.Tensor:
    """
    Computes per-sample gate values in [0, 1].
    gate = 1 → apply FiLM (hint is informative)
    gate = 0 → skip FiLM (hint is too uncertain)

    Uses soft gate (sigmoid of standardised entropy deviation) rather than
    hard threshold so gradient flows through during fine-tuning.
    """
    max_entropy = math.log(max(n_classes, 2))
    threshold   = FILM_ENTROPY_GATE_THRESHOLD.get(
        organ, FILM_ENTROPY_GATE_THRESHOLD["default"]) * max_entropy

    # Temperature scaling before entropy computation
    T = FILM_HINT_TEMPERATURE.get(organ, FILM_HINT_TEMPERATURE["default"])
    # (temperature applied below: on logits, or as p**(1/T) on probabilities)

    # V22-FIX: get_gateway_hint() already returns PROBABILITIES. Softmaxing them
    # again flattens every hint to ~uniform → gate ≈ 0.05 for all inputs, i.e.
    # FiLM was silently switched off (and its head fed features it never saw).
    _h = hint_logits.float()
    _is_prob = bool((_h >= 0).all() and torch.allclose(_h.sum(-1), torch.ones_like(_h.sum(-1)), atol=1e-3))
    probs   = _h.clamp_min(1e-8) ** (1.0 / T) if _is_prob else torch.softmax(_h, dim=-1)
    probs   = probs / probs.sum(-1, keepdim=True)                # (B, n_cls)
    entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=-1)  # (B,)

    # Soft gate: 1 when entropy << threshold, 0 when entropy >> threshold
    # Scale = 5.0 controls sharpness of transition
    gate = torch.sigmoid((threshold - entropy) * 5.0)  # (B,) in (0,1)
    return gate                                         # shape (B,)


# V22: FiLM checkpoints were trained WITHOUT this gate, so the gate changes the
# model that was validated. Default OFF → forward identical to training. Set True
# only as an ablation, and report it as such.
FILM_USE_ENTROPY_GATE = False
_ORIG_FILM_FWD = globals().get("_ORIG_FILM_FWD", FiLMConditionedSpecialist.forward)

def _patched_film_forward(self, x, hint_logits=None):
    if not FILM_USE_ENTROPY_GATE:
        return _ORIG_FILM_FWD(self, x, hint_logits)
    """
    Entropy-gated FiLM forward pass.
    Replaces FiLMConditionedSpecialist.forward() via monkey-patch.

    When hint entropy is high (gateway uncertain):
        output = head(backbone(x))           ← identical to base model
    When hint entropy is low (gateway confident):
        output = head(FiLM(backbone(x), hint))  ← FiLM activates

    Soft blending ensures gradient flows throughout.
    """
    if self.needs_resize:
        x = F.interpolate(x, (self.img_size, self.img_size),
                          mode="bilinear", align_corners=False)

    feat = self.backbone(x)  # (B, D)

    if hint_logits is None:
        # No hint provided — use raw features
        return self.head(feat)

    # Retrieve organ from context (stored as attribute by _patched_film_forward)
    organ = getattr(self, '_current_organ', 'default')
    n_cls = hint_logits.shape[-1]

    # Compute entropy gate (B,) → (B, 1) for broadcasting
    gate = _film_entropy_gate(hint_logits, n_cls, organ).unsqueeze(1)  # (B, 1)

    # FiLM-conditioned features
    film_feat = self.film(feat, hint_logits.float())    # (B, D)

    # Soft blend: high gate → use FiLM; low gate → use raw
    blended = gate * film_feat + (1. - gate) * feat     # (B, D)

    return self.head(blended)


# Apply patch to the class method
FiLMConditionedSpecialist.forward = _patched_film_forward
print("  ✅ FiLMConditionedSpecialist.forward patched with entropy gate")


# ════════════════════════════════════════════════════════════════════════════════
# PART 2 — Override smart_fwd to pass organ to FiLM model
# WHY: _patched_film_forward needs to know the organ to look up its gate
#      threshold, but the original API passes hint_logits only.
#      We store organ as an attribute on the model object before calling.
# ════════════════════════════════════════════════════════════════════════════════

def smart_fwd_v2(base_model, film_model, inp, hint, organ, class_names):
    """
    v2 of smart_fwd: sets _current_organ on film_model before forward.
    All other logic identical to original.
    """
    threshold = FILM_HURT_THRESHOLD_ORGAN.get(organ, FILM_HURT_THRESHOLD_DEFAULT)
    base_f1   = ORGAN_BASE_F1.get(organ, 0.0)
    film_frozen = base_f1 >= FILM_FREEZE_F1

    with torch.no_grad():
        base_logits = _safe_fwd(base_model, inp, None, organ)
        base_prbs   = torch.softmax(base_logits, 1)[0].cpu().numpy()
        base_pred   = int(base_prbs.argmax())
        base_conf   = float(base_prbs[base_pred])

        film_disabled = (film_model is None)
        film_status   = SPECIALISTS.get(organ, {}).get("film_status", "ABSENT")

        if film_disabled or film_frozen or film_status in ("FROZEN", "ABSENT"):
            return base_logits, base_prbs, base_pred, base_conf, False

        # Set organ context on film model for gate lookup
        film_model._current_organ = organ

        film_logits = _safe_fwd(film_model, inp, hint, organ)
        film_prbs   = torch.softmax(film_logits, 1)[0].cpu().numpy()
        film_pred   = int(film_prbs.argmax())
        film_conf   = float(film_prbs[film_pred])

        if film_pred != base_pred:
            base_ent = float(-np.sum(base_prbs * np.log(base_prbs + 1e-8)))
            film_ent = float(-np.sum(film_prbs * np.log(film_prbs + 1e-8)))
            if film_ent < base_ent:
                return film_logits, film_prbs, film_pred, film_conf, True
            return base_logits, base_prbs, base_pred, base_conf, False

        if film_conf - base_conf < threshold:
            return base_logits, base_prbs, base_pred, base_conf, False

        return film_logits, film_prbs, film_pred, film_conf, True


# Replace global smart_fwd with v2
smart_fwd = smart_fwd_v2
print("  ✅ smart_fwd replaced with smart_fwd_v2 (organ context passing)")


# ════════════════════════════════════════════════════════════════════════════════
# PART 3 — Re-check disabled FiLM organs with gated forward
# ════════════════════════════════════════════════════════════════════════════════

FILM_RECHECK_ORGANS = [
    org for org, info in SPECIALISTS.items()
    if info.get("film_status") == "DISABLED"
       and info.get("film_model") is not None
]

print(f"\n  Re-checking {len(FILM_RECHECK_ORGANS)} disabled FiLM organs "
      f"with entropy gate: {FILM_RECHECK_ORGANS}")

for organ in FILM_RECHECK_ORGANS:
    spec_info  = SPECIALISTS[organ]
    film_model = spec_info["film_model"]
    base_model = spec_info["model"]
    class_names = spec_info["class_names"]
    base_f1    = ORGAN_BASE_F1.get(organ, 0.)

    # Set organ context
    film_model._current_organ = organ

    try:
        heavy = AUGMENT_CFG.get(organ) == "heavy"
        _, vl_check, _, _ = build_loaders(organ, batch_size=32, heavy=heavy)

        # Quick eval with gated FiLM
        film_preds_r, labels_r = [], []
        film_model.eval()
        for imgs, lbs in vl_check:
            imgs = imgs.to(DEVICE)
            hint = get_gateway_hint(gw, imgs, organ) if gw else None
            film_model._current_organ = organ
            out  = _safe_fwd(film_model, imgs, hint, organ)
            film_preds_r.extend(out.argmax(1).cpu().numpy())
            labels_r.extend(lbs.numpy())

        from sklearn.metrics import f1_score as _f1_check
        gated_f1 = float(_f1_check(labels_r, film_preds_r,
                                    average="macro", zero_division=0))
        delta    = gated_f1 - base_f1
        improved = gated_f1 >= (base_f1 - 0.002)

        if improved:
            SPECIALISTS[organ]["film_status"] = "ACTIVE"
            print(f"  ✅ [{organ:<12}] DISABLED → ACTIVE  "
                  f"gated_f1={gated_f1:.4f}  base={base_f1:.4f}  "
                  f"Δ={delta:+.4f}")
        else:
            print(f"  ⚠️  [{organ:<12}] still hurts  "
                  f"gated_f1={gated_f1:.4f}  base={base_f1:.4f}  "
                  f"Δ={delta:+.4f}")

    except Exception as e:
        print(f"  ❌ [{organ}] recheck failed: {e}")


# ════════════════════════════════════════════════════════════════════════════════
# PART 4 — Smoke tests
# ════════════════════════════════════════════════════════════════════════════════

print("\n  Smoke tests...")

# Test gate with uniform vs peaked hint
try:
    _uniform_hint = torch.ones(2, 7) / 7.       # entropy = log(7) ≈ 1.946
    _peaked_hint  = torch.zeros(2, 7)
    _peaked_hint[:, 2] = 8.                      # almost all mass on class 2
    _peaked_hint  = F.softmax(_peaked_hint, -1)  # probabilities, like get_gateway_hint()

    _gate_unif = _film_entropy_gate(_uniform_hint, 7, "skin")
    _gate_peak = _film_entropy_gate(_peaked_hint,  7, "skin")

    print(f"  Uniform hint gate  : {_gate_unif.mean().item():.4f}  (expect ≈ 0.0)")
    print(f"  Peaked hint gate   : {_gate_peak.mean().item():.4f}  (expect ≈ 1.0)")
    assert _gate_unif.mean().item() < 0.2, "Uniform hint should be gated out"
    assert _gate_peak.mean().item() > 0.8, "Peaked hint should pass through"
    print("  ✅ Entropy gate smoke test passed")
except Exception as _e:
    print(f"  ❌ Entropy gate FAILED: {_e}")

# Test FiLM forward with mock model
try:
    _mock_film = FiLMConditionedSpecialist(
        "convnext_tiny", 2, n_hint_dims=2,
        dropout=0.1, pretrained=False).to(DEVICE)
    _mock_film._current_organ = "breast"
    _mock_x    = torch.randn(2, 3, 224, 224).to(DEVICE)
    _mock_hint = torch.softmax(torch.randn(2, 2), dim=-1).to(DEVICE)
    with torch.no_grad():
        _out = _mock_film(_mock_x, hint_logits=_mock_hint)
    assert _out.shape == (2, 2)
    print(f"  ✅ Patched FiLM forward smoke test passed  shape={_out.shape}")
except Exception as _e:
    print(f"  ❌ Patched FiLM forward FAILED: {_e}")

# Count now-active organs
n_active = sum(1 for info in SPECIALISTS.values()
               if info.get("film_status") == "ACTIVE")
n_frozen = sum(1 for info in SPECIALISTS.values()
               if info.get("film_status") == "FROZEN")
n_disabled = sum(1 for info in SPECIALISTS.values()
                 if info.get("film_status") == "DISABLED")
n_absent = sum(1 for info in SPECIALISTS.values()
               if info.get("film_status") == "ABSENT")

print(f"\n  FiLM status after Cell 5c:")
print(f"    ACTIVE   : {n_active}")
print(f"    FROZEN   : {n_frozen}")
print(f"    DISABLED : {n_disabled}")
print(f"    ABSENT   : {n_absent}")

print(f"\n  FILM_USE_ENTROPY_GATE = {FILM_USE_ENTROPY_GATE}  (False = trained FiLM forward, recommended)")
print("\n✅ Cell 5c complete")
print("   FiLMConditionedSpecialist.forward  patched with entropy gate")
print("   smart_fwd                          replaced with smart_fwd_v2")
print("   FILM_ENTROPY_GATE_THRESHOLD        per-organ gate config")
print("   FILM_HINT_TEMPERATURE              per-organ hint sharpening")
print("   Disabled organs re-checked         expect skin/breast/gastric/ovarian → ACTIVE")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 6 — XAI Engine v18 ULTIMATE
#
# Base: v17 ULTIMATE (all sections 0–13 preserved, none removed)
# === UPGRADE v18 — Advanced XAI === new sections added AFTER Section 13
#
# CHANGES SUMMARY:
#   v18-CAM-1  AdvancedCAMEnsemble: confidence-weighted fusion of up to 4 CAMs
#   v18-CAM-2  CAMPostProcessor: bilateral filter + adaptive threshold + sharpen
#   v18-CAM-3  CAMAgreementOptimizer: dynamic selection based on spatial entropy
#   v18-IG-1   Improved IG baseline: mean-background instead of pure zeros
#   v18-IG-2   IG skip threshold lowered for high-T* organs via per-organ config
#   v18-SHAP-1 Contrastive SHAP background: class-targeted background masks
#   v18-SEG-1  HoVerNet confidence gate: only flag seg_net_used when neoplastic
#              confidence > threshold (fixes 0% seg_net_used bug)
#   v18-AGREE  Post-processing agreement: Spearman on post-processed CAMs
#              (increases correlation from ~0.073 to expected ~0.35+)
#
# All v17 functions unchanged. run_full_xai() gets minimal additions with
# fallback to v17 behavior on any exception.
# ════════════════════════════════════════════════════════════════════════════════

import math, gc, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.stats               import spearmanr
from numpy.polynomial.legendre import leggauss
from skimage.filters           import threshold_otsu
from skimage.measure           import label as sk_label, regionprops
from skimage.morphology        import binary_closing, binary_opening, disk
from scipy.ndimage             import binary_dilation, binary_fill_holes
from PIL                       import Image
import shap

warnings.filterwarnings("ignore")

if not hasattr(np, 'trapezoid'):
    np.trapezoid = np.trapz

print("═" * 68)
print("  CELL 6 — XAI Engine v18 ULTIMATE")
print("  v17 ULTIMATE base + v18 Advanced CAM + IG + SHAP + Seg fixes")
print("═" * 68)
print(f"  NumPy {np.__version__} | trapezoid: "
      f"{'native' if hasattr(np, 'trapezoid') else 'shimmed via trapz'}")


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 0 — Constants (v17 preserved + v18 additions)
# ════════════════════════════════════════════════════════════════════════════════

HOOK_BROKEN_ORGANS = {
    "cervical",
    "kidney",
    "thyroid",
}

LAYER_CAM_ORGANS = {"skin", "melanoma", "lymphoma", "blood"}

IG_SKIP_T_THRESHOLD = 0.15
IG_GL_STEPS         = 64

CAM_DISAGREE_THRESH = 0.25

GRADCAM_SMOOTH_CFG = {
    "skin":     (8,  0.05),
    "melanoma": (8,  0.05),
    "brain_2d": (12, 0.08),
    "default":  (20, 0.10),
}

SHAP_N_BACKGROUND = 16
SHAP_SEG_H        = 14
SHAP_SEG_W        = 14
SHAP_N_SEGS       = SHAP_SEG_H * SHAP_SEG_W
SHAP_NSAMPLES     = max(2048, SHAP_N_SEGS * 12)

SHAP_SEG_H_VIT    = 14
SHAP_SEG_W_VIT    = 14
SHAP_N_SEGS_VIT   = SHAP_SEG_H_VIT * SHAP_SEG_W_VIT
SHAP_NSAMPLES_VIT = 600

assert 224 % SHAP_SEG_H     == 0
assert 224 % SHAP_SEG_H_VIT == 0

# === UPGRADE v18 === New constants
# WHY v18-IG-2: Organs with large T* have very flat logit surfaces — IG on
# zero baseline doesn't integrate over any meaningful variation. Per-organ
# skip thresholds are set higher for high-T* organs.
IG_SKIP_T_THRESHOLD_PER_ORGAN = {
    "kidney":     6.0,   # T*=4.33 → still skip
    "thyroid":    2.5,   # T*=1.95 → relax slightly
    "pancreatic": 2.0,   # T*=1.49 → allow IG
    "cervical":   2.5,   # T*=1.76 → relax
    "default":    IG_SKIP_T_THRESHOLD,
}

# WHY v18-CAM: These organs get AdvancedCAMEnsemble for better agreement
ADVANCED_CAM_ORGANS = {
    "skin", "cervical", "gastric", "lymphoma", "ovarian",
    "kidney", "pancreatic", "breast", "bladder", "thyroid",
}

# WHY v18-SEG: Neoplastic confidence threshold for HoVerNet
# Fixes 0% seg_net_used: random images → all neoplastic (untrained classifier)
HOVERNET_NEO_CONF_THRESHOLD = 0.55   # only count nucleus as neoplastic if conf > this

# WHY v18-CAM-3: spatial entropy threshold for CAM selector
CAM_SPATIAL_ENTROPY_HIGH = 4.5   # above this = diffuse CAM → prefer focused one

# Bilateral filter params per organ
# WHY: Skin/cervical have fine texture detail that needs edge preservation
BILATERAL_PARAMS = {
    "skin":     {"d": 7,  "sigmaColor": 35, "sigmaSpace": 35},
    "cervical": {"d": 5,  "sigmaColor": 30, "sigmaSpace": 30},
    "gastric":  {"d": 9,  "sigmaColor": 45, "sigmaSpace": 45},
    "default":  {"d": 9,  "sigmaColor": 50, "sigmaSpace": 50},
}

print(f"  HOOK_BROKEN_ORGANS   : {HOOK_BROKEN_ORGANS}")
print(f"  ADVANCED_CAM_ORGANS  : {ADVANCED_CAM_ORGANS}  (v18)")
print(f"  HOVERNET_NEO_CONF    : {HOVERNET_NEO_CONF_THRESHOLD}  (v18 seg fix)")
print(f"  IG per-organ thresholds : {IG_SKIP_T_THRESHOLD_PER_ORGAN}")


# ════════════════════════════════════════════════════════════════════════════════
# SECTIONS 1–13: v17 ULTIMATE (ALL PRESERVED, NOT MODIFIED)
# ════════════════════════════════════════════════════════════════════════════════

# [All original v17 functions from Sections 1-13 go here unchanged]
# For brevity in this diff, they are denoted with their function names.
# The complete implementations are IDENTICAL to Cell 6 v17 ULTIMATE above.
# In your notebook, paste the full v17 Sections 1-13 here, then append
# the v18 sections below.

# --- PASTE v17 SECTIONS 1-13 HERE (unchanged) ---
# Section 1:  _norm01, _ensure_2d, _find_layer, _vit_to_spatial,
#             _tissue_mask, _blur_tensor, _cam_agreement
# Section 2:  _remove_all_hooks, _disable_inplace, _restore_inplace
# Section 3:  _raw_model_fwd
# Section 4:  gradcam_plus_plus, layer_cam_smooth, eigen_cam,
#             attention_gradcam_vit, attention_rollout, rise_cam,
#             score_cam, multi_scale_cam
# Section 5:  integrated_gradients
# Section 6:  _parse_shap_values, shap_attribution_gpu
# Section 7:  _apply_crf, _morphological_cleanup, _keep_largest_component,
#             _compute_seg_metrics, generate_segmentation_mask
# Section 8:  mc_dropout_uncertainty, auc_deletion_v17, auc_deletion (alias),
#             auc_insertion, tta_confidence
# Section 9:  compute_trust_score, get_concept_label, _compute_cam_seg_iou
# Section 10: _overlay, _overlay_seg, _build_seg_result_from_row
# Section 11: run_full_xai (v17 version, will be overridden by v18 below)
# Section 12: _empty_xai
# Section 13: Smoke tests

# NOTE: In your actual notebook, copy the ENTIRE v17 Cell 6 content above,
# then add the v18 sections below. The v18 run_full_xai OVERRIDES the v17 one.

# ════════════════════════════════════════════════════════════════════════════════
# CELL 6 — XAI Engine v17 ULTIMATE
#
# FULL MERGE: v14 clean decomposition + v17 all fixes + v7 cherry-picks
#
# ── From v14 (cherry-picked for robustness) ──────────────────────────────────
#   V14-A  _parse_shap_values()      — dedicated 7-branch SHAP shape parser
#   V14-B  _apply_crf()              — dedicated CRF fn, C-contiguous, (mask,bool)
#   V14-C  _morphological_cleanup()  — dedicated fn, explicit per-organ r_close/r_open
#   V14-D  _keep_largest_component() — dedicated fn (v17 inlined this)
#   V14-E  _compute_seg_metrics()    — dedicated fn returning full metrics dict
#   V14-F  auc_deletion alias        — auc_deletion = auc_deletion_v17 for compat
#
# ── From v7 (cherry-picked for quality) ──────────────────────────────────────
#   V7-A  EigenCAM gram-matrix eigh (primary) + SVD fallback
#   V7-B  LayerCAM F.relu(A * F.relu(g)) element-wise formula
#   V7-C  MC-dropout saves/restores model.training state
#   V7-D  3-map blend for LAYER_CAM_ORGANS (skin, melanoma, lymphoma, blood)
#
# ── From v17 UPGRADED (all retained, none regressed) ─────────────────────────
#   BUG-1  IG clone().detach() per GL node ✅
#   BUG-2  IG zero baseline — completeness axiom satisfied ✅
#   FIX-A  HOOK_BROKEN_ORGANS → RISE-only, no IG ✅
#   FIX-B  _remove_all_hooks() before every backward ✅
#   FIX-C  _raw_model_fwd() in IG — T* bypassed ✅
#   FIX-D  generate_segmentation_mask → hovernet_predict ✅
#   FIX-E  KernelSHAP-only, always returns (ndarray, str) ✅
#   FIX-G  _ensure_2d() guards every output map ✅
#   FIX-H  np.trapezoid NumPy 2.0 compat ✅
#   FIX-I  _overlay_seg accepts seg_result dict ✅
#   FIX-J  SHAP _rebuild pads/crops to exactly 224×224 ✅
#   ISS-9  Spearman rank-corr for CAM agreement ✅
#   UPG-1  ViT attention block index from XAI_LAYERS ✅
#   UPG-2  ViT SHAP 14×14 grid (exact 224×224) ✅
#   UPG-3  ViT SHAP 600 samples ✅
#   UPG-4  flagged = 95th-pctile entropy only ✅
#   UPG-5  per-organ GradCAM++ smooth config ✅
#   UPG-6  skin MS-CAM stage.3 dominant weights ✅
#   UPG-7  benign seg always shows explicit message ✅
#   UPG-8  pred_class_name always from argmax ✅
#   UPG-9  CAM_DISAGREE_THRESH = 0.25 ✅
# ════════════════════════════════════════════════════════════════════════════════

import math, gc, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.stats               import spearmanr
from numpy.polynomial.legendre import leggauss
from skimage.filters           import threshold_otsu
from skimage.measure           import label as sk_label, regionprops
from skimage.morphology        import binary_closing, binary_opening, disk
from scipy.ndimage             import binary_dilation, binary_fill_holes
from PIL                       import Image
import shap

warnings.filterwarnings("ignore")

# FIX-H: NumPy 2.0 compatibility shim
if not hasattr(np, 'trapezoid'):
    np.trapezoid = np.trapz

print("═" * 68)
print("  CELL 6 — XAI Engine v17 ULTIMATE")
print("  v14 clean decomposition + v7 CAM quality + v17 all fixes")
print("═" * 68)
print(f"  NumPy {np.__version__} | trapezoid: "
      f"{'native' if hasattr(np, 'trapezoid') else 'shimmed via trapz'}")


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 0 — Constants
# ════════════════════════════════════════════════════════════════════════════════

# FIX-A: inplace-op backbones → RISE-only, no GradCAM backward
HOOK_BROKEN_ORGANS = {
    "cervical",   # EfficientViT-M5 — inplace SqueezeExcite
    "kidney",     # EfficientNet-B3 — inplace silu_
    "thyroid",    # EfficientNet-B2 — inplace silu_
}

# V7-D: organs that get 3-map blend (LayerCAM + GradCAM++ + EigenCAM)
LAYER_CAM_ORGANS = {"skin", "melanoma", "lymphoma", "blood"}

# ISS-7 / UPG-5
IG_SKIP_T_THRESHOLD = 0.15
IG_GL_STEPS         = 64

# UPG-9
CAM_DISAGREE_THRESH = 0.25

# Per-organ GradCAM++ smoothing (UPG-5)
GRADCAM_SMOOTH_CFG = {
    "skin":     (8,  0.05),
    "melanoma": (8,  0.05),
    "brain_2d": (12, 0.08),
    "default":  (20, 0.10),
}

# KernelSHAP — CNN grid (14×14=196, 224÷14=16px cells → exact 224×224)
SHAP_N_BACKGROUND = 16
SHAP_SEG_H        = 14
SHAP_SEG_W        = 14
SHAP_N_SEGS       = SHAP_SEG_H * SHAP_SEG_W          # 196
SHAP_NSAMPLES     = max(2048, SHAP_N_SEGS * 12)       # 2352

# KernelSHAP — ViT grid (UPG-2: 14×14 same → exact 224×224)
SHAP_SEG_H_VIT    = 14
SHAP_SEG_W_VIT    = 14
SHAP_N_SEGS_VIT   = SHAP_SEG_H_VIT * SHAP_SEG_W_VIT  # 196
SHAP_NSAMPLES_VIT = 600                                # UPG-3

# Grid divisibility assertion — catches any future accidental change
assert 224 % SHAP_SEG_H     == 0, "CNN SHAP grid does not divide 224 evenly"
assert 224 % SHAP_SEG_H_VIT == 0, "ViT SHAP grid does not divide 224 evenly"

print(f"  HOOK_BROKEN_ORGANS  : {HOOK_BROKEN_ORGANS}")
print(f"  LAYER_CAM_ORGANS    : {LAYER_CAM_ORGANS}  (V7-D 3-map blend)")
print(f"  CAM_DISAGREE_THRESH : {CAM_DISAGREE_THRESH}  (UPG-9)")
print(f"  IG_SKIP_T_THRESHOLD : {IG_SKIP_T_THRESHOLD}")
print(f"  SHAP CNN  grid      : {SHAP_SEG_H}×{SHAP_SEG_W}={SHAP_N_SEGS} "
      f"cell={224//SHAP_SEG_H}px @ {SHAP_NSAMPLES} samples")
print(f"  SHAP ViT  grid      : {SHAP_SEG_H_VIT}×{SHAP_SEG_W_VIT}={SHAP_N_SEGS_VIT} "
      f"cell={224//SHAP_SEG_H_VIT}px @ {SHAP_NSAMPLES_VIT} samples")
print(f"  Grid divisibility   : ✅ verified")


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 1 — Shared utilities
# ════════════════════════════════════════════════════════════════════════════════

def _norm01(arr: np.ndarray) -> np.ndarray:
    mn, mx = float(arr.min()), float(arr.max())
    if mx - mn < 1e-8:
        return np.zeros_like(arr, dtype=np.float32)
    return ((arr - mn) / (mx - mn)).astype(np.float32)


def _ensure_2d(v, shape=(224, 224)) -> np.ndarray:
    """FIX-G: unwrap tuples, enforce 2-D float32."""
    if isinstance(v, tuple):
        v = v[0] if len(v) > 0 else None
    if v is None or not isinstance(v, np.ndarray) or v.ndim != 2:
        return np.zeros(shape, dtype=np.float32)
    return v.astype(np.float32)


def _find_layer(model: nn.Module, target_name: str):
    for name, mod in model.named_modules():
        if name == target_name:
            return mod
    fb = None
    for _, mod in model.named_modules():
        if isinstance(mod, nn.Conv2d) and mod.kernel_size not in ((1, 1), (1,)):
            fb = mod
    return fb


def _vit_to_spatial(tensor: torch.Tensor):
    if tensor.ndim == 2: tensor = tensor.unsqueeze(0)
    if tensor.ndim != 3: return None
    B, N, C = tensor.shape
    patch = tensor[:, 1:, :]
    n = patch.shape[1]; h = w = int(math.sqrt(n))
    if h * w != n: return None
    return patch.permute(0, 2, 1).reshape(B, C, h, w)


def _tissue_mask(img_np_hwc: np.ndarray) -> np.ndarray:
    """V14-FIX-9: handles near-all-tissue images correctly."""
    gray = (0.299 * img_np_hwc[:, :, 0] +
            0.587 * img_np_hwc[:, :, 1] +
            0.114 * img_np_hwc[:, :, 2]).astype(np.float32)
    # If <5% near-white pixels → mostly tissue → return full mask
    if (gray > 0.85).mean() < 0.05:
        return np.ones((224, 224), dtype=np.float32)
    try:
        t    = threshold_otsu(gray)
        mask = (gray < t).astype(np.float32)
    except Exception:
        mask = np.ones((224, 224), dtype=np.float32)
    return binary_dilation(mask.astype(bool), iterations=3).astype(np.float32)


def _blur_tensor(img_t: torch.Tensor, sigma: float = 15) -> torch.Tensor:
    import torchvision.transforms.functional as TF
    ks = int(sigma * 3) | 1
    bl = TF.gaussian_blur(img_t[0].cpu(),
                          kernel_size=[ks, ks], sigma=[sigma, sigma])
    return bl.unsqueeze(0)


def _cam_agreement(cam1: np.ndarray, cam2: np.ndarray) -> float:
    """ISS-9: Spearman rank-correlation (replaces SSIM from v14)."""
    f1 = cam1.flatten().astype(np.float64)
    f2 = cam2.flatten().astype(np.float64)
    if f1.std() < 1e-8 or f2.std() < 1e-8:
        return 0.0
    corr, _ = spearmanr(f1, f2)
    return float(max(0.0, corr))


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 2 — Hook management (FIX-B)
# ════════════════════════════════════════════════════════════════════════════════

def _remove_all_hooks(model: nn.Module) -> None:
    """FIX-B: sweep every hook before any backward — prevents stale-hook crashes."""
    for m in model.modules():
        for attr in ('_forward_hooks', '_backward_hooks',
                     '_forward_pre_hooks', '_full_backward_hooks'):
            if hasattr(m, attr):
                try: getattr(m, attr).clear()
                except Exception: pass


def _disable_inplace(model: nn.Module) -> list:
    saved = []
    for m in model.modules():
        if hasattr(m, 'inplace'):
            saved.append((m, m.inplace)); m.inplace = False
    return saved


def _restore_inplace(saved: list) -> None:
    for m, val in saved:
        m.inplace = val


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 3 — Forward pass wrappers
# ════════════════════════════════════════════════════════════════════════════════

def _raw_model_fwd(model: nn.Module, inp: torch.Tensor,
                   hint=None) -> torch.Tensor:
    """FIX-C: no T* scaling — used ONLY inside integrated_gradients."""
    with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
        logits = (model(inp, hint_logits=hint)
                  if hint is not None and hasattr(model, 'film')
                  else model(inp))
    return torch.clamp(
        torch.nan_to_num(logits.float(), nan=0., posinf=30., neginf=-30.),
        -30., 30.)


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 4 — CAM methods
# ════════════════════════════════════════════════════════════════════════════════

def gradcam_plus_plus(model, img_t, target_layer_name, pred_class_idx,
                      is_vit=False, smooth_n=20, smooth_sigma=0.10,
                      hint=None, organ=None) -> np.ndarray:
    """GradCAM++ with SmoothGrad. UPG-5: per-organ smooth config."""
    cfg_key      = organ if organ in GRADCAM_SMOOTH_CFG else "default"
    smooth_n, smooth_sigma = GRADCAM_SMOOTH_CFG[cfg_key]

    model.eval()
    target_layer = _find_layer(model, target_layer_name)
    if target_layer is None:
        return np.zeros((224, 224), dtype=np.float32)

    cam_accum = np.zeros((224, 224), dtype=np.float32)

    for _ in range(smooth_n):
        features, grads = [], []

        def _fwd(m, i, o):
            out = o.detach().clone()
            if is_vit: out = _vit_to_spatial(out)
            if out is not None and out.ndim == 4: features.append(out)

        def _bwd(m, gi, go):
            g = go[0]
            if g is None: return
            g = g.contiguous().clone()
            if is_vit: g = _vit_to_spatial(g)
            if g is not None and g.ndim == 4: grads.append(g)

        _remove_all_hooks(model)
        hf = target_layer.register_forward_hook(_fwd)
        hb = target_layer.register_full_backward_hook(_bwd)
        saved = _disable_inplace(model)
        try:
            noise = torch.randn_like(img_t) * smooth_sigma
            inp   = (img_t + noise).float().to(DEVICE)
            inp.requires_grad_(True)
            with torch.enable_grad():
                out   = _safe_fwd(model, inp, hint, organ)
                score = out[0, pred_class_idx] * 100.
                model.zero_grad(); score.backward()
        except Exception:
            continue
        finally:
            _restore_inplace(saved)
            for h in (hf, hb):
                try: h.remove()
                except Exception: pass

        if not features or not grads: continue
        A = features[0]; g = grads[0]
        if A.ndim == 4: A = A[0]
        if g.ndim == 4: g = g[0]
        if A.ndim != 3: continue

        g2      = g.pow(2); g3 = g.pow(3)
        sum_Ag3 = (A * g3).sum(dim=(1, 2), keepdim=True)
        alpha   = g2 / (2. * g2 + sum_Ag3 + 1e-8)
        weights = (alpha * F.relu(g)).sum(dim=(1, 2))
        cam_raw = F.relu((weights[:, None, None] * A).sum(0))
        cam_up  = F.interpolate(cam_raw.unsqueeze(0).unsqueeze(0).float(),
                                (224, 224), mode="bilinear", align_corners=False)
        cam_accum += cam_up[0, 0].detach().cpu().numpy().astype(np.float32)

    return (_norm01(cam_accum / smooth_n) if cam_accum.max() > 0
            else np.zeros((224, 224), dtype=np.float32))


def layer_cam_smooth(model, img_t, target_layer_name, pred_class_idx,
                     is_vit=False, hint=None, organ=None) -> np.ndarray:
    """V7-B: element-wise LayerCAM F.relu(A * F.relu(g)) — preserves fine local structure."""
    cfg_key      = organ if organ in GRADCAM_SMOOTH_CFG else "default"
    _n, _s       = GRADCAM_SMOOTH_CFG[cfg_key]
    smooth_n     = max(8, _n // 2)
    smooth_sigma = _s

    model.eval()
    target_layer = _find_layer(model, target_layer_name)
    if target_layer is None:
        return np.zeros((224, 224), dtype=np.float32)

    cam_accum = np.zeros((224, 224), dtype=np.float32)

    for _ in range(smooth_n):
        features, grads = [], []

        def _fwd(m, i, o):
            out = o.detach().clone()
            if is_vit: out = _vit_to_spatial(out)
            if out is not None and out.ndim == 4: features.append(out)

        def _bwd(m, gi, go):
            g = go[0]
            if g is None: return
            g = g.contiguous().clone()
            if is_vit: g = _vit_to_spatial(g)
            if g is not None and g.ndim == 4: grads.append(g)

        _remove_all_hooks(model)
        hf = target_layer.register_forward_hook(_fwd)
        hb = target_layer.register_full_backward_hook(_bwd)
        saved = _disable_inplace(model)
        try:
            noise = torch.randn_like(img_t) * smooth_sigma
            inp   = (img_t + noise).float().to(DEVICE)
            inp.requires_grad_(True)
            with torch.enable_grad():
                out   = _safe_fwd(model, inp, hint, organ)
                score = out[0, pred_class_idx] * 100.
                model.zero_grad(); score.backward()
        except Exception:
            continue
        finally:
            _restore_inplace(saved)
            for h in (hf, hb):
                try: h.remove()
                except Exception: pass

        if not features or not grads: continue
        A = features[0]; g = grads[0]
        if A.ndim == 4: A = A[0]
        if g.ndim == 4: g = g[0]
        if A.ndim != 3: continue

        # V7-B: element-wise formula (not global alpha like GradCAM++)
        cam_raw = F.relu(A * F.relu(g)).sum(0)
        cam_up  = F.interpolate(cam_raw.unsqueeze(0).unsqueeze(0).float(),
                                (224, 224), mode="bilinear", align_corners=False)
        cam_accum += cam_up[0, 0].detach().cpu().numpy().astype(np.float32)

    return (_norm01(cam_accum / smooth_n) if cam_accum.max() > 0
            else np.zeros((224, 224), dtype=np.float32))


def eigen_cam(model, img_t, target_layer_name,
              is_vit=False, hint=None, organ=None) -> np.ndarray:
    """V7-A: gram-matrix eigh (primary) + SVD fallback. More stable for deep layers."""
    model.eval()
    target_layer = _find_layer(model, target_layer_name)
    if target_layer is None:
        return np.zeros((224, 224), dtype=np.float32)
    features = []

    def _fwd(m, i, o):
        out = o.detach().clone()
        if is_vit: out = _vit_to_spatial(out)
        if out is not None and out.ndim == 4: features.append(out)

    hf = target_layer.register_forward_hook(_fwd)
    with torch.no_grad():
        _safe_fwd(model, img_t.float().to(DEVICE), hint, organ)
    hf.remove()

    if not features:
        return np.zeros((224, 224), dtype=np.float32)
    A = features[0]
    if A.ndim == 4: A = A[0]
    if A.ndim != 3:
        return np.zeros((224, 224), dtype=np.float32)

    C, H, W  = A.shape
    A_flat   = A.reshape(C, H * W).float().cpu()   # (C, H*W)
    cam      = None

    # V7-A primary: gram-matrix eigh — stable when C ≫ H*W
    try:
        gram     = A_flat @ A_flat.T               # (C, C)
        _, evecs = torch.linalg.eigh(gram)         # ascending eigenvalues
        v1       = evecs[:, -1]                    # principal eigenvector
        cam_flat = (v1.unsqueeze(0) @ A_flat).squeeze(0).numpy()
        if (cam_flat < 0).sum() > (cam_flat > 0).sum():
            cam_flat = -cam_flat
        cam = np.maximum(cam_flat, 0).reshape(H, W).astype(np.float32)
    except Exception:
        pass

    # V7-A fallback: SVD on transposed F_mat
    if cam is None:
        try:
            F_mat = A_flat.T                       # (H*W, C)
            F_mat = F_mat - F_mat.mean(dim=0, keepdim=True)
            _, _, Vt = torch.linalg.svd(F_mat, full_matrices=False)
            v1 = Vt[0]
            cam_flat = (F_mat @ v1).numpy()
            if (cam_flat < 0).sum() > (cam_flat > 0).sum():
                cam_flat = -cam_flat
            cam = np.maximum(cam_flat, 0).reshape(H, W).astype(np.float32)
        except Exception:
            cam = np.zeros((H, W), dtype=np.float32)

    cam_up = F.interpolate(
        torch.from_numpy(cam).unsqueeze(0).unsqueeze(0),
        (224, 224), mode="bilinear", align_corners=False)
    return _norm01(cam_up[0, 0].numpy().astype(np.float32))


def attention_gradcam_vit(model, img_t, pred_class_idx,
                           hint=None, organ=None) -> np.ndarray:
    """Gradient-weighted attention for ViT. UPG-1: block index from XAI_LAYERS."""
    model.eval()
    attn_maps, attn_grads, hooks = [], [], []

    for _, mod in model.named_modules():
        if hasattr(mod, "attn_drop"):
            hooks.append(mod.attn_drop.register_forward_hook(
                lambda m, i, o: attn_maps.append(o.detach().clone())))
            hooks.append(mod.attn_drop.register_full_backward_hook(
                lambda m, gi, go:
                attn_grads.append(go[0].detach().clone())
                if go[0] is not None else None))

    saved = _disable_inplace(model)
    model.zero_grad()
    try:
        with torch.enable_grad():
            inp = img_t.float().to(DEVICE).clone().detach().requires_grad_(True)
            out = _safe_fwd(model, inp, hint, organ)
            model.zero_grad(); out[0, pred_class_idx].backward()
    finally:
        _restore_inplace(saved)
        for h in hooks:
            try: h.remove()
            except Exception: pass

    if not attn_maps or not attn_grads:
        return np.zeros((224, 224), dtype=np.float32)

    # UPG-1: resolve block index from XAI_LAYERS
    target_idx = -1
    tgt_name   = XAI_LAYERS.get(organ, "")
    if "blocks." in tgt_name:
        try:
            bnum = int(tgt_name.split("blocks.")[1].split(".")[0])
            if bnum < len(attn_maps):
                target_idx = bnum
        except (ValueError, IndexError):
            pass

    attn = attn_maps[target_idx]
    grad = (attn_grads[target_idx]
            if abs(target_idx) < len(attn_grads) else attn_grads[-1])

    if attn.ndim != 4:
        return np.zeros((224, 224), dtype=np.float32)
    weighted = ((grad.abs() * attn).mean(dim=1)
                if grad.shape == attn.shape else attn.mean(dim=1))
    tok_imp  = weighted[0].mean(dim=0)[1:]
    n = tok_imp.shape[0]; h = w = int(math.sqrt(n))
    if h * w != n:
        return np.zeros((224, 224), dtype=np.float32)
    cam = F.interpolate(
        tok_imp.reshape(h, w).cpu().float().unsqueeze(0).unsqueeze(0),
        (224, 224), mode="bilinear", align_corners=False)
    return _norm01(cam[0, 0].numpy().astype(np.float32))


def attention_rollout(model, img_t, is_vit=True,
                       hint=None, organ=None) -> np.ndarray:
    """Pure rollout — secondary map for Spearman agreement on ViT."""
    if not is_vit:
        return np.zeros((224, 224), dtype=np.float32)
    model.eval(); attn_maps = []; hooks = []
    for _, mod in model.named_modules():
        if hasattr(mod, "attn_drop"):
            hooks.append(mod.attn_drop.register_forward_hook(
                lambda m, i, o: attn_maps.append(o.detach().contiguous().clone())))
    with torch.no_grad():
        _safe_fwd(model, img_t.float().to(DEVICE), hint, organ)
    for h in hooks:
        try: h.remove()
        except Exception: pass
    if not attn_maps:
        return np.zeros((224, 224), dtype=np.float32)
    rollout = None
    for attn in attn_maps:
        if attn.ndim != 4: continue
        attn_avg = attn.mean(dim=1)
        eye = torch.eye(attn_avg.shape[-1], device=attn_avg.device).unsqueeze(0)
        A_L = 0.5 * attn_avg + 0.5 * eye
        A_L = A_L / (A_L.sum(dim=-1, keepdim=True) + 1e-8)
        rollout = A_L if rollout is None else torch.bmm(A_L, rollout)
    if rollout is None:
        return np.zeros((224, 224), dtype=np.float32)
    tok_imp = rollout.mean(dim=1).mean(dim=0)[1:]
    n = tok_imp.shape[0]; h = w = int(math.sqrt(n))
    if h * w != n:
        return np.zeros((224, 224), dtype=np.float32)
    cam = F.interpolate(
        tok_imp.reshape(h, w).cpu().float().unsqueeze(0).unsqueeze(0),
        (224, 224), mode="bilinear", align_corners=False)
    return _norm01(cam[0, 0].numpy().astype(np.float32))


def rise_cam(model, img_t, pred_class_idx,
             n_masks=500, s=8, p=0.5,
             hint=None, organ=None) -> np.ndarray:
    """RISE: Randomised Input Sampling for Explanation."""
    model.eval()
    H, W    = 224, 224
    cpu_img = img_t[0].detach().cpu()
    cam     = np.zeros((H, W), dtype=np.float32)
    total_w = 0.
    with torch.no_grad():
        for _ in range(n_masks):
            mask_s  = np.random.binomial(1, p, (s, s)).astype(np.float32)
            mask_up = np.array(
                Image.fromarray(mask_s).resize((W + s, H + s), Image.BILINEAR))
            ox = np.random.randint(0, s + 1)
            oy = np.random.randint(0, s + 1)
            mask_up = mask_up[oy:oy + H, ox:ox + W]
            mask_t  = torch.from_numpy(mask_up).float().to(DEVICE)
            masked  = cpu_img.clone().to(DEVICE) * mask_t.unsqueeze(0)
            out     = _safe_fwd(model, masked.unsqueeze(0), hint, organ)
            sc      = float(torch.softmax(out, 1)[0, pred_class_idx].item())
            cam    += sc * mask_up; total_w += sc
    return (_norm01(cam / total_w) if total_w > 0
            else np.zeros((H, W), dtype=np.float32))


def score_cam(model, img_t, target_layer_name, pred_class_idx,
              is_vit=False, top_k=32, hint=None, organ=None) -> np.ndarray:
    """ScoreCAM — channel weight by masked-image score delta."""
    model.eval()
    target_layer = _find_layer(model, target_layer_name)
    if target_layer is None:
        return np.zeros((224, 224), dtype=np.float32)
    features = []

    def _fwd(m, i, o):
        out = o.detach().clone()
        if is_vit: out = _vit_to_spatial(out)
        if out is not None and out.ndim == 4: features.append(out)

    hf = target_layer.register_forward_hook(_fwd)
    with torch.no_grad():
        base_out = _safe_fwd(model, img_t.float().to(DEVICE), hint, organ)
    hf.remove()

    if not features:
        return np.zeros((224, 224), dtype=np.float32)
    A = features[0]
    if A.ndim == 4: A = A[0]
    if A.ndim != 3: return np.zeros((224, 224), dtype=np.float32)
    C, H, W    = A.shape
    ch_energy  = A.reshape(C, -1).abs().mean(dim=1)
    topk_ch    = ch_energy.topk(min(top_k, C)).indices
    base_score = float(torch.softmax(base_out, 1)[0, pred_class_idx])
    weights    = []
    with torch.no_grad():
        for ch_idx in topk_ch:
            mask   = A[ch_idx:ch_idx + 1].unsqueeze(0).float()
            mask_u = F.interpolate(mask, (224, 224),
                                   mode="bilinear", align_corners=False)
            m_min, m_max = mask_u.min(), mask_u.max()
            if m_max - m_min < 1e-8: weights.append(0.); continue
            mask_n = (mask_u - m_min) / (m_max - m_min)
            masked = img_t.float().to(DEVICE) * mask_n.to(DEVICE)
            sc_out = _safe_fwd(model, masked, hint, organ)
            sc     = float(torch.softmax(sc_out, 1)[0, pred_class_idx])
            weights.append(max(0., sc - base_score))
    cam = torch.zeros(H, W)
    for i, ch_idx in enumerate(topk_ch):
        mask_u = F.interpolate(A[ch_idx].cpu().float().unsqueeze(0).unsqueeze(0),
                               (H, W), mode="bilinear", align_corners=False)[0, 0]
        cam   += weights[i] * mask_u
    cam_up = F.interpolate(F.relu(cam).unsqueeze(0).unsqueeze(0),
                           (224, 224), mode="bilinear", align_corners=False)
    return _norm01(cam_up[0, 0].numpy().astype(np.float32))


def multi_scale_cam(model, img_t, organ, pred_class_idx,
                    is_vit=False, hook_broken=False,
                    hint=None) -> np.ndarray | None:
    """Multi-layer GradCAM++ blend. UPG-6: skin uses stage.3-dominant weights."""
    layers = MS_CAM_LAYERS.get(organ, [])
    if hook_broken or not layers or is_vit:
        return None
    MS_W = {
        "skin":     [0.20, 0.35, 0.45],
        "melanoma": [0.20, 0.35, 0.45],
        "default":  [0.25, 0.35, 0.40],
    }
    organ_w = MS_W.get(organ, MS_W["default"])
    cams = []
    for layer_name in layers:
        cam = gradcam_plus_plus(model, img_t, layer_name, pred_class_idx,
                                is_vit=False, hint=hint, organ=organ)
        if cam.max() > 1e-6: cams.append(cam)
    if not cams: return None
    n = len(cams)
    if n == 1: return cams[0]
    w = organ_w[:n]; total = sum(w); w = [x / total for x in w]
    return _norm01(sum(wt * c for wt, c in zip(w, cams)))


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 5 — Integrated Gradients (BUG-1 + BUG-2 + FIX-B + FIX-C)
# ════════════════════════════════════════════════════════════════════════════════

def integrated_gradients(model, img_t, pred_class_idx,
                          steps=IG_GL_STEPS,
                          hint=None, organ=None) -> tuple[np.ndarray, float]:
    """
    GL-quadrature IG with zero baseline (BUG-2: completeness axiom).
    BUG-1: fresh clone().detach() each node.
    FIX-B: hook sweep before every backward.
    FIX-C: _raw_model_fwd — T* not applied.
    ISS-7: skipped for HOOK_BROKEN_ORGANS or T*≤0.15.
    """
    T_opt = OPTIMIZED_TEMPERATURES.get(organ, 1.0)
    if T_opt <= IG_SKIP_T_THRESHOLD or organ in HOOK_BROKEN_ORGANS:
        return np.zeros((224, 224), dtype=np.float32), float('nan')

    model.eval()
    baseline = torch.zeros_like(img_t).to(DEVICE)   # BUG-2: zero baseline
    inp      = img_t.float().to(DEVICE)
    nodes, gl_w = leggauss(steps)
    alphas   = (nodes + 1) / 2; gl_w = gl_w / 2
    grads    = []; nodes_ok = 0

    for alpha, wt in zip(alphas, gl_w):
        x = (baseline + float(alpha) * (inp - baseline)).clone().detach()  # BUG-1
        x.requires_grad_(True)
        _remove_all_hooks(model); model.zero_grad()  # FIX-B
        saved = _disable_inplace(model)
        try:
            with torch.enable_grad():
                out   = _raw_model_fwd(model, x, hint)  # FIX-C
                score = out[0, pred_class_idx]; score.backward()
            if x.grad is not None:
                grads.append(float(wt) * x.grad.detach().clone()); nodes_ok += 1
        except RuntimeError as e:
            if nodes_ok == 0:
                print(f"    IG node 0 failed ({str(e)[:60]}) — skipping IG")
        finally:
            _restore_inplace(saved); del x
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    if not grads:
        return np.zeros((224, 224), dtype=np.float32), float('nan')

    ig = (inp - baseline).detach() * sum(grads)
    completeness_err = float('nan')
    try:
        _remove_all_hooks(model)
        with torch.no_grad():
            fi = float(torch.softmax(
                _raw_model_fwd(model, inp,      hint), 1)[0, pred_class_idx])
            fb = float(torch.softmax(
                _raw_model_fwd(model, baseline, hint), 1)[0, pred_class_idx])
        completeness_err = abs(float(ig.sum().item()) - (fi - fb))
    except Exception:
        pass

    return _norm01(ig[0].abs().mean(0).cpu().numpy().astype(np.float32)), completeness_err


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 6 — KernelSHAP (FIX-E + V14-A + FIX-J + UPG-2/3)
# ════════════════════════════════════════════════════════════════════════════════

def _parse_shap_values(sv, pred_class_idx: int,
                        n_cls: int, n_segs: int) -> np.ndarray:
    """
    V14-A: dedicated 7-branch SHAP shape parser.
    Handles every shape SHAP can return across versions and binary/multi-class.
    """
    sv_arr = np.array(sv) if isinstance(sv, list) else sv

    if sv_arr.ndim == 3 and sv_arr.shape[0] == n_cls and sv_arr.shape[2] == n_segs:
        return sv_arr[min(pred_class_idx, n_cls-1), 0, :].astype(np.float32)
    if sv_arr.ndim == 3 and sv_arr.shape[0] == 1 and sv_arr.shape[2] == n_segs:
        vals = sv_arr[0, 0, :]
        return (-vals if pred_class_idx == 0 else vals).astype(np.float32)
    if sv_arr.ndim == 3 and sv_arr.shape[2] == n_cls and sv_arr.shape[1] == n_segs:
        return sv_arr[0, :, min(pred_class_idx, n_cls-1)].astype(np.float32)
    if sv_arr.ndim == 2 and sv_arr.shape[0] == n_segs and sv_arr.shape[1] == n_cls:
        return sv_arr[:, min(pred_class_idx, n_cls-1)].astype(np.float32)
    if sv_arr.ndim == 2 and sv_arr.shape[0] == n_cls and sv_arr.shape[1] == n_segs:
        return sv_arr[min(pred_class_idx, n_cls-1), :].astype(np.float32)
    if sv_arr.ndim == 2 and sv_arr.shape[1] == n_segs:
        return sv_arr[0, :].astype(np.float32)
    if sv_arr.ndim == 1 and sv_arr.shape[0] == n_segs:
        return sv_arr.astype(np.float32)
    # Last resort
    return sv_arr.flatten()[:n_segs].astype(np.float32)


def shap_attribution_gpu(model, img_t, class_names, pred_class_idx,
                          val_loader_ref=None, n_background=SHAP_N_BACKGROUND,
                          hint=None, organ=None) -> tuple[np.ndarray, str]:
    """
    KernelSHAP with organ-aware grid sizing.
    FIX-E: always returns (ndarray, str).
    V14-A: uses _parse_shap_values() — robust 7-branch parser.
    FIX-J: _rebuild guarantees exactly 224×224.
    UPG-2: ViT uses 14×14 grid (exact 224×224).
    UPG-3: ViT uses 600 samples.
    """
    n_cls = len(class_names)
    if organ in VIT_ORGANS:
        SEG_H, SEG_W = SHAP_SEG_H_VIT, SHAP_SEG_W_VIT
        N_SEGS       = SHAP_N_SEGS_VIT
        N_SAMPLES    = SHAP_NSAMPLES_VIT
    else:
        SEG_H, SEG_W = SHAP_SEG_H, SHAP_SEG_W
        N_SEGS       = SHAP_N_SEGS
        N_SAMPLES    = SHAP_NSAMPLES
    CELL_H = 224 // SEG_H
    CELL_W = 224 // SEG_W

    try:
        model.eval()
        img_np   = img_t[0].cpu().numpy().transpose(1, 2, 0)  # (224,224,3)
        hint_gpu = (hint[0:1].to(DEVICE)
                    if hint is not None and hasattr(model, 'film') else None)

        def _rebuild(mr: np.ndarray) -> torch.Tensor:
            """FIX-J: guarantees exactly 224×224 via pad+crop."""
            pm = np.kron(mr.reshape(SEG_H, SEG_W),
                         np.ones((CELL_H, CELL_W), dtype=np.float32))
            # FIX-J: pad if undershoots (safety for any future grid change)
            ph = 224 - pm.shape[0]; pw = 224 - pm.shape[1]
            if ph > 0 or pw > 0:
                pm = np.pad(pm, ((0, ph), (0, pw)), mode='edge')
            pm   = pm[:224, :224]                 # crop if overshoots
            mhw  = img_np * pm[:, :, None]        # (224,224,3) * (224,224,1)
            return (torch.from_numpy(mhw.transpose(2, 0, 1))
                    .float().unsqueeze(0).to(DEVICE))

        def _predict(mask_rows: np.ndarray) -> np.ndarray:
            out = []
            for row in mask_rows:
                t = _rebuild(row)
                with torch.no_grad():
                    lo = (model(t, hint_logits=hint_gpu.expand(t.shape[0], -1))
                          if hint_gpu is not None else model(t))
                out.append(torch.softmax(lo, 1)[0].cpu().numpy().astype(np.float64))
            return np.array(out, dtype=np.float64)

        bg  = np.random.binomial(1, 0.5,
                                 (n_background, N_SEGS)).astype(np.float64)
        exp = shap.KernelExplainer(_predict, bg)
        sv  = exp.shap_values(np.ones((1, N_SEGS), np.float64),
                               nsamples=N_SAMPLES, silent=True)

        # V14-A: robust shape parser
        vals = _parse_shap_values(sv, pred_class_idx, n_cls, N_SEGS)
        vals = vals.flatten()[:N_SEGS]
        vmap = _norm01(np.abs(
            np.kron(vals.reshape(SEG_H, SEG_W),
                    np.ones((CELL_H, CELL_W), dtype=np.float32))[:224, :224]))
        return vmap, "KernelSHAP"

    except Exception as e:
        print(f"  ⚠️  SHAP failed: {e}")
        return np.zeros((224, 224), dtype=np.float32), "failed"


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 7 — Segmentation helpers (V14-B/C/D/E + FIX-D + ISS-11)
# ════════════════════════════════════════════════════════════════════════════════

def _apply_crf(img_np_hwc: np.ndarray, cam_map: np.ndarray,
               n_iter: int = 5) -> tuple[np.ndarray, bool]:
    """
    V14-B: dedicated CRF function.
    All arrays explicitly C-contiguous (V14-FIX-1).
    Returns (mask, crf_actually_used) — truthful (V14-FIX-2).
    """
    threshold    = SEG_CAM_THRESHOLD.get("default", 0.30)
    fallback_mask = (cam_map > threshold).astype(np.float32)

    if not HAS_CRF:
        return fallback_mask, False

    try:
        H, W = 224, 224
        img_u8 = np.ascontiguousarray(
            (img_np_hwc * 255).clip(0, 255).astype(np.uint8))
        cam_c  = np.ascontiguousarray(cam_map.astype(np.float32))
        fg     = np.ascontiguousarray(np.clip(cam_c, 0.01, 0.99))
        bg     = np.ascontiguousarray(1.0 - fg)
        prob   = np.ascontiguousarray(np.stack([bg, fg], axis=0))
        from pydensecrf.utils import unary_from_softmax as _ufs
        import pydensecrf.densecrf as _dcrf
        unary  = np.ascontiguousarray(_ufs(prob))
        d = _dcrf.DenseCRF2D(W, H, 2)
        d.setUnaryEnergy(unary)
        d.addPairwiseGaussian(sxy=3, compat=3)
        d.addPairwiseBilateral(sxy=40, srgb=13, rgbim=img_u8, compat=10)
        Q    = d.inference(n_iter)
        mask = (np.array(Q).reshape(2, H, W)[1] > 0.5).astype(np.float32)
        return mask, True
    except Exception as e:
        print(f"  ⚠️  CRF failed: {e} — using raw threshold")
        return fallback_mask, False


def _morphological_cleanup(mask: np.ndarray, organ: str) -> np.ndarray:
    """
    V14-C: dedicated morphological cleanup.
    Explicit per-organ r_close / r_open values (same as v14).
    """
    mask = mask.astype(bool)
    mask = binary_fill_holes(mask)
    if organ == "brain_2d":
        r_c, r_o = 5, 3
    elif organ == "skin":
        r_c, r_o = 4, 2
    elif organ in {"blood", "lymphoma", "pancreatic"}:
        r_c, r_o = 2, 1
    else:
        r_c, r_o = 3, 2
    mask = binary_closing(mask, disk(r_c))
    mask = binary_opening(mask, disk(r_o))
    mask = binary_fill_holes(mask)
    return mask.astype(np.float32)


def _keep_largest_component(mask: np.ndarray,
                             organ: str = "") -> np.ndarray:
    """
    V14-D: dedicated largest-component selection.
    Extended vs v14: respects MULTI_FOCAL_K for lymphoma/ovarian/gastric.
    """
    labeled = sk_label(mask.astype(bool))
    if labeled.max() == 0:
        return mask
    regions = sorted(regionprops(labeled), key=lambda r: r.area, reverse=True)
    n_keep  = (MULTI_FOCAL_K if organ in MULTI_FOCAL_ORGANS else 1)
    n_keep  = min(n_keep, len(regions))
    out     = np.zeros_like(mask)
    for r in regions[:n_keep]:
        out[labeled == r.label] = 1.
    return out


def _compute_seg_metrics(mask: np.ndarray,
                          total_pixels: int = 224 * 224) -> dict:
    """
    V14-E: dedicated metrics function.
    Returns full dict: tumor_area_pct, circularity, n_regions,
    boundary_regularity, seg_ok.
    """
    mask_bool = mask.astype(bool)
    area      = float(mask_bool.sum())
    tumor_area_pct = round(100. * area / total_pixels, 2)

    if area < 10:
        return {"tumor_area_pct": 0., "seg_circularity": 0.,
                "n_regions": 0, "boundary_regularity": 0., "seg_ok": False}

    labeled  = sk_label(mask_bool)
    regions  = regionprops(labeled)
    n_regions = len(regions)
    largest  = max(regions, key=lambda r: r.area)
    perim    = largest.perimeter
    seg_circ = (round(4. * math.pi * largest.area / (perim ** 2), 4)
                if perim > 0 else 0.)
    seg_circ = min(seg_circ, 1.)
    try:
        bdry_reg = round(float(largest.area) / (largest.convex_area + 1e-8), 4)
        bdry_reg = min(bdry_reg, 1.)
    except Exception:
        bdry_reg = 0.

    return {"tumor_area_pct": tumor_area_pct, "seg_circularity": seg_circ,
            "n_regions": n_regions, "boundary_regularity": bdry_reg,
            "seg_ok": True}


def generate_segmentation_mask(organ, pred_class_name, final_cam, img_np_hwc,
                                model=None, img_t=None, pred_class_idx=None,
                                hint=None) -> dict:
    """
    FIX-D + ISS-11: delegates to hovernet_predict() when available.
    V14-B/C/D/E: uses dedicated helper functions for CRF, morphology, metrics.
    V14-FIX-3: seg_skipped_reason always set — no silent skips.
    UPG-7: every skip reason has an explicit string.
    """
    mal_set      = MALIGNANT_CLASSES.get(organ, set())
    is_malignant = pred_class_name in mal_set

    _empty = dict(
        mask=np.zeros((224, 224), dtype=np.float32),
        inst_map=np.zeros((224, 224), dtype=np.int32),
        np_map=np.zeros((224, 224), dtype=np.float32),
        nc_probs=np.zeros((6, 224, 224), dtype=np.float32),
        nuclear_types={}, morphometrics=[], til={},
        is_malignant=is_malignant, seg_is_malignant=is_malignant,
        tumor_area_pct=0., seg_circularity=0.,
        n_regions=0, n_neoplastic=0, boundary_regularity=0.,
        seg_ok=False, crf_used=False, seg_net_used=False, overlay=None,
        # V14-FIX-3: always set a reason
        seg_skipped_reason="benign_class" if not is_malignant else "cam_too_weak",
    )

    # FIX-D: HoVerNet primary path
    if img_np_hwc is not None and "HOVERNET_MODEL" in dir() and HOVERNET_MODEL is not None:
        try:
            result = hovernet_predict_v21(img_np_hwc, organ=organ, return_overlay=True)
            result["is_malignant"]     = is_malignant
            result["seg_is_malignant"] = is_malignant
            if not is_malignant:
                result["seg_skipped_reason"] = "benign_class"
            for k, v in _empty.items():
                result.setdefault(k, v)
            return result
        except Exception as e:
            print(f"  ⚠️  hovernet_predict failed ({e}) — CAM fallback")

    if not is_malignant or final_cam.max() < 0.05:
        return _empty

    # ISS-5: denser CAM for ViT organs
    if (organ in VIT_ORGANS and model is not None
            and img_t is not None and pred_class_idx is not None):
        try:
            att = attention_gradcam_vit(model, img_t, pred_class_idx,
                                        hint=hint, organ=organ)
            ris = rise_cam(model, img_t, pred_class_idx,
                           n_masks=200, hint=hint, organ=organ)
            cam_for_seg = _norm01(0.5 * att + 0.5 * ris)
        except Exception:
            cam_for_seg = final_cam
    else:
        cam_for_seg = final_cam

    # V14-B: dedicated CRF function — C-contiguous, truthful bool return
    if HAS_CRF and img_np_hwc is not None:
        raw_mask, crf_used = _apply_crf(img_np_hwc, cam_for_seg, n_iter=5)
    else:
        threshold = SEG_CAM_THRESHOLD.get(organ, SEG_CAM_THRESHOLD.get("default", 0.30))
        raw_mask  = (cam_for_seg > threshold).astype(np.float32)
        crf_used  = False

    # V14-C: dedicated morphological cleanup
    clean_mask = _morphological_cleanup(raw_mask, organ)

    # V14-D: dedicated component selection (extended: MULTI_FOCAL_K)
    if clean_mask.sum() > 0:
        final_mask = _keep_largest_component(clean_mask, organ)
    else:
        final_mask = clean_mask

    # V14-E: dedicated metrics
    metrics = _compute_seg_metrics(final_mask)
    if not metrics["seg_ok"]:
        return {**_empty, "seg_ok": False, "seg_skipped_reason": "mask_too_small"}

    labeled2 = sk_label(final_mask.astype(bool))
    return {**_empty,
            "mask":               final_mask,
            "inst_map":           labeled2.astype(np.int32),
            "is_malignant":       True,
            "seg_is_malignant":   True,
            "crf_used":           crf_used,           # V14-FIX-2: truthful
            "seg_skipped_reason": None,               # V14-FIX-3: explicitly None
            "seg_ok":             True,
            "seg_net_used":       False,
            **{k: v for k, v in metrics.items() if k != "seg_ok"}}


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 8 — Uncertainty & Faithfulness
# ════════════════════════════════════════════════════════════════════════════════

def mc_dropout_uncertainty(model, img_t, passes=20,
                            hint=None, organ=None) -> tuple:
    """V7-C: saves and restores model.training state exactly."""
    was_training = model.training      # V7-C: save state
    model.eval()
    for m in model.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout2d)): m.train()
    preds = []
    with torch.no_grad():
        for _ in range(passes):
            out = _safe_fwd(model, img_t.to(DEVICE), hint, organ)
            preds.append(torch.softmax(out, 1))
    # V7-C: restore exactly
    if was_training: model.train()
    else:            model.eval()

    stack   = torch.stack(preds, 0)
    mean_p  = stack.mean(0)
    std_unc = float(stack.std(0).mean().item())
    entropy = float(-(mean_p * torch.log(mean_p + 1e-8)).sum(1).mean().item())
    return mean_p, entropy, std_unc


def auc_deletion_v17(model, img_t, final_cam, pred_class_idx,
                      steps=None, hint=None, organ=None) -> float:
    """
    AUC-deletion using logit-margin.
    FIX-H: np.trapezoid.
    V14-FIX-5: DELETION_FRACTION=0.10 (defined in Cell 1).
    """
    if steps is None: steps = DELETION_STEPS
    model.eval()
    flat_idx = np.argsort(final_cam.flatten())[::-1].copy()

    def _margin(lo):
        lg  = lo[0]
        idx = [i for i in range(lg.shape[0]) if i != pred_class_idx]
        return (float(lg[pred_class_idx] - lg[idx].max())
                if idx else float(lg[pred_class_idx]))

    try:
        cpu_inp = img_t[0].detach().cpu()
        with torch.no_grad():
            base_out = _safe_fwd(model,
                                 cpu_inp.unsqueeze(0).to(DEVICE), hint, organ)
        scores = [_margin(base_out)]
        for s in range(1, steps + 1):
            n_del    = int(s / steps * 224 * 224 * DELETION_FRACTION)
            del_mask = torch.ones(224 * 224, dtype=torch.bool)
            del_mask[flat_idx[:n_del]] = False
            masked   = cpu_inp.clone()
            masked[:, ~del_mask.reshape(224, 224)] = 0.
            with torch.no_grad():
                sc_out = _safe_fwd(model,
                                   masked.unsqueeze(0).to(DEVICE), hint, organ)
            scores.append(_margin(sc_out))
        result = float(np.trapezoid(scores) / max(len(scores) - 1, 1))
        return result if math.isfinite(result) else 0.
    except Exception as e:
        print(f"  ⚠️  AUC-del: {e}"); return 0.


# V14-F: alias so old code calling auc_deletion() still works
auc_deletion = auc_deletion_v17


def auc_insertion(model, img_t, final_cam, pred_class_idx,
                  steps=None, hint=None, organ=None) -> float:
    """
    AUC-insertion using softmax probability.
    FIX-H: np.trapezoid.
    V14-FIX-5: INSERTION_FRACTION=0.10.
    """
    if steps is None: steps = INSERTION_STEPS
    model.eval()
    flat_idx = np.argsort(final_cam.flatten())[::-1].copy()
    try:
        cpu_inp  = img_t[0].detach().cpu()
        baseline = _blur_tensor(img_t, sigma=15)[0].detach().cpu()
        with torch.no_grad():
            base_out = _safe_fwd(model,
                                 baseline.unsqueeze(0).to(DEVICE), hint, organ)
        scores = [float(torch.softmax(base_out, 1)[0, pred_class_idx])]
        for s in range(1, steps + 1):
            n_rev    = int(s / steps * 224 * 224 * INSERTION_FRACTION)
            rev_mask = torch.zeros(224 * 224, dtype=torch.bool)
            rev_mask[flat_idx[:n_rev]] = True
            masked   = baseline.clone()
            for c in range(3):
                ch = masked[c].flatten()
                ch[rev_mask] = cpu_inp[c].flatten()[rev_mask]
                masked[c] = ch.reshape(224, 224)
            with torch.no_grad():
                sc_out = _safe_fwd(model,
                                   masked.unsqueeze(0).to(DEVICE), hint, organ)
            scores.append(float(torch.softmax(sc_out, 1)[0, pred_class_idx]))
        result = float(np.trapezoid(scores) / max(steps, 1))
        return result if math.isfinite(result) else 0.
    except Exception as e:
        print(f"  ⚠️  AUC-ins: {e}"); return 0.


def tta_confidence(model, img_t, pred_class_idx,
                   n_aug=4, hint=None, organ=None) -> tuple[float, float]:
    augs = [lambda x: x, lambda x: torch.flip(x, [3]),
            lambda x: torch.flip(x, [2]), lambda x: torch.rot90(x, 1, [2, 3])]
    model.eval(); probs = []
    with torch.no_grad():
        for aug in augs[:n_aug]:
            out = _safe_fwd(model, aug(img_t).to(DEVICE), hint, organ)
            probs.append(float(torch.softmax(out, 1)[0, pred_class_idx]))
    return float(np.mean(probs)), float(np.std(probs))


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 9 — Trust score & concept labelling
# ════════════════════════════════════════════════════════════════════════════════

def compute_trust_score(agreement, uncertainty, auc_del, spec_conf,
                         focus_ratio=1., pointing=True, is_vit=False,
                         seg_iou=0., is_malignant=False) -> float:
    """
    7-component trust score.
    V14-FIX-7: benign predictions redistribute seg weight to conf — no penalty.
    """
    focus_ratio = min(float(focus_ratio), FOCUS_RATIO_MAX)
    s_ssim  = min(agreement / TRUST_SSIM_MAX, 1.) * TRUST_W_SSIM
    faith   = max(0., min(auc_del / 3.0 + 0.5, 1.0))
    s_unc   = max(0., 1. - uncertainty / TRUST_UNCERT_MAX) * TRUST_W_UNCERT
    s_auc   = min(faith / 0.70, 1.) * TRUST_W_AUC_DEL
    s_focus = min(focus_ratio / 3., 1.) * TRUST_W_FOCUS
    s_point = float(pointing) * TRUST_W_POINT
    if is_malignant and seg_iou > 0:
        # Malignant: full seg component
        s_seg  = min(float(seg_iou) / 0.50, 1.) * TRUST_W_SEG
        s_conf = float(spec_conf) * TRUST_W_CONF
    else:
        # V14-FIX-7: benign — redistribute seg weight to conf, no penalty
        s_seg  = 0.
        s_conf = float(spec_conf) * (TRUST_W_CONF + TRUST_W_SEG)
    return round(s_ssim + s_unc + s_auc + s_conf + s_focus + s_point + s_seg, 4)


def get_concept_label(organ, pred_name, final_cam, img_np_hwc) -> tuple:
    key    = (organ, pred_name)
    labels = CONCEPTS.get(key, ["Model focus region"])
    if not labels: return "Model focus region", (112, 112)
    if final_cam.max() < 0.05: return labels[0], (112, 112)
    peak   = np.argmax(final_cam); py, px = divmod(int(peak), 224)
    bright = img_np_hwc[max(0, py-3):min(224, py+4),
                        max(0, px-3):min(224, px+4), :].mean()
    idx    = 0 if (bright > 0.5 or len(labels) == 1) else 1
    return labels[idx], (px, py)


def _compute_cam_seg_iou(cam_map: np.ndarray,
                          seg_mask: np.ndarray,
                          organ=None) -> float:
    thr   = SEG_CAM_THRESHOLD.get(organ, 0.30) if organ else 0.30
    cam_b = (cam_map > thr).astype(bool)
    seg_b = seg_mask.astype(bool)
    inter = (cam_b & seg_b).sum(); union = (cam_b | seg_b).sum()
    return float(inter / union) if union > 0 else 0.


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 10 — Rendering helpers (FIX-G + FIX-I + UPG-7)
# ════════════════════════════════════════════════════════════════════════════════

def _overlay(ax, bg, cam, cmap, label, lc="white"):
    """FIX-G: tuple-safe, shape-validated before imshow."""
    import matplotlib.pyplot as _plt
    ax.imshow(bg)
    cam = _ensure_2d(cam)
    if cam.max() > 0.05:
        try:
            hm = ax.imshow(cam, cmap=cmap, alpha=0.65, vmin=0, vmax=1)
            cb = _plt.colorbar(hm, ax=ax, fraction=0.04, pad=0.02, ticks=[0, 1])
            cb.ax.yaxis.set_tick_params(color="white", labelcolor="white")
        except Exception: pass
    ax.axis("off"); ax.set_title(label, color=lc, fontsize=7, pad=2)


def _overlay_seg(ax, bg, seg_result: dict, label: str = "Segmentation"):
    """
    FIX-I: accepts seg_result DICT — no more flat keyword args.
    Fixes Cell 11 crash: TypeError: _overlay_seg() got unexpected kwarg 'seg_ok'.
    UPG-7: always renders a visible title — no blank right column.
    """
    ax.imshow(bg)
    overlay_img  = seg_result.get("overlay")
    seg_mask     = seg_result.get("mask")
    seg_ok       = seg_result.get("seg_ok", False)
    crf_used     = seg_result.get("crf_used", False)
    seg_net_used = seg_result.get("seg_net_used", False)
    til          = seg_result.get("til", {})
    area         = float(seg_result.get("tumor_area_pct") or 0.)
    circ         = float(seg_result.get("seg_circularity") or 0.)
    reason       = seg_result.get("seg_skipped_reason")
    is_malignant = seg_result.get("is_malignant", False)

    if overlay_img is not None:
        ax.imshow(overlay_img)
        til_score = til.get("til_score", 0.) if til else 0.
        ax.set_title(f"HoVerNet  TIL={til_score:.3f}  Area={area:.1f}%",
                     color="#00BFA6", fontsize=7, pad=2)
    elif seg_ok and seg_mask is not None and seg_mask.sum() > 0:
        rgba = np.zeros((*seg_mask.shape, 4), dtype=np.float32)
        rgba[seg_mask > 0.5] = [0., 1., 0.5, 0.35]
        ax.imshow(rgba)
        try:
            from skimage import measure as _m
            for c in _m.find_contours(seg_mask, 0.5):
                ax.plot(c[:, 1], c[:, 0], lw=1.5, color="#00BFA6", alpha=0.9)
        except Exception: pass
        parts = ["CRF✓" if crf_used else "Morph"]
        if seg_net_used: parts.append("SegNet✓")
        ax.set_title(
            f"Seg({'|'.join(parts)})  Area:{area:.1f}%  Circ:{circ:.2f}",
            color="#00BFA6", fontsize=7, pad=2)
    # UPG-7: explicit messages for every skip reason
    elif reason == "benign_class" or not is_malignant:
        ax.set_title("Benign class\nSeg not applicable",
                     color="#888888", fontsize=7, pad=2)
    elif reason == "mask_too_small":
        ax.set_title("Seg mask too small\nCAM signal weak",
                     color="#e67e22", fontsize=7, pad=2)
    elif reason == "cam_too_weak":
        ax.set_title("CAM too weak\nfor segmentation",
                     color="#e67e22", fontsize=7, pad=2)
    elif reason == "hovernet_not_available":
        ax.set_title("HoVerNet not loaded\nCAM fallback used",
                     color="#e67e22", fontsize=7, pad=2)
    else:
        ax.set_title(label, color="#e74c3c", fontsize=7, pad=2)
    ax.axis("off")


def _build_seg_result_from_row(r: dict) -> dict:
    """
    FIX-I helper: builds a seg_result dict from a Cell 11 result row.
    Prefers r['seg_result'] if already present (set by run_full_xai).
    Falls back to flat keys for backward compat.

    Usage in Cell 11:
        _overlay_seg(ax11, r["img_np"],
                     seg_result=_build_seg_result_from_row(r),
                     label="Seg—malignant but failed")
    """
    if r.get("seg_result"):
        return r["seg_result"]
    return {
        "seg_ok":             r.get("seg_ok", False),
        "mask":               r.get("seg_mask"),
        "overlay":            r.get("overlay"),
        "tumor_area_pct":     r.get("seg_tumor_area_pct", 0.),
        "seg_circularity":    r.get("seg_circularity", 0.) or 0.,
        "crf_used":           r.get("seg_crf_used", False),
        "seg_net_used":       r.get("seg_net_used", False),
        "seg_skipped_reason": r.get("seg_skipped_reason", None),
        "is_malignant":       r.get("seg_is_malignant", False),
        "til":                r.get("til", {}),
    }


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 11 — Master XAI function
# ════════════════════════════════════════════════════════════════════════════════

def run_full_xai(organ, model, img_t, pred_class_idx, class_names,
                 img_np_hwc=None, hint=None, val_loader_ref=None) -> dict:
    """
    v17 ULTIMATE XAI engine — full merge of v14 + v7 + v17.

    V7-D: LAYER_CAM_ORGANS get 3-map blend:
          LayerCAM(0.30) + GradCAM++(0.40) + EigenCAM(0.30)
    Others: GradCAM++(0.60) + EigenCAM(0.40).
    ScoreCAM overrides if Spearman < 0.25 (UPG-9).

    V14 improvements active:
    - _apply_crf:            C-contiguous, (mask, bool) return
    - _morphological_cleanup: per-organ r_close/r_open
    - _keep_largest_component: MULTI_FOCAL_K aware
    - _compute_seg_metrics:   full dict
    - _parse_shap_values:     7-branch robust parser
    - V14-FIX-7 trust score: benign redistribution
    - V14-FIX-9 tissue mask: near-all-tissue handling

    UPG-8: pred_class_name from argmax — staging always populated.
    UPG-4: flagged = 95th-pctile entropy only.
    FIX-I: seg_result dict in return for _overlay_seg.
    """
    tgt_layer       = XAI_LAYERS.get(organ, "backbone.stages.2")
    is_vit          = organ in VIT_ORGANS
    hook_broken     = organ in HOOK_BROKEN_ORGANS
    use_layer_cam   = organ in LAYER_CAM_ORGANS and not is_vit and not hook_broken
    pred_class_name = class_names[pred_class_idx]  # UPG-8
    T_opt           = OPTIMIZED_TEMPERATURES.get(organ, 1.0)

    _remove_all_hooks(model); model.zero_grad()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ── Step 1: Primary CAM ──────────────────────────────────────────────────
    if hook_broken:
        gcam = rise_cam(model, img_t, pred_class_idx,
                        n_masks=500, hint=hint, organ=organ)
    elif is_vit:
        gcam = attention_gradcam_vit(model, img_t, pred_class_idx,
                                     hint=hint, organ=organ)
    else:
        gcam = gradcam_plus_plus(model, img_t, tgt_layer, pred_class_idx,
                                 is_vit=False, hint=hint, organ=organ)

    # V7-D: LayerCAM for texture/structure organs
    lcam = None
    if use_layer_cam:
        lcam = layer_cam_smooth(model, img_t, tgt_layer, pred_class_idx,
                                is_vit=False, hint=hint, organ=organ)

    # ── Step 2: Secondary CAM ─────────────────────────────────────────────────
    if is_vit:
        ecam = attention_rollout(model, img_t, is_vit=True,
                                 hint=hint, organ=organ)
    else:
        ecam = eigen_cam(model, img_t, tgt_layer, is_vit=False,
                         hint=hint, organ=organ)

    # ISS-9: Spearman rank-correlation (replaces SSIM from v14)
    agreement = _cam_agreement(gcam, ecam)

    # ── Step 3: ScoreCAM fallback (UPG-9: threshold 0.25) ────────────────────
    scam = None
    if agreement < CAM_DISAGREE_THRESH and not hook_broken and not is_vit:
        scam = score_cam(model, img_t, tgt_layer, pred_class_idx,
                         is_vit=False, hint=hint, organ=organ)

    # ── Step 4: Multi-scale CAM (UPG-6) ──────────────────────────────────────
    ms_cam_result = multi_scale_cam(model, img_t, organ, pred_class_idx,
                                    is_vit=is_vit, hook_broken=hook_broken,
                                    hint=hint)

    # ── Step 5: Final CAM composition (V7-D: 3-map blend) ────────────────────
    if scam is not None:
        final_cam = scam
    elif use_layer_cam and lcam is not None and ms_cam_result is not None:
        final_cam = _norm01(0.30 * lcam + 0.40 * ms_cam_result + 0.30 * ecam)
    elif use_layer_cam and lcam is not None:
        final_cam = _norm01(0.30 * lcam + 0.40 * gcam + 0.30 * ecam)
    elif ms_cam_result is not None:
        final_cam = _norm01(0.60 * ms_cam_result + 0.40 * ecam)
    else:
        final_cam = _norm01(0.60 * gcam + 0.40 * ecam)

    # ── Step 6: Tissue masking (V14-FIX-9: near-all-tissue handling) ─────────
    if img_np_hwc is not None:
        tmask = _tissue_mask(img_np_hwc)
        gcam      = _norm01(gcam * tmask)
        ecam      = _norm01(ecam * tmask)
        if lcam is not None:          lcam          = _norm01(lcam * tmask)
        if scam is not None:          scam          = _norm01(scam * tmask)
        if ms_cam_result is not None: ms_cam_result = _norm01(ms_cam_result * tmask)
        final_cam = _norm01(final_cam * tmask)

    # ── Step 7: Integrated Gradients ─────────────────────────────────────────
    _remove_all_hooks(model); model.zero_grad()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    ig_skipped = hook_broken or (T_opt <= IG_SKIP_T_THRESHOLD)
    if ig_skipped:
        ig_map = np.zeros((224, 224), dtype=np.float32)
        ig_completeness_err = float('nan')
    else:
        ig_map, ig_completeness_err = integrated_gradients(
            model, img_t, pred_class_idx, steps=IG_GL_STEPS,
            hint=hint, organ=organ)
        if img_np_hwc is not None:
            ig_map = _norm01(ig_map * _tissue_mask(img_np_hwc))
    ig_ok = (not ig_skipped and ig_map.max() > 0.02 and
             (math.isnan(ig_completeness_err) or ig_completeness_err < 2.0))

    # ── Step 8: KernelSHAP (V14-A + FIX-J + UPG-2/3) ────────────────────────
    shap_map, shap_method = shap_attribution_gpu(
        model, img_t, class_names, pred_class_idx,
        val_loader_ref=None, n_background=SHAP_N_BACKGROUND,
        hint=hint, organ=organ)
    shap_map = _ensure_2d(shap_map)

    # ── Step 9: Segmentation (V14-B/C/D/E + FIX-D + ISS-11) ─────────────────
    seg_result   = generate_segmentation_mask(
        organ, pred_class_name, final_cam, img_np_hwc,
        model=model, img_t=img_t, pred_class_idx=pred_class_idx, hint=hint)
    seg_mask     = _ensure_2d(seg_result.get("mask"))
    is_malignant = bool(seg_result.get("is_malignant", False))
    seg_iou      = (_compute_cam_seg_iou(final_cam, seg_mask, organ)
                    if seg_result.get("seg_ok", False) else 0.)

    # ── Step 10: Uncertainty & Faithfulness (V7-C + V14-FIX-5) ──────────────
    mean_p, entropy, std_unc = mc_dropout_uncertainty(
        model, img_t, passes=20, hint=hint, organ=organ)
    auc_del = auc_deletion_v17(model, img_t, final_cam, pred_class_idx,
                                hint=hint, organ=organ)
    auc_ins = auc_insertion(model, img_t, final_cam, pred_class_idx,
                             hint=hint, organ=organ)
    tta_conf_v, tta_std_v = tta_confidence(
        model, img_t, pred_class_idx, hint=hint, organ=organ)

    # ── Step 11: Focus ratio & pointing game ──────────────────────────────────
    pred_mask = (gcam > 0.3).astype(bool)
    focus = float(final_cam[pred_mask].mean() / (final_cam[~pred_mask].mean() + 1e-8)
                  if pred_mask.any() else 0.)
    peak_y, peak_x = divmod(int(np.argmax(final_cam)), 224)
    pointing = (bool(pred_mask[peak_y, peak_x])
                if pred_mask.shape == (224, 224) else False)

    # ── Step 12: Trust score (V14-FIX-7: benign redistribution) ──────────────
    trust = compute_trust_score(
        agreement, entropy, auc_del,
        float(mean_p[0, pred_class_idx].item()),
        focus, pointing, is_vit=is_vit,
        seg_iou=seg_iou, is_malignant=is_malignant)

    # ── Step 13: Concept labelling ────────────────────────────────────────────
    concept_label, concept_loc = get_concept_label(
        organ, pred_class_name, final_cam,
        img_np_hwc if img_np_hwc is not None else np.zeros((224, 224, 3)))

    # ── Step 14: Uncertainty flag (UPG-4: 95th-pctile entropy only) ──────────
    unc_thresh = ORGAN_VAL_UNC_STATS.get(organ, {}).get("threshold", 0.10)
    flagged    = entropy > unc_thresh

    return {
        # CAMs
        "layer_cam":      gcam,
        "eigen_cam":      ecam,
        "score_cam":      scam,
        "ms_cam":         ms_cam_result,
        "layercam_v7":    lcam,
        "final_cam":      final_cam,
        "gradcam_pp":     gcam,
        # Attribution
        "shap_map":       shap_map,
        "shap_method":    str(shap_method),
        "shap_ok":        bool(shap_map.max() > 0.05),
        "ig_map":         ig_map,
        "ig_completeness_err": ig_completeness_err,
        "ig_skipped":     ig_skipped,
        "ig_ok":          ig_ok,
        # Segmentation — FIX-I: seg_result dict for _overlay_seg
        "seg_result":               seg_result,
        "seg_mask":                 seg_mask,
        "seg_ok":                   seg_result.get("seg_ok", False),
        "seg_is_malignant":         is_malignant,
        "seg_tumor_area_pct":       seg_result.get("tumor_area_pct", 0.),
        "seg_circularity":          float(seg_result.get("seg_circularity") or 0.),
        "seg_n_regions":            seg_result.get("n_regions", 0),
        "seg_boundary_regularity":  seg_result.get("boundary_regularity", 0.),
        "seg_crf_used":             seg_result.get("crf_used", False),
        "seg_net_used":             seg_result.get("seg_net_used", False),
        "seg_skipped_reason":       seg_result.get("seg_skipped_reason", None),
        "seg_iou":                  seg_iou,
        # HoVerNet
        "nuclear_types":  seg_result.get("nuclear_types", {}),
        "morphometrics":  seg_result.get("morphometrics", []),
        "til":            seg_result.get("til", {}),
        "n_neoplastic":   seg_result.get("n_neoplastic", 0),
        "overlay":        seg_result.get("overlay"),
        # Uncertainty
        "agreement":      agreement,
        "uncertainty":    entropy,
        "entropy":        entropy,
        "std_unc":        std_unc,
        "mean_probs":     mean_p,
        # Faithfulness
        "auc_deletion":   auc_del,
        "auc_insertion":  auc_ins,
        "tta_conf":       tta_conf_v,
        "tta_std":        tta_std_v,
        # Focus
        "focus_ratio":    focus,
        "pointing_game":  pointing,
        # Trust & flagging
        "trust_score":    trust,
        "flagged":        flagged,
        # Concepts
        "concept_label":  concept_label,
        "concept_loc":    concept_loc,
        # Method flags
        "score_cam_used":         scam is not None,
        "ms_cam_used":            ms_cam_result is not None,
        "layer_cam_v7_used":      lcam is not None,
        "rise_used":              hook_broken,
        "attention_rollout_used": is_vit,
        "T_opt":                  T_opt,
    }


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 12 — Safe empty result
# ════════════════════════════════════════════════════════════════════════════════

def _empty_xai(organ, class_names, pred_class_idx) -> dict:
    z = np.zeros((224, 224), dtype=np.float32); n = len(class_names)
    empty_seg = {
        "seg_ok": False, "mask": z, "overlay": None,
        "tumor_area_pct": 0., "seg_circularity": 0.,
        "crf_used": False, "seg_net_used": False,
        "seg_skipped_reason": "xai_crash",
        "is_malignant": False, "til": {},
    }
    return dict(
        layer_cam=z, eigen_cam=z, score_cam=None, ms_cam=None,
        layercam_v7=None, final_cam=z, gradcam_pp=z,
        shap_map=z, shap_method="failed", shap_ok=False,
        ig_map=z, ig_completeness_err=float('nan'),
        ig_skipped=True, ig_ok=False,
        seg_result=empty_seg,
        seg_mask=z, seg_ok=False, seg_is_malignant=False,
        seg_tumor_area_pct=0., seg_circularity=0., seg_n_regions=0,
        seg_boundary_regularity=0., seg_crf_used=False, seg_net_used=False,
        seg_skipped_reason="xai_crash", seg_iou=0.,
        nuclear_types={}, morphometrics=[], til={}, n_neoplastic=0, overlay=None,
        agreement=0., uncertainty=0., entropy=0., std_unc=0.,
        mean_probs=torch.zeros(1, n),
        auc_deletion=0., auc_insertion=0., tta_conf=0., tta_std=0.,
        focus_ratio=0., pointing_game=False, trust_score=0., flagged=True,
        concept_label="XAI error", concept_loc=(112, 112),
        score_cam_used=False, ms_cam_used=False,
        layer_cam_v7_used=False,
        rise_used=organ in HOOK_BROKEN_ORGANS,
        attention_rollout_used=organ in VIT_ORGANS,
        T_opt=OPTIMIZED_TEMPERATURES.get(organ, 1.0),
    )


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 13 — Smoke test
# ════════════════════════════════════════════════════════════════════════════════

print("\n  Smoke tests...")
print(f"  _norm01([0,1,2])          = {_norm01(np.array([0.,1.,2.]))}")
print(f"  _ensure_2d(None).shape    = {_ensure_2d(None).shape}")
print(f"  _ensure_2d(tuple).shape   = "
      f"{_ensure_2d((np.ones((224,224)), 'str')).shape}")
print(f"  np.trapezoid available    = {hasattr(np, 'trapezoid')}")
print(f"  _parse_shap_values test   = ", end="")
try:
    _sv_test = np.random.rand(4, 1, 196)   # shape: (n_cls, 1, n_segs)
    _r = _parse_shap_values(_sv_test, 2, 4, 196)
    print(f"OK  shape={_r.shape}")
except Exception as _e:
    print(f"FAIL: {_e}")

print(f"  auc_deletion alias        = {auc_deletion is auc_deletion_v17}")
_ph = 224 - (SHAP_SEG_H_VIT * (224 // SHAP_SEG_H_VIT))
_pw = 224 - (SHAP_SEG_W_VIT * (224 // SHAP_SEG_W_VIT))
print(f"  ViT SHAP grid padding     = ({_ph},{_pw})  "
      f"{'✅ exact' if _ph == 0 and _pw == 0 else '⚠ will pad'}")
print(f"  HOOK_BROKEN_ORGANS        = {HOOK_BROKEN_ORGANS}")
print(f"  LAYER_CAM_ORGANS          = {LAYER_CAM_ORGANS}")
print(f"  CAM_DISAGREE_THRESH       = {CAM_DISAGREE_THRESH}")
print(f"  IG_SKIP_T_THRESHOLD       = {IG_SKIP_T_THRESHOLD}")
print(f"  CRF available             = {HAS_CRF}")
print(f"  HoVerNet available        = "
      f"{'YES' if 'HOVERNET_MODEL' in dir() and HOVERNET_MODEL is not None else 'NO'}")
print(f"  XAI_LAYERS[lymphoma]      = "
      f"{XAI_LAYERS.get('lymphoma', '⚠ NOT SET')}"
      f"  {'✅' if XAI_LAYERS.get('lymphoma') == 'backbone.blocks.6' else '⚠ should be backbone.blocks.6'}")

print()
print("✅ Cell 6 — XAI Engine v17 ULTIMATE complete")
print()
print("  Merged from v14:")
print("  V14-A  _parse_shap_values()       7-branch robust shape parser")
print("  V14-B  _apply_crf()               C-contiguous + (mask,bool) return")
print("  V14-C  _morphological_cleanup()   per-organ r_close/r_open")
print("  V14-D  _keep_largest_component()  MULTI_FOCAL_K aware")
print("  V14-E  _compute_seg_metrics()     full metrics dict")
print("  V14-F  auc_deletion alias         backward compat for old callers")
print("  V14-FIX-7 trust score             benign weight redistribution")
print("  V14-FIX-9 tissue mask             near-all-tissue handling")
print()
print("  Merged from v7:")
print("  V7-A  EigenCAM gram-matrix eigh + SVD fallback")
print("  V7-B  LayerCAM F.relu(A*relu(g)) element-wise")
print("  V7-C  MC-dropout saves/restores model.training")
print("  V7-D  3-map blend for LAYER_CAM_ORGANS")
print()
print("  All v17 fixes retained:")
print("  BUG-1/2 IG  FIX-A/B/C/D/E/G/H/I/J  ISS-9  UPG-1..9")
print()
print("  Cell 11 change needed:")
print("    _overlay_seg(ax, img, seg_result=_build_seg_result_from_row(r), label=...)")
print("  Cell 1 change needed:")
print("    XAI_LAYERS['lymphoma'] = 'backbone.blocks.6'")

# ════════════════════════════════════════════════════════════════════════════════
# === UPGRADE v18-FIX === SECTION 14 — CAM Post-Processor (FIXED)
#
# FIX-PP-1: bilateral filter params reduced significantly
#   - d: 9→5, sigmaColor: 50→20, sigmaSpace: 50→20 (default)
#   - skin/cervical get even gentler params
#   WHY: v18 agreement dropped -0.115 because aggressive bilateral smoothing
#   destroyed the spatial structure that Spearman correlation relies on.
#   Gentler smoothing preserves edge information while still reducing noise.
#
# FIX-PP-2: sharpen_strength reduced 0.4→0.2 by default
#   WHY: Over-sharpening creates artificial high-frequency noise that
#   makes two CAMs look more different, not less.
#
# FIX-PP-3: skip bilateral for hook_broken organs (RISE already smooth)
#   WHY: RISE output is already spatially averaged; bilateral adds nothing
#   and destroys the agreement between RISE and rollout.
# ════════════════════════════════════════════════════════════════════════════════

# FIX-PP-1: Gentler bilateral params (was d=9, sigmaColor=50)
BILATERAL_PARAMS = {
    "skin":     {"d": 5,  "sigmaColor": 15, "sigmaSpace": 15},
    "cervical": {"d": 5,  "sigmaColor": 12, "sigmaSpace": 12},
    "gastric":  {"d": 5,  "sigmaColor": 18, "sigmaSpace": 18},
    "lymphoma": {"d": 0,  "sigmaColor":  0, "sigmaSpace":  0},  # skip (RISE)
    "default":  {"d": 5,  "sigmaColor": 20, "sigmaSpace": 20},
}

# FIX-PP-2: organs where post-processing is skipped entirely
# (hook_broken → RISE → already smooth, bilateral hurts agreement)
_PP_SKIP_ORGANS = {"lymphoma", "cervical", "kidney", "thyroid"}


def cam_post_process(cam: np.ndarray,
                     img_np_hwc=None,
                     organ: str = "default",
                     sharpen_strength: float = 0.20,   # FIX-PP-2: was 0.5
                     apply_bilateral: bool = True,
                     apply_adaptive_threshold: bool = False) -> np.ndarray:
    """
    === UPGRADE v18-FIX — CAM Post-Processor ===

    FIX-PP-1: gentler bilateral (d=5, σ=20 instead of d=9, σ=50)
    FIX-PP-2: sharpen_strength default 0.20 (was 0.5)
    FIX-PP-3: returns raw cam for hook_broken/skip organs
    """
    import cv2 as _cv2

    if cam is None or not isinstance(cam, np.ndarray) or cam.ndim != 2:
        return np.zeros((224, 224), dtype=np.float32)

    # FIX-PP-3: skip post-processing for hook-broken organs
    if organ in _PP_SKIP_ORGANS:
        return cam.astype(np.float32)

    cam_f = cam.astype(np.float32).copy()

    # Step 1: Gentle bilateral filter
    if apply_bilateral:
        try:
            bp = BILATERAL_PARAMS.get(organ, BILATERAL_PARAMS["default"])
            if bp["d"] <= 0:
                pass  # skip for this organ
            else:
                cam_u8 = (cam_f * 255.).clip(0., 255.).astype(np.uint8)
                # Always use regular bilateral (joint bilateral often unavailable)
                cam_filt = _cv2.bilateralFilter(
                    cam_u8,
                    d=bp["d"],
                    sigmaColor=bp["sigmaColor"],
                    sigmaSpace=bp["sigmaSpace"])
                cam_f = cam_filt.astype(np.float32) / 255.
        except Exception:
            pass

    # Step 2: Adaptive threshold (optional)
    if apply_adaptive_threshold and cam_f.max() > 0.05:
        try:
            import cv2 as _cv2_inner
            cam_u8_thresh = (cam_f * 255.).clip(0., 255.).astype(np.uint8)
            otsu_val, _ = _cv2_inner.threshold(
                cam_u8_thresh, 0, 255,
                _cv2_inner.THRESH_BINARY + _cv2_inner.THRESH_OTSU)
            if otsu_val > 0:
                cam_f = cam_f / (otsu_val / 255. + 1e-6) * 0.5
                cam_f = np.clip(cam_f, 0., 1.)
        except Exception:
            pass

    # Step 3: Mild unsharp mask (FIX-PP-2: reduced strength)
    if sharpen_strength > 0.01:
        try:
            import cv2 as _cv2_s
            blur  = _cv2_s.GaussianBlur(cam_f, (5, 5), sigmaX=1.5)
            cam_f = cam_f + sharpen_strength * (cam_f - blur)
            cam_f = np.clip(cam_f, 0., 1.)
        except Exception:
            pass

    mn, mx = float(cam_f.min()), float(cam_f.max())
    if mx - mn > 1e-8:
        cam_f = (cam_f - mn) / (mx - mn)

    return cam_f.astype(np.float32)


def batch_post_process_cams(cam_dict: dict,
                              img_np_hwc,
                              organ: str) -> dict:
    processed = {}
    for key, cam in cam_dict.items():
        if cam is not None and isinstance(cam, np.ndarray) and cam.ndim == 2:
            processed[key + "_pp"] = cam_post_process(
                cam, img_np_hwc, organ,
                sharpen_strength=0.20,   # FIX-PP-2
                apply_bilateral=True)
        else:
            processed[key + "_pp"] = None
    return processed


# ════════════════════════════════════════════════════════════════════════════════
# === UPGRADE v18-FIX === SECTION 15 — CAM Agreement Optimizer (FIXED)
#
# FIX-OPT-1: entropy weighting softened (temperature 0.5→1.5)
#   WHY: sharp softmax at T=0.5 always picks one CAM and ignores others.
#   At T=1.5, weights are more uniform → true ensemble, not winner-take-all.
#
# FIX-OPT-2: primary boost reduced 2.0→1.3
#   WHY: boosting primary by 2x was making the "optimizer" identical to
#   just using GradCAM++ alone, defeating the ensemble purpose.
#
# FIX-OPT-3: agreement measured by Spearman on ORIGINAL (pre-PP) maps
#   WHY: post-processed agreement was confounded by smoothing artifacts
#   making disparate CAMs look artificially similar. Pre-PP Spearman
#   is the honest measure of model-level agreement.
# ════════════════════════════════════════════════════════════════════════════════

def _spatial_entropy(cam: np.ndarray, eps: float = 1e-8) -> float:
    if cam is None or not isinstance(cam, np.ndarray) or cam.max() < 1e-6:
        return float('inf')
    flat = cam.flatten().astype(np.float64)
    flat = flat / (flat.sum() + eps)
    return float(-(flat * np.log(flat + eps)).sum())


def _cam_sharpness(cam: np.ndarray) -> float:
    if cam is None or cam.max() < 1e-6:
        return 0.
    try:
        gy, gx = np.gradient(cam.astype(np.float64))
        return float((gx**2 + gy**2).mean())
    except Exception:
        return 0.


def cam_agreement_optimizer(cam_primary: np.ndarray,
                              cam_secondary: np.ndarray,
                              cam_tertiary=None,
                              cam_primary_raw: np.ndarray = None,  # FIX-OPT-3
                              cam_secondary_raw: np.ndarray = None,
                              organ: str = "default",
                              confidence: float = 0.5) -> tuple:
    """
    === UPGRADE v18-FIX — CAM Agreement Optimizer ===

    FIX-OPT-1: softmax temperature 0.5→1.5 (softer selection)
    FIX-OPT-2: primary boost 2.0→1.3
    FIX-OPT-3: Spearman computed on raw (pre-PP) maps when available
    """
    cams = [c for c in [cam_primary, cam_secondary, cam_tertiary]
            if c is not None and isinstance(c, np.ndarray) and c.max() > 1e-6]

    if not cams:
        return np.zeros((224, 224), dtype=np.float32), 0., "empty"
    if len(cams) == 1:
        return cams[0], 1., "single"

    # Compute quality per CAM
    qualities = []
    for cam in cams:
        entropy   = _spatial_entropy(cam)
        sharpness = _cam_sharpness(cam)
        q = sharpness / (entropy + 1e-3)
        qualities.append(float(q))

    qualities_arr = np.array(qualities, dtype=np.float64)

    # FIX-OPT-2: reduced primary boost (was 2.0)
    if confidence >= 0.80:
        qualities_arr[0] *= 1.3

    # FIX-OPT-1: softer temperature (was 0.5)
    T_blend = 1.5
    q_exp   = np.exp(qualities_arr / T_blend)
    weights = q_exp / (q_exp.sum() + 1e-8)

    fused = sum(w * c for w, c in zip(weights, cams))
    mn, mx = float(fused.min()), float(fused.max())
    if mx - mn > 1e-8:
        fused = (fused - mn) / (mx - mn)
    fused = fused.astype(np.float32)

    # FIX-OPT-3: use raw (pre-PP) maps for agreement when available
    raw_p = cam_primary_raw if cam_primary_raw is not None else cam_primary
    raw_s = cam_secondary_raw if cam_secondary_raw is not None else cam_secondary
    agreement = 0.
    if raw_p is not None and raw_s is not None:
        f1 = raw_p.flatten().astype(np.float64)
        f2 = raw_s.flatten().astype(np.float64)
        if f1.std() > 1e-8 and f2.std() > 1e-8:
            corr, _ = spearmanr(f1, f2)
            agreement = float(max(0., corr))

    method_str = (f"SoftBlend(T={T_blend},w={[round(x,2) for x in weights.tolist()]})"
                  f"@boost=1.3")
    return fused, agreement, method_str


# ════════════════════════════════════════════════════════════════════════════════
# === UPGRADE v18-FIX === SECTION 16 — Improved IG Baseline (FIXED)
#
# FIX-IG-1: use per-batch mean image instead of fixed H&E color constant
#   WHY: Fixed off-white [240,235,240] is too close to actual H&E slide
#   background for some organs (brain, kidney), causing near-zero delta
#   between baseline and image → integral ≈ 0 → high completeness error.
#   Using the per-image spatial mean provides a meaningful baseline
#   while respecting each organ's actual staining characteristics.
#
# FIX-IG-2: reduce GL steps 64→48 to save VRAM and time
#   WHY: No measurable accuracy difference between 48 and 64 GL nodes
#   in our experiments, but saves ~25% VRAM per IG call.
#
# FIX-IG-3: per-organ skip thresholds more conservative
#   blood T*=0.585 was wrongly running IG (sharp logit surface, fast)
#   but giving error=2.06. Skip blood IG too.
# ════════════════════════════════════════════════════════════════════════════════

# FIX-IG-3: expanded skip set + updated thresholds
IG_SKIP_T_THRESHOLD_PER_ORGAN = {
    "kidney":     6.0,   # T*=4.33 → still skip
    "thyroid":    2.5,   # T*=1.95 → marginal, run
    "pancreatic": 2.0,   # T*=1.49 → allow IG
    "cervical":   2.5,   # hook-broken, skipped anyway
    "blood":      0.70,  # FIX-IG-3: T*=0.585 gives err=2.06 → skip
    "esophageal": 0.50,  # T*=0.452 gives err=1.27 → borderline
    "default":    IG_SKIP_T_THRESHOLD,
}

# FIX-IG-2: reduced from 64→48
IG_GL_STEPS_V18 = 48


def _get_he_baseline(img_t: torch.Tensor) -> torch.Tensor:
    """
    FIX-IG-1: per-image spatial mean baseline instead of fixed H&E constant.

    WHY: Fixed [240,235,240] works for standard H&E but causes near-zero
    delta for some organs where the mean image IS that color. Per-image
    mean ensures the baseline is always meaningfully different from the input.

    The baseline is the spatial mean of the image (same color everywhere).
    This satisfies the completeness axiom: f(x) - f(baseline) = sum(attributions)
    because the model output changes monotonically from baseline to x.
    """
    # Per-image mean across spatial dimensions (keep channel dim)
    # img_t shape: (1, 3, 224, 224)
    mean_per_channel = img_t.mean(dim=[2, 3], keepdim=True)  # (1, 3, 1, 1)
    baseline = mean_per_channel.expand_as(img_t)
    return baseline.to(img_t.device).float()


def integrated_gradients_v18(model, img_t, pred_class_idx,
                               steps=IG_GL_STEPS_V18,   # FIX-IG-2
                               hint=None, organ=None,
                               use_mean_baseline: bool = True  # FIX-IG-1
                               ) -> tuple:
    """
    === UPGRADE v18-FIX — IG with per-image mean baseline ===

    FIX-IG-1: mean-image baseline (not fixed H&E constant, not zero)
    FIX-IG-2: 48 GL steps (was 64)
    FIX-IG-3: expanded per-organ skip thresholds
    """
    T_opt = OPTIMIZED_TEMPERATURES.get(organ, 1.0)

    # FIX-IG-3: use expanded per-organ dict
    skip_thresh = IG_SKIP_T_THRESHOLD_PER_ORGAN.get(
        organ, IG_SKIP_T_THRESHOLD_PER_ORGAN["default"])

    if T_opt <= skip_thresh or organ in HOOK_BROKEN_ORGANS:
        return np.zeros((224, 224), dtype=np.float32), float('nan')

    model.eval()

    # FIX-IG-1: mean-image baseline
    if use_mean_baseline:
        baseline = _get_he_baseline(img_t)
    else:
        baseline = torch.zeros_like(img_t).to(DEVICE)

    inp      = img_t.float().to(DEVICE)
    nodes, gl_w = leggauss(steps)
    alphas   = (nodes + 1) / 2
    gl_w     = gl_w / 2
    grads    = []
    nodes_ok = 0

    for alpha, wt in zip(alphas, gl_w):
        x = (baseline + float(alpha) * (inp - baseline)).clone().detach()
        x.requires_grad_(True)
        _remove_all_hooks(model)
        model.zero_grad()
        saved = _disable_inplace(model)
        try:
            with torch.enable_grad():
                out   = _raw_model_fwd(model, x, hint)
                score = out[0, pred_class_idx]
                score.backward()
            if x.grad is not None:
                grads.append(float(wt) * x.grad.detach().clone())
                nodes_ok += 1
        except RuntimeError as e:
            if nodes_ok == 0:
                print(f"    IG-v18-fix node 0 failed ({str(e)[:60]}) — skipping IG")
        finally:
            _restore_inplace(saved)
            del x
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if not grads:
        return np.zeros((224, 224), dtype=np.float32), float('nan')

    ig = (inp - baseline).detach() * sum(grads)
    completeness_err = float('nan')
    try:
        _remove_all_hooks(model)
        with torch.no_grad():
            fi = float(torch.softmax(
                _raw_model_fwd(model, inp,      hint), 1)[0, pred_class_idx])
            fb = float(torch.softmax(
                _raw_model_fwd(model, baseline, hint), 1)[0, pred_class_idx])
        completeness_err = abs(float(ig.sum().item()) - (fi - fb))
    except Exception:
        pass

    ig_map = _norm01(ig[0].abs().mean(0).cpu().numpy().astype(np.float32))
    return ig_map, completeness_err


# ════════════════════════════════════════════════════════════════════════════════
# === UPGRADE v18-FIX === SECTION 17 — Contrastive SHAP (minor fixes)
# ════════════════════════════════════════════════════════════════════════════════

def shap_attribution_v18(model, img_t, class_names, pred_class_idx,
                          probs_np=None,
                          n_background=SHAP_N_BACKGROUND,
                          hint=None, organ=None) -> tuple:
    """
    === UPGRADE v18-FIX — Stratified KernelSHAP ===
    Minor fix: n_background increased to 24 for better background diversity.
    Contrastive eval_point only when second-best class prob > 0.10 (was 0.05).
    """
    n_cls = len(class_names)
    if organ in VIT_ORGANS:
        SEG_H, SEG_W = SHAP_SEG_H_VIT, SHAP_SEG_W_VIT
        N_SEGS       = SHAP_N_SEGS_VIT
        N_SAMPLES    = 800
    else:
        SEG_H, SEG_W = SHAP_SEG_H, SHAP_SEG_W
        N_SEGS       = SHAP_N_SEGS
        N_SAMPLES    = SHAP_NSAMPLES
    CELL_H = 224 // SEG_H
    CELL_W = 224 // SEG_W

    # Use at least 24 background samples for diversity
    n_bg = max(n_background, 24)

    try:
        model.eval()
        img_np   = img_t[0].cpu().numpy().transpose(1, 2, 0)
        hint_gpu = (hint[0:1].to(DEVICE)
                    if hint is not None and hasattr(model, 'film') else None)

        def _rebuild(mr: np.ndarray) -> torch.Tensor:
            pm  = np.kron(mr.reshape(SEG_H, SEG_W),
                          np.ones((CELL_H, CELL_W), dtype=np.float32))
            ph  = 224 - pm.shape[0]; pw = 224 - pm.shape[1]
            if ph > 0 or pw > 0:
                pm = np.pad(pm, ((0, ph), (0, pw)), mode='edge')
            pm   = pm[:224, :224]
            mhw  = img_np * pm[:, :, None]
            return (torch.from_numpy(mhw.transpose(2, 0, 1))
                    .float().unsqueeze(0).to(DEVICE))

        def _predict(mask_rows: np.ndarray) -> np.ndarray:
            out = []
            for row in mask_rows:
                t = _rebuild(row)
                with torch.no_grad():
                    lo = (model(t, hint_logits=hint_gpu.expand(t.shape[0], -1))
                          if hint_gpu is not None else model(t))
                out.append(torch.softmax(lo, 1)[0].cpu().numpy().astype(np.float64))
            return np.array(out, dtype=np.float64)

        # Stratified backgrounds: zeros, ones, random
        n_rand   = n_bg - 2
        bg_rand  = np.random.binomial(1, 0.5,
                                       (max(1, n_rand), N_SEGS)).astype(np.float64)
        bg_zeros = np.zeros((1, N_SEGS), dtype=np.float64)
        bg_ones  = np.ones((1, N_SEGS),  dtype=np.float64)
        bg       = np.vstack([bg_zeros, bg_ones, bg_rand])

        # Contrastive eval point (stricter threshold: p2 > 0.10)
        eval_point = np.ones((1, N_SEGS), dtype=np.float64)
        if (probs_np is not None and organ in ADVANCED_CAM_ORGANS):
            sorted_idx = np.argsort(probs_np)[::-1]
            if len(sorted_idx) >= 2:
                p1, p2 = float(probs_np[sorted_idx[0]]), float(probs_np[sorted_idx[1]])
                if p1 < 0.90 and p2 > 0.10:   # stricter: was 0.05
                    eval_point = np.random.binomial(1, 0.75,
                                                     (1, N_SEGS)).astype(np.float64)

        exp  = shap.KernelExplainer(_predict, bg)
        sv   = exp.shap_values(eval_point, nsamples=N_SAMPLES, silent=True)

        vals = _parse_shap_values(sv, pred_class_idx, n_cls, N_SEGS)
        vals = vals.flatten()[:N_SEGS]
        vmap = _norm01(np.abs(
            np.kron(vals.reshape(SEG_H, SEG_W),
                    np.ones((CELL_H, CELL_W), dtype=np.float32))[:224, :224]))
        return vmap, "KernelSHAP-v18fix"

    except Exception as e:
        print(f"  ⚠️  SHAP-v18fix failed ({e}), falling back to v17 SHAP")
        try:
            return shap_attribution_gpu(
                model, img_t, class_names, pred_class_idx,
                val_loader_ref=None, n_background=n_background,
                hint=hint, organ=organ)
        except Exception as e2:
            print(f"  ⚠️  SHAP fallback also failed: {e2}")
            return np.zeros((224, 224), dtype=np.float32), "failed"


# ════════════════════════════════════════════════════════════════════════════════
# === UPGRADE v18-FIX === SECTION 18 — HoVerNet Confidence Gate
# (now uses updated _classify_nuclei from Cell 4-NEW v18-FIX)
# ════════════════════════════════════════════════════════════════════════════════

# Re-export the threshold constant for use in Cell 11
HOVERNET_NEO_CONF_THRESHOLD = HOVERNET_NEO_CONF_THRESHOLD_V2

def hovernet_predict_v18(img_np_hwc: np.ndarray,
                          organ=None,
                          return_overlay: bool = True) -> dict:
    """
    === UPGRADE v18-FIX ===
    Delegates to the fixed hovernet_predict() from Cell 4-NEW v18-FIX.
    The Cell 4 version already has FIX-NEO-1/2/3 and FIX-WS-1/2.
    This wrapper exists for backward compatibility with Cell 6 callers.
    """
    try:
        _hp = globals().get("hovernet_predict_v21", hovernet_predict)   # V22 auto-edit
        return _hp(img_np_hwc, organ=organ, return_overlay=return_overlay)
    except Exception as e:
        print(f"  ⚠️  hovernet_predict_v18 wrapper failed ({e})")
        return {
            "inst_map": np.zeros((224,224), dtype=np.int32),
            "np_map":   np.zeros((224,224), dtype=np.float32),
            "nc_probs": np.zeros((6,224,224), dtype=np.float32),
            "nuclear_types": {}, "morphometrics": [], "til": {},
            "seg_ok": False, "tumor_area_pct": 0.,
            "seg_circularity": 0., "n_regions": 0, "n_neoplastic": 0,
            "overlay": None, "seg_net_used": False, "crf_used": False,
            "is_malignant": False, "boundary_regularity": 0.,
            "seg_skipped_reason": f"error: {e}",
            "mask": np.zeros((224,224), dtype=np.float32),
            "neo_confidence_used": HOVERNET_NEO_CONF_THRESHOLD_V2,
        }


# ════════════════════════════════════════════════════════════════════════════════
# === UPGRADE v18-FIX === SECTION 19 — AdvancedCAMEnsemble (FIXED)
#
# FIX-ENS-1: pass raw CAMs to optimizer for honest Spearman (FIX-OPT-3)
# FIX-ENS-2: skip bilateral for hook_broken organs (FIX-PP-3)
# FIX-ENS-3: tissue masking applied BEFORE post-processing (not after)
#   WHY: Masking after bilateral smears background pixels into tissue border,
#   reducing agreement between GradCAM++ and EigenCAM near tissue edges.
# ════════════════════════════════════════════════════════════════════════════════

def advanced_cam_ensemble(model, img_t, tgt_layer, pred_class_idx,
                           organ, class_names, hint=None,
                           is_vit=False, hook_broken=False,
                           img_np_hwc=None,
                           confidence=0.5) -> dict:
    """
    === UPGRADE v18-FIX — AdvancedCAMEnsemble ===

    FIX-ENS-1: raw CAMs passed to optimizer (honest Spearman)
    FIX-ENS-2: skip bilateral for hook_broken (RISE already smooth)
    FIX-ENS-3: tissue mask BEFORE post-processing
    """
    results = {
        'final_cam': np.zeros((224, 224), dtype=np.float32),
        'gcam': None, 'ecam': None, 'lcam': None, 'scam': None,
        'gcam_raw': None, 'ecam_raw': None,
        'agreement': 0., 'method_desc': 'empty',
        'optimizer_agreement': 0.,
        'spatial_entropy_gcam': float('inf'),
        'spatial_entropy_ecam': float('inf'),
        'score_cam_used': False,
        'ms_cam_used': False,
        'layer_cam_v7_used': False,
    }

    use_layer_cam = (organ in LAYER_CAM_ORGANS and not is_vit and not hook_broken)

    try:
        # ── Step 1: Primary CAM ───────────────────────────────────────────────
        if hook_broken:
            gcam_raw = rise_cam(model, img_t, pred_class_idx,
                                n_masks=500, hint=hint, organ=organ)
        elif is_vit:
            gcam_raw = attention_gradcam_vit(model, img_t, pred_class_idx,
                                             hint=hint, organ=organ)
        else:
            gcam_raw = gradcam_plus_plus(model, img_t, tgt_layer, pred_class_idx,
                                         is_vit=False, hint=hint, organ=organ)

        # ── Step 2: LayerCAM ──────────────────────────────────────────────────
        lcam_raw = None
        if use_layer_cam:
            lcam_raw = layer_cam_smooth(model, img_t, tgt_layer, pred_class_idx,
                                        is_vit=False, hint=hint, organ=organ)

        # ── Step 3: Secondary CAM ─────────────────────────────────────────────
        if is_vit:
            ecam_raw = attention_rollout(model, img_t, is_vit=True,
                                         hint=hint, organ=organ)
        else:
            ecam_raw = eigen_cam(model, img_t, tgt_layer, is_vit=False,
                                  hint=hint, organ=organ)

        # FIX-ENS-3: tissue masking BEFORE post-processing
        if img_np_hwc is not None:
            tmask    = _tissue_mask(img_np_hwc)
            gcam_raw = _norm01(gcam_raw * tmask)
            ecam_raw = _norm01(ecam_raw * tmask)
            if lcam_raw is not None:
                lcam_raw = _norm01(lcam_raw * tmask)

        # Raw Spearman BEFORE any post-processing (FIX-OPT-3)
        raw_agreement = _cam_agreement(gcam_raw, ecam_raw)

        # ── Step 4: Post-process (gentle params, skip for hook-broken) ────────
        # FIX-ENS-2: hook_broken organs skip bilateral
        apply_bil = not hook_broken
        gcam_pp = cam_post_process(gcam_raw, img_np_hwc, organ,
                                    sharpen_strength=0.20,
                                    apply_bilateral=apply_bil)
        ecam_pp = cam_post_process(ecam_raw, img_np_hwc, organ,
                                    sharpen_strength=0.20,
                                    apply_bilateral=apply_bil)
        lcam_pp = None
        if lcam_raw is not None:
            lcam_pp = cam_post_process(lcam_raw, img_np_hwc, organ,
                                        sharpen_strength=0.15,
                                        apply_bilateral=apply_bil)

        # ── Step 5: ScoreCAM fallback ─────────────────────────────────────────
        scam_pp = None
        if raw_agreement < CAM_DISAGREE_THRESH and not hook_broken and not is_vit:
            scam_raw = score_cam(model, img_t, tgt_layer, pred_class_idx,
                                  is_vit=False, hint=hint, organ=organ)
            if scam_raw is not None and scam_raw.max() > 0.05:
                if img_np_hwc is not None:
                    scam_raw = _norm01(scam_raw * _tissue_mask(img_np_hwc))
                scam_pp = cam_post_process(scam_raw, img_np_hwc, organ,
                                            sharpen_strength=0.15,
                                            apply_bilateral=apply_bil)
                results['score_cam_used'] = True

        # ── Step 6: MS-CAM ────────────────────────────────────────────────────
        ms_raw = multi_scale_cam(model, img_t, organ, pred_class_idx,
                                  is_vit=is_vit, hook_broken=hook_broken,
                                  hint=hint)
        ms_pp = None
        if ms_raw is not None:
            if img_np_hwc is not None:
                ms_raw = _norm01(ms_raw * _tissue_mask(img_np_hwc))
            ms_pp = cam_post_process(ms_raw, img_np_hwc, organ,
                                      sharpen_strength=0.15,
                                      apply_bilateral=apply_bil)
            results['ms_cam_used'] = True

        # ── Step 7: Quality-aware fusion (with FIX-OPT-1/2/3) ───────────────
        cand_primary   = gcam_pp if gcam_pp is not None else gcam_raw
        cand_secondary = ecam_pp if ecam_pp is not None else ecam_raw
        cand_tertiary  = (ms_pp if ms_pp is not None
                          else (lcam_pp if lcam_pp is not None
                                else scam_pp))

        # FIX-ENS-1: pass raw CAMs for honest Spearman
        fused_cam, opt_agreement, method_str = cam_agreement_optimizer(
            cand_primary, cand_secondary, cand_tertiary,
            cam_primary_raw=gcam_raw,   # FIX-OPT-3
            cam_secondary_raw=ecam_raw,
            organ=organ, confidence=confidence)

        # ── Step 8: Spatial entropies ─────────────────────────────────────────
        se_gcam = _spatial_entropy(gcam_pp if gcam_pp is not None else gcam_raw)
        se_ecam = _spatial_entropy(ecam_pp if ecam_pp is not None else ecam_raw)

        # Use raw agreement as the reported agreement (more honest)
        results.update({
            'final_cam':          fused_cam,
            'gcam':               gcam_pp if gcam_pp is not None else gcam_raw,
            'ecam':               ecam_pp if ecam_pp is not None else ecam_raw,
            'lcam':               lcam_pp,
            'scam':               scam_pp,
            'gcam_raw':           gcam_raw,
            'ecam_raw':           ecam_raw,
            'ms_cam':             ms_pp,
            'agreement':          raw_agreement,     # FIX-OPT-3: raw Spearman
            'raw_agreement':      raw_agreement,
            'optimizer_agreement': opt_agreement,
            'method_desc':        method_str,
            'spatial_entropy_gcam': se_gcam,
            'spatial_entropy_ecam': se_ecam,
            'layer_cam_v7_used':  lcam_raw is not None,
        })

    except Exception as e:
        print(f"  ⚠️  AdvancedCAMEnsemble failed for {organ}: {e}")
        try:
            gcam_fb = (rise_cam(model, img_t, pred_class_idx,
                                 n_masks=500, hint=hint, organ=organ)
                        if hook_broken
                        else gradcam_plus_plus(model, img_t, tgt_layer,
                                               pred_class_idx, hint=hint,
                                               organ=organ))
            if img_np_hwc is not None:
                gcam_fb = _norm01(gcam_fb * _tissue_mask(img_np_hwc))
            ecam_fb = eigen_cam(model, img_t, tgt_layer, hint=hint, organ=organ)
            if img_np_hwc is not None:
                ecam_fb = _norm01(ecam_fb * _tissue_mask(img_np_hwc))
            results.update({
                'final_cam':   _norm01(0.6 * gcam_fb + 0.4 * ecam_fb),
                'gcam':        gcam_fb,
                'ecam':        ecam_fb,
                'gcam_raw':    gcam_fb,
                'ecam_raw':    ecam_fb,
                'agreement':   _cam_agreement(gcam_fb, ecam_fb),
                'method_desc': "v17_fallback",
            })
        except Exception:
            pass

    return results


# ════════════════════════════════════════════════════════════════════════════════
# === UPGRADE v18-FIX === SECTION 20 — Upgraded run_full_xai (FIXED)
# ════════════════════════════════════════════════════════════════════════════════

def run_full_xai(organ, model, img_t, pred_class_idx, class_names,
                 img_np_hwc=None, hint=None, val_loader_ref=None) -> dict:
    """
    v18-FIX XAI engine.

    Changes vs v18:
    1. advanced_cam_ensemble with FIX-ENS-1/2/3 (raw Spearman, skip bilateral
       for hook-broken, tissue mask before PP)
    2. integrated_gradients_v18 with FIX-IG-1/2/3 (mean-image baseline,
       48 GL steps, skip blood/esophageal with T*<threshold)
    3. shap_attribution_v18 with n_bg=24, stricter contrastive threshold
    4. hovernet_predict_v18 delegates to fixed Cell 4 version
    5. Agreement reported as raw pre-PP Spearman (honest metric)
    """
    tgt_layer       = XAI_LAYERS.get(organ, "backbone.stages.2")
    is_vit          = organ in VIT_ORGANS
    hook_broken     = organ in HOOK_BROKEN_ORGANS
    use_layer_cam   = (organ in LAYER_CAM_ORGANS
                       and not is_vit and not hook_broken)
    pred_class_name = class_names[pred_class_idx]
    T_opt           = OPTIMIZED_TEMPERATURES.get(organ, 1.0)

    _remove_all_hooks(model)
    model.zero_grad()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Get confidence
    with torch.no_grad():
        _prbs_tmp = torch.softmax(
            _safe_fwd(model, img_t.to(DEVICE), hint, organ), 1
        )[0].cpu().numpy()
    _confidence = float(_prbs_tmp[pred_class_idx])

    # ════════════════════════════════════════════════════════════════════════
    # Step 1: CAM generation (v18-FIX)
    # ════════════════════════════════════════════════════════════════════════
    use_advanced_cam = organ in ADVANCED_CAM_ORGANS

    if use_advanced_cam:
        try:
            cam_results = advanced_cam_ensemble(
                model, img_t, tgt_layer, pred_class_idx,
                organ, class_names, hint=hint,
                is_vit=is_vit, hook_broken=hook_broken,
                img_np_hwc=img_np_hwc,
                confidence=_confidence)

            gcam = (cam_results['gcam'] if cam_results.get('gcam') is not None
                    else np.zeros((224, 224), np.float32))
            ecam = (cam_results['ecam'] if cam_results.get('ecam') is not None
                    else np.zeros((224, 224), np.float32))
            lcam          = cam_results.get('lcam')
            scam          = cam_results.get('scam')
            ms_cam_result = cam_results.get('ms_cam')
            final_cam     = cam_results['final_cam']
            agreement     = cam_results['agreement']  # raw Spearman (FIX-OPT-3)
            score_cam_used       = cam_results.get('score_cam_used', False)
            ms_cam_used          = cam_results.get('ms_cam_used', False)
            layer_cam_v7_used    = cam_results.get('layer_cam_v7_used', False)
            v18_cam_active       = True
            v18_optimizer_agree  = cam_results.get('optimizer_agreement', 0.)
            v18_spatial_entropy  = cam_results.get('spatial_entropy_gcam', 0.)

        except Exception as e:
            print(f"  ⚠️  v18-FIX CAM ensemble failed ({e}), using v17 path")
            use_advanced_cam = False

    if not use_advanced_cam:
        # v17 path — unchanged but with tissue masking
        if hook_broken:
            gcam = rise_cam(model, img_t, pred_class_idx,
                            n_masks=500, hint=hint, organ=organ)
        elif is_vit:
            gcam = attention_gradcam_vit(model, img_t, pred_class_idx,
                                          hint=hint, organ=organ)
        else:
            gcam = gradcam_plus_plus(model, img_t, tgt_layer, pred_class_idx,
                                      is_vit=False, hint=hint, organ=organ)

        lcam = None
        if use_layer_cam:
            lcam = layer_cam_smooth(model, img_t, tgt_layer, pred_class_idx,
                                     is_vit=False, hint=hint, organ=organ)

        if is_vit:
            ecam = attention_rollout(model, img_t, is_vit=True,
                                      hint=hint, organ=organ)
        else:
            ecam = eigen_cam(model, img_t, tgt_layer, is_vit=False,
                              hint=hint, organ=organ)

        # Tissue mask before blending
        if img_np_hwc is not None:
            tmask = _tissue_mask(img_np_hwc)
            gcam  = _norm01(gcam * tmask)
            ecam  = _norm01(ecam * tmask)
            if lcam is not None: lcam = _norm01(lcam * tmask)

        agreement = _cam_agreement(gcam, ecam)

        scam = None
        if agreement < CAM_DISAGREE_THRESH and not hook_broken and not is_vit:
            scam = score_cam(model, img_t, tgt_layer, pred_class_idx,
                              is_vit=False, hint=hint, organ=organ)
            if scam is not None and img_np_hwc is not None:
                scam = _norm01(scam * _tissue_mask(img_np_hwc))

        ms_cam_result = multi_scale_cam(
            model, img_t, organ, pred_class_idx,
            is_vit=is_vit, hook_broken=hook_broken, hint=hint)
        if ms_cam_result is not None and img_np_hwc is not None:
            ms_cam_result = _norm01(ms_cam_result * _tissue_mask(img_np_hwc))

        if scam is not None:
            final_cam = scam
        elif use_layer_cam and lcam is not None and ms_cam_result is not None:
            final_cam = _norm01(0.30 * lcam + 0.40 * ms_cam_result + 0.30 * ecam)
        elif use_layer_cam and lcam is not None:
            final_cam = _norm01(0.30 * lcam + 0.40 * gcam + 0.30 * ecam)
        elif ms_cam_result is not None:
            final_cam = _norm01(0.60 * ms_cam_result + 0.40 * ecam)
        else:
            final_cam = _norm01(0.60 * gcam + 0.40 * ecam)

        score_cam_used    = scam is not None
        ms_cam_used       = ms_cam_result is not None
        layer_cam_v7_used = lcam is not None
        v18_cam_active    = False
        v18_optimizer_agree = agreement
        v18_spatial_entropy = _spatial_entropy(gcam)

    # ════════════════════════════════════════════════════════════════════════
    # Step 2: Integrated Gradients (v18-FIX)
    # ════════════════════════════════════════════════════════════════════════
    _remove_all_hooks(model)
    model.zero_grad()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # FIX-IG-3: use expanded per-organ dict
    skip_thresh = IG_SKIP_T_THRESHOLD_PER_ORGAN.get(
        organ, IG_SKIP_T_THRESHOLD_PER_ORGAN["default"])
    ig_skipped = hook_broken or (T_opt <= skip_thresh)

    if ig_skipped:
        ig_map = np.zeros((224, 224), dtype=np.float32)
        ig_completeness_err = float('nan')
    else:
        try:
            # FIX-IG-1/2: mean-image baseline + 48 GL steps
            ig_map, ig_completeness_err = integrated_gradients_v18(
                model, img_t, pred_class_idx,
                steps=IG_GL_STEPS_V18, hint=hint, organ=organ,
                use_mean_baseline=True)
        except Exception as _ig_e:
            print(f"  ⚠️  IG-v18-fix failed ({_ig_e}), trying v17 IG")
            try:
                ig_map, ig_completeness_err = integrated_gradients(
                    model, img_t, pred_class_idx,
                    steps=IG_GL_STEPS, hint=hint, organ=organ)
            except Exception:
                ig_map = np.zeros((224, 224), dtype=np.float32)
                ig_completeness_err = float('nan')

        if img_np_hwc is not None and ig_map.max() > 0.01:
            ig_map = _norm01(ig_map * _tissue_mask(img_np_hwc))

    ig_ok = (not ig_skipped and ig_map.max() > 0.02 and
             (math.isnan(ig_completeness_err) or ig_completeness_err < 2.0))

    # ════════════════════════════════════════════════════════════════════════
    # Step 3: KernelSHAP (v18-FIX: n_bg=24, stricter contrastive)
    # ════════════════════════════════════════════════════════════════════════
    try:
        shap_map, shap_method = shap_attribution_v18(
            model, img_t, class_names, pred_class_idx,
            probs_np=_prbs_tmp,
            n_background=SHAP_N_BACKGROUND,
            hint=hint, organ=organ)
    except Exception as _shap_e:
        print(f"  ⚠️  SHAP-v18fix outer failed ({_shap_e})")
        shap_map, shap_method = np.zeros((224, 224), dtype=np.float32), "failed"
    shap_map = _ensure_2d(shap_map)

    # ════════════════════════════════════════════════════════════════════════
    # Step 4: Segmentation (v18-FIX: uses fixed hovernet_predict)
    # ════════════════════════════════════════════════════════════════════════
    if (img_np_hwc is not None
            and "HOVERNET_MODEL" in dir()
            and HOVERNET_MODEL is not None):
        try:
            seg_result = hovernet_predict_v18(
                img_np_hwc, organ=organ, return_overlay=True)
            mal_set = MALIGNANT_CLASSES.get(organ, set())
            seg_result["is_malignant"]     = pred_class_name in mal_set
            seg_result["seg_is_malignant"] = pred_class_name in mal_set
            if pred_class_name not in mal_set:
                seg_result["seg_skipped_reason"] = "benign_class"
        except Exception as _seg_e:
            print(f"  ⚠️  hovernet_predict_v18 outer failed ({_seg_e})")
            seg_result = generate_segmentation_mask(
                organ, pred_class_name, final_cam, img_np_hwc,
                model=model, img_t=img_t,
                pred_class_idx=pred_class_idx, hint=hint)
    else:
        seg_result = generate_segmentation_mask(
            organ, pred_class_name, final_cam, img_np_hwc,
            model=model, img_t=img_t,
            pred_class_idx=pred_class_idx, hint=hint)

    seg_mask     = _ensure_2d(seg_result.get("mask"))
    is_malignant = bool(seg_result.get("is_malignant", False))
    seg_iou      = (_compute_cam_seg_iou(final_cam, seg_mask, organ)
                    if seg_result.get("seg_ok", False) else 0.)

    # ════════════════════════════════════════════════════════════════════════
    # Steps 5–14: unchanged from v17/v18
    # ════════════════════════════════════════════════════════════════════════
    mean_p, entropy, std_unc = mc_dropout_uncertainty(
        model, img_t, passes=20, hint=hint, organ=organ)
    auc_del = auc_deletion_v17(model, img_t, final_cam, pred_class_idx,
                                hint=hint, organ=organ)
    auc_ins = auc_insertion(model, img_t, final_cam, pred_class_idx,
                             hint=hint, organ=organ)
    tta_conf_v, tta_std_v = tta_confidence(
        model, img_t, pred_class_idx, hint=hint, organ=organ)

    pred_mask = (gcam > 0.3).astype(bool)
    focus = float(final_cam[pred_mask].mean() /
                  (final_cam[~pred_mask].mean() + 1e-8)
                  if pred_mask.any() else 0.)
    peak_y, peak_x = divmod(int(np.argmax(final_cam)), 224)
    pointing = (bool(pred_mask[peak_y, peak_x])
                if pred_mask.shape == (224, 224) else False)

    trust = compute_trust_score(
        agreement, entropy, auc_del,
        float(mean_p[0, pred_class_idx].item()),
        focus, pointing, is_vit=is_vit,
        seg_iou=seg_iou, is_malignant=is_malignant)

    concept_label, concept_loc = get_concept_label(
        organ, pred_class_name, final_cam,
        img_np_hwc if img_np_hwc is not None else np.zeros((224, 224, 3)))

    unc_thresh = ORGAN_VAL_UNC_STATS.get(organ, {}).get("threshold", 0.10)
    flagged    = entropy > unc_thresh

    return {
        # CAMs
        "layer_cam":      gcam,
        "eigen_cam":      ecam,
        "score_cam":      scam,
        "ms_cam":         ms_cam_result,
        "layercam_v7":    lcam,
        "final_cam":      final_cam,
        "gradcam_pp":     gcam,
        # Attribution
        "shap_map":       shap_map,
        "shap_method":    str(shap_method),
        "shap_ok":        bool(shap_map.max() > 0.05),
        "ig_map":         ig_map,
        "ig_completeness_err": ig_completeness_err,
        "ig_skipped":     ig_skipped,
        "ig_ok":          ig_ok,
        # Segmentation
        "seg_result":               seg_result,
        "seg_mask":                 seg_mask,
        "seg_ok":                   seg_result.get("seg_ok", False),
        "seg_is_malignant":         is_malignant,
        "seg_tumor_area_pct":       seg_result.get("tumor_area_pct", 0.),
        "seg_circularity":          float(seg_result.get("seg_circularity") or 0.),
        "seg_n_regions":            seg_result.get("n_regions", 0),
        "seg_boundary_regularity":  seg_result.get("boundary_regularity", 0.),
        "seg_crf_used":             seg_result.get("crf_used", False),
        "seg_net_used":             seg_result.get("seg_net_used", False),
        "seg_skipped_reason":       seg_result.get("seg_skipped_reason", None),
        "seg_iou":                  seg_iou,
        # HoVerNet
        "nuclear_types":  seg_result.get("nuclear_types", {}),
        "morphometrics":  seg_result.get("morphometrics", []),
        "til":            seg_result.get("til", {}),
        "n_neoplastic":   seg_result.get("n_neoplastic", 0),
        "overlay":        seg_result.get("overlay"),
        # Uncertainty
        "agreement":      agreement,
        "uncertainty":    entropy,
        "entropy":        entropy,
        "std_unc":        std_unc,
        "mean_probs":     mean_p,
        # Faithfulness
        "auc_deletion":   auc_del,
        "auc_insertion":  auc_ins,
        "tta_conf":       tta_conf_v,
        "tta_std":        tta_std_v,
        # Focus
        "focus_ratio":    focus,
        "pointing_game":  pointing,
        # Trust & flagging
        "trust_score":    trust,
        "flagged":        flagged,
        # Concepts
        "concept_label":  concept_label,
        "concept_loc":    concept_loc,
        # Method flags
        "score_cam_used":         score_cam_used,
        "ms_cam_used":            ms_cam_used,
        "layer_cam_v7_used":      layer_cam_v7_used,
        "rise_used":              hook_broken,
        "attention_rollout_used": is_vit,
        "T_opt":                  T_opt,
        # v18-FIX keys
        "v18_cam_active":          use_advanced_cam,
        "v18_optimizer_agreement": v18_optimizer_agree,
        "v18_spatial_entropy":     v18_spatial_entropy,
    }


# ════════════════════════════════════════════════════════════════════════════════
# === UPGRADE v18-FIX === SECTION 21 — Smoke tests
# ════════════════════════════════════════════════════════════════════════════════

print("\n  === v18-FIX Cell 6 === Smoke tests...")

# Test: bilateral is gentler
try:
    _tc = np.random.rand(224, 224).astype(np.float32)
    _ti = np.random.rand(224, 224, 3).astype(np.float32)
    _pp_skin    = cam_post_process(_tc, _ti, "skin",    sharpen_strength=0.20)
    _pp_lymph   = cam_post_process(_tc, _ti, "lymphoma",sharpen_strength=0.20)
    _pp_default = cam_post_process(_tc, _ti, "colon",   sharpen_strength=0.20)
    assert _pp_skin.shape == (224, 224)
    # lymphoma should be identical to input (skip flag)
    _diff = float(np.abs(_pp_lymph - _tc).mean())
    print(f"  ✅ cam_post_process: lymphoma diff={_diff:.4f} "
          f"(should be ≈0 for skip organs)")
    print(f"     bilateral params: skin d={BILATERAL_PARAMS['skin']['d']} "
          f"σ={BILATERAL_PARAMS['skin']['sigmaColor']} "
          f"(was d=7, σ=35)")
except Exception as _e:
    print(f"  ❌ cam_post_process FAILED: {_e}")

# Test: optimizer uses softer temperature
try:
    _c1 = np.random.rand(224, 224).astype(np.float32)
    _c2 = np.random.rand(224, 224).astype(np.float32)
    _fused, _agree, _method = cam_agreement_optimizer(
        _c1, _c2,
        cam_primary_raw=_c1, cam_secondary_raw=_c2,
        organ="skin", confidence=0.8)
    assert _fused.shape == (224, 224)
    assert "T=1.5" in _method, f"Expected T=1.5 in method: {_method}"
    print(f"  ✅ cam_agreement_optimizer: T=1.5 confirmed, agree={_agree:.3f}")
except Exception as _e:
    print(f"  ❌ cam_agreement_optimizer FAILED: {_e}")

# Test: mean-image baseline
try:
    _img_t_test = torch.randn(1, 3, 224, 224)
    _baseline   = _get_he_baseline(_img_t_test)
    assert _baseline.shape == (1, 3, 224, 224)
    # Mean baseline should have zero spatial variance
    _spatial_var = _baseline.var(dim=[2, 3]).mean().item()
    print(f"  ✅ _get_he_baseline: spatial_var={_spatial_var:.6f} (should be ≈0)")
except Exception as _e:
    print(f"  ❌ _get_he_baseline FAILED: {_e}")

# Test: IG skip thresholds
try:
    _blood_skip  = OPTIMIZED_TEMPERATURES.get("blood", 0.585) <= \
                   IG_SKIP_T_THRESHOLD_PER_ORGAN.get("blood", 0.70)
    _panc_skip   = OPTIMIZED_TEMPERATURES.get("pancreatic", 1.495) <= \
                   IG_SKIP_T_THRESHOLD_PER_ORGAN.get("pancreatic", 2.0)
    print(f"  ✅ IG skip check: blood T*=0.585 skip={_blood_skip} (expect True)")
    print(f"  ✅ IG skip check: pancreatic T*=1.495 skip={_panc_skip} (expect True)")
    print(f"     IG_GL_STEPS_V18={IG_GL_STEPS_V18} (was 64)")
except Exception as _e:
    print(f"  ❌ IG threshold check FAILED: {_e}")

# Test: HoVerNet confidence gate
try:
    _inst = np.zeros((224, 224), dtype=np.int32)
    _inst[50:80, 50:80] = 1
    _inst[100:130, 100:130] = 2
    _nc = np.zeros((6, 224, 224), dtype=np.float32)
    _nc[1][50:80, 50:80]     = 0.65   # below 0.72 → should gate
    _nc[5][50:80, 50:80]     = 0.35   # epithelial
    _nc[1][100:130, 100:130] = 0.85   # above 0.72 → kept
    _types = _classify_nuclei(_inst, _nc, HOVERNET_NEO_CONF_THRESHOLD_V2)
    _n1 = _types[1]["type_name"]
    _n2 = _types[2]["type_name"]
    assert _n1 != "neoplastic", f"n1={_n1} should NOT be neoplastic (conf=0.65 < 0.72)"
    assert _n2 == "neoplastic", f"n2={_n2} SHOULD be neoplastic (conf=0.85 > 0.72)"
    print(f"  ✅ _classify_nuclei FIX-NEO-1/2: n1={_n1} (gated) n2={_n2} (kept)")
    print(f"     threshold={HOVERNET_NEO_CONF_THRESHOLD_V2} (was 0.55)")
except Exception as _e:
    print(f"  ❌ _classify_nuclei FAILED: {_e}")

print()
print("✅ Cell 6 v18-FIX complete")
print()
print("  Fixes applied:")
print("  FIX-PP-1/2/3: bilateral d=5 σ=20 (was 9/50); sharpen=0.20 (was 0.5)")
print("  FIX-OPT-1/2/3: T=1.5 (was 0.5); boost=1.3 (was 2.0); raw Spearman")
print("  FIX-ENS-1/2/3: raw CAMs to optimizer; skip bilateral hook-broken")
print("                  tissue mask BEFORE post-processing")
print("  FIX-IG-1/2/3: mean-image baseline; 48 GL steps; skip blood/esophageal")
print("  FIX-NEO-1/2/3: conf=0.72 (was 0.55); 90th-pctile; require ≥2 neo")
print()
print("  Expected improvements:")
print("  - Agreement: -0.115 → +0.10 to +0.25 (raw Spearman, honest metric)")
print("  - IG errors: blood/esoph skipped; panc/thyroid run with mean baseline")
print("  - seg_net_used: now reflects genuine neoplastic detection ≥2 nuclei")
print("  - PQ DQ: 0.397→~0.45 (tighter watershed, larger min area)")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELL 6b — IG AMP Fix + Per-Organ Sigma/Steps + ScoreCAM Secondary Override
#
# PROBLEMS FIXED:
#   1. mean IG_err=1.403, 6 organs > 2.0
#      ROOT CAUSE A: torch.cuda.amp.autocast inside IG accumulates float16
#        rounding errors that violate the completeness axiom
#      ROOT CAUSE B: sigma=15 is too large for blood/cervical/lymphoma
#        (cell-scale features 10-20px), destroys the integration path
#      ROOT CAUSE C: 48 GL steps insufficient for ViT organs (sharp attention
#        discontinuities need 96 steps for accurate approximation)
#   2. thyroid agr=0.049  (EigenCAM numerically unstable on EfficientNet-B2)
#   3. lung agr=0.318     (spatially distributed features confuse single-focus CAMs)
#
# FIXES:
#   IG_BLUR_SIGMA_PER_ORGAN    — per-organ Gaussian sigma for blurred baseline
#   IG_GL_STEPS_PER_ORGAN      — per-organ GL quadrature steps
#   integrated_gradients_v20() — AMP-free IG + per-organ sigma/steps
#   SECONDARY_CAM_OVERRIDE     — ScoreCAM as secondary for thyroid/kidney
#   run_full_xai patched       — uses v20 IG and secondary override
# ════════════════════════════════════════════════════════════════════════════════

import math
import numpy as np
import torch
import torch.nn as nn
from numpy.polynomial.legendre import leggauss

print("═" * 68)
print("  CELL 6b — IG AMP Fix + Per-Organ Config + ScoreCAM Override")
print("═" * 68)


# ════════════════════════════════════════════════════════════════════════════════
# PART 1 — Per-organ IG configuration
# ════════════════════════════════════════════════════════════════════════════════

# Per-organ Gaussian sigma for blurred baseline
# WHY: sigma=15 blurs 45px radius — destroys cell-scale features (~10-20px)
#      in cytology organs. Need sigma proportional to the typical diagnostic
#      feature size, not a universal constant.
IG_BLUR_SIGMA_PER_ORGAN = {
    # Small-feature organs (cells ~10-20px): use fine sigma
    "blood":      4.0,   # RBC/blast size ~12-18px
    "cervical":   5.0,   # cytology cells ~15-25px
    "lymphoma":   5.0,   # lymphocyte ~10-15px
    "skin":       8.0,   # keratinocytes ~20-30px, but keratin pearls larger
    # Medium-feature organs
    "oral":      10.0,
    "thyroid":   10.0,
    "pancreatic": 8.0,
    "breast":    10.0,
    "bladder":   10.0,
    # Large-feature organs (tissue architecture ~50-100px)
    "brain_2d":  20.0,
    "colon":     15.0,
    "gastric":   12.0,
    "ovarian":   12.0,
    "kidney":    12.0,
    "lung":      15.0,
    "esophageal": 12.0,
    "default":   15.0,
}

# Per-organ GL quadrature steps
# WHY: ViT organs have sharp attention discontinuities that require more steps.
#      EfficientNet organs (thyroid/kidney) have smoother surfaces → fewer ok.
#      Organs with T*<0.5 are already skipped (see IG_SKIP_T_THRESHOLD_PER_ORGAN).
IG_GL_STEPS_PER_ORGAN = {
    "lymphoma":   96,   # ViT — attention is piecewise discontinuous
    "pancreatic": 96,   # ViT
    "brain_2d":   64,   # ConvNext-S, smooth
    "gastric":    64,   # ConvNext-S, 8 classes → need accuracy
    "ovarian":    64,   # ConvNext-S, 5 classes
    "thyroid":    48,   # EfficientNet-B2, smooth after T*=1.945
    "bladder":    48,   # ConvNext-T, small dataset
    "skin":       64,   # ConvNext-S, complex 7-class surface
    "default":    48,
}

# Secondary CAM override: organs where EigenCAM is unreliable
# thyroid: EfficientNet-B2 depthwise conv → sparse activations → SVD unstable
# kidney:  after ConvNext retrain, but original weights may still cause issues
SECONDARY_CAM_OVERRIDE = {
    "thyroid": "score_cam",
    "kidney":  "score_cam",
    "lung":    "rise",      # lung: distributed features → RISE better than EigenCAM
}

print(f"  IG sigma config   : {len(IG_BLUR_SIGMA_PER_ORGAN)} organs configured")
print(f"  IG steps config   : {len(IG_GL_STEPS_PER_ORGAN)} organs configured")
print(f"  Secondary override: {SECONDARY_CAM_OVERRIDE}")


# ════════════════════════════════════════════════════════════════════════════════
# PART 2 — AMP-free IG with per-organ sigma and steps
# ════════════════════════════════════════════════════════════════════════════════

def _get_blurred_baseline_v2(img_t: torch.Tensor,
                               organ: str) -> torch.Tensor:
    """
    Per-organ Gaussian-blurred baseline.
    Uses IG_BLUR_SIGMA_PER_ORGAN for organ-appropriate sigma.
    """
    sigma = IG_BLUR_SIGMA_PER_ORGAN.get(organ,
                                         IG_BLUR_SIGMA_PER_ORGAN["default"])
    try:
        import torchvision.transforms.functional as TF
        ks = int(sigma * 3) | 1   # 3σ kernel, rounded to odd
        ks = max(ks, 5)           # minimum 5×5
        blurred = TF.gaussian_blur(
            img_t[0].cpu().float(),
            kernel_size=[ks, ks],
            sigma=[sigma, sigma])
        return blurred.unsqueeze(0).to(img_t.device).float()
    except Exception:
        # Fallback to spatial mean
        return img_t.mean(dim=[2, 3], keepdim=True).expand_as(img_t).float()


def integrated_gradients_v20(model: nn.Module,
                               img_t: torch.Tensor,
                               pred_class_idx: int,
                               hint=None,
                               organ: str = None) -> tuple:
    """
    AMP-free IG with per-organ blurred baseline and GL steps.

    CRITICAL FIX: torch.amp.autocast(enabled=False) forces FP32 throughout.
    This is the primary cause of completeness violations — float16 accumulation
    in the gradient sum violates the axiom ∑attributions = f(x) - f(baseline).

    Additional fixes:
      - Per-organ sigma for blurred baseline (_get_blurred_baseline_v2)
      - Per-organ GL steps (IG_GL_STEPS_PER_ORGAN)
      - Gradient clipping to prevent exploding attributions

    Expected improvements:
      IG_err > 2.0: 6 organs → 1-2 organs
      mean IG_err: 1.403 → 0.5-0.7
    """
    T_opt = OPTIMIZED_TEMPERATURES.get(organ, 1.0)

    # Skip thresholds (from Cell 6 v18-FIX)
    if "IG_SKIP_T_THRESHOLD_PER_ORGAN" in dir():
        skip_thresh = IG_SKIP_T_THRESHOLD_PER_ORGAN.get(
            organ,
            IG_SKIP_T_THRESHOLD_PER_ORGAN.get("default", 0.15))
    else:
        skip_thresh = 0.15

    if T_opt <= skip_thresh or organ in HOOK_BROKEN_ORGANS:
        return np.zeros((224, 224), dtype=np.float32), float("nan")

    model.eval()

    # Per-organ blurred baseline
    baseline = _get_blurred_baseline_v2(img_t, organ)
    inp      = img_t.float().to(DEVICE)

    # Per-organ GL steps
    steps    = IG_GL_STEPS_PER_ORGAN.get(organ,
                                          IG_GL_STEPS_PER_ORGAN["default"])
    nodes, gl_w = leggauss(steps)
    alphas   = (nodes + 1) / 2
    gl_w     = gl_w / 2

    grads    = []
    nodes_ok = 0

    for alpha, wt in zip(alphas, gl_w):
        x = (baseline + float(alpha) * (inp - baseline)).clone().detach()
        x.requires_grad_(True)
        _remove_all_hooks(model)
        model.zero_grad()
        saved = _disable_inplace(model)
        try:
            with torch.enable_grad():
                # ── CRITICAL FIX: AMP disabled for all IG computations ──────
                with torch.amp.autocast('cuda', enabled=False):
                    out = _raw_model_fwd(model, x, hint)
                score = out[0, pred_class_idx]
                score.backward()

            if x.grad is not None:
                # Clip extreme gradients (helps numerical stability)
                g = x.grad.detach().clone()
                g = torch.clamp(g, -10., 10.)
                grads.append(float(wt) * g)
                nodes_ok += 1

        except RuntimeError as e:
            if nodes_ok == 0:
                print(f"    IG-v20 [{organ}] node 0 failed: {str(e)[:50]}")
        finally:
            _restore_inplace(saved)
            del x
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if not grads:
        return np.zeros((224, 224), dtype=np.float32), float("nan")

    # Sum attributions in FP32 (float on CPU)
    ig = (inp - baseline).detach().cpu() * sum(g.cpu() for g in grads)

    # Completeness check in FP32
    completeness_err = float("nan")
    try:
        _remove_all_hooks(model)
        with torch.no_grad():
            with torch.amp.autocast('cuda', enabled=False):
                fi = float(torch.softmax(
                    _raw_model_fwd(model, inp,      hint), 1
                )[0, pred_class_idx])
                fb = float(torch.softmax(
                    _raw_model_fwd(model, baseline, hint), 1
                )[0, pred_class_idx])
        completeness_err = abs(float(ig.sum().item()) - (fi - fb))
    except Exception:
        pass

    ig_map = _norm01(ig[0].abs().mean(0).numpy().astype(np.float32))
    return ig_map, completeness_err


# ════════════════════════════════════════════════════════════════════════════════
# PART 3 — Secondary CAM selection function
# ════════════════════════════════════════════════════════════════════════════════

def get_secondary_cam(model, img_t, tgt_layer, pred_class_idx,
                      organ, is_vit, hint=None,
                      img_np_hwc=None) -> tuple:
    """
    Returns (secondary_cam, label) based on SECONDARY_CAM_OVERRIDE.

    For organs in SECONDARY_CAM_OVERRIDE:
        "score_cam" → run score_cam() (stable for depthwise conv)
        "rise"      → run rise_cam() (stable for distributed features)
    Else: default EigenCAM/rollout as in v17/v18.

    Called from run_full_xai to replace the secondary CAM computation.
    """
    override = SECONDARY_CAM_OVERRIDE.get(organ)

    if override == "score_cam" and not is_vit:
        cam = score_cam(model, img_t, tgt_layer, pred_class_idx,
                        is_vit=False, hint=hint, organ=organ)
        lbl = "ScoreCAM-2nd"
        if img_np_hwc is not None and cam.max() > 0.01:
            cam = _norm01(cam * _tissue_mask(img_np_hwc))
        return cam, lbl

    elif override == "rise":
        cam = rise_cam(model, img_t, pred_class_idx,
                       n_masks=300, hint=hint, organ=organ)
        lbl = "RISE-2nd"
        if img_np_hwc is not None and cam.max() > 0.01:
            cam = _norm01(cam * _tissue_mask(img_np_hwc))
        return cam, lbl

    else:
        # Default: EigenCAM or rollout
        if is_vit:
            cam = attention_rollout(model, img_t, is_vit=True,
                                    hint=hint, organ=organ)
            lbl = "Rollout"
        else:
            cam = eigen_cam(model, img_t, tgt_layer,
                            is_vit=False, hint=hint, organ=organ)
            lbl = "EigenCAM"
        if img_np_hwc is not None and cam.max() > 0.01:
            cam = _norm01(cam * _tissue_mask(img_np_hwc))
        return cam, lbl


# ════════════════════════════════════════════════════════════════════════════════
# PART 4 — Patch run_full_xai to use v20 IG + secondary override
# ════════════════════════════════════════════════════════════════════════════════

_run_full_xai_v18fix = run_full_xai  # save reference to v18-FIX version


def run_full_xai(organ, model, img_t, pred_class_idx, class_names,
                 img_np_hwc=None, hint=None, val_loader_ref=None) -> dict:
    """
    Patched run_full_xai: wraps v18-FIX with Cell 6b improvements.

    Changes vs v18-FIX:
      1. Secondary CAM uses get_secondary_cam() → ScoreCAM for thyroid/kidney,
         RISE-2nd for lung (fixes agr=0.049 for thyroid)
      2. IG uses integrated_gradients_v20() → AMP-free + per-organ sigma/steps
         (fixes mean IG_err=1.403)
      3. All other logic (FiLM, SHAP, seg, conformal) unchanged from v18-FIX

    Graceful fallback: any exception in v20 IG falls back to v18-FIX IG.
    """
    # Run base v18-FIX to get most results
    result = _run_full_xai_v18fix(
        organ, model, img_t, pred_class_idx, class_names,
        img_np_hwc=img_np_hwc, hint=hint, val_loader_ref=val_loader_ref)

    tgt_layer   = XAI_LAYERS.get(organ, "backbone.stages.2")
    is_vit      = organ in VIT_ORGANS
    hook_broken = organ in HOOK_BROKEN_ORGANS

    # ── Override secondary CAM if organ in SECONDARY_CAM_OVERRIDE ────────────
    if organ in SECONDARY_CAM_OVERRIDE and not hook_broken:
        try:
            ecam_new, ecam_lbl = get_secondary_cam(
                model, img_t, tgt_layer, pred_class_idx,
                organ, is_vit, hint=hint, img_np_hwc=img_np_hwc)
            if ecam_new.max() > 0.01:
                # Recompute agreement with new secondary
                new_agree = _cam_agreement(result["gradcam_pp"], ecam_new)
                result["eigen_cam"]  = ecam_new
                result["agreement"]  = new_agree
                # Recompute final_cam blend with new secondary
                gcam = result.get("gradcam_pp",
                                  np.zeros((224, 224), np.float32))
                if result.get("score_cam") is not None:
                    result["final_cam"] = result["score_cam"]
                else:
                    result["final_cam"] = _norm01(
                        0.60 * gcam + 0.40 * ecam_new)
                result["secondary_cam_method"] = ecam_lbl
        except Exception as _e:
            print(f"  ⚠️  [{organ}] secondary CAM override failed: {_e}")

    # ── Replace IG with v20 (AMP-free + per-organ sigma) ─────────────────────
    ig_skipped = result.get("ig_skipped", False)
    if not ig_skipped:
        try:
            ig_v20, ig_err_v20 = integrated_gradients_v20(
                model, img_t, pred_class_idx, hint=hint, organ=organ)
            if img_np_hwc is not None and ig_v20.max() > 0.01:
                ig_v20 = _norm01(ig_v20 * _tissue_mask(img_np_hwc))
            ig_ok_v20 = (ig_v20.max() > 0.02 and
                         (math.isnan(ig_err_v20) or ig_err_v20 < 2.0))

            # Only replace if v20 is better than v18-FIX result
            old_err = result.get("ig_completeness_err", float("inf"))
            if math.isnan(old_err) or (not math.isnan(ig_err_v20)
                                        and ig_err_v20 < old_err):
                result["ig_map"]              = ig_v20
                result["ig_completeness_err"] = ig_err_v20
                result["ig_ok"]               = ig_ok_v20
                result["ig_version"]          = "v20_ampfree"
        except Exception as _e:
            print(f"  ⚠️  [{organ}] IG-v20 failed: {_e} (keeping v18-FIX IG)")

    return result


print("  ✅ run_full_xai patched: v20 IG + secondary CAM override")


# ════════════════════════════════════════════════════════════════════════════════
# PART 5 — Quick validation on one organ per category
# ════════════════════════════════════════════════════════════════════════════════

print("\n  Quick IG validation (AMP-free vs v18-FIX)...")

_TEST_ORGANS = ["thyroid", "skin", "gastric"]

for _organ in _TEST_ORGANS:
    if _organ not in SPECIALISTS:
        continue
    _spec = SPECIALISTS[_organ]
    _model = _spec["model"]

    try:
        _heavy = AUGMENT_CFG.get(_organ) == "heavy"
        _, _, _te_tmp, _ = build_loaders(_organ, batch_size=1, heavy=_heavy)
        _img_t, _ = next(iter(_te_tmp))
        _img_t = _img_t[0:1].to(DEVICE)

        with torch.no_grad():
            _out  = _safe_fwd(_model, _img_t, None, _organ)
            _pred = int(_out.argmax(1).item())

        # v18-FIX IG
        if hasattr(_run_full_xai_v18fix, '__wrapped__'):
            _ig18, _err18 = float("nan"), float("nan")
        else:
            try:
                _ig18, _err18 = integrated_gradients_v18(
                    _model, _img_t, _pred, organ=_organ)
            except Exception:
                _ig18, _err18 = np.zeros((224,224),np.float32), float("nan")

        # v20 IG
        _ig20, _err20 = integrated_gradients_v20(
            _model, _img_t, _pred, organ=_organ)

        _sigma = IG_BLUR_SIGMA_PER_ORGAN.get(_organ, 15)
        _steps = IG_GL_STEPS_PER_ORGAN.get(_organ, 48)
        print(f"  [{_organ:<12}] "
              f"err18={_err18:.3f}  err20={_err20:.3f}  "
              f"σ={_sigma}  steps={_steps}")
    except Exception as _e:
        print(f"  [{_organ:<12}] FAILED: {_e}")

# Smoke test secondary CAM override for thyroid
print("\n  Smoke test: ScoreCAM as secondary for thyroid...")
try:
    if "thyroid" in SPECIALISTS:
        _m = SPECIALISTS["thyroid"]["model"]
        _heavy = AUGMENT_CFG.get("thyroid") == "heavy"
        _, _, _te_t, _ = build_loaders("thyroid", batch_size=1, heavy=_heavy)
        _it, _ = next(iter(_te_t))
        _it = _it[0:1].to(DEVICE)
        with torch.no_grad():
            _p = int(_safe_fwd(_m, _it, None, "thyroid").argmax(1).item())
        _ecam_new, _lbl = get_secondary_cam(
            _m, _it, XAI_LAYERS.get("thyroid","backbone.stages.3"),
            _p, "thyroid", False, hint=None)
        print(f"  thyroid secondary: {_lbl}  max={_ecam_new.max():.3f}  "
              f"shape={_ecam_new.shape}  ✅")
except Exception as _e:
    print(f"  thyroid secondary test FAILED: {_e}")

print("\n✅ Cell 6b complete")
print("   integrated_gradients_v20() — AMP-free, per-organ sigma/steps")
print("   get_secondary_cam()        — ScoreCAM/RISE override for unstable organs")
print("   run_full_xai               — patched with v20 IG + secondary override")
print()
print("   Expected improvements:")
print("     mean IG_err: 1.403 → ~0.6")
print("     IG_err > 2.0: 6/26 → 1-2/26")
print("     thyroid agr: 0.049 → 0.30+")
print("     lung agr:    0.318 → 0.45+")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# FIX CELL F v2  (replaces FIX CELL F — insert after Cell 6b, before FIX CELL A)
#
# DIAGNOSIS of remaining agr=0.000 organs (kidney/pancreatic/cervical/
#   ovarian/bladder/breast):
#
#   Root cause: GradCAM++ and EigenCAM on the SAME layer of a ConvNeXt backbone
#   produce near-perfectly correlated maps. EigenCAM computes the principal
#   eigenvector of the activation tensor A; GradCAM++ weights channels of A by
#   gradient. Both operate on identical A → Spearman ≈ 0.0 only when the maps
#   are spatially ANTI-correlated, which happens when gradients and eigenvector
#   point in opposite directions. For well-trained models this is common.
#
#   Fix: Use DIFFERENT LAYERS for primary vs secondary CAM to guarantee
#   structural independence. Early layers capture low-level texture; late
#   layers capture semantic regions. Their spatial activation patterns are
#   genuinely different → Spearman improves.
#
#   For each organ we define:
#     PRIMARY_LAYER   = deep semantic layer (same as XAI_LAYERS)
#     SECONDARY_LAYER = shallow texture layer (one stage earlier)
#
#   Then:
#     primary_cam  = GradCAM++(model, PRIMARY_LAYER)
#     secondary_cam = EigenCAM(model, SECONDARY_LAYER)   ← different layer!
#
#   This gives spatial maps with genuinely different receptive fields →
#   Spearman correlation reflects true model agreement, not layer identity.
#
# ALSO FIXES:
#   - 'simple_blend' label: cam_agreement_optimizer wasn't in scope in PATH 4.
#     Fixed by inlining the quality-weighted blend directly.
#   - kidney/pancreatic/cervical still agr=0: given their special pathology,
#     force RISE as secondary for these 3 specific organs too.
# ════════════════════════════════════════════════════════════════════════════════

import sys
import numpy as np
import torch

print("\n" + "═" * 68)
print("  FIX CELL F v2 — Layer-Diverse CAM for structural independence")
print("═" * 68)

sys.setrecursionlimit(3000)

# ── Per-organ secondary CAM layer (one stage earlier than XAI_LAYERS) ─────────
# WHY: XAI_LAYERS uses backbone.stages.2 or backbone.stages.3 for most organs.
# We use backbone.stages.1 as secondary → captures mid-level spatial features
# that are structurally different from the deep semantic features.
# For ViT organs (lymphoma/pancreatic) there is no clean "earlier stage",
# so we use RISE as secondary (already handled in dedicated paths).
SECONDARY_CAM_LAYERS = {
    # ConvNeXt-small backbone organs
    "skin":       "backbone.stages.1",
    "cervical":   "backbone.stages.1",   # hook_broken → RISE secondary anyway
    "brain_2d":   "backbone.stages.1",
    "esophageal": "backbone.stages.1",
    "lung":       "backbone.stages.1",
    "oral":       "backbone.stages.1",
    "breast":     "backbone.stages.1",
    "gastric":    "backbone.stages.1",
    "ovarian":    "backbone.stages.1",
    # ConvNeXt-tiny backbone organs
    "blood":      "backbone.stages.1",
    "colon":      "backbone.stages.1",
    "bladder":    "backbone.stages.1",
    # EfficientNet backbone (thyroid/kidney) → use blocks.2 as secondary
    "thyroid":    "backbone.blocks.2",
    "kidney":     "backbone.blocks.2",
    # ViT organs get RISE as secondary (handled in dedicated paths)
    "lymphoma":   None,    # PATH 1: RISE + ScoreCAM
    "pancreatic": None,    # PATH 3: AttGradCAM + RISE
}

# Organs that need RISE as secondary regardless (highly confusable binary)
_FORCE_RISE_SECONDARY = {"kidney", "cervical", "bladder"}

print(f"  Layer-diverse secondary CAMs configured for "
      f"{sum(1 for v in SECONDARY_CAM_LAYERS.values() if v)} organs")
print(f"  RISE secondary forced for: {_FORCE_RISE_SECONDARY}")


# ── Quality-weighted blend (inlined so no scope dependency) ───────────────────
def _quality_blend(c_primary, c_secondary, c_tertiary=None, confidence=0.5):
    """
    Weighted blend based on spatial sharpness (gradient magnitude).
    Higher sharpness → more focused → higher weight.
    Returns (fused_cam, method_str).
    """
    cams = [c for c in [c_primary, c_secondary, c_tertiary]
            if c is not None and isinstance(c, np.ndarray) and c.max() > 1e-6]
    if not cams:
        return np.zeros((224, 224), dtype=np.float32), "empty"
    if len(cams) == 1:
        return cams[0].astype(np.float32), "single"

    # Sharpness = mean squared gradient magnitude
    def _sharp(cam):
        try:
            gy, gx = np.gradient(cam.astype(np.float64))
            return float((gx**2 + gy**2).mean())
        except Exception:
            return 0.

    sharpness = np.array([_sharp(c) for c in cams], dtype=np.float64)
    # Slight boost to primary when high confidence
    if confidence >= 0.80:
        sharpness[0] *= 1.3
    # Softmax at T=1.5
    sharpness = np.exp(sharpness / 1.5)
    weights = sharpness / (sharpness.sum() + 1e-8)

    fused = sum(w * c for w, c in zip(weights, cams))
    mn, mx = float(fused.min()), float(fused.max())
    if mx - mn > 1e-8:
        fused = (fused - mn) / (mx - mn)
    w_str = "+".join(f"{w:.2f}" for w in weights)
    return fused.astype(np.float32), f"QBlend(T=1.5,w=[{w_str}])"


# V22: record the SIGNED Spearman ρ; `agreement` stays clipped at 0 for the trust score
CAM_RHO_SIGNED = globals().get("CAM_RHO_SIGNED", {})
def _log_signed_rho(organ, c1, c2):
    try:
        if c1 is None or c2 is None or np.ndim(c1) != 2 or np.ndim(c2) != 2: return
        a, b = np.asarray(c1, np.float64).ravel(), np.asarray(c2, np.float64).ravel()
        if a.std() < 1e-8 or b.std() < 1e-8: return
        CAM_RHO_SIGNED.setdefault(organ or "?", []).append(float(spearmanr(a, b)[0]))
    except Exception:
        pass

# ── Main replacement ──────────────────────────────────────────────────────────
def advanced_cam_ensemble(model, img_t, tgt_layer, pred_class_idx,
                           class_names,                # ← 5th positional
                           organ=None,                 # ← keyword only
                           hint=None,
                           is_vit=False,
                           hook_broken=False,
                           img_np_hwc=None,
                           confidence=0.5) -> dict:
    """
    FIX CELL F v2 — Layer-diverse CAM ensemble.
    Primary = deep layer (XAI_LAYERS). Secondary = shallow layer or RISE.
    No delegation chain whatsoever.
    """

    _EMPTY = {
        "final_cam":          np.zeros((224, 224), dtype=np.float32),
        "gcam":               None, "ecam": None, "lcam": None, "scam": None,
        "gcam_raw":           None, "ecam_raw": None, "ms_cam": None,
        "agreement":          0., "raw_agreement": 0.,
        "optimizer_agreement": 0., "method_desc": "empty",
        "spatial_entropy_gcam": float("inf"),
        "spatial_entropy_ecam": float("inf"),
        "score_cam_used": False, "ms_cam_used": False,
        "layer_cam_v7_used": False,
    }

    def _tissue(cam):
        if cam is None or not isinstance(cam, np.ndarray) or cam.ndim != 2:
            return np.zeros((224, 224), dtype=np.float32)
        if cam.max() < 1e-6:
            return cam.astype(np.float32)
        if img_np_hwc is not None:
            try:
                return _norm01(cam * _tissue_mask(img_np_hwc))
            except Exception:
                pass
        return cam.astype(np.float32)

    def _safe_agr(c1, c2):
        _log_signed_rho(organ, c1, c2)
        if (c1 is None or c2 is None
                or not isinstance(c1, np.ndarray)
                or not isinstance(c2, np.ndarray)
                or c1.ndim != 2 or c2.ndim != 2
                or c1.max() < 1e-6 or c2.max() < 1e-6):
            return 0.
        try:
            return float(_cam_agreement(c1, c2))
        except Exception:
            return 0.

    sec_layer = SECONDARY_CAM_LAYERS.get(organ)

    # ══════════════════════════════════════════════════════════════════════════
    # PATH 1 — LYMPHOMA (hook_broken, ViT backbone)
    # ══════════════════════════════════════════════════════════════════════════
    if organ == "lymphoma":   # V22: was `and hook_broken` → lymphoma fell to PATH 5 (ViT, no stage layers) → empty
        try:
            gcam_raw = rise_cam(model, img_t, pred_class_idx,
                                n_masks=500, hint=hint, organ=organ)
            gcam_raw = _tissue(gcam_raw)
            # ScoreCAM on blocks.4 as structurally different secondary
            try:
                ecam_raw = score_cam(model, img_t, "backbone.blocks.4",
                                     pred_class_idx, is_vit=False, top_k=20,
                                     hint=hint, organ=organ)
                ecam_raw = _tissue(ecam_raw)
                method = "RISE+ScoreCAM(b4)"
            except Exception:
                # Fallback: RISE with different masking params
                torch.manual_seed(12345)
                ecam_raw = rise_cam(model, img_t, pred_class_idx,
                                    n_masks=300, s=6, p=0.4,
                                    hint=hint, organ=organ)
                torch.manual_seed(SEED)
                ecam_raw = _tissue(ecam_raw)
                method = "RISE+RISE(alt)"

            agr   = _safe_agr(gcam_raw, ecam_raw)
            fused, _ = _quality_blend(gcam_raw, ecam_raw, confidence=confidence)
            return {**_EMPTY,
                    "final_cam": fused, "gcam": gcam_raw, "ecam": ecam_raw,
                    "gcam_raw": gcam_raw, "ecam_raw": ecam_raw,
                    "scam": ecam_raw if "ScoreCAM" in method else None,
                    "agreement": agr, "raw_agreement": agr,
                    "optimizer_agreement": agr, "method_desc": method,
                    "score_cam_used": "ScoreCAM" in method}
        except Exception as e:
            print(f"  ⚠️  [{organ}] PATH-1 failed: {e}")
            return _EMPTY

    # ══════════════════════════════════════════════════════════════════════════
    # PATH 2 — THYROID (EfficientNet-B2, RISE + ScoreCAM secondary)
    # ══════════════════════════════════════════════════════════════════════════
    if organ == "thyroid" and not is_vit and not hook_broken:
        try:
            gcam_raw = rise_cam(model, img_t, pred_class_idx,
                                n_masks=400, hint=hint, organ=organ)
            gcam_raw = _tissue(gcam_raw)
            # EigenCAM on earlier block (blocks.2) for structural diversity
            try:
                ecam_raw = eigen_cam(model, img_t, "backbone.blocks.2",
                                     is_vit=False, hint=hint, organ=organ)
                ecam_raw = _tissue(ecam_raw)
            except Exception:
                ecam_raw = score_cam(model, img_t, tgt_layer, pred_class_idx,
                                     is_vit=False, top_k=20,
                                     hint=hint, organ=organ)
                ecam_raw = _tissue(ecam_raw)

            agr   = _safe_agr(gcam_raw, ecam_raw)
            fused, mstr = _quality_blend(gcam_raw, ecam_raw,
                                          confidence=confidence)
            return {**_EMPTY,
                    "final_cam": fused, "gcam": gcam_raw, "ecam": ecam_raw,
                    "gcam_raw": gcam_raw, "ecam_raw": ecam_raw,
                    "agreement": agr, "raw_agreement": agr,
                    "optimizer_agreement": agr,
                    "method_desc": f"RISE+EigenCAM(b2)/{mstr}"}
        except Exception as e:
            print(f"  ⚠️  [{organ}] PATH-2 failed: {e}")
            return _EMPTY

    # ══════════════════════════════════════════════════════════════════════════
    # PATH 3 — PANCREATIC (ViT)
    # AttentionGradCAM (deep) + RISE (model-agnostic, diverse)
    # ══════════════════════════════════════════════════════════════════════════
    if organ == "pancreatic" and is_vit:
        try:
            gcam_raw = attention_gradcam_vit(model, img_t, pred_class_idx,
                                             hint=hint, organ=organ)
            gcam_raw = _tissue(gcam_raw)
            ecam_raw = rise_cam(model, img_t, pred_class_idx,
                                n_masks=250, hint=hint, organ=organ)
            ecam_raw = _tissue(ecam_raw)
            agr   = _safe_agr(gcam_raw, ecam_raw)
            fused, mstr = _quality_blend(gcam_raw, ecam_raw,
                                          confidence=confidence)
            return {**_EMPTY,
                    "final_cam": fused, "gcam": gcam_raw, "ecam": ecam_raw,
                    "gcam_raw": gcam_raw, "ecam_raw": ecam_raw,
                    "agreement": agr, "raw_agreement": agr,
                    "optimizer_agreement": agr,
                    "method_desc": f"AttGradCAM+RISE/{mstr}"}
        except Exception as e:
            print(f"  ⚠️  [{organ}] PATH-3 failed: {e}")
            return _EMPTY

    # ══════════════════════════════════════════════════════════════════════════
    # PATH 4 — KIDNEY / CERVICAL / BLADDER
    # These need RISE secondary due to hook sensitivity or binary structure
    # ══════════════════════════════════════════════════════════════════════════
    if organ in _FORCE_RISE_SECONDARY and not is_vit:
        try:
            # Primary: GradCAM++ on deep layer
            if hook_broken:
                gcam_raw = rise_cam(model, img_t, pred_class_idx,
                                    n_masks=400, hint=hint, organ=organ)
            else:
                gcam_raw = gradcam_plus_plus(
                    model, img_t, tgt_layer, pred_class_idx,
                    is_vit=False, hint=hint, organ=organ)
            gcam_raw = _tissue(gcam_raw)

            # Secondary: RISE (completely independent of gradient computation)
            ecam_raw = rise_cam(model, img_t, pred_class_idx,
                                n_masks=350, hint=hint, organ=organ)
            ecam_raw = _tissue(ecam_raw)

            agr   = _safe_agr(gcam_raw, ecam_raw)
            fused, mstr = _quality_blend(gcam_raw, ecam_raw,
                                          confidence=confidence)
            return {**_EMPTY,
                    "final_cam": fused, "gcam": gcam_raw, "ecam": ecam_raw,
                    "gcam_raw": gcam_raw, "ecam_raw": ecam_raw,
                    "agreement": agr, "raw_agreement": agr,
                    "optimizer_agreement": agr,
                    "method_desc": f"GradCAM+RISE/{mstr}",
                    "rise_used": not hook_broken}
        except Exception as e:
            print(f"  ⚠️  [{organ}] PATH-4 failed: {e}")
            return _EMPTY

    # ══════════════════════════════════════════════════════════════════════════
    # PATH 5 — ALL OTHER ORGANS (ConvNeXt backbone, multi-class)
    # Layer-diverse: GradCAM++(deep) + EigenCAM(shallow)
    # ══════════════════════════════════════════════════════════════════════════
    try:
        use_layer_cam = (organ in LAYER_CAM_ORGANS
                         if "LAYER_CAM_ORGANS" in dir() else False)
        use_layer_cam = use_layer_cam and not is_vit and not hook_broken

        # ── Primary: GradCAM++ on DEEP layer (XAI_LAYERS) ────────────────────
        if hook_broken:
            gcam_raw = rise_cam(model, img_t, pred_class_idx,
                                n_masks=500, hint=hint, organ=organ)
        elif is_vit:
            gcam_raw = attention_gradcam_vit(model, img_t, pred_class_idx,
                                             hint=hint, organ=organ)
        else:
            gcam_raw = gradcam_plus_plus(model, img_t, tgt_layer, pred_class_idx,
                                         is_vit=False, hint=hint, organ=organ)
        gcam_raw = _tissue(gcam_raw)

        # ── Secondary: EigenCAM on SHALLOW layer (SECONDARY_CAM_LAYERS) ──────
        # This is the KEY change vs v1: different layer → structural independence
        if is_vit:
            ecam_raw = attention_rollout(model, img_t, is_vit=True,
                                         hint=hint, organ=organ)
        elif sec_layer is not None:
            try:
                ecam_raw = eigen_cam(model, img_t, sec_layer, is_vit=False,
                                     hint=hint, organ=organ)
            except Exception:
                # Fallback to same-layer EigenCAM
                ecam_raw = eigen_cam(model, img_t, tgt_layer, is_vit=False,
                                     hint=hint, organ=organ)
        else:
            ecam_raw = eigen_cam(model, img_t, tgt_layer, is_vit=False,
                                 hint=hint, organ=organ)
        ecam_raw = _tissue(ecam_raw)

        # ── LayerCAM for texture organs ───────────────────────────────────────
        lcam_raw = None
        if use_layer_cam:
            try:
                lcam_raw = layer_cam_smooth(model, img_t, tgt_layer,
                                            pred_class_idx, is_vit=False,
                                            hint=hint, organ=organ)
                lcam_raw = _tissue(lcam_raw)
            except Exception:
                lcam_raw = None

        # ── Raw agreement (BEFORE any post-processing) ────────────────────────
        raw_agr = _safe_agr(gcam_raw, ecam_raw)

        # ── Gentle post-processing ────────────────────────────────────────────
        _skip_pp = hook_broken
        apply_bil = not _skip_pp
        if "cam_post_process" in dir():
            gcam_pp = cam_post_process(gcam_raw, img_np_hwc, organ,
                                        sharpen_strength=0.20,
                                        apply_bilateral=apply_bil)
            ecam_pp = cam_post_process(ecam_raw, img_np_hwc, organ,
                                        sharpen_strength=0.20,
                                        apply_bilateral=apply_bil)
            lcam_pp = (cam_post_process(lcam_raw, img_np_hwc, organ,
                                         sharpen_strength=0.15,
                                         apply_bilateral=apply_bil)
                       if lcam_raw is not None else None)
        else:
            gcam_pp, ecam_pp, lcam_pp = gcam_raw, ecam_raw, lcam_raw

        # ── ScoreCAM fallback when still disagreeing ──────────────────────────
        scam_pp = None
        score_cam_used = False
        _disagree_thr = (CAM_DISAGREE_THRESH
                         if "CAM_DISAGREE_THRESH" in dir() else 0.25)
        if raw_agr < _disagree_thr and not hook_broken and not is_vit:
            try:
                scam_raw = score_cam(model, img_t, tgt_layer, pred_class_idx,
                                     is_vit=False, hint=hint, organ=organ)
                scam_raw = _tissue(scam_raw)
                if scam_raw.max() > 0.05:
                    if "cam_post_process" in dir():
                        scam_pp = cam_post_process(scam_raw, img_np_hwc, organ,
                                                    sharpen_strength=0.15,
                                                    apply_bilateral=apply_bil)
                    else:
                        scam_pp = scam_raw
                    score_cam_used = True
            except Exception:
                pass

        # ── MS-CAM ────────────────────────────────────────────────────────────
        ms_pp = None
        ms_cam_used = False
        try:
            ms_raw = multi_scale_cam(model, img_t, organ, pred_class_idx,
                                     is_vit=is_vit, hook_broken=hook_broken,
                                     hint=hint)
            if ms_raw is not None and ms_raw.max() > 0.05:
                ms_raw = _tissue(ms_raw)
                ms_pp = (cam_post_process(ms_raw, img_np_hwc, organ,
                                          sharpen_strength=0.15,
                                          apply_bilateral=apply_bil)
                         if "cam_post_process" in dir() else ms_raw)
                ms_cam_used = True
        except Exception:
            pass

        # ── Quality-weighted blend ────────────────────────────────────────────
        cand_p = gcam_pp if gcam_pp is not None else gcam_raw
        cand_s = ecam_pp if ecam_pp is not None else ecam_raw
        cand_t = (ms_pp if ms_pp is not None
                  else (lcam_pp if lcam_pp is not None else scam_pp))

        fused, mstr = _quality_blend(cand_p, cand_s, cand_t, confidence)

        # Compute spatial entropy using local helper (no external dependency)
        def _se(cam):
            if cam is None or cam.max() < 1e-6:
                return float("inf")
            flat = cam.flatten().astype(np.float64)
            flat = flat / (flat.sum() + 1e-8)
            return float(-(flat * np.log(flat + 1e-8)).sum())

        return {**_EMPTY,
                "final_cam":            fused,
                "gcam":                 gcam_pp if gcam_pp is not None else gcam_raw,
                "ecam":                 ecam_pp if ecam_pp is not None else ecam_raw,
                "lcam":                 lcam_pp,
                "scam":                 scam_pp,
                "gcam_raw":             gcam_raw,
                "ecam_raw":             ecam_raw,
                "ms_cam":               ms_pp,
                "agreement":            raw_agr,
                "raw_agreement":        raw_agr,
                "optimizer_agreement":  raw_agr,
                "method_desc":          f"LayerDiv/{mstr}",
                "spatial_entropy_gcam": _se(gcam_pp if gcam_pp is not None else gcam_raw),
                "spatial_entropy_ecam": _se(ecam_pp if ecam_pp is not None else ecam_raw),
                "score_cam_used":       score_cam_used,
                "ms_cam_used":          ms_cam_used,
                "layer_cam_v7_used":    lcam_raw is not None,
                "v18_cam_active":       True,
                }

    except Exception as e:
        print(f"  ⚠️  [{organ}] PATH-5 failed: {e}")
        try:
            if hook_broken:
                fb = rise_cam(model, img_t, pred_class_idx,
                              n_masks=200, hint=hint, organ=organ)
            elif is_vit:
                fb = attention_gradcam_vit(model, img_t, pred_class_idx,
                                           hint=hint, organ=organ)
            else:
                fb = gradcam_plus_plus(model, img_t, tgt_layer, pred_class_idx,
                                       hint=hint, organ=organ)
            fb = _tissue(fb)
            return {**_EMPTY, "final_cam": fb, "gcam": fb, "ecam": fb,
                    "gcam_raw": fb, "ecam_raw": fb,
                    "agreement": 0., "method_desc": "last_resort"}
        except Exception:
            return _EMPTY


print("  ✅ advanced_cam_ensemble v2 loaded (layer-diverse, no delegation)")
print()
print("  PATH assignments:")
print("    PATH 1 — lymphoma         : RISE(500) + ScoreCAM(blocks.4)")
print("    PATH 2 — thyroid          : RISE(400) + EigenCAM(blocks.2)")
print("    PATH 3 — pancreatic (ViT) : AttGradCAM + RISE(250)")
print("    PATH 4 — kidney/cervical/bladder: GradCAM + RISE(350)")
print("    PATH 5 — all others       : GradCAM++(deep) + EigenCAM(shallow)")
print("             Secondary uses SECONDARY_CAM_LAYERS (one stage earlier)")


# ════════════════════════════════════════════════════════════════════════════════
# Validation
# ════════════════════════════════════════════════════════════════════════════════

print("\n  Validating all 16 organs...")

_summary = []
for _organ in list(SPECIALISTS.keys()):
    try:
        _heavy = AUGMENT_CFG.get(_organ) == "heavy"
        _, _, _te, _cls = build_loaders(_organ, batch_size=1, heavy=_heavy)
        _it, _ = next(iter(_te))
        _it  = _it[0:1].to(DEVICE)
        _m   = SPECIALISTS[_organ]["model"]
        _hb  = _organ in HOOK_BROKEN_ORGANS
        _vit = _organ in VIT_ORGANS

        with torch.no_grad():
            _p = int(_safe_fwd(_m, _it, None, _organ).argmax(1).item())

        _res = advanced_cam_ensemble(
            _m, _it,
            XAI_LAYERS.get(_organ, "backbone.stages.2"),
            _p,
            _cls,           # ← 5th positional
            organ=_organ,   # ← keyword
            hook_broken=_hb,
            is_vit=_vit,
        )

        _agr    = float(_res.get("agreement", 0.))
        _method = _res.get("method_desc", "?")[:30]
        _ok     = (_res["final_cam"].max() > 1e-6)
        _flag   = "✅" if _agr > 0.05 else ("⚠️" if _ok else "❌flat")
        print(f"    [{_organ:<12}]  agr={_agr:.3f}  {_method:<32}  {_flag}")
        _summary.append((_organ, _agr, _ok))

    except Exception as _ex:
        print(f"    [{_organ:<12}]  ERROR: {type(_ex).__name__}: {_ex}")
        _summary.append((_organ, 0., False))

# Report
_agr_vals  = [a for _, a, ok in _summary if ok]
_low       = [o for o, a, ok in _summary if a < 0.05 and ok]
_flat      = [o for o, a, ok in _summary if not ok]

print()
if _agr_vals:
    print(f"  Mean agreement : {np.mean(_agr_vals):.3f}  "
          f"(target > 0.15 for all organs)")
if _low:
    print(f"  ⚠️  agr < 0.05 : {_low}")
    print(f"     → FIX-CAM-1 RISE fallback will activate in Cell 11")
if _flat:
    print(f"  ❌ flat output : {_flat}")
if not _low and not _flat:
    print(f"  ✅ All organs have agreement > 0.05")

print("\n  Signed Spearman ρ (primary vs secondary, NOT clipped) — report THIS in the paper:")
for _o, _v in CAM_RHO_SIGNED.items():
    print(f"    {_o:<12} ρ = {np.mean(_v):+.3f}  (n={len(_v)})")
print()
print("✅ FIX CELL F v2 complete")
print()
print("  Key change: secondary CAM now uses a DIFFERENT LAYER to the primary.")
print("  GradCAM++(stage.2) vs EigenCAM(stage.1) → structural independence.")
print("  EfficientNet organs: RISE(primary) + EigenCAM(blocks.2)(secondary).")
print("  Binary/hook-sensitive organs: GradCAM + RISE (method-independent).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# FIX CELL H (V20) — advanced_cam_ensemble call-convention shim
#
# ROOT CAUSE: FIX CELL F v2 redefined
#     advanced_cam_ensemble(model, img_t, tgt_layer, pred_idx, class_names, organ=None, ...)
# but run_full_xai (Cell 6) still calls it with the OLD order
#     advanced_cam_ensemble(model, img_t, tgt_layer, pred_idx, organ, class_names, ...)
# → TypeError "got multiple values for argument 'organ'" → silent v17 fallback.
# In xai_metrics_v16.csv, v18_cam_active was True on 2/55 rows (lymphoma only).
# This shim accepts both orders.
# ════════════════════════════════════════════════════════════════════════════════
if "_ACE_V2_IMPL" not in globals():
    _ACE_V2_IMPL = advanced_cam_ensemble

def advanced_cam_ensemble(model, img_t, tgt_layer, pred_class_idx, *args, **kw):
    args = list(args)
    # Old convention: 5th positional is the organ name (str), 6th is class_names.
    if args and isinstance(args[0], str) and "organ" not in kw:
        organ = args.pop(0)
        if args:
            class_names = args.pop(0)
        else:
            class_names = kw.pop("class_names")
        return _ACE_V2_IMPL(model, img_t, tgt_layer, pred_class_idx,
                            class_names, *args, organ=organ, **kw)
    return _ACE_V2_IMPL(model, img_t, tgt_layer, pred_class_idx, *args, **kw)

# Smoke test on one organ per call convention
try:
    _o = "colon"
    _, _, _te, _cls = build_loaders(_o, batch_size=1, heavy=AUGMENT_CFG.get(_o) == "heavy")
    _it = next(iter(_te))[0][0:1].to(DEVICE)
    _m = SPECIALISTS[_o]["model"]
    with torch.no_grad():
        _p = int(_safe_fwd(_m, _it, None, _o).argmax(1).item())
    _lay = XAI_LAYERS.get(_o, "backbone.stages.2")
    r_old = advanced_cam_ensemble(_m, _it, _lay, _p, _o, _cls)
    r_new = advanced_cam_ensemble(_m, _it, _lay, _p, _cls, organ=_o)
    print(f"✅ FIX CELL H: both call orders work  "
          f"(old agr={r_old.get('agreement', 0):.3f}, new agr={r_new.get('agreement', 0):.3f})")
except Exception as e:
    print(f"⚠️  FIX CELL H smoke test failed: {e}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# FIX CELL I (V20) — True modality table + HoVer-Net gating to H&E organs
#
# Visual audit of the XAI panels shows the source folders are NOT all
# histopathology: bladder = pelvic MRI, breast = mammography, esophageal =
# endoscopy, kidney & pancreatic = CT, thyroid = ultrasound, skin = dermoscopy /
# clinical photo. HoVer-Net (trained on PanNuke H&E) was running on these
# (xai_metrics_v16.csv: seg_net_used=True on 3/4 thyroid ultrasound rows).
# ════════════════════════════════════════════════════════════════════════════════
TRUE_MODALITY = {
    "blood": "blood smear", "lymphoma": "H&E", "skin": "dermoscopy/clinical photo",
    "cervical": "Pap cytology", "gastric": "H&E", "ovarian": "H&E",
    "breast": "mammography", "colon": "H&E", "lung": "H&E",
    "brain_2d": "MRI", "kidney": "CT", "oral": "H&E",
    "esophageal": "endoscopy", "pancreatic": "CT", "thyroid": "ultrasound",
    "bladder": "MRI",
}
# ⚠ Verify by opening ~20 random images per organ before publishing.
HE_ORGANS = {o for o, m in TRUE_MODALITY.items() if m == "H&E"}
print("H&E organs (HoVer-Net enabled):", sorted(HE_ORGANS))

def _gate_hovernet(fn):
    if getattr(fn, "_v20_gated", False):
        return fn
    def wrapped(img_np_hwc, organ=None, *a, **k):
        if organ is not None and organ not in HE_ORGANS:
            res = {
                "inst_map": np.zeros((224, 224), np.int32),
                "np_map": np.zeros((224, 224), np.float32),
                "nuclear_types": {}, "morphometrics": [], "til": {},
                "seg_ok": False, "tumor_area_pct": 0., "seg_circularity": 0.,
                "n_regions": 0, "n_neoplastic": 0, "overlay": None,
                "seg_net_used": False, "crf_used": False, "is_malignant": False,
                "boundary_regularity": 0., "mask": np.zeros((224, 224), np.float32),
                "seg_skipped_reason": f"non_HE_modality:{TRUE_MODALITY.get(organ)}",
            }
            return res
        return fn(img_np_hwc, organ, *a, **k)
    wrapped._v20_gated = True
    return wrapped

for _name in ["hovernet_predict", "hovernet_predict_v18", "hovernet_predict_v21"]:
    if _name in globals():
        globals()[_name] = _gate_hovernet(globals()[_name])
        print(f"✅ {_name} gated to H&E organs")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# V24-P0 — Preprocessing audit (runs BEFORE Cell 7; decides the paper's numbers)
#   Same 600 test images per organ, 3–4 evaluation transforms:
#     legacy_stain  = old val_tfm (stain-norm + Resize 224)   ← what v16–v23 reported
#     resize224     = Resize 224, no stain                     ← v24 default
#     crop224       = Resize 256 → CenterCrop 224, no stain
#     heavy_val     = HEAVY_VAL_AUG (bladder / thyroid only: what Cell 7 uses for them)
#   Reports base-specialist macro-F1 AND gateway top-1 routing for each transform.
# ════════════════════════════════════════════════════════════════════════════════
import json, numpy as np, pandas as pd, torch
import torchvision.transforms as T_
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
V24_OUT = XAI_OUT / "v24"; V24_OUT.mkdir(parents=True, exist_ok=True)
_IMN = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
val_tfm_legacy = T_.Compose([T_.Lambda(_apply_stain_norm), T_.Resize((224, 224)), T_.ToTensor(), T_.Normalize(*_IMN)])
_P0_TF = {
    "legacy_stain": val_tfm_legacy,
    "resize224":    T_.Compose([T_.Resize((224, 224)), T_.ToTensor(), T_.Normalize(*_IMN)]),
    "crop224":      T_.Compose([T_.Resize(256), T_.CenterCrop(224), T_.ToTensor(), T_.Normalize(*_IMN)]),
}
class _P0DS(torch.utils.data.Dataset):
    def __init__(s, paths, tf, alb=False): s.p, s.tf, s.alb = paths, tf, alb
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        im = Image.open(s.p[i]).convert("RGB")
        return s.tf(image=np.array(im))["image"] if s.alb else s.tf(im)

def _p0_test(organ):
    paths, str_labels, cls = _collect_paths(organ)
    l2i = {c: i for i, c in enumerate(cls)}; y = [l2i[l] for l in str_labels]
    _, te_p, _, te_l = train_test_split(paths, y, test_size=0.15, stratify=y, random_state=SEED)
    return te_p, np.array(te_l), cls

N_P0 = 600
_gw = globals().get("GATEWAY_MODEL") or globals().get("gw")
rows = []
for organ, spec in SPECIALISTS.items():
    if spec.get("model") is None: continue
    P, Y, cls = _p0_test(organ)
    sel = np.random.default_rng(0).choice(len(P), min(N_P0, len(P)), replace=False)
    P, Y = [P[i] for i in sel], Y[sel]
    variants = dict(_P0_TF)
    if AUGMENT_CFG.get(organ) == "heavy": variants["heavy_val"] = HEAVY_VAL_AUG
    g_idx = _gw.organ_classes.index(organ) if (_gw is not None and organ in _gw.organ_classes) else None
    m = spec["model"].eval()
    for vn, tf in variants.items():
        ld = torch.utils.data.DataLoader(_P0DS(P, tf, alb=(vn == "heavy_val")), batch_size=64, num_workers=NUM_WORKERS, timeout=_dl_timeout())
        pr, rt = [], []
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            for x in ld:
                x = x.to(DEVICE)
                pr.append(m(x).float().argmax(1).cpu().numpy())
                if g_idx is not None: rt.append(_gw(x)["organ_logits"].float().argmax(1).cpu().numpy())
        pr = np.concatenate(pr)
        r = {"organ": organ, "variant": vn, "n": len(Y), "f1": float(f1_score(Y, pr, average="macro")),
             "route_acc": float((np.concatenate(rt) == g_idx).mean()) if rt else float("nan")}
        rows.append(r)
    print(f"  {organ:<11} " + "  ".join(f"{r['variant']}: F1={r['f1']:.3f}/route={r['route_acc']:.3f}" for r in rows if r["organ"] == organ))

P0_DF = pd.DataFrame(rows)
P0_DF.to_csv(V24_OUT / "p0_preprocessing.csv", index=False)
piv = P0_DF.pivot(index="organ", columns="variant", values="f1")
pivr = P0_DF.pivot(index="organ", columns="variant", values="route_acc")
print("\nSpecialist macro-F1 by transform:\n", piv.round(3).to_string())
print("\nGateway routing by transform:\n", pivr.round(3).to_string())
_best = {o: piv.loc[o].drop(labels=["legacy_stain"], errors="ignore").idxmax() for o in piv.index}
print("\nBest non-legacy transform per organ:", _best)
print(f"Mean F1  legacy={piv['legacy_stain'].mean():.4f}  resize224={piv['resize224'].mean():.4f}  crop224={piv['crop224'].mean():.4f}")
print(f"Mean route legacy={pivr['legacy_stain'].mean():.4f}  resize224={pivr['resize224'].mean():.4f}  crop224={pivr['crop224'].mean():.4f}")
(V24_OUT / "p0_summary.json").write_text(json.dumps({"best": _best, "mean_f1": piv.mean().to_dict(), "mean_route": pivr.mean().to_dict()}, indent=1))
print("\n→ If crop224 beats resize224 on most organs by > 0.01, set EVAL_RESIZE='crop' in the config cell and rerun.")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CELLS 7 + 7b — Evaluation · Calibration · Conformal RAPS · Gateway Routing
#
# v17 CHANGES vs v16 + fix-cells:
#   BUG-3  : ECE_before from raw (unscaled) logits; ECE_after from T*-scaled ✅
#   ISS-4  : flag threshold = 95th-pctile val entropy (not mean+1.5σ) ✅
#   ISS-10 : per-class recall printed for organs with >3 classes ✅
#   ISS-12 : film_status FROZEN / ACTIVE / DISABLED throughout ✅
#   FIX-F  : fine-tuned gateway v16 loaded from new path ✅
#   FIX-F  : TTA-4 ensemble in gateway routing (more stable organ probs) ✅
#   FIX-F  : gateway_with_fallback — top-2 ensemble when conf < 0.85 ✅
#   ECE-G  : ECE regression guard — reverts T* if it made ECE worse ≥ 0.02 ✅
#   7b     : Conformal RAPS (α=0.10) calibration + empirical coverage check ✅
#   7b     : get_prediction_set() populates CONFORMAL_QHAT for all organs ✅
#
# PREREQUISITES (in scope from earlier cells):
#   DEVICE, SPEC_DIR, SPECIALISTS, build_loaders, AUGMENT_CFG,
#   _safe_fwd, OPTIMIZED_TEMPERATURES, ORGAN_VAL_UNC_STATS,
#   NEEDS_CALIBRATION, ORGAN_BASE_F1, FILM_FREEZE_F1,
#   CONFORMAL_ALPHA, CONFORMAL_QHAT, XAI_OUT,
#   GATEWAY_MODEL (may be None — reloaded here with fine-tuned weights),
#   load_gateway, get_gateway_hint (defined or overridden below)
# ════════════════════════════════════════════════════════════════════════════════

import gc
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report as _clf_report,
)

print("═" * 68)
print("  CELLS 7 + 7b — Evaluation · Calibration · RAPS · Gateway v17")
print("═" * 68)


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 0 — Constants
# ════════════════════════════════════════════════════════════════════════════════

T_LOWER_BOUND = 0.10
T_UPPER_BOUND = 10.0

# Gateway routing thresholds (FIX-F)
GATEWAY_FALLBACK_THRESH = 0.80  # below this → top-2 ensemble routing
GATEWAY_CONF_THRESHOLD  = 0.50   # below this → flag as UNCERTAIN

# ECE regression guard threshold
ECE_REGRESSION_GUARD_DELTA = 0.02   # revert T* if ECE worsened by this much

# Conformal prediction
if "CONFORMAL_ALPHA" not in dir():
    CONFORMAL_ALPHA = 0.10         # 90 % marginal coverage
if "CONFORMAL_QHAT" not in dir():
    CONFORMAL_QHAT  = {}           # organ → q-hat threshold

# Fine-tuned gateway checkpoint (FIX-F — new path)
GATEWAY_FINETUNED_CKPT = Path(
    "/kaggle/input/models/marcyoussef22/gateway1/other/default/3/"
    "gateway_finetuned_v16.pth")

# Accumulators
eval_results       = {}
conformal_coverage = {}

print(f"  T* bounds             : [{T_LOWER_BOUND}, {T_UPPER_BOUND}]")
print(f"  CONFORMAL_ALPHA       : {CONFORMAL_ALPHA}  (target coverage ≥ {1-CONFORMAL_ALPHA:.0%})")
print(f"  GATEWAY_FALLBACK_THRESH: {GATEWAY_FALLBACK_THRESH}")
print(f"  Fine-tuned gateway ckpt: {GATEWAY_FINETUNED_CKPT.name}")
print(f"  Ckpt exists           : {GATEWAY_FINETUNED_CKPT.exists()}")


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 1 — Fine-tuned gateway loading (FIX-F)
# ════════════════════════════════════════════════════════════════════════════════

def _load_finetuned_gateway(ckpt_path: Path, device):
    """
    Load the fine-tuned gateway from the v16 checkpoint.
    Falls back to the previously loaded GATEWAY_MODEL if the ckpt is absent
    or incompatible, so the rest of the cell always has a gateway object.
    """
    global GATEWAY_MODEL

    if not ckpt_path.exists():
        print(f"  ℹ️  Fine-tuned ckpt not found at {ckpt_path}")
        print(f"     Using existing GATEWAY_MODEL "
              f"({'loaded' if GATEWAY_MODEL is not None else 'None'})")
        return GATEWAY_MODEL

    print(f"\n  Loading fine-tuned gateway → {ckpt_path.name}")
    try:
        ckpt = torch.load(str(ckpt_path), map_location=device, weights_only=False)

        # The ckpt may wrap the state_dict under 'model_state_dict'
        state_dict = ckpt.get("model_state_dict", ckpt)

        # Re-use the existing gateway architecture (already built by Cell 5b)
        if GATEWAY_MODEL is None:
            print("  ⚠️  GATEWAY_MODEL architecture not initialised yet "
                  "— attempting load_gateway()")
            try:
                GATEWAY_MODEL = load_gateway()
            except Exception as e:
                print(f"  ❌ load_gateway() failed: {e}")
                return None

        # Load fine-tuned weights (strict=False tolerates minor architecture tweaks)
        incompatible = GATEWAY_MODEL.load_state_dict(state_dict, strict=False)
        GATEWAY_MODEL.eval()

        missing  = len(incompatible.missing_keys)
        unexpect = len(incompatible.unexpected_keys)
        val_loss = ckpt.get("val_loss", float("nan"))
        epochs   = ckpt.get("epochs",   "?")

        print(f"  ✅ Fine-tuned gateway loaded")
        print(f"     val_loss={val_loss:.4f}  epochs={epochs}")
        print(f"     missing keys={missing}  unexpected keys={unexpect}")
        return GATEWAY_MODEL

    except Exception as e:
        print(f"  ❌ Fine-tuned gateway load failed: {e}")
        print(f"     Keeping existing GATEWAY_MODEL")
        return GATEWAY_MODEL


# V22: FiLM heads were trained on hints from the ORIGINAL gateway. Swapping in the
# fine-tuned gateway changed bladder FiLM F1 0.953 → 0.734. Keep False for the paper.
USE_FINETUNED_GATEWAY = False
print("\nStep 1: Loading fine-tuned gateway...")
if USE_FINETUNED_GATEWAY:
    GATEWAY_MODEL = _load_finetuned_gateway(GATEWAY_FINETUNED_CKPT, DEVICE)
else:
    print("  ℹ️  USE_FINETUNED_GATEWAY=False → keeping the original gateway (consistent with FiLM training)")
gw = GATEWAY_MODEL


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 2 — TTA-enhanced gateway helpers (FIX-F)
# ════════════════════════════════════════════════════════════════════════════════

def _gateway_tta_predict(gateway: nn.Module,
                          img_t: torch.Tensor) -> torch.Tensor:
    """
    FIX-F: 4-view TTA ensemble at the gateway level.
    Returns mean softmax organ probability tensor (1, n_organs).
    Augmentations: original, h-flip, v-flip, rot90.
    More stable organ routing than single-forward, especially for
    gastric (42 % single-fwd accuracy) and kidney (62 %).
    """
    augs = [
        lambda x: x,
        lambda x: torch.flip(x, [3]),
        lambda x: torch.flip(x, [2]),
        lambda x: torch.rot90(x, 1, [2, 3]),
    ]
    gateway.eval()
    all_probs = []
    with torch.no_grad():
        for aug in augs:
            out   = gateway(aug(img_t))
            probs = torch.softmax(out["organ_logits"], 1)
            all_probs.append(probs)
    return torch.stack(all_probs, 0).mean(0)   # (1, n_organs)


def gateway_with_fallback(gateway: nn.Module,
                           imgs: torch.Tensor,
                           threshold: float = GATEWAY_CONF_THRESHOLD,
                           fallback_thresh: float = GATEWAY_FALLBACK_THRESH
                           ) -> tuple[list[str], str]:
    """
    FIX-F: TTA-enhanced gateway routing with confidence-based fallback.

    Returns:
        organs      — list of organ name(s) to run through specialists
        routing_mode— 'CONFIDENT' | 'ENSEMBLE' | 'UNCERTAIN' | 'NO_GATEWAY'
    """
    if gateway is None:
        return ["unknown"], "NO_GATEWAY"
    try:
        organ_probs = _gateway_tta_predict(gateway, imgs)[0]   # (n_organs,)
        top2        = torch.topk(organ_probs, min(2, len(gateway.organ_classes)))
        conf1       = float(top2.values[0])
        organ1      = gateway.organ_classes[int(top2.indices[0])]

        if conf1 >= fallback_thresh:
            return [organ1], "CONFIDENT"
        elif conf1 >= threshold:
            organ2 = gateway.organ_classes[int(top2.indices[1])]
            return [organ1, organ2], "ENSEMBLE"
        else:
            return [organ1], "UNCERTAIN"
    except Exception as e:
        print(f"  ⚠️  gateway_with_fallback: {e}")
        return ["unknown"], "ERROR"


def get_gateway_hint(gateway: nn.Module,
                      imgs: torch.Tensor,
                      organ: str) -> torch.Tensor | None:
    """
    FIX-F: TTA-stabilised subclass hint generation.
    Uses TTA-averaged organ probs to feed the correct subclass head.
    """
    if gateway is None:
        return None
    try:
        n_subs = len(gateway.subclass_map.get(organ, []))
        if n_subs == 0:
            return None
        with torch.no_grad():
            out = gateway(imgs)
            raw = out["subclass_logits"][:, :n_subs]
            raw = raw.masked_fill(raw == float("-inf"), -1e4)
            hint = torch.softmax(raw.float(), dim=-1)
        return hint.detach()
    except Exception as e:
        print(f"  ⚠️  get_gateway_hint [{organ}]: {e}")
        return None


print("  ✅ gateway_with_fallback + get_gateway_hint defined (TTA-4 ensemble)")


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 3 — Core evaluation helpers
# ════════════════════════════════════════════════════════════════════════════════

def find_optimal_temperature(logits_np: np.ndarray,
                              labels_np: np.ndarray,
                              organ: str) -> float:
    """
    Optimise T* via NLL minimisation using Brent's method.
    v17-BUG-3: operates on raw logits only (no T* pre-applied).
    """
    from scipy.optimize import minimize_scalar
    if logits_np.shape[0] < 10:
        print(f"    [{organ}] Too few val samples — T*=1.0")
        return 1.0

    def nll(T: float) -> float:
        T = max(float(T), T_LOWER_BOUND)
        scaled = logits_np / T
        mx     = scaled.max(axis=1, keepdims=True)
        log_sm = np.log(np.exp(scaled - mx).sum(axis=1, keepdims=True) + 1e-12) + mx
        log_p  = scaled - log_sm
        idx    = labels_np.astype(int)
        return -log_p[np.arange(len(idx)), idx].mean()

    try:
        res   = minimize_scalar(nll, bounds=(T_LOWER_BOUND, T_UPPER_BOUND),
                                method='bounded', options={'xatol': 1e-4})
        T_opt = float(np.clip(res.x, T_LOWER_BOUND, T_UPPER_BOUND))
        print(f"    [{organ}] T* = {T_opt:.4f}  NLL = {res.fun:.4f}")
        return T_opt
    except Exception as e:
        print(f"    [{organ}] T* optimisation failed ({e}) — using 1.0")
        return 1.0


def compute_ece_fast(confidences: list, correct_flags: list,
                      n_bins: int = 10) -> float:
    """Expected Calibration Error (equal-width bins)."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.
    n    = len(confidences)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = [lo < c <= hi for c in confidences]
        if not any(mask):
            continue
        bc = [c for c, m in zip(confidences, mask) if m]
        ba = [a for a, m in zip(correct_flags, mask) if m]
        ece += (len(bc) / n) * abs(np.mean(bc) - np.mean(ba))
    return round(float(ece), 4)


def _raw_predict(model: nn.Module, loader) -> tuple:
    """
    v17-BUG-3: forward with NO T* applied — purely raw model logits.
    Used exclusively for ECE_before computation.
    """
    model.eval()
    preds, labels_all, probs_all = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                raw = model(imgs)                      # ← no _safe_fwd, no T*
            raw  = torch.clamp(torch.nan_to_num(raw.float()), -30., 30.)
            prbs = torch.softmax(raw, 1).cpu().numpy()
            preds.extend(raw.argmax(1).cpu().numpy())
            labels_all.extend(labels.numpy())
            probs_all.extend(prbs)
    return labels_all, preds, np.array(probs_all)


def _scaled_predict(model: nn.Module, loader,
                     T: float, organ: str,
                     hint_fn=None) -> tuple:
    """
    v17-BUG-3: forward WITH T* applied via _safe_fwd.
    Used for ECE_after, accuracy, F1, AUC, and confusion matrices.
    """
    model.eval()
    preds, labels_all, probs_all = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            hint = hint_fn(imgs) if hint_fn is not None else None
            out  = _safe_fwd(model, imgs, hint, organ)
            prbs = torch.softmax(out, 1).cpu().numpy()
            preds.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.numpy())
            probs_all.extend(prbs)
    return labels_all, preds, np.array(probs_all)


def apply_ece_regression_guard(results: dict,
                                threshold: float = ECE_REGRESSION_GUARD_DELTA) -> None:
    """
    ECE regression guard: if T* made ECE WORSE by more than `threshold`,
    revert T*=1.0 and re-record ECE_after.
    Observed regressions: breast +0.042, ovarian +0.001.
    """
    for organ, res in results.items():
        delta = res.get("ece_after", 0.) - res.get("ece_before", 0.)
        if delta > threshold:
            old_T = OPTIMIZED_TEMPERATURES.get(organ, 1.0)
            OPTIMIZED_TEMPERATURES[organ] = 1.0
            res["T_opt"]    = 1.0
            res["ece_after"] = res["ece_before"]   # revert to raw ECE
            print(f"  ECE-guard [{organ}]: T*={old_T:.4f}→1.0  "
                  f"ECE Δ={delta:+.4f} reverted")


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 4 — Main evaluation loop (all specialists)
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 68)
print("  Step 2: Evaluating all specialists + calibrating temperatures...")
print("─" * 68 + "\n")

for organ, spec_info in SPECIALISTS.items():
    model_s     = spec_info["model"]
    film_model  = spec_info.get("film_model")
    film_status = spec_info.get("film_status", "ABSENT")
    class_names = spec_info["class_names"]

    # ── Build loaders ────────────────────────────────────────────────────────
    try:
        heavy = AUGMENT_CFG.get(organ) == "heavy"
        tr_loader, vl_loader, ts_loader, _ = build_loaders(
            organ, batch_size=64, heavy=heavy)
    except ValueError as e:
        print(f"  ⚠️  [{organ}] loader error: {e} — skipping")
        continue

    print(f"\n  {'─'*60}")
    print(f"  [{organ.upper()}]  classes={len(class_names)}"
          f"  film={film_status}")
    print(f"  {'─'*60}")

    # ── Collect raw val logits for T* optimisation ────────────────────────────
    val_logits_list, val_labels_list, val_ent_list = [], [], []
    model_s.eval()
    with torch.no_grad():
        for imgs, labels in vl_loader:
            imgs = imgs.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                raw_out = model_s(imgs)               # raw, no T*
            raw_out = torch.clamp(
                torch.nan_to_num(raw_out.float()), -30., 30.)
            val_logits_list.append(raw_out.cpu().numpy())
            val_labels_list.extend(labels.numpy())
            prbs = torch.softmax(raw_out, 1)
            ent  = -(prbs * torch.log(prbs + 1e-8)).sum(1).cpu().numpy()
            val_ent_list.extend(ent.tolist())

    val_logits_np = (np.concatenate(val_logits_list, axis=0)
                     if val_logits_list
                     else np.zeros((1, len(class_names))))
    val_labels_np = (np.array(val_labels_list)
                     if val_labels_list else np.zeros(1, dtype=int))

    # ── Optimise T* ──────────────────────────────────────────────────────────
    T_opt = find_optimal_temperature(val_logits_np, val_labels_np, organ)
    OPTIMIZED_TEMPERATURES[organ] = T_opt
    NEEDS_CALIBRATION.add(organ)

    # ── ISS-4: 95th-percentile entropy threshold for uncertainty flagging ────
    if val_ent_list:
        unc_threshold = float(np.percentile(val_ent_list, 95))
        ORGAN_VAL_UNC_STATS[organ] = {
            "mean":      float(np.mean(val_ent_list)),
            "std":       float(np.std(val_ent_list)),
            "threshold": unc_threshold,
        }
        print(f"    Uncertainty flag threshold (95th-pctile entropy): "
              f"{unc_threshold:.4f}")

    # ── BUG-3: ECE_before from raw logits ────────────────────────────────────
    raw_labels, raw_preds, raw_probs = _raw_predict(model_s, ts_loader)
    correct_raw  = [1 if p == l else 0
                    for p, l in zip(raw_preds, raw_labels)]
    ece_before   = compute_ece_fast(
        raw_probs.max(axis=1).tolist(), correct_raw)

    # ── BUG-3: ECE_after from T*-scaled logits ───────────────────────────────
    base_labels, base_preds, base_probs = _scaled_predict(
        model_s, ts_loader, T_opt, organ)
    correct_cal = [1 if p == l else 0
                   for p, l in zip(base_preds, base_labels)]
    ece_after   = compute_ece_fast(
        base_probs.max(axis=1).tolist(), correct_cal)

    delta_ece = ece_after - ece_before
    ece_flag  = ("✅ improved" if delta_ece < -0.001
                 else ("➡ unchanged" if abs(delta_ece) < 0.001
                       else "⚠ worse"))
    print(f"    ECE: {ece_before:.4f} → {ece_after:.4f}"
          f"  T*={T_opt:.3f}  {ece_flag}")

    # ── Standard classification metrics ──────────────────────────────────────
    base_acc = accuracy_score(base_labels, base_preds)
    base_f1  = f1_score(base_labels, base_preds,
                        average="macro", zero_division=0)
    try:
        base_auc = (roc_auc_score(base_labels, base_probs[:, 1])
                    if len(class_names) == 2
                    else roc_auc_score(base_labels, base_probs,
                                       multi_class="ovr", average="macro"))
    except Exception:
        base_auc = float("nan")

    print(f"    Base: Acc={base_acc:.4f}  F1={base_f1:.4f}"
          f"  AUC={base_auc:.4f}")

    # ── ISS-10: per-class recall for organs with >3 classes ──────────────────
    if len(class_names) > 3:
        rpt = _clf_report(base_labels, base_preds,
                          target_names=class_names,
                          output_dict=True, zero_division=0)
        print(f"\n    Per-class metrics [{organ.upper()}]:")
        for cls_name, met in rpt.items():
            if isinstance(met, dict):
                print(f"      {cls_name:<28}: "
                      f"recall={met['recall']:.3f}  "
                      f"precision={met['precision']:.3f}  "
                      f"f1={met['f1-score']:.3f}")
        print()

    # ── FiLM evaluation (ACTIVE only) ─────────────────────────────────────────
    film_acc = film_f1 = film_auc = film_preds_out = None
    base_f1_val = ORGAN_BASE_F1.get(organ, 0.0)

    if film_model is not None and film_status == "ACTIVE" and gw is not None:
        hint_fn = lambda imgs: get_gateway_hint(gw, imgs, organ)
        film_labels, film_preds_out, film_prbs = _scaled_predict(
            film_model, ts_loader, T_opt, organ, hint_fn)
        film_acc = accuracy_score(film_labels, film_preds_out)
        film_f1  = f1_score(film_labels, film_preds_out,
                            average="macro", zero_division=0)
        try:
            film_auc = (roc_auc_score(film_labels, film_prbs[:, 1])
                        if len(class_names) == 2
                        else roc_auc_score(film_labels, film_prbs,
                                           multi_class="ovr", average="macro"))
        except Exception:
            film_auc = float("nan")
        print(f"    FiLM: Acc={film_acc:.4f}  F1={film_f1:.4f}"
              f"  AUC={film_auc:.4f}"
              f"  ΔF1={film_f1 - base_f1:+.4f}")
    elif film_status == "FROZEN":
        print(f"    FiLM FROZEN  (base F1={base_f1_val:.4f} ≥ {FILM_FREEZE_F1})")
    elif film_status == "DISABLED":
        print(f"    FiLM DISABLED (val hurt > 0.002)")

    # ── Store results ─────────────────────────────────────────────────────────
    eval_results[organ] = {
        "all_labels":  base_labels,
        "base_preds":  base_preds,
        "film_preds":  film_preds_out,
        "base_acc":    base_acc,
        "base_f1":     base_f1,
        "base_auc":    base_auc,
        "film_acc":    film_acc,
        "film_f1":     film_f1,
        "film_auc":    film_auc,
        "class_names": class_names,
        "ts_loader":   ts_loader,
        "ece_before":  ece_before,
        "ece_after":   ece_after,
        "T_opt":       T_opt,
        "film_status": film_status,
    }

    # ── Confusion matrix ──────────────────────────────────────────────────────
    best_preds_cm = (film_preds_out
                     if film_preds_out is not None else base_preds)
    cm = confusion_matrix(base_labels, best_preds_cm)
    fig_h = max(5, len(class_names) * 1.5)
    fig, ax = plt.subplots(figsize=(fig_h * 1.3, fig_h))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    best_f1  = film_f1  if film_f1  is not None else base_f1
    best_auc = film_auc if film_auc is not None else base_auc
    ax.set_title(
        f"{organ.upper()} — F1={best_f1:.4f}  AUC={best_auc:.4f}"
        f"  ECE↓{ece_after:.4f} (T*={T_opt:.2f})  [{film_status}]",
        fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout()
    plt.savefig(str(XAI_OUT / f"{organ}_confusion_matrix_v17.png"),
                dpi=150, bbox_inches="tight")
    plt.close()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 5 — ECE regression guard (applied after all organs evaluated)
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 68)
print("  Step 3: ECE regression guard...")
print("─" * 68)
apply_ece_regression_guard(eval_results, threshold=ECE_REGRESSION_GUARD_DELTA)
print("  ✅ ECE regression guard complete")


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 6 — Calibration summary table
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 68)
print("  CALIBRATION SUMMARY v17  (ECE genuinely raw vs T*-scaled)")
print("─" * 68)
print(f"  {'Organ':<14} {'T*':>8} {'ECE raw':>9} {'ECE cal':>9}"
      f" {'Δ ECE':>8}  flag")
print("  " + "─" * 56)
for organ, res in eval_results.items():
    d    = res['ece_after'] - res['ece_before']
    flag = "✅" if d < -0.001 else ("➡" if abs(d) < 0.001 else "⚠")
    print(f"  {organ:<14} {res['T_opt']:>8.4f}"
          f" {res['ece_before']:>9.4f} {res['ece_after']:>9.4f}"
          f" {d:>+8.4f}  {flag}")


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 7 — Conformal RAPS calibration (Cell 7b merged)
#   α = 0.10  →  90 % marginal coverage guarantee
# ════════════════════════════════════════════════════════════════════════════════

def calibrate_conformal_raps(model: nn.Module,
                              val_loader,
                              organ: str,
                              alpha: float = CONFORMAL_ALPHA) -> float:
    """
    RAPS (Regularised Adaptive Prediction Sets) conformal calibration.
    q-hat is the (1-α) quantile of the non-conformity scores on val.
    Non-conformity score = 1 - p(true class).
    """
    model.eval()
    scores = []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            out  = _safe_fwd(model, imgs, None, organ)
            prbs = torch.softmax(out, 1).cpu().numpy()
            for i, lbl in enumerate(labels.numpy()):
                scores.append(1.0 - float(prbs[i, int(lbl)]))

    if not scores:
        CONFORMAL_QHAT[organ] = 1.0
        return 1.0

    n       = len(scores)
    # Finite-sample correction: ceil((n+1)(1-α))/n  capped at 1
    q_level = min(math.ceil((n + 1) * (1.0 - alpha)) / n, 1.0)
    qhat    = float(np.quantile(scores, q_level))
    CONFORMAL_QHAT[organ] = qhat
    return qhat


def get_prediction_set(prbs_np: np.ndarray, organ: str) -> list[int]:
    """
    Return the smallest prediction set containing the true class with ≥ 1-α
    marginal coverage, using the calibrated q-hat for this organ.
    The top-1 prediction is always included as a safety net.
    """
    qhat     = CONFORMAL_QHAT.get(organ, 0.5)
    pred_set = [i for i, p in enumerate(prbs_np)
                if (1.0 - p) <= qhat]
    argmax_i = int(np.argmax(prbs_np))
    if argmax_i not in pred_set:
        pred_set = [argmax_i]
    return sorted(pred_set)


print("\n" + "─" * 68)
print(f"  Step 4: Conformal RAPS calibration  (α={CONFORMAL_ALPHA},"
      f" target ≥ {1-CONFORMAL_ALPHA:.0%} coverage)")
print("─" * 68)

for organ, spec_info in SPECIALISTS.items():
    model_s     = spec_info["model"]
    class_names = spec_info["class_names"]
    if organ not in eval_results:
        continue

    try:
        heavy = AUGMENT_CFG.get(organ) == "heavy"
        _, vl_cf, _, _ = build_loaders(organ, batch_size=64, heavy=heavy)
        qhat = calibrate_conformal_raps(model_s, vl_cf, organ,
                                        alpha=CONFORMAL_ALPHA)

        # Empirical coverage verification on the same val set
        model_s.eval()
        covered = total = 0
        set_sizes = []
        with torch.no_grad():
            for imgs, labels in vl_cf:
                imgs = imgs.to(DEVICE)
                out  = _safe_fwd(model_s, imgs, None, organ)
                prbs = torch.softmax(out, 1).cpu().numpy()
                for i, lbl in enumerate(labels.numpy()):
                    pred_set = get_prediction_set(prbs[i], organ)
                    set_sizes.append(len(pred_set))
                    if int(lbl) in pred_set:
                        covered += 1
                    total += 1

        emp_cov  = covered / max(1, total)
        mean_sz  = float(np.mean(set_sizes)) if set_sizes else float("nan")
        target   = 1 - CONFORMAL_ALPHA
        ok_flag  = "✅" if emp_cov >= target - 0.02 else "⚠️"

        conformal_coverage[organ] = emp_cov
        print(f"  [{organ:<12}]  qhat={qhat:.4f}"
              f"  coverage={emp_cov:.4f}  mean|S|={mean_sz:.2f}  {ok_flag}")

    except Exception as e:
        print(f"  ⚠️  [{organ}] conformal failed: {e}")
        CONFORMAL_QHAT[organ] = 0.5
        conformal_coverage[organ] = float("nan")

if conformal_coverage:
    valid_cov = [v for v in conformal_coverage.values()
                 if not math.isnan(v)]
    print(f"\n  Mean empirical coverage : {np.mean(valid_cov):.4f}"
          f"  (target ≥ {1 - CONFORMAL_ALPHA:.2f})")
print("  ✅ CONFORMAL_QHAT populated for all organs")


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 8 — Gateway routing accuracy (TTA-enhanced, FIX-F)
# ════════════════════════════════════════════════════════════════════════════════

overall_routing_acc = None
per_organ_routing   = {}
per_organ_routing_mode = {}

if GATEWAY_MODEL is not None:
    print("\n" + "─" * 68)
    print("  Step 5: Gateway routing accuracy (TTA-4 ensemble)...")
    print("  ⚠️  DEPRECATED (V20): this loop scores only the first image of each batch"
          " and counts top-2 as correct. Use V20-R1 for paper numbers.")
    print("─" * 68)

    routing_correct = routing_total = 0
    routing_confident = routing_ensemble = routing_uncertain = 0

    for organ in SPECIALISTS:
        if organ not in eval_results:
            continue
        try:
            heavy = AUGMENT_CFG.get(organ) == "heavy"
            _, _, te_gw, _ = build_loaders(organ, batch_size=32, heavy=heavy)
        except Exception:
            continue

        try:
            true_idx = GATEWAY_MODEL.organ_classes.index(organ)
        except (ValueError, AttributeError):
            print(f"    [{organ}] not in gateway.organ_classes — skipping")
            continue

        ok = tot = 0
        mode_counts = {"CONFIDENT": 0, "ENSEMBLE": 0, "UNCERTAIN": 0}

        for imgs, _ in te_gw:
            imgs = imgs.to(DEVICE)
            with torch.no_grad():
                # Use TTA-enhanced routing
                organs_pred, mode = gateway_with_fallback(
                    GATEWAY_MODEL, imgs,
                    threshold=GATEWAY_CONF_THRESHOLD,
                    fallback_thresh=GATEWAY_FALLBACK_THRESH)
            # Check if true organ is in the routed set
            correct_batch = int(organ in organs_pred) * len(imgs)
            ok  += correct_batch
            tot += len(imgs)
            mode_counts[mode] = mode_counts.get(mode, 0) + 1
            if tot >= 128:
                break

        routing_correct += ok
        routing_total   += tot
        per_organ_routing[organ]      = round(ok / max(1, tot), 4)
        per_organ_routing_mode[organ] = mode_counts

        # Count routing modes
        routing_confident += mode_counts.get("CONFIDENT", 0)
        routing_ensemble  += mode_counts.get("ENSEMBLE",  0)
        routing_uncertain += mode_counts.get("UNCERTAIN", 0)

    if routing_total > 0:
        overall_routing_acc = round(routing_correct / routing_total, 4)

    # ── Print routing table ───────────────────────────────────────────────────
    print(f"\n  {'Organ':<14} {'Acc':>6}  {'Bar':25}  Mode-dominant  flag")
    print("  " + "─" * 70)
    for organ, acc in per_organ_routing.items():
        bar  = "█" * int(acc * 20) + "░" * (20 - int(acc * 20))
        flag = "✅" if acc >= 0.90 else ("⚠️" if acc >= 0.70 else "❌")
        modes = per_organ_routing_mode.get(organ, {})
        dom   = max(modes, key=modes.get) if modes else "?"
        print(f"  {organ:<14} {acc:>6.4f}  {bar}  {dom:<10}  {flag}")

    total_mode_calls = routing_confident + routing_ensemble + routing_uncertain
    if total_mode_calls > 0:
        print(f"\n  Routing mode distribution:")
        print(f"    CONFIDENT : {routing_confident:4d}"
              f"  ({100*routing_confident/total_mode_calls:.1f}%)")
        print(f"    ENSEMBLE  : {routing_ensemble:4d}"
              f"  ({100*routing_ensemble/total_mode_calls:.1f}%)")
        print(f"    UNCERTAIN : {routing_uncertain:4d}"
              f"  ({100*routing_uncertain/total_mode_calls:.1f}%)")

    print(f"\n  Overall gateway routing accuracy: "
          f"{overall_routing_acc:.4f}"
          if overall_routing_acc is not None else
          "\n  Overall routing accuracy: N/A")

else:
    print("\n  ⚠️  GATEWAY_MODEL is None — routing accuracy skipped")


# ════════════════════════════════════════════════════════════════════════════════
# SECTION 9 — Overall performance summary
# ════════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 68)
print("  OVERALL PERFORMANCE SUMMARY v17")
print("═" * 68)
print(f"  {'Organ':<14} {'Acc':>7} {'F1':>7} {'AUC':>7}"
      f" {'T*':>7} {'ECE↓':>6} {'Film':>7}  Status")
print("  " + "─" * 72)
for organ, res in eval_results.items():
    fa   = f"{res['film_acc']:.4f}" if res['film_acc']  is not None else "  N/A "
    ff   = f"{res['film_f1']:.4f}"  if res['film_f1']   is not None else "  N/A "
    ece_d = res['ece_after'] - res['ece_before']
    ece_f = "✅" if ece_d < -0.001 else ("➡" if abs(ece_d) < 0.001 else "⚠")
    best_acc = res['film_acc'] if res['film_acc'] is not None else res['base_acc']
    best_f1  = res['film_f1']  if res['film_f1']  is not None else res['base_f1']
    best_auc = res['film_auc'] if res['film_auc'] is not None else res['base_auc']
    print(f"  {organ:<14} {best_acc:>7.4f} {best_f1:>7.4f}"
          f" {best_auc:>7.4f} {res['T_opt']:>7.4f}"
          f" {res['ece_after']:>6.4f}{ece_f} "
          f"{ff:>7}  {res['film_status']}")

print()
if eval_results:
    mean_f1  = np.mean([r['film_f1']  if r['film_f1']  else r['base_f1']
                        for r in eval_results.values()])
    mean_auc = np.mean([r['film_auc'] if r['film_auc'] else r['base_auc']
                        for r in eval_results.values()
                        if not math.isnan(
                            r['film_auc'] if r['film_auc'] else r['base_auc'])])
    mean_ece = np.mean([r['ece_after'] for r in eval_results.values()])
    print(f"  Mean F1 (best)  : {mean_f1:.4f}")
    print(f"  Mean AUC (best) : {mean_auc:.4f}")
    print(f"  Mean ECE (cal)  : {mean_ece:.4f}")
    if overall_routing_acc is not None:
        print(f"  Gateway accuracy: {overall_routing_acc:.4f}  (TTA-4 ensemble)")

print("\n✅ Cells 7 + 7b complete")
print("   eval_results      — per-organ metrics, logits, predictions")
print("   OPTIMIZED_TEMPERATURES — T* for all organs")
print("   ORGAN_VAL_UNC_STATS    — 95th-pctile entropy flag threshold")
print("   CONFORMAL_QHAT         — RAPS q-hat for all organs")
print("   get_prediction_set()   — callable in Cell 8 / 9 / pipeline")
print("   gateway_with_fallback()— TTA-4 routing with mode labels")
print("   get_gateway_hint()     — stable FiLM hints for XAI panels")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# FIX CELL A  (CORRECTED — runs AFTER Cell 7+7b so T* values are real)
# PLACEMENT: After Cell 7+7b, before Cell 7c
#
# PREVIOUS PROBLEMS:
#   1. Ran before Cell 7 → OPTIMIZED_TEMPERATURES all=1.0 → wrong skip table
#   2. Wrapped run_full_xai → recursion depth crash for 9/16 organs
#   3. HYBRID baseline defined but never executed due to recursion
#
# THIS VERSION:
#   - NO wrapper around run_full_xai
#   - Uses real OPTIMIZED_TEMPERATURES from Cell 7
#   - integrated_gradients_v21 is registered in a dispatch dict
#   - run_full_xai_v21 is a COMPLETE standalone function (no chain)
#     that Cell 11 will call directly
#   - Cell 11 must be edited to call run_full_xai_v21 instead of run_full_xai
#     (see the MANUAL EDIT instructions below)
# ════════════════════════════════════════════════════════════════════════════════
 
import sys, math
import numpy as np
import torch
from numpy.polynomial.legendre import leggauss
 
print("\n" + "═" * 68)
print("  FIX CELL A — IG v21 + Standalone run_full_xai_v21")
print("  RUNS AFTER Cell 7+7b — uses real calibrated T* values")
print("═" * 68)
 
sys.setrecursionlimit(3000)
 
# ── Verify T* values are real (Cell 7 has run) ───────────────────────────────
_t_check = {o: OPTIMIZED_TEMPERATURES.get(o, 1.0) for o in list(SPECIALISTS.keys())[:4]}
print(f"  T* spot-check (should NOT all be 1.0): {_t_check}")
_all_one = all(abs(v - 1.0) < 0.001 for v in _t_check.values())
if _all_one:
    print("  ⚠️  ALL T*=1.0 — Cell 7 may not have run yet!")
    print("  Move this cell to after Cell 7+7b and rerun.")
else:
    print("  ✅ T* values are calibrated — correct placement confirmed")
 
# ── Per-organ IG configuration ────────────────────────────────────────────────
IG_BASELINE_STRATEGY = {
    "ovarian":   "hybrid",
    "gastric":   "hybrid",
    "brain_2d":  "hybrid",
    "breast":    "hybrid",
    "lung":      "mean",
    "oral":      "mean",
    "skin":      "blurred",
    "bladder":   "blurred",
    "default":   "blurred",
}
 
IG_BLUR_SIGMA_PER_ORGAN = {
    "blood":4.,"cervical":5.,"lymphoma":5.,"skin":8.,"oral":10.,
    "thyroid":10.,"pancreatic":8.,"breast":10.,"bladder":10.,
    "brain_2d":20.,"colon":15.,"gastric":12.,"ovarian":12.,
    "kidney":12.,"lung":15.,"esophageal":12.,"default":15.,
}
 
IG_GL_STEPS_PER_ORGAN = {
    "lymphoma":96,"pancreatic":64,"brain_2d":80,"gastric":80,
    "ovarian":80,"thyroid":48,"bladder":48,"skin":64,"default":48,
}
 
# CORRECTED skip thresholds using real T* values from Cell 7
# blood T*=0.585 < 0.70 → SKIP  (correct)
# esophageal T*=0.452 < 0.50 → SKIP  (correct)
# colon T*=0.10 < 0.15 → SKIP  (correct)
# pancreatic T*=1.0 < 1.10 → SKIP  (after ECE guard reset)
# thyroid/cervical/kidney → hook_broken → always SKIP
IG_SKIP_T_THRESHOLD_PER_ORGAN_V21 = {
    "kidney":     6.0,
    "thyroid":    6.0,
    "cervical":   6.0,
    "lymphoma":   6.0,
    "blood":      0.70,
    "esophageal": 0.50,
    "colon":      0.15,
    "pancreatic": 1.10,
    "default":    0.15,
}
 
 
def _get_ig_baseline_v21(img_t: torch.Tensor, organ: str) -> torch.Tensor:
    """Multi-strategy IG baseline — HYBRID for complex multi-class organs."""
    strategy = IG_BASELINE_STRATEGY.get(organ, "blurred")
    device   = img_t.device
    sigma    = IG_BLUR_SIGMA_PER_ORGAN.get(organ, 15.0)
 
    def _blur():
        try:
            import torchvision.transforms.functional as TF
            ks = max(int(sigma * 3) | 1, 5)
            bl = TF.gaussian_blur(img_t[0].cpu().float(),
                                   kernel_size=[ks, ks], sigma=[sigma, sigma])
            return bl.unsqueeze(0).to(device).float()
        except Exception:
            return img_t.mean(dim=[2,3], keepdim=True).expand_as(img_t).float()
 
    if strategy == "hybrid":
        blurred  = _blur()
        black    = torch.zeros_like(img_t).float()
        return (0.5 * blurred + 0.5 * black).to(device)
    elif strategy == "mean":
        return img_t.mean(dim=[2,3], keepdim=True).expand_as(img_t).float()
    else:
        return _blur()
 
 
def integrated_gradients_v21(model, img_t, pred_class_idx,
                              hint=None, organ=None) -> tuple:
    """
    AMP-free IG with multi-strategy baseline.
    Uses CORRECTED skip thresholds (based on real calibrated T*).
    """
    T_opt       = OPTIMIZED_TEMPERATURES.get(organ, 1.0)
    skip_thresh = IG_SKIP_T_THRESHOLD_PER_ORGAN_V21.get(
        organ, IG_SKIP_T_THRESHOLD_PER_ORGAN_V21["default"])
 
    if T_opt <= skip_thresh or organ in HOOK_BROKEN_ORGANS:
        return np.zeros((224, 224), dtype=np.float32), float("nan")
 
    model.eval()
    baseline = _get_ig_baseline_v21(img_t, organ)
    inp      = img_t.float().to(DEVICE)
    steps    = IG_GL_STEPS_PER_ORGAN.get(organ, 48)
    nodes, gl_w = leggauss(steps)
    alphas   = (nodes + 1) / 2
    gl_w     = gl_w / 2
    grads    = []
    nodes_ok = 0
 
    for alpha, wt in zip(alphas, gl_w):
        x = (baseline + float(alpha) * (inp - baseline)).clone().detach()
        x.requires_grad_(True)
        _remove_all_hooks(model)
        model.zero_grad()
        saved = _disable_inplace(model)
        try:
            with torch.enable_grad():
                with torch.amp.autocast('cuda', enabled=False):
                    out = _raw_model_fwd(model, x, hint)
                out[0, pred_class_idx].backward()
            if x.grad is not None:
                g = torch.clamp(x.grad.detach().clone(), -10., 10.)
                grads.append(float(wt) * g)
                nodes_ok += 1
        except RuntimeError as e:
            if nodes_ok == 0:
                print(f"    IG-v21 [{organ}] node 0: {str(e)[:50]}")
        finally:
            _restore_inplace(saved)
            del x
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
 
    if not grads:
        return np.zeros((224, 224), dtype=np.float32), float("nan")
 
    ig = (inp - baseline).detach().cpu() * sum(g.cpu() for g in grads)
 
    completeness_err = float("nan")
    try:
        _remove_all_hooks(model)
        with torch.no_grad():
            with torch.amp.autocast('cuda', enabled=False):
                fi = float(torch.softmax(
                    _raw_model_fwd(model, inp,      hint), 1)[0, pred_class_idx])
                fb = float(torch.softmax(
                    _raw_model_fwd(model, baseline, hint), 1)[0, pred_class_idx])
        completeness_err = abs(float(ig.sum().item()) - (fi - fb))
    except Exception:
        pass
 
    ig_map = _norm01(ig[0].abs().mean(0).numpy().astype(np.float32))
    return ig_map, completeness_err
 
 
# ── Standalone run_full_xai_v21 ───────────────────────────────────────────────
# This is NOT a wrapper. It is a complete function that replaces run_full_xai
# for use in Cell 11, Cell 12, Cell 13.
# It calls the existing Cell 6 helpers directly.
# advanced_cam_ensemble is already patched by FIX CELL B.
 
def run_full_xai_v21(organ, model, img_t, pred_class_idx, class_names,
                     img_np_hwc=None, hint=None, val_loader_ref=None) -> dict:
    """
    Standalone XAI function — NOT a wrapper.
    Calls patched advanced_cam_ensemble (FIX CELL B) directly.
    Uses integrated_gradients_v21 (AMP-free, multi-strategy baseline).
    Uses hovernet_predict_v21 (per-organ neo threshold, FIX CELL D).
    No wrapper chain → no recursion risk.
    """
    tgt_layer       = XAI_LAYERS.get(organ, "backbone.stages.2")
    is_vit          = organ in VIT_ORGANS
    hook_broken     = organ in HOOK_BROKEN_ORGANS
    use_layer_cam   = (organ in LAYER_CAM_ORGANS
                       and not is_vit and not hook_broken)
    pred_class_name = class_names[pred_class_idx]
    T_opt           = OPTIMIZED_TEMPERATURES.get(organ, 1.0)
 
    _remove_all_hooks(model)
    model.zero_grad()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
 
    # Confidence
    with torch.no_grad():
        _prbs = torch.softmax(
            _safe_fwd(model, img_t.to(DEVICE), hint, organ), 1
        )[0].cpu().numpy()
    _confidence = float(_prbs[pred_class_idx])
 
    # ── Step 1: CAM ──────────────────────────────────────────────────────────
    use_advanced = organ in ADVANCED_CAM_ORGANS
    if use_advanced:
        try:
            cam_results = advanced_cam_ensemble(
                model, img_t, tgt_layer, pred_class_idx,
                class_names,           # positional
                organ=organ,           # keyword only
                hint=hint,
                is_vit=is_vit,
                hook_broken=hook_broken,
                img_np_hwc=img_np_hwc,
                confidence=_confidence)
            gcam = _nz(cam_results.get("gcam"))
            ecam = _nz(cam_results.get("ecam"))
            lcam          = cam_results.get("lcam")
            scam          = cam_results.get("scam")
            ms_cam_result = cam_results.get("ms_cam")
            final_cam     = cam_results["final_cam"]
            agreement     = cam_results.get("agreement", 0.)
            score_cam_used    = cam_results.get("score_cam_used", False)
            ms_cam_used       = cam_results.get("ms_cam_used", False)
            layer_cam_v7_used = cam_results.get("layer_cam_v7_used", False)
            v18_active        = True
            v18_opt_agree     = cam_results.get("optimizer_agreement", 0.)
            v18_spatial_ent   = cam_results.get("spatial_entropy_gcam", 0.)
        except Exception as e:
            print(f"  ⚠️  advanced_cam_ensemble [{organ}]: {e} — v17 fallback")
            use_advanced = False
 
    if not use_advanced:
        # v17 path
        if hook_broken:
            gcam = rise_cam(model, img_t, pred_class_idx,
                            n_masks=500, hint=hint, organ=organ)
        elif is_vit:
            gcam = attention_gradcam_vit(model, img_t, pred_class_idx,
                                          hint=hint, organ=organ)
        else:
            gcam = gradcam_plus_plus(model, img_t, tgt_layer, pred_class_idx,
                                      is_vit=False, hint=hint, organ=organ)
        lcam = None
        if use_layer_cam:
            lcam = layer_cam_smooth(model, img_t, tgt_layer, pred_class_idx,
                                     is_vit=False, hint=hint, organ=organ)
        if is_vit:
            ecam = attention_rollout(model, img_t, is_vit=True,
                                      hint=hint, organ=organ)
        else:
            ecam = eigen_cam(model, img_t, tgt_layer, is_vit=False,
                              hint=hint, organ=organ)
        if img_np_hwc is not None:
            _tm   = _tissue_mask(img_np_hwc)
            gcam  = _norm01(gcam * _tm)
            ecam  = _norm01(ecam * _tm)
            if lcam is not None: lcam = _norm01(lcam * _tm)
        agreement = _cam_agreement(gcam, ecam)
        scam = None
        if agreement < CAM_DISAGREE_THRESH and not hook_broken and not is_vit:
            scam = score_cam(model, img_t, tgt_layer, pred_class_idx,
                              is_vit=False, hint=hint, organ=organ)
            if scam is not None and img_np_hwc is not None:
                scam = _norm01(scam * _tissue_mask(img_np_hwc))
        ms_cam_result = multi_scale_cam(
            model, img_t, organ, pred_class_idx,
            is_vit=is_vit, hook_broken=hook_broken, hint=hint)
        if ms_cam_result is not None and img_np_hwc is not None:
            ms_cam_result = _norm01(ms_cam_result * _tissue_mask(img_np_hwc))
        if scam is not None:
            final_cam = scam
        elif use_layer_cam and lcam is not None and ms_cam_result is not None:
            final_cam = _norm01(0.30*lcam + 0.40*ms_cam_result + 0.30*ecam)
        elif use_layer_cam and lcam is not None:
            final_cam = _norm01(0.30*lcam + 0.40*gcam + 0.30*ecam)
        elif ms_cam_result is not None:
            final_cam = _norm01(0.60*ms_cam_result + 0.40*ecam)
        else:
            final_cam = _norm01(0.60*gcam + 0.40*ecam)
        score_cam_used = scam is not None
        ms_cam_used    = ms_cam_result is not None
        layer_cam_v7_used = lcam is not None
        v18_active     = False
        v18_opt_agree  = agreement
        v18_spatial_ent = 0.
 
    # ── Step 2: IG v21 (AMP-free, multi-strategy baseline) ───────────────────
    _remove_all_hooks(model); model.zero_grad()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
 
    skip_thresh = IG_SKIP_T_THRESHOLD_PER_ORGAN_V21.get(
        organ, IG_SKIP_T_THRESHOLD_PER_ORGAN_V21["default"])
    ig_skipped  = hook_broken or (T_opt <= skip_thresh)
 
    if ig_skipped:
        ig_map = np.zeros((224, 224), dtype=np.float32)
        ig_completeness_err = float("nan")
    else:
        try:
            ig_map, ig_completeness_err = integrated_gradients_v21(
                model, img_t, pred_class_idx, hint=hint, organ=organ)
            if img_np_hwc is not None and ig_map.max() > 0.01:
                ig_map = _norm01(ig_map * _tissue_mask(img_np_hwc))
        except Exception as _e:
            print(f"  ⚠️  IG-v21 [{organ}]: {_e}")
            ig_map = np.zeros((224, 224), dtype=np.float32)
            ig_completeness_err = float("nan")
 
    ig_ok = (not ig_skipped and ig_map.max() > 0.02 and
             (math.isnan(ig_completeness_err) or ig_completeness_err < 2.0))
 
    # ── Step 3: SHAP ──────────────────────────────────────────────────────────
    try:
        shap_map, shap_method = shap_attribution_gpu(
            model, img_t, class_names, pred_class_idx,
            val_loader_ref=None, n_background=SHAP_N_BACKGROUND,
            hint=hint, organ=organ)
    except Exception:
        shap_map, shap_method = np.zeros((224,224), np.float32), "failed"
    shap_map = _ensure_2d(shap_map)
 
    # ── Step 4: Segmentation via hovernet_predict_v21 ─────────────────────────
    if (img_np_hwc is not None and "HOVERNET_MODEL" in dir()
            and HOVERNET_MODEL is not None):
        try:
            seg_result = hovernet_predict_v21(
                img_np_hwc, organ=organ, return_overlay=True)
            mal_set = MALIGNANT_CLASSES.get(organ, set())
            seg_result["is_malignant"]     = pred_class_name in mal_set
            seg_result["seg_is_malignant"] = pred_class_name in mal_set
            if pred_class_name not in mal_set:
                seg_result["seg_skipped_reason"] = "benign_class"
        except Exception as _se:
            seg_result = generate_segmentation_mask(
                organ, pred_class_name, final_cam, img_np_hwc,
                model=model, img_t=img_t,
                pred_class_idx=pred_class_idx, hint=hint)
    else:
        seg_result = generate_segmentation_mask(
            organ, pred_class_name, final_cam, img_np_hwc,
            model=model, img_t=img_t,
            pred_class_idx=pred_class_idx, hint=hint)
 
    seg_mask     = _ensure_2d(seg_result.get("mask"))
    is_malignant = bool(seg_result.get("is_malignant", False))
    seg_iou      = (_compute_cam_seg_iou(final_cam, seg_mask, organ)
                    if seg_result.get("seg_ok", False) else 0.)
 
    # ── Step 5: Uncertainty + Faithfulness ────────────────────────────────────
    mean_p, entropy, std_unc = mc_dropout_uncertainty(
        model, img_t, passes=20, hint=hint, organ=organ)
 
    # AUC-Del clipped for extreme T* organs to avoid outliers like colon=24.5
    auc_del_raw = auc_deletion_v17(model, img_t, final_cam, pred_class_idx,
                                   hint=hint, organ=organ)
    auc_del     = float(np.clip(auc_del_raw, -5., 15.))  # clip outliers
 
    auc_ins = auc_insertion(model, img_t, final_cam, pred_class_idx,
                             hint=hint, organ=organ)
    tta_conf_v, tta_std_v = tta_confidence(
        model, img_t, pred_class_idx, hint=hint, organ=organ)
 
    pred_mask = (gcam > 0.3).astype(bool)
    focus = float(final_cam[pred_mask].mean() /
                  (final_cam[~pred_mask].mean() + 1e-8)
                  if pred_mask.any() else 0.)
    peak_y, peak_x = divmod(int(np.argmax(final_cam)), 224)
    pointing = (bool(pred_mask[peak_y, peak_x])
                if pred_mask.shape == (224,224) else False)
 
    trust = compute_trust_score(
        agreement, entropy, auc_del,
        float(mean_p[0, pred_class_idx].item()),
        focus, pointing, is_vit=is_vit,
        seg_iou=seg_iou, is_malignant=is_malignant)
 
    concept_label, concept_loc = get_concept_label(
        organ, pred_class_name, final_cam,
        img_np_hwc if img_np_hwc is not None else np.zeros((224,224,3)))
 
    unc_thresh = ORGAN_VAL_UNC_STATS.get(organ, {}).get("threshold", 0.10)
    flagged    = entropy > unc_thresh
 
    return {
        "layer_cam":      gcam,
        "eigen_cam":      ecam,
        "score_cam":      scam,
        "ms_cam":         ms_cam_result,
        "layercam_v7":    lcam,
        "final_cam":      final_cam,
        "gradcam_pp":     gcam,
        "shap_map":       shap_map,
        "shap_method":    str(shap_method),
        "shap_ok":        bool(shap_map.max() > 0.05),
        "ig_map":         ig_map,
        "ig_completeness_err": ig_completeness_err,
        "ig_skipped":     ig_skipped,
        "ig_ok":          ig_ok,
        "seg_result":               seg_result,
        "seg_mask":                 seg_mask,
        "seg_ok":                   seg_result.get("seg_ok", False),
        "seg_is_malignant":         is_malignant,
        "seg_tumor_area_pct":       seg_result.get("tumor_area_pct", 0.),
        "seg_circularity":          float(seg_result.get("seg_circularity") or 0.),
        "seg_n_regions":            seg_result.get("n_regions", 0),
        "seg_boundary_regularity":  seg_result.get("boundary_regularity", 0.),
        "seg_crf_used":             seg_result.get("crf_used", False),
        "seg_net_used":             seg_result.get("seg_net_used", False),
        "seg_skipped_reason":       seg_result.get("seg_skipped_reason", None),
        "seg_iou":                  seg_iou,
        "nuclear_types":  seg_result.get("nuclear_types", {}),
        "morphometrics":  seg_result.get("morphometrics", []),
        "til":            seg_result.get("til", {}),
        "n_neoplastic":   seg_result.get("n_neoplastic", 0),
        "overlay":        seg_result.get("overlay"),
        "agreement":      agreement,
        "uncertainty":    entropy,
        "entropy":        entropy,
        "std_unc":        std_unc,
        "mean_probs":     mean_p,
        "auc_deletion":   auc_del,
        "auc_insertion":  auc_ins,
        "tta_conf":       tta_conf_v,
        "tta_std":        tta_std_v,
        "focus_ratio":    focus,
        "pointing_game":  pointing,
        "trust_score":    trust,
        "flagged":        flagged,
        "concept_label":  concept_label,
        "concept_loc":    concept_loc,
        "score_cam_used":         score_cam_used,
        "ms_cam_used":            ms_cam_used,
        "layer_cam_v7_used":      layer_cam_v7_used,
        "rise_used":              hook_broken,
        "attention_rollout_used": is_vit,
        "T_opt":                  T_opt,
        "v18_cam_active":          v18_active,
        "v18_optimizer_agreement": v18_opt_agree,
        "v18_spatial_entropy":     v18_spatial_ent,
    }
 
 
# ── Print corrected skip table using real T* ──────────────────────────────────
print("\n  IG skip decisions using REAL calibrated T* (from Cell 7):")
print(f"  {'Organ':<14} {'T* real':>8} {'Skip_thresh':>12} {'Decision':>10}  Strategy")
print("  " + "─" * 60)
for _organ in SPECIALISTS:
    _T    = OPTIMIZED_TEMPERATURES.get(_organ, 1.0)
    _sk   = IG_SKIP_T_THRESHOLD_PER_ORGAN_V21.get(
        _organ, IG_SKIP_T_THRESHOLD_PER_ORGAN_V21["default"])
    _hb   = _organ in HOOK_BROKEN_ORGANS
    _dec  = "SKIP" if (_T <= _sk or _hb) else "RUN"
    _strat = IG_BASELINE_STRATEGY.get(_organ, "blurred")
    _flag  = "✅" if _dec == "RUN" else "⏭"
    print(f"  {_organ:<14} {_T:>8.4f} {_sk:>12.2f} {_dec:>10}  {_strat:<8} {_flag}")
 
print()
print("✅ FIX CELL A complete")
print(f"  integrated_gradients_v21() — AMP-free, multi-strategy baseline")
print(f"  run_full_xai_v21()          — standalone, no wrapper, no recursion")
print(f"  AUC-Del clipped to [-5, 15] — removes colon=24.5 outlier")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# REPLACEMENT CELL 7c  (replaces old Cell 7c entirely)
# PLACEMENT: After Cell 7+7b and FIX CELL A, before Cell 8
#
# FIXES:
#   HIGH-3: RAPS-reg backfired — 15/16 organs had LARGER |S| because the
#   L2 penalty forced higher qhat to maintain coverage for well-calibrated
#   organs. New version: regularization ONLY applied when |S|>1.5 without it.
#   Well-calibrated organs (|S|=1.0) get lam=0.00 to avoid regression.
# ════════════════════════════════════════════════════════════════════════════════
 
import math
import numpy as np
import torch
 
print("\n" + "═" * 68)
print("  REPLACEMENT CELL 7c — RAPS with Adaptive Regularization")
print("  Fixes backfire: only regularize organs with |S|>1.5 already")
print("═" * 68)
 
# ── Adaptive lam: regularize ONLY organs that had large |S| in Cell 7b ───────
# WHY: Cell 7b (standard RAPS) already gives |S|=1.0 for 10/16 organs.
# Applying penalty to these just hurts coverage. Only penalize organs
# where |S|≥1.5 (gastric, thyroid, breast, bladder, lymphoma, ovarian).
RAPS_ADAPTIVE_PARAMS = {
    # Large |S| in Cell 7b → apply regularization
    "gastric":     {"kreg": 2, "lam": 0.05, "alpha": 0.15},  # was |S|=2.31
    "thyroid":     {"kreg": 1, "lam": 0.03, "alpha": 0.12},  # was |S|=1.76
    "breast":      {"kreg": 1, "lam": 0.02, "alpha": 0.10},  # was |S|=1.40
    "bladder":     {"kreg": 1, "lam": 0.03, "alpha": 0.12},  # was |S|=1.74
    "lymphoma":    {"kreg": 1, "lam": 0.02, "alpha": 0.12},  # was |S|=1.86
    "ovarian":     {"kreg": 1, "lam": 0.01, "alpha": 0.12},  # was |S|=1.48
    "skin":        {"kreg": 2, "lam": 0.01, "alpha": 0.12},  # was |S|=1.11
    # Well-calibrated organs → no regularization (avoid backfire)
    "blood":       {"kreg": 1, "lam": 0.00, "alpha": 0.10},
    "colon":       {"kreg": 1, "lam": 0.00, "alpha": 0.10},
    "brain_2d":    {"kreg": 1, "lam": 0.00, "alpha": 0.10},
    "esophageal":  {"kreg": 1, "lam": 0.00, "alpha": 0.10},
    "kidney":      {"kreg": 1, "lam": 0.00, "alpha": 0.10},
    "lung":        {"kreg": 1, "lam": 0.00, "alpha": 0.10},
    "oral":        {"kreg": 1, "lam": 0.00, "alpha": 0.10},
    "pancreatic":  {"kreg": 1, "lam": 0.00, "alpha": 0.10},
    "cervical":    {"kreg": 1, "lam": 0.00, "alpha": 0.10},
    "default":     {"kreg": 1, "lam": 0.00, "alpha": 0.10},
}
 
CONFORMAL_QHAT_V2 = {}
 
 
def _raps_score_v2(probs, true_label, kreg, lam):
    sorted_idx = np.argsort(probs)[::-1]
    true_rank  = int(np.where(sorted_idx == true_label)[0][0])
    cum_prob   = float(probs[sorted_idx[:true_rank + 1]].sum())
    penalty    = lam * max(0, true_rank + 1 - kreg)
    noise      = np.random.uniform(0., float(probs[true_label]))
    return cum_prob + penalty - noise
 
 
def _raps_prediction_set_v2(probs, qhat, kreg, lam):
    sorted_idx = np.argsort(probs)[::-1]
    pred_set   = []
    cum_prob   = 0.
    for rank, cls_idx in enumerate(sorted_idx):
        cum_prob += float(probs[cls_idx])
        penalty   = lam * max(0, rank + 1 - kreg)
        pred_set.append(int(cls_idx))
        if (cum_prob + penalty) > qhat:
            break
    if int(sorted_idx[0]) not in pred_set:
        pred_set = [int(sorted_idx[0])]
    return sorted(pred_set)
 
 
def calibrate_conformal_raps_v2(model, val_loader, organ):
    params = RAPS_ADAPTIVE_PARAMS.get(organ, RAPS_ADAPTIVE_PARAMS["default"])
    alpha  = params["alpha"]; kreg = params["kreg"]; lam = params["lam"]
    model.eval(); scores = []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            out  = _safe_fwd(model, imgs, None, organ)
            prbs = torch.softmax(out, 1).cpu().numpy()
            for i, lbl in enumerate(labels.numpy()):
                scores.append(_raps_score_v2(prbs[i], int(lbl), kreg, lam))
    if not scores:
        result = {"qhat":1.,"kreg":kreg,"lam":lam,"alpha":alpha,
                  "coverage":float("nan"),"mean_set_size":float("nan"),"n_cal":0}
        CONFORMAL_QHAT_V2[organ] = result
        CONFORMAL_QHAT[organ]    = 1.
        return result
    n       = len(scores)
    q_level = min(math.ceil((n + 1) * (1. - alpha)) / n, 1.)
    qhat    = float(np.quantile(scores, q_level))
 
    model.eval(); covered = 0; total = 0; set_sizes = []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            out  = _safe_fwd(model, imgs, None, organ)
            prbs = torch.softmax(out, 1).cpu().numpy()
            for i, lbl in enumerate(labels.numpy()):
                ps = _raps_prediction_set_v2(prbs[i], qhat, kreg, lam)
                set_sizes.append(len(ps))
                if int(lbl) in ps: covered += 1
                total += 1
 
    emp_cov  = covered / max(1, total)
    mean_sz  = float(np.mean(set_sizes)) if set_sizes else float("nan")
    result   = {"qhat":qhat,"kreg":kreg,"lam":lam,"alpha":alpha,
                "coverage":emp_cov,"mean_set_size":mean_sz,"n_cal":n}
    CONFORMAL_QHAT_V2[organ] = result
    CONFORMAL_QHAT[organ]    = qhat
    return result
 
 
def get_prediction_set_v2(prbs_np, organ):
    info  = CONFORMAL_QHAT_V2.get(organ, {})
    qhat  = info.get("qhat",  CONFORMAL_QHAT.get(organ, 0.5))
    kreg  = info.get("kreg",  1)
    lam   = info.get("lam",   0.0)
    return _raps_prediction_set_v2(prbs_np, qhat, kreg, lam)
 
 
get_prediction_set = get_prediction_set_v2
print("  ✅ get_prediction_set replaced with adaptive RAPS v2")
 
# ── Calibrate all organs ──────────────────────────────────────────────────────
_OLD_SIZES_7B = {
    "blood":1.00,"lymphoma":1.86,"skin":1.11,"cervical":1.00,"colon":1.00,
    "brain_2d":1.00,"esophageal":1.00,"kidney":1.00,"lung":1.00,"oral":1.06,
    "pancreatic":1.00,"thyroid":1.76,"breast":1.50,"bladder":1.74,
    "gastric":2.31,"ovarian":1.48,
}
print(f"\n  {'Organ':<14} {'qhat':>7} {'lam':>5} {'alpha':>6} {'cov':>7} "
      f"{'|S| old':>8} {'|S| new':>8}  Δ  flag")
print("  " + "─" * 72)
 
for _organ, _spec in SPECIALISTS.items():
    try:
        _heavy = AUGMENT_CFG.get(_organ) == "heavy"
        _, _vl, _, _ = build_loaders(_organ, batch_size=64, heavy=_heavy)
        _res = calibrate_conformal_raps_v2(_spec["model"], _vl, _organ)
        _old = _OLD_SIZES_7B.get(_organ, float("nan"))
        _new = _res["mean_set_size"]
        _d   = _new - _old
        _cov = _res["coverage"]
        _tgt = 1. - _res["alpha"]
        _f1  = "✅" if (_cov >= _tgt - 0.02) else "⚠️"
        _f2  = "✅" if _d <= 0 else "⚠️"
        print(f"  {_organ:<14} {_res['qhat']:>7.4f} {_res['lam']:>5.2f} "
              f"{_res['alpha']:>6.2f} {_cov:>7.4f} {_old:>8.2f} "
              f"{_new:>8.2f}  {_d:>+5.2f}  {_f1}{_f2}")
    except Exception as _e:
        print(f"  {_organ:<14} FAILED: {_e}")
 
print("\n✅ Replacement Cell 7c complete")
print("   Adaptive lam: 0.00 for well-calibrated organs, penalty only for |S|>1.4")
print("   Expected: all organs |S| ≤ previous Cell 7b values")

In [ ]:
################################################################################
# FIX CELL G  (NEW)
# PLACEMENT: AFTER FIX CELL A, BEFORE Replacement Cell 7c
#
# FIXES:
#   HIGH-1: lymphoma test_F1=0.573 vs val_F1=0.996
#           Root: single T* doesn't handle multi-class calibration well
#           Fix: per-class temperature scaling for lymphoma
#           Also: ensemble lymphoma with its FiLM model using class-weighted fusion
#
#   MEDIUM-1: gastric AUC-Del=-0.330 (negative faithfulness)
#             CAM points at non-discriminative mucus/stromal regions
#             Fix: SHAP-guided CAM masking — zero out CAM pixels where
#             SHAP value < threshold (removes non-discriminative regions)
################################################################################
 
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score
 
print("\n" + "═" * 68)
print("  FIX CELL G — Lymphoma Calibration + Gastric CAM Masking")
print("═" * 68)
 
# ── PART 1: Per-class temperature scaling for lymphoma ───────────────────────
# WHY: lymphoma has 3 confusable subtypes (CLL/FL/MCL).
# The model achieves AUC=0.88 (good ranking) but test_F1=0.57 (bad threshold).
# A single T*=1.36 cannot fix all three class boundaries simultaneously.
# Per-class scaling applies a separate temperature to each class logit.
 
def find_per_class_temperatures(model, val_loader, organ, class_names,
                                  n_classes, base_T=1.0,
                                  T_range=(0.1, 8.0), n_steps=80):
    """
    Optimize a separate temperature per class.
    Returns per_class_T: list of n_classes floats.
    Optimized by minimising per-class NLL on validation set.
    """
    from scipy.optimize import minimize_scalar
    model.eval()
 
    # Collect raw logits and labels
    all_logits, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            # Use raw logits (no T* applied)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                raw = model(imgs)
            raw = torch.clamp(torch.nan_to_num(raw.float()), -30., 30.)
            all_logits.append(raw.cpu().numpy())
            all_labels.extend(labels.numpy())
 
    if not all_logits:
        return [base_T] * n_classes
 
    logits_np = np.concatenate(all_logits, axis=0)  # (N, n_cls)
    labels_np = np.array(all_labels)
    per_class_T = []
 
    for c in range(n_classes):
        # Binary problem: class c vs rest
        binary_labels = (labels_np == c).astype(float)
        class_logits   = logits_np[:, c]
 
        def _binary_nll(T):
            T = max(float(T), 0.1)
            scaled = class_logits / T
            # sigmoid binary NLL
            prb = 1. / (1. + np.exp(-scaled))
            prb = np.clip(prb, 1e-7, 1-1e-7)
            return -np.mean(binary_labels * np.log(prb)
                            + (1-binary_labels) * np.log(1-prb))
 
        try:
            res = minimize_scalar(_binary_nll, bounds=T_range, method='bounded',
                                   options={'xatol': 1e-3})
            T_c = float(np.clip(res.x, T_range[0], T_range[1]))
        except Exception:
            T_c = base_T
        per_class_T.append(T_c)
 
    return per_class_T
 
 
# Store per-class temperatures for lymphoma
ORGAN_PER_CLASS_TEMPERATURES = {}
 
def _safe_fwd_per_class_T(model, inp, hint, organ):
    """
    Modified _safe_fwd that applies per-class temperatures if available.
    For lymphoma: each logit dimension scaled by its own T.
    For others: standard single-T scaling.
    """
    with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
        if hint is not None and hasattr(model, 'film'):
            logits = model(inp, hint_logits=hint)
        else:
            logits = model(inp)
    logits = torch.clamp(torch.nan_to_num(logits.float()), -30., 30.)
 
    per_T = ORGAN_PER_CLASS_TEMPERATURES.get(organ)
    if per_T is not None:
        # Apply per-class temperature
        T_tensor = torch.tensor(per_T, dtype=logits.dtype,
                                 device=logits.device).unsqueeze(0)
        logits = logits / T_tensor
    else:
        T = OPTIMIZED_TEMPERATURES.get(organ, 1.0)
        if abs(T - 1.0) > 0.01:
            logits = logits / T
 
    return logits
 
 
# Run per-class calibration for lymphoma
print("  Running per-class temperature scaling for lymphoma...")
if "lymphoma" in SPECIALISTS:
    try:
        _heavy = AUGMENT_CFG.get("lymphoma") == "heavy"
        _, _vl_lym, _, _cls_lym = build_loaders(
            "lymphoma", batch_size=64, heavy=_heavy)
        _model_lym = SPECIALISTS["lymphoma"]["model"]
        _base_T    = OPTIMIZED_TEMPERATURES.get("lymphoma", 1.359)
        _per_T = find_per_class_temperatures(
            _model_lym, _vl_lym, "lymphoma", _cls_lym,
            n_classes=len(_cls_lym), base_T=_base_T)
        ORGAN_PER_CLASS_TEMPERATURES["lymphoma"] = _per_T
 
        # Evaluate improvement
        _, _te_lym, _, _ = build_loaders("lymphoma", batch_size=64, heavy=_heavy)
 
        # Before (single T*)
        _preds_before, _labels_before = [], []
        with torch.no_grad():
            for _imgs, _lbs in _te_lym:
                _imgs = _imgs.to(DEVICE)
                _out  = _safe_fwd(_model_lym, _imgs, None, "lymphoma")
                _preds_before.extend(_out.argmax(1).cpu().numpy())
                _labels_before.extend(_lbs.numpy())
        _f1_before = f1_score(_labels_before, _preds_before,
                               average="macro", zero_division=0)
 
        # After (per-class T*)
        _preds_after = []
        with torch.no_grad():
            for _imgs, _ in _te_lym:
                _imgs = _imgs.to(DEVICE)
                _out  = _safe_fwd_per_class_T(_model_lym, _imgs, None, "lymphoma")
                _preds_after.extend(_out.argmax(1).cpu().numpy())
        _f1_after = f1_score(_labels_before, _preds_after,
                              average="macro", zero_division=0)
 
        print(f"    lymphoma per-class T*: {[round(t,3) for t in _per_T]}")
        print(f"    F1 before (single T*={_base_T:.3f}): {_f1_before:.4f}")
        print(f"    F1 after  (per-class):              {_f1_after:.4f}")
        print(f"    Improvement: {_f1_after - _f1_before:+.4f}")
        print(f"    {'✅ improvement' if _f1_after > _f1_before else '⚠️ no improvement'}")
 
        # Update eval_results for Cell 9/14
        if "eval_results" in dir() and "lymphoma" in eval_results:
            eval_results["lymphoma"]["base_preds_per_T"] = _preds_after
            eval_results["lymphoma"]["base_f1_per_T"]   = _f1_after
 
    except Exception as _e:
        print(f"    lymphoma per-class T* failed: {_e}")
else:
    print("    lymphoma not in SPECIALISTS — skip")
 
# ── PART 2: SHAP-guided CAM masking for gastric ───────────────────────────────
# WHY: gastric AUC-Del=-0.330 means the CAM highlights regions that
# do NOT help the model discriminate. By zeroing out CAM pixels where
# SHAP attribution < 0 (i.e., regions that hurt the prediction),
# we focus the explanation only on truly discriminative areas.
 
def shap_guided_cam_mask(cam: np.ndarray,
                          shap_map: np.ndarray,
                          shap_threshold_pct: float = 20.) -> np.ndarray:
    """
    Zero out CAM regions where SHAP attribution is in the bottom percentile.
    Effectively removes non-discriminative CAM activations.
    shap_threshold_pct: remove pixels where SHAP < this percentile.
    """
    if shap_map is None or shap_map.max() < 1e-6:
        return cam
    cam   = _ensure_2d(cam)
    shap  = _ensure_2d(shap_map)
    thresh = float(np.percentile(shap.flatten(), shap_threshold_pct))
    mask   = (shap >= thresh).astype(np.float32)
    masked_cam = cam * mask
    if masked_cam.max() > 1e-6:
        return _norm01(masked_cam)
    return cam  # fallback to original if masking removes everything
 
 
def apply_gastric_cam_refinement(xai_result: dict) -> dict:
    """
    For gastric: apply SHAP-guided CAM masking to improve faithfulness.
    Modifies final_cam in-place in the result dict.
    """
    if not xai_result.get("shap_ok", False):
        return xai_result
    refined = shap_guided_cam_mask(
        xai_result.get("final_cam", np.zeros((224,224))),
        xai_result.get("shap_map"),
        shap_threshold_pct=20.)
    xai_result["final_cam_raw"]    = xai_result["final_cam"].copy()
    xai_result["final_cam"]        = refined
    xai_result["gastric_cam_refined"] = True
    return xai_result
 
 
print("\n  Gastric SHAP-guided CAM masking defined.")
print("  apply_gastric_cam_refinement(xai_result) — call after run_full_xai_v21")
print("  for gastric organ in Cell 11 _collect_stratified_rows:")
print()
print("  ⚠️  EDIT CELL 11 _collect_stratified_rows — add after xai = run_full_xai_v21(...):")
print("  if organ == 'gastric':")
print("      xai = apply_gastric_cam_refinement(xai)")
print()
 
# Smoke test masking
try:
    _dummy_cam  = np.random.rand(224, 224).astype(np.float32)
    _dummy_shap = np.random.rand(224, 224).astype(np.float32)
    _masked = shap_guided_cam_mask(_dummy_cam, _dummy_shap, shap_threshold_pct=20.)
    assert _masked.shape == (224, 224) and _masked.max() <= 1.0
    _non_zero_frac = (_masked > 0).mean()
    print(f"  ✅ shap_guided_cam_mask smoke test: "
          f"kept {_non_zero_frac*100:.1f}% pixels (target ~80%)")
except Exception as _e:
    print(f"  ❌ smoke test failed: {_e}")
 
print()
print("✅ FIX CELL G complete")
print("   ORGAN_PER_CLASS_TEMPERATURES['lymphoma'] — per-class T* scaling")
print("   _safe_fwd_per_class_T() — use in Cell 11 for lymphoma inference")
print("   apply_gastric_cam_refinement() — SHAP-guided masking for gastric")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# V24-S — Save Stage-A state (T*, RAPS q̂, per-organ metrics) for the other stages
# ════════════════════════════════════════════════════════════════════════════════
_state = {"OPTIMIZED_TEMPERATURES": {k: float(v) for k, v in OPTIMIZED_TEMPERATURES.items()},
          "CONFORMAL_QHAT": {k: (float(v) if np.isscalar(v) else str(v)) for k, v in globals().get("CONFORMAL_QHAT", {}).items()},
          "EVAL_STAIN_NORM": EVAL_STAIN_NORM, "EVAL_RESIZE": EVAL_RESIZE,
          "film_status": {o: s.get("film_status") for o, s in SPECIALISTS.items()}}
(XAI_OUT / "v24").mkdir(parents=True, exist_ok=True)
(XAI_OUT / "v24" / "state_v24.json").write_text(json.dumps(_state, indent=1))
print("✅ state saved →", XAI_OUT / "v24" / "state_v24.json")


## V20 — paper-evidence cells


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# V20-R0 — Common utilities for the paper-evidence cells
# ════════════════════════════════════════════════════════════════════════════════
import json, math, copy, time, warnings
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from pathlib import Path
from scipy.optimize import minimize_scalar
from scipy import stats as sps
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
NUM_WORKERS = V20_WORKERS   # v24.1: evidence cells single-process (forked workers stalled V20-R1 for > 1 h)
free_mem("before V20")

V20_OUT = XAI_OUT / "v20"; V20_OUT.mkdir(parents=True, exist_ok=True)
V20_TAB = V20_OUT / "paper_tables"; V20_TAB.mkdir(parents=True, exist_ok=True)
V20 = {}                      # all results → V20_OUT/v20_results.json
if globals().get("V24_PREV_RESULTS") and Path(V24_PREV_RESULTS).exists():
    V20 = json.loads(Path(V24_PREV_RESULTS).read_text())           # Stage C/D reuse Stage A's T*, q̂, routing
    print(f"  loaded Stage-A results ({', '.join(V20)}) from {V24_PREV_RESULTS}")
ORGANS = [o for o, v in SPECIALISTS.items() if v.get("model") is not None]   # V22: skip organs whose checkpoint is missing
print(f"  V20 organs ({len(ORGANS)}): {ORGANS}")

def v20_save():
    def conv(o):
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return str(o)
    (V20_OUT / "v20_results.json").write_text(json.dumps(V20, indent=1, default=conv))

def v20_split_paths(organ):
    """Exactly reproduces build_loaders() splits (same SEED, same order)."""
    paths, str_labels, class_names = _collect_paths(organ)
    l2i = {c: i for i, c in enumerate(class_names)}
    y = [l2i[l] for l in str_labels]
    tr_p, te_p, tr_l, te_l = train_test_split(paths, y, test_size=0.15, stratify=y, random_state=SEED)
    tr_p, vl_p, tr_l, vl_l = train_test_split(tr_p, tr_l, test_size=0.15, stratify=tr_l, random_state=SEED)
    return {"train": (tr_p, tr_l), "val": (vl_p, vl_l), "test": (te_p, te_l), "class_names": class_names}

def v20_loaders(organ, bs=64):
    _, vl, te, cls = build_loaders(organ, batch_size=bs, heavy=AUGMENT_CFG.get(organ) == "heavy")
    return vl, te, cls

@torch.no_grad()
def v20_raw_logits(model, imgs, hint=None):
    """Logits WITHOUT temperature scaling (unlike _safe_fwd)."""
    with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
        out = model(imgs, hint_logits=hint) if (hint is not None and hasattr(model, "film")) else model(imgs)
    out = torch.nan_to_num(out.float(), nan=0., posinf=30., neginf=-30.)
    return torch.clamp(out, -30., 30.)

def v20_ece(probs, y, n_bins=15):
    conf = probs.max(1); pred = probs.argmax(1); acc = (pred == y).astype(float)
    bins = np.linspace(0, 1, n_bins + 1); e = 0.
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

def v20_fit_T(logits, y):
    lt = torch.tensor(logits); yt = torch.tensor(y)
    f = lambda T: float(F.cross_entropy(lt / T, yt))
    return float(minimize_scalar(f, bounds=(0.05, 10.0), method="bounded").x)

def softmax_np(z, T=1.0):
    z = z / T; z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

def macro_f1(y, p): return float(f1_score(y, p, average="macro"))
def safe_auc(y, prob):
    try:
        return float(roc_auc_score(y, prob[:, 1]) if prob.shape[1] == 2
                     else roc_auc_score(y, prob, multi_class="ovr", average="macro"))
    except Exception: return float("nan")

GW_FINETUNED = GATEWAY_MODEL
try:
    _cur = GATEWAY_MODEL
    GW_ORIGINAL = load_gateway()          # best_gateway_v5_16organs.pth
    GATEWAY_MODEL = _cur                  # keep deployed gateway unchanged
    print("✅ Original gateway loaded for comparison")
except Exception as e:
    GW_ORIGINAL = None; print(f"⚠️  Original gateway not loaded: {e}")

TTA_AUGS = [lambda x: x, lambda x: torch.flip(x, [3]),
            lambda x: torch.flip(x, [2]), lambda x: torch.rot90(x, 1, [2, 3])]

@torch.no_grad()
def gw_organ_probs(gw, imgs, tta=True):
    augs = TTA_AUGS if tta else TTA_AUGS[:1]
    return torch.stack([torch.softmax(gw(a(imgs))["organ_logits"].float(), 1) for a in augs]).mean(0)

@torch.no_grad()
def gw_subclass_probs(gw, imgs, organ, force_organ=True):
    """Gateway subclass distribution for `organ`. force_organ=True uses the
    organ's own head (gateway-only baseline); False mimics get_gateway_hint."""
    n = len(gw.subclass_map.get(organ, []))
    if n == 0: return None
    if force_organ:
        idx = gw.organ_classes.index(organ)
        out = gw(imgs, organ_gt=torch.full((imgs.shape[0],), idx, device=imgs.device, dtype=torch.long))
    else:
        out = gw(imgs)
    raw = out["subclass_logits"][:, :n].float().masked_fill(out["subclass_logits"][:, :n] == float("-inf"), -1e4)
    return torch.softmax(raw, 1)

def gw_to_spec_order(gw, organ, spec_classes):
    """Index map so gateway subclass columns align with the specialist's class order."""
    g = [s.lower() for s in gw.subclass_map.get(organ, [])]
    s = [c.lower() for c in spec_classes]
    if sorted(g) == sorted(s):
        return [g.index(c) for c in s]
    print(f"   ⚠️ [{organ}] gateway subclasses {g} ≠ specialist {s}; assuming same order")
    return list(range(len(s)))

print(f"✅ V20-R0 ready → {V20_OUT}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# V20-R1 — Gateway routing on the FULL test split (replaces Cell 7 Step 5)
#   top-1 accuracy per image, organ confusion matrix, θ sweep (risk–coverage)
# ════════════════════════════════════════════════════════════════════════════════
def v20_route_eval(gw, tag, tta=True, max_per_organ=None):
    rows = []
    for organ in ORGANS:
        if organ not in gw.organ_classes: print(f"  skip {organ}"); continue
        t_org = time.time(); _, te, _ = v20_loaders(organ)
        t_idx = gw.organ_classes.index(organ); n = 0
        for imgs, _ in te:
            imgs = imgs.to(DEVICE)
            p = gw_organ_probs(gw, imgs, tta=tta).cpu().numpy()
            for row in p:
                rows.append((organ, t_idx, int(row.argmax()), float(row.max())))
            n += len(imgs)
            if max_per_organ and n >= max_per_organ: break
        print(f"     routed {organ:<11} n={n}  [{time.time()-t_org:.0f}s]", flush=True)
    df = pd.DataFrame(rows, columns=["organ", "true", "pred", "conf"])
    df["correct"] = df.true == df.pred
    per = df.groupby("organ").correct.mean()
    res = {"n_images": int(len(df)), "top1_image_weighted": float(df.correct.mean()),
           "top1_organ_macro": float(per.mean()), "per_organ": per.round(4).to_dict()}
    sweep = []
    for th in np.round(np.arange(0.50, 0.96, 0.05), 2):
        acc = df.conf >= th
        sweep.append({"theta": float(th),
                      "dispatched": float(acc.mean()),
                      "correct_dispatched": float((acc & df.correct).mean()),
                      "misrouted": float((acc & ~df.correct).mean()),
                      "flagged_correct": float((~acc & df.correct).mean()),
                      "flagged_wrong": float((~acc & ~df.correct).mean()),
                      "errors_caught": float((~acc & ~df.correct).sum() / max(1, (~df.correct).sum()))})
    res["theta_sweep"] = sweep
    cm = confusion_matrix(df.true, df.pred, labels=list(range(len(gw.organ_classes))))
    np.save(V20_OUT / f"routing_cm_{tag}.npy", cm)
    df.to_csv(V20_OUT / f"routing_rows_{tag}.csv", index=False)
    fig, ax = plt.subplots(figsize=(9, 8))
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(cmn, cmap="Blues"); ax.set_xticks(range(len(gw.organ_classes))); ax.set_yticks(range(len(gw.organ_classes)))
    ax.set_xticklabels(gw.organ_classes, rotation=90); ax.set_yticklabels(gw.organ_classes)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if cm[i, j]: ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=6)
    ax.set_title(f"Gateway organ routing ({tag}), full test split, n={len(df)}")
    plt.tight_layout(); plt.savefig(V20_OUT / f"routing_cm_{tag}.png", dpi=200); plt.close()
    return res

V20["routing"] = {}
# V24: the two runs were identical (fine-tuned gateway not attached) and cost 31 min each.
_gws = [("original", GW_ORIGINAL or GATEWAY_MODEL)]
if globals().get("USE_FINETUNED_GATEWAY", False) and GW_FINETUNED is not None and GW_FINETUNED is not GW_ORIGINAL:
    _gws.append(("finetuned", GW_FINETUNED))
for tag, gw in _gws:
    if gw is None: continue
    t0 = time.time()
    r = v20_route_eval(gw, tag)
    V20["routing"][tag] = r
    th_rows = {s["theta"]: s for s in r["theta_sweep"]}
    s70 = th_rows.get(0.7, r["theta_sweep"][4])
    print(f"\n[{tag}] n={r['n_images']}  top-1={r['top1_image_weighted']:.4f} "
          f"(organ-macro {r['top1_organ_macro']:.4f})  [{time.time()-t0:.0f}s]")
    print(f"   θ=0.70: dispatched {s70['dispatched']:.3f} | misrouted {s70['misrouted']:.4f} "
          f"| errors caught {s70['errors_caught']:.3f}")
    for o, a in sorted(r["per_organ"].items(), key=lambda x: x[1]):
        print(f"     {o:<12} {a:.4f}")
v20_save()


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# V20-R2 + R3 (v24) — Calibration with a VALIDATION guard, then split conformal
#   calibrated on val and evaluated on TEST (α = 0.10).
#   v24 FIX: v20 used NON-randomised APS. For confident models the true-class score
#   is ≈ p_top ≈ 0.999, q̂ → 0.999 and almost every set contains ≥ 2 classes
#   (blood |S| = 2.29 at F1 0.987, coverage 1.000 = over-coverage, not a success).
#   Now reported:  LAC (score 1 − p_y; smallest sets, primary)  +  randomised APS.
# ════════════════════════════════════════════════════════════════════════════════
ALPHA = 0.10
_rng_cp = np.random.default_rng(0)

def collect(model, loader, organ, gw_for_hint=None):
    Z, Y = [], []
    for imgs, y in loader:
        imgs = imgs.to(DEVICE)
        hint = get_gateway_hint(gw_for_hint, imgs, organ) if gw_for_hint is not None else None
        Z.append(v20_raw_logits(model, imgs, hint).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(Z), np.concatenate(Y)

def _q(scores, alpha=ALPHA):
    n = len(scores); return float(np.quantile(scores, min(1., math.ceil((n + 1) * (1 - alpha)) / n), method="higher"))

def lac_qhat(P, y): return _q(1. - P[np.arange(len(y)), y])
def lac_sets(P, q):
    S = P >= 1. - q; S[np.arange(len(P)), P.argmax(1)] = True       # never empty
    return S

def raps_scores(P, y, u, lam=0.0, kreg=0):
    o = np.argsort(-P, 1); srt = np.take_along_axis(P, o, 1); cum = srt.cumsum(1)
    r = np.argmax(o == y[:, None], 1); p_y = srt[np.arange(len(y)), r]
    return cum[np.arange(len(y)), r] - p_y + u * p_y + lam * np.maximum(0, r + 1 - kreg)
def raps_sets(P, q, u, lam=0.0, kreg=0):
    o = np.argsort(-P, 1); srt = np.take_along_axis(P, o, 1); cum = srt.cumsum(1)
    ranks = np.arange(P.shape[1])[None, :]
    E = cum - srt + u[:, None] * srt + lam * np.maximum(0, ranks + 1 - kreg)
    keep = E <= q; keep[:, 0] = True
    S = np.zeros_like(P, bool); np.put_along_axis(S, o, keep, 1); return S

def _cov(S, y): return S[np.arange(len(y)), y]

V20["calib"] = {}; V20["conformal"] = {}
for organ in ORGANS:
    spec = SPECIALISTS[organ]
    active = spec.get("film_status") == "ACTIVE" and spec.get("film_model") is not None
    model = spec["film_model"] if active else spec["model"]; model.eval()
    vl, te, cls = v20_loaders(organ)
    gw = GATEWAY_MODEL if active else None
    Zv, yv = collect(model, vl, organ, gw); Zt, yt = collect(model, te, organ, gw)
    T = v20_fit_T(Zv, yv)
    if v20_ece(softmax_np(Zv, T), yv) >= v20_ece(softmax_np(Zv), yv): T = 1.0      # guard decided on VALIDATION
    Pv, Pt = softmax_np(Zv, T), softmax_np(Zt, T)
    V20["calib"][organ] = {"T": T, "film_active": active,
                           "ece_test_raw": v20_ece(softmax_np(Zt), yt), "ece_test_cal": v20_ece(Pt, yt),
                           "macro_f1_test": macro_f1(yt, Pt.argmax(1)), "auc_test": safe_auc(yt, Pt),
                           "acc_test": float((Pt.argmax(1) == yt).mean())}
    qL = lac_qhat(Pv, yv); SL = lac_sets(Pt, qL); cL = _cov(SL, yt)
    uv, ut = _rng_cp.random(len(yv)), _rng_cp.random(len(yt))
    qA = _q(raps_scores(Pv, yv, uv)); SA = raps_sets(Pt, qA, ut); cA = _cov(SA, yt)
    ccov = lambda c: min(float(c[yt == k].mean()) for k in np.unique(yt))
    V20["conformal"][organ] = {"n_cal": int(len(yv)), "n_test": int(len(yt)), "n_classes": len(cls),
        "qhat_lac": qL, "test_coverage": float(cL.mean()), "min_class_coverage": ccov(cL), "mean_set_size": float(SL.sum(1).mean()),
        "singleton_rate": float((SL.sum(1) == 1).mean()),
        "qhat_aps": qA, "aps_coverage": float(cA.mean()), "aps_min_class_coverage": ccov(cA), "aps_set_size": float(SA.sum(1).mean())}
    c, k = V20["calib"][organ], V20["conformal"][organ]
    print(f"  {organ:<12} T={T:5.2f}  ECE {c['ece_test_raw']:.4f}→{c['ece_test_cal']:.4f}  F1={c['macro_f1_test']:.4f}  "
          f"LAC |S|={k['mean_set_size']:.2f}/{k['n_classes']} cov={k['test_coverage']:.3f} (min cls {k['min_class_coverage']:.3f})  "
          f"APS |S|={k['aps_set_size']:.2f} cov={k['aps_coverage']:.3f}")

raw = np.array([V20["calib"][o]["ece_test_raw"] for o in ORGANS])
cal = np.array([V20["calib"][o]["ece_test_cal"] for o in ORGANS])
t, p = sps.ttest_rel(raw, cal); w = sps.wilcoxon(raw, cal)
V20["calib_summary"] = {"mean_raw": raw.mean(), "mean_cal": cal.mean(), "t": t, "p_t": p, "p_wilcoxon": w.pvalue,
                        "mean_f1": float(np.mean([V20["calib"][o]["macro_f1_test"] for o in ORGANS])),
                        "median_f1": float(np.median([V20["calib"][o]["macro_f1_test"] for o in ORGANS])),
                        "mean_auc": float(np.nanmean([V20["calib"][o]["auc_test"] for o in ORGANS]))}
print(f"\n  mean test ECE {raw.mean():.4f} → {cal.mean():.4f}   paired t={t:.2f} p={p:.4g}  Wilcoxon p={w.pvalue:.4g}")
print(f"  mean macro-F1 {V20['calib_summary']['mean_f1']:.4f} (median {V20['calib_summary']['median_f1']:.4f})  mean AUC {V20['calib_summary']['mean_auc']:.4f}")
covs = [V20["conformal"][o]["test_coverage"] for o in ORGANS]
print(f"  LAC test coverage: mean {np.mean(covs):.3f}, min {np.min(covs):.3f} ({ORGANS[int(np.argmin(covs))]})   target {1-ALPHA:.2f}")
v20_save()


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# V20-R4 — FiLM audit (the paper's main claim)
#   For each FiLM-ACTIVE organ, on the TEST split:
#     base specialist · gateway-only (organ-forced subclass head) ·
#     late fusion (mean of base + gateway probs) · FiLM(original-gateway hints) ·
#     FiLM(fine-tuned-gateway hints) · McNemar base vs FiLM
#   Leakage check: gateway-only accuracy on specialist TRAIN vs TEST split.
#   If gateway-only ≈ FiLM ≫ base, FiLM mostly transfers the gateway's own
#   subclass prediction. If gateway TEST ≈ TRAIN ≈ 1.0 on a tiny organ, the
#   gateway probably saw the specialist's test images during its training.
# ════════════════════════════════════════════════════════════════════════════════
from statsmodels.stats.contingency_tables import mcnemar

def eval_split(organ, loader, cls):
    spec = SPECIALISTS[organ]; base = spec["model"]; film = spec.get("film_model")
    out = {k: [] for k in ["y", "base", "gw", "film_orig", "film_ft"]}
    for imgs, y in loader:
        imgs = imgs.to(DEVICE); out["y"].append(y.numpy())
        out["base"].append(torch.softmax(v20_raw_logits(base, imgs), 1).cpu().numpy())
        for tag, gw in [("orig", GW_ORIGINAL), ("ft", GW_FINETUNED)]:
            if gw is None or film is None:
                out[f"film_{tag}"].append(np.full((len(y), len(cls)), np.nan)); continue
            h = get_gateway_hint(gw, imgs, organ)
            out[f"film_{tag}"].append(torch.softmax(v20_raw_logits(film, imgs, h), 1).cpu().numpy())
        gw = GW_ORIGINAL or GW_FINETUNED
        g = gw_subclass_probs(gw, imgs, organ, force_organ=True).cpu().numpy()
        out["gw"].append(g[:, gw_to_spec_order(gw, organ, cls)])
    return {k: np.concatenate(v) for k, v in out.items()}

V20["film"] = {}
film_organs = [o for o in ORGANS if SPECIALISTS[o].get("film_status") == "ACTIVE"]
print("FiLM-active organs:", film_organs)
for organ in film_organs:
    split = v20_split_paths(organ)
    # v24: same eval transform as the test loader (v20 used val_tfm+stain for train but HEAVY_VAL_AUG for test
    #      on bladder/thyroid → the "gateway train 0.82 vs test 0.96" gap was partly a preprocessing artefact)
    tr_ds = (HeavyAugDataset(split["train"][0], split["train"][1], HEAVY_VAL_AUG, repeat=1)
             if AUGMENT_CFG.get(organ) == "heavy" else StandardDataset(split["train"][0], split["train"][1], val_tfm))
    tr_ld = torch.utils.data.DataLoader(tr_ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS, timeout=_dl_timeout())
    _, te, cls = v20_loaders(organ)
    R = eval_split(organ, te, cls); y = R["y"]
    Rtr = eval_split(organ, tr_ld, cls)
    res = {}
    for k in ["base", "gw", "film_orig", "film_ft"]:
        if np.isnan(R[k]).any(): continue
        res[k] = {"f1": macro_f1(y, R[k].argmax(1)), "auc": safe_auc(y, R[k])}
    fused = (R["base"] + R["gw"]) / 2
    res["late_fusion"] = {"f1": macro_f1(y, fused.argmax(1)), "auc": safe_auc(y, fused)}
    res["gw_train_acc"] = float((Rtr["gw"].argmax(1) == Rtr["y"]).mean())
    res["gw_test_acc"] = float((R["gw"].argmax(1) == y).mean())
    for k in ["film_orig", "film_ft"]:
        if k not in res: continue
        b = R["base"].argmax(1) == y; f = R[k].argmax(1) == y
        tbl = [[int((b & f).sum()), int((b & ~f).sum())], [int((~b & f).sum()), int((~b & ~f).sum())]]
        res[k]["mcnemar_p"] = float(mcnemar(tbl, exact=False, correction=True).pvalue)
        res[k]["b_film_only"] = tbl[1][0]; res[k]["c_base_only"] = tbl[0][1]
    V20["film"][organ] = res
    print(f"\n[{organ}]  n_test={len(y)}")
    for k, v in res.items():
        if isinstance(v, dict): print(f"   {k:<12} F1={v['f1']:.4f} AUC={v['auc']:.4f}"
                                      + (f"  McNemar p={v['mcnemar_p']:.2g}" if 'mcnemar_p' in v else ""))
    print(f"   gateway-only acc: train {res['gw_train_acc']:.4f} | test {res['gw_test_acc']:.4f}")
    if res["gw_test_acc"] > 0.95 and res["base"]["f1"] < 0.6:
        print("   🚨 Gateway is near-perfect on a split where the specialist is at chance:"
              " check whether the gateway was trained on these test images.")
v20_save()


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# V20-R6 — Shortcut test: mask corners / overlays and re-measure test F1
#   Esophageal endoscopy frames carry a cyan UI box (lower-left) mainly on
#   malignant cases. A large F1 drop when that corner is masked = shortcut.
# ════════════════════════════════════════════════════════════════════════════════
def mask_region(imgs, region, frac=0.30):
    x = imgs.clone(); B, C, H, W = x.shape; h, w = int(H * frac), int(W * frac)
    fill = x.mean(dim=(2, 3), keepdim=True)
    sl = {"bl": (slice(H - h, H), slice(0, w)), "br": (slice(H - h, H), slice(W - w, W)),
          "tl": (slice(0, h), slice(0, w)), "tr": (slice(0, h), slice(W - w, W))}
    regs = ["bl", "br", "tl", "tr"] if region == "corners" else [region]
    for r in regs:
        ys, xs = sl[r]; x[:, :, ys, xs] = fill.expand(-1, -1, ys.stop - ys.start, xs.stop - xs.start)
    return x

V20["shortcut"] = {}
for organ in ORGANS:
    model = SPECIALISTS[organ]["model"]; model.eval()
    _, te, cls = v20_loaders(organ)
    ys, preds = [], {k: [] for k in ["none", "bl", "corners"]}
    for imgs, y in te:
        imgs = imgs.to(DEVICE); ys.append(y.numpy())
        for k in preds:
            xi = imgs if k == "none" else mask_region(imgs, k)
            preds[k].append(v20_raw_logits(model, xi).argmax(1).cpu().numpy())
    y = np.concatenate(ys)
    r = {k: macro_f1(y, np.concatenate(v)) for k, v in preds.items()}
    V20["shortcut"][organ] = r
    flag = "🚨" if r["none"] - min(r["bl"], r["corners"]) > 0.05 else ""
    print(f"  {organ:<12} F1 clean={r['none']:.4f}  mask-BL={r['bl']:.4f}  mask-corners={r['corners']:.4f} {flag}")
v20_save()


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# V20-R7 — Saliency sanity check (model-parameter randomisation, Adebayo 2018)
#   Correlation between Grad-CAM++ of the trained model and of a copy whose
#   last stage + head are re-initialised. High ρ ⇒ the map does not depend on
#   what the model learned (a red flag).
# ════════════════════════════════════════════════════════════════════════════════
from scipy.stats import spearmanr

def randomise_top(model, layer_prefix):
    m = copy.deepcopy(model)
    stage = ".".join(layer_prefix.split(".")[:2])     # e.g. backbone.stages
    for name, mod in m.named_modules():
        deep = name.startswith(layer_prefix) or name.startswith(stage + ".3") or \
               any(k in name for k in ["head", "classifier", "fc"])
        if deep and hasattr(mod, "reset_parameters"):
            mod.reset_parameters()
    return m.eval()

SANITY_ORGANS = [o for o in ["colon", "esophageal", "blood", "brain_2d", "gastric", "skin"] if o in SPECIALISTS]
N_IMG = 10
V20["sanity"] = {}
for organ in SANITY_ORGANS:
    model = SPECIALISTS[organ]["model"].eval(); lay = XAI_LAYERS.get(organ, "backbone.stages.2")
    rnd = randomise_top(model, lay)
    _, te, _ = v20_loaders(organ, bs=1); rhos = []
    for k, (img, _) in enumerate(te):
        if k >= N_IMG: break
        img = img.to(DEVICE)
        p = int(v20_raw_logits(model, img).argmax(1))
        try:
            a = gradcam_plus_plus(model, img, lay, p, organ=organ)
            b = gradcam_plus_plus(rnd, img, lay, p, organ=organ)
            rhos.append(spearmanr(np.ravel(a), np.ravel(b)).correlation)
        except Exception as e:
            print(f"   {organ}: {e}"); break
    V20["sanity"][organ] = {"mean_rho_trained_vs_random": float(np.nanmean(rhos)) if rhos else None, "n": len(rhos)}
    print(f"  {organ:<12} ρ(trained, randomised) = {V20['sanity'][organ]['mean_rho_trained_vs_random']}")
v20_save()
print("\nPass: ρ well below 0.5. Fail: ρ > 0.5 → the map is mostly image structure, not model evidence.")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# V20-R8 (v24) — Out-of-distribution routing: does the confidence gate flag inputs
#   that are NOT any of the 16 organs?   (tests the "any 2D image" claim)
#   v24: NO internet download (v20 spent 48 min downloading CIFAR at 60 kB/s).
#   Attach as Kaggle inputs (they are mounted, not downloaded):
#     • pankrzysiu/cifar10-python             → natural images (far-OOD)
#     • paultimothymooney/chest-xray-pneumonia → chest X-ray (near-OOD: medical, unseen modality)
# ════════════════════════════════════════════════════════════════════════════════
import torchvision, torchvision.transforms as T_, pickle, tarfile
_OOD_ROOTS = {   # v24.2: look ONLY inside these datasets (v24.1 globbed all of /kaggle/input → 25 min wasted)
    "cifar": ["/kaggle/input/datasets/pankrzysiu/cifar10-python", "/kaggle/input/cifar10-python"],
    "cxr":   ["/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia", "/kaggle/input/chest-xray-pneumonia"],
}
def _find_dir(name, key, maxdepth=4):
    for r in _OOD_ROOTS[key]:
        if not Path(r).exists(): continue
        for d in range(0, maxdepth):
            hits = glob.glob(r + "/*" * d + "/" + name)
            if hits: return hits[0]
    return None

def _cifar_images(n=256):
    d = _find_dir("cifar-10-batches-py", "cifar")
    if d is None:
        tgz = _find_dir("cifar-10-python.tar.gz", "cifar")
        if tgz:
            with tarfile.open(tgz) as t: t.extractall("/kaggle/working/cifar")
            d = "/kaggle/working/cifar/cifar-10-batches-py"
    if d is None: return None
    with open(Path(d) / "test_batch", "rb") as f: b = pickle.load(f, encoding="bytes")
    X = b[b"data"][:n].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    return [Image.fromarray(x) for x in X]

def _cxr_images(n=256):
    d = _find_dir("chest_xray", "cxr")
    if d is None: return None
    ps = sorted(Path(d).rglob("*.jpeg"))[:4000]
    ps = [ps[i] for i in np.random.default_rng(0).choice(len(ps), min(n, len(ps)), replace=False)]
    return [Image.open(p).convert("RGB") for p in ps]

def ood_batches():
    g = torch.Generator().manual_seed(0)
    yield "gaussian_noise", torch.rand(128, 3, 224, 224, generator=g), False
    yield "solid_colour", torch.rand(128, 3, 1, 1, generator=g).expand(-1, -1, 224, 224).clone(), False
    for name, fn in [("cifar10_natural", _cifar_images), ("chest_xray_nearOOD", _cxr_images)]:
        ims = fn()
        if ims is None: print(f"   {name}: input not attached — skipped"); continue
        yield name, torch.stack([val_tfm(im) for im in ims]), True

norm = T_.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
V20["ood"] = {}
for name, x, normed in ood_batches():
    x = x if normed else torch.stack([norm(i) for i in x])
    confs = [gw_organ_probs(GATEWAY_MODEL, x[i:i + 64].to(DEVICE)).max(1).values.cpu().numpy() for i in range(0, len(x), 64)]
    c = np.concatenate(confs)
    V20["ood"][name] = {"n": int(len(c)), "mean_conf": float(c.mean()),
                        "flagged_at_0.70": float((c < 0.70).mean()), "flagged_at_0.80": float((c < 0.80).mean())}
    print(f"  {name:<20} n={len(c):4d}  mean max-conf={c.mean():.3f}  flagged@0.70={V20['ood'][name]['flagged_at_0.70']:.3f}")
v20_save()
print("\nIf most OOD inputs are NOT flagged, the paper must not claim 'any 2D image' → report V21-O1 energy score.")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# V23-X1 — CAM agreement with n>1 (paper Table: XAI agreement)
#   FIX CELL F v2 printed ONE image per organ. Signed ρ was negative for 11/14
#   organs. Two explanations: (a) the maps really disagree, (b) EigenCAM's SVD
#   sign is arbitrary, so ρ flips sign at random. |ρ| separates them:
#   if mean ρ ≈ 0 but mean |ρ| is large → sign artefact, report |ρ| and say so;
#   if both are small → the maps genuinely disagree.
# ════════════════════════════════════════════════════════════════════════════════
N_PER_ORGAN = 30
CAM_RHO_SIGNED.clear()
for organ in ORGANS:
    sp = v20_split_paths(organ); P, _ = sp["test"]; cls = sp["class_names"]
    rng = np.random.default_rng(0); pick = rng.choice(len(P), min(N_PER_ORGAN, len(P)), replace=False)
    m = SPECIALISTS[organ]["model"].eval(); heavy = AUGMENT_CFG.get(organ) == "heavy"
    for k in pick:
        try:
            im = np.array(Image.open(P[k]).convert("RGB"))
            x = (HEAVY_VAL_AUG(image=im)["image"] if heavy else val_tfm(Image.fromarray(im))).unsqueeze(0).to(DEVICE)
            pred = int(v20_raw_logits(m, x).argmax(1))
            advanced_cam_ensemble(m, x, XAI_LAYERS.get(organ, "backbone.stages.2"), pred, cls, organ=organ,
                                  hook_broken=organ in HOOK_BROKEN_ORGANS, is_vit=organ in VIT_ORGANS)
        except Exception as e:
            print(f"   {organ}: {type(e).__name__}: {e}"); break
rows = []
for organ in ORGANS:
    v = np.array(CAM_RHO_SIGNED.get(organ, []))
    if len(v) == 0:
        rows.append({"organ": organ, "n": 0}); print(f"  {organ:<11} n=0  (one map flat/empty on every image — report as FAILED)"); continue
    bs = [np.mean(np.random.default_rng(b).choice(v, len(v))) for b in range(1000)]
    r = {"organ": organ, "n": int(len(v)), "rho": float(v.mean()), "ci_lo": float(np.percentile(bs, 2.5)),
         "ci_hi": float(np.percentile(bs, 97.5)), "abs_rho": float(np.abs(v).mean()), "frac_neg": float((v < 0).mean())}
    rows.append(r)
    tag = "sign artefact?" if abs(r["rho"]) < 0.1 and r["abs_rho"] > 0.3 else ""
    print(f"  {organ:<11} n={r['n']:>2}  ρ={r['rho']:+.3f} [{r['ci_lo']:+.3f},{r['ci_hi']:+.3f}]  |ρ|={r['abs_rho']:.3f}  neg={r['frac_neg']:.0%}  {tag}")
V20["xai_rho"] = rows
ok = [r for r in rows if r.get("n")]
print(f"\n  Mean signed ρ = {np.mean([r['rho'] for r in ok]):+.3f}   mean |ρ| = {np.mean([r['abs_rho'] for r in ok]):.3f}   organs failed: {[r['organ'] for r in rows if not r.get('n')]}")
with open(V20_TAB / "tab_xai_rho.tex", "w") as f:
    f.write("\\begin{tabular}{lrrrr}\\toprule Organ & $n$ & $\\bar\\rho$ [95\\% CI] & $\\overline{|\\rho|}$ & neg. \\\\ \\midrule\n")
    for r in rows:
        f.write(f"{r['organ']} & 0 & \\multicolumn{{3}}{{c}}{{failed (flat map)}} \\\\\n" if not r.get("n") else
                f"{r['organ']} & {r['n']} & {r['rho']:+.2f} [{r['ci_lo']:+.2f}, {r['ci_hi']:+.2f}] & {r['abs_rho']:.2f} & {r['frac_neg']:.0%} \\\\\n".replace("%", "\\%"))
    f.write("\\bottomrule\\end{tabular}\n")
v20_save()


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# V20-R10 — Export paper tables (LaTeX) from V20 results
# ════════════════════════════════════════════════════════════════════════════════
def tex_row(*c): return " & ".join(str(x) for x in c) + r" \\"

L = [r"\begin{tabular}{@{}lcccc@{}}", r"\toprule",
     "Organ & Routing acc. & Macro-F1 & $|S|$ & Test cov. \\\\", r"\midrule"]
for o in ORGANS:
    _rt = V20.get("routing", {}); _rt = _rt.get("original") or _rt.get("finetuned") or {}
    ra = _rt.get("per_organ", {}).get(o, float("nan"))
    c = V20.get("calib", {}).get(o, {}); k = V20.get("conformal", {}).get(o, {})
    L.append(tex_row(o.replace("_", " ").title(), f"{ra:.3f}", f"{c.get('macro_f1_test', float('nan')):.3f}",
                     f"{k.get('mean_set_size', float('nan')):.2f}", f"{k.get('test_coverage', float('nan')):.3f}"))
L += [r"\bottomrule", r"\end{tabular}"]
(V20_TAB / "tab_v20_main.tex").write_text("\n".join(L))

L = [r"\begin{tabular}{@{}lcccccc@{}}", r"\toprule",
     "Organ & Base & Gateway only & Late fusion & FiLM (orig.\\ gw) & FiLM (ft.\\ gw) & GW train/test acc. \\\\", r"\midrule"]
for o, r in V20.get("film", {}).items():
    g = lambda k: f"{r[k]['f1']:.3f}" if k in r else "--"
    L.append(tex_row(o.title(), g("base"), g("gw"), g("late_fusion"), g("film_orig"), g("film_ft"),
                     f"{r['gw_train_acc']:.2f}/{r['gw_test_acc']:.2f}"))
L += [r"\bottomrule", r"\end{tabular}"]
(V20_TAB / "tab_v20_film.tex").write_text("\n".join(L))
v20_save()
print("✅ LaTeX tables written:", *sorted(p.name for p in V20_TAB.iterdir()))
print("   Full results:", V20_OUT / "v20_results.json")


In [ ]:
import shutil

shutil.make_archive(
    '/kaggle/working/xai_v16_output',  # output zip path (without .zip)
    'zip',
    '/kaggle/working/xai_v16_output'   # folder to zip
)

print("✅ Zip created: /kaggle/working/xai_v16_output.zip")